# Més Que Un Data · Ball Possession from Automatic Tracking

**More than a Hack 2027, Phase 1 (Ball Possession)** · FC Barcelona vs Athletic Club, LaLiga EA Sports 2024-25

| | |
|---|---|
| **Team** | Més Que Un Data |
| **Members** | Raya Ramadha Fitroh · Fauzan Ahsanudin Alfikri · Gabriel Edbert Liandrew · Benedict Brian Joel Purba · Yan Ardhinaya Ardika |
| **Version** | final submission |
| **Output** | `submissions/MesQueUnData.csv`: one label `H` / `A` / `D` / `X` for each of the 199,319 frames of the tactical video |

## The problem

Every frame needs a label: the home team has the ball (`H`), the away team has it (`A`), or the ball is dead (`D`).
The leaderboard uses **spellF1**. Every *possession* (a maximal run of one team with the ball alive) is matched one to
one with a true possession through time IoU at thresholds 0.3–0.7. What counts is therefore the **whole possession**,
not per-frame accuracy:

- one spurious switch splits one possession into three;
- one missed short possession merges the two long possessions around it.

The input is **uncorrected automatic tracking**: no player identities, broken tracks, and a **ball that is missing on
± 41% of the graded frames** (Cioppa et al., 2022). The EDA in section 4b shows that ± 90% of those ball-less frames
fall during dead-ball time. During live play the ball is visible on ± 92% of frames, but the remaining gaps fall at the
moments that matter most: challenges and changes of possession.

## The approach

| Stage | Main idea |
|---|---|
| 1–2. Data & ball | Tactical ATD in metres; ball gaps filled with the **ball from the broadcast tracking**, aligned through player positions; the fill is **bridged at the gap edges** so the ball does not jump when the source changes |
| 2b. Ball from video | **YOLO26 detector trained for the tactical camera** from automatic labels (trackers agree → COCO YOLO in a small window), detection on the whole video, Viterbi tracking, label-free validation; corrects floating tactical balls and fills gaps for the possession model; fills come only from YOLO tracks anchored to the tracker ball |
| 3–4. Dead ball | `D` detector based on collective behaviour (calm players, no passing, ball out/missing, broadcast replay) + Viterbi |
| 4. `D` boundaries | end of `D` snapped to the **restart kick**, start of `D` to the **ball crossing a line** (tested on 7 IDSSE matches with `BallStatus`) |
| 4b. Data EDA | when and where the ball goes missing, broadcast camera content (replay, close-up), per-minute alignment, long balls, Smart Tagging |
| 4c. Data quality | physical outliers (track jumps, balls that are too fast, positions off the pitch), duplicate tracks, tracks per team |
| 5–7. H/A possession | **sequence models** (BiGRU & TCN with a player-set encoder) **trained on 23 other labelled matches** (open data), with augmentation that copies this match's ball-gap pattern, plus **ball-motion and arrival-time** features that came out of the error analysis |
| 6c. Evaluation & errors | metrics per IoU, switch timing, calibration, errors per context, possession error types, permutation importance, sensitivity to outliers |
| 7b. Video touches | who touched the ball, read from the tactical and broadcast video (kit colour + contact) |
| 8. Decoder | 3-state Viterbi: the switching cost keeps possessions stable, and a restart prior gives a clear owner after a dead ball |
| 8b–8d. Rules | referee whistle from the audio, switch re-timing (rejected), last toucher before the ball goes out (final rule) |
| 9. Validation | official validator + correctness check on the 5-minute public GT |
| 9b–9c. Final EDA | does the public 5 minutes represent 90 minutes, H/A possession balance, error diagnosis for post-processing |

Every design decision comes from **leave-match-out cross-validation on labelled external data** (± 6,000
possessions). The public GT (17 possessions) is used only as a correctness check at the end, because it is far too
small to choose parameters on (one possession ≈ 6 spellF1 points).

## Reproduction

- The possession model weights live in `notebooks/pipeline/models/` (with their SHA-256 in `meta.json`). When they are
  present, the notebook loads them instead of retraining.
- The ball detector, its automatic labels and the detections on the whole video are stored in `cache/yolo/` and
  reloaded when their parameters match. The first run of section 2b takes ± 1.5–2 GPU hours.
- Match data never leaves this machine: `cache/` and `submissions/` are git-ignored, and images/video clips are shown
  only in the notebook's local output (nbstripout removes them on commit).

## Development history

| Stage | What changed |
|---|---|
| **Base pipeline** | Ball fusion, YOLO ball, dead-ball detector and possession models up to the model log-odds (section 7). Then **section 7b** reads *who touched the ball* in the tactical and broadcast video, and the **base decoder** (after section 8) uses those touches as extra evidence; section 8b adds the referee whistle. |
| **Switch re-timing** (rejected) | **Section 8c** re-times every H↔A switch of the base decoder **without changing the number of possessions**: the base decoder decides *whether* a switch happens, 8c decides *exactly when* (base log-odds + clear video touches). Method and window are chosen by a **nested simulation on the 23 external matches**, not by the public GT. It did not pass (log #94). |
| **Smoothed-ball training** (rejected) | **Section 6d**: a label-free diagnosis shows that the BAR-ATH ball seen by the model is **smoother** than the ball in the 23 training matches (ball turns and acceleration, the signs of a touch, are damped). 6d measures that smoothing from BAR-ATH's own features, trains the model with equally smoothed training balls, and **uses the new model only if it wins the cross-validation on the 23 external matches**. It did not pass (log #96). |
| **Nearest-player method** (rejected) | **Section 6e, the "video method"**: possession = team of the **player nearest to the ball** (like the YOLO detection + homography + 2D tactical-map pipeline the team reviewed), tested (a) alone and (b) as extra evidence on top of the model log-odds, with the out-of-fold log-odds of the 23 external matches from section 6d. It did not pass (log #99). |
| **Stacking** (rejected) | **Section 6f**: the **embeddings** before each architecture's *classification head* (gru_set 192 + tcn 128 + tcn_set 128 dimensions) are combined with **"video method" features** (nearest player to the ball, ball holder, length/width/defensive line/block height, hull area of each team), and an **MLP** produces the H vs A log-odds for the base decoder. Training embeddings are *out-of-fold*. It did not pass (log #102). |
| **Final** | Base pipeline + **section 8d, the last toucher before the ball goes out**. The Laws of the Game award corners and goal kicks to the *opponents* of the team that touched the ball last; on BAR-ATH, the base labels often give possession to the restart taker right up to the moment the ball goes out (17 of 20 corners). Section 8d tests two ways to fix this on **7 IDSSE matches** (DFL toucher flag; the possession model has never seen this data) and applies the fix to BAR-ATH only if it passes. It passed (log #104–#105). |

**In short:** the submitted file = the base pipeline (sections 0–8b) + the ball-out rule (section 8d). The re-timing,
smoothed-ball, nearest-player and stacking sections are kept in the notebook for the record, but their switches are off
because each idea was rejected by its pre-registered decision rule.


## How to run this notebook

The notebook regenerates the submitted file `submissions/MesQueUnData.csv` from the raw match data. It does **not**
retrain the possession models: their weights ship with the repository.

1. **Install** (once, from the repository root; see `README.md` for details):
   ```bash
   py -3.11 -m venv .venv
   .venv\Scripts\python -m pip install -r requirements-lock.txt
   .venv\Scripts\python -m ipykernel install --sys-prefix --name mesqueundata
   .venv\Scripts\python notebooks\pipeline\pasang_artefak.py      # installs the trained YOLO / kit-colour weights into cache/
   ```
2. **Place the data**: the match files (tracking, homographies, Smart Tagging and both videos) go in `data/tactical/`
   and `data/broadcast/` (or point the environment variable `MTAH_DATA_DIR` to them). IDSSE goes in
   `external_data/idsse/`; it is needed only for the ball-out rule validation cell (section 8d). Without it, set
   `BALLOUT_STAGE = "force"` in the parameter cell to apply the already validated variant.
3. **Run**: close other Jupyter kernels (the YOLO stage needs GPU memory), choose the kernel
   `Python 3.11 (.venv Més Que Un Data)` and *Run All*, or use `run.ps1` / `run.sh`, which also run the official
   validator and print the SHA-256.
4. **Runtime**: the first run takes ± 2.5–3 hours, mostly ball detection with YOLO over the whole video (± 1.5–2 GPU
   hours). Later runs read `cache/` and take ± 15–20 minutes.
5. **Expected result**: the last cell prints `DONE` and the SHA-256 of the CSV. The submitted file has SHA-256
   `8ba09b17d35612bcd39e362819f75aef22c16a2416b426803e60fddf49cbacf2`.

**Data privacy.** `data/`, `cache/` and `submissions/` are git-ignored, and `nbstripout` removes all cell outputs (which
contain match frames) on commit. Only aggregate numbers appear in the committed notebook.

## Notebook map

The notebook runs top to bottom; each step uses the results of the previous ones. Steps marked *rejected* are kept for
the record but are switched off in the parameter cell, so they do not change the submission.

| Step | Section | What it produces |
|---|---|---|
| 0 | Parameters, environment, utilities, visual style | switches for every stage, paths, the official scorer, `viterbi` / `runs` / `check` |
| 1 | Tactical tracking | player and ball positions in metres, attacking direction per half |
| 2 | Ball from the broadcast | broadcast offset per half, merged ball (tactical + broadcast, bridged gap edges) |
| 2b | Ball from video (YOLO) | automatic labels → trained ball detector → detections → one ball track → merged ball for the model |
| 3 | Smart Tagging | set-piece tags, restart taker teams |
| 4 | Dead-ball detector | `D` segments, snapped to the restart kick and the ball crossing a line |
| 4b–4c | EDA and data quality | where the ball goes missing, broadcast content, alignment, long balls, tags, outliers |
| 5 | Features | 47 per-step features + an 8-player set per team, at 10 fps |
| 6 | Possession model | 6a train (skipped: weights included) · 6b load · 6c evaluation & error analysis on external data |
| 6d–6f | Rejected experiments | *rejected* (smoothed ball, nearest-player method, stacking) |
| 7 | Log-odds for BAR-ATH | H vs A log-odds per frame from the 6-model ensemble |
| 7b | Video touches | who touched the ball and when (tactical + broadcast video) |
| 8 | Decoder | labels without touches (Viterbi) and base labels (Viterbi + touch evidence) |
| 8b | Whistle rule | `D` starts at the referee's whistle |
| 8c | Switch re-timing | *rejected* |
| 8d | ball-out rule | the last toucher before a corner or goal kick → **the final labels** |
| 9 | Write & validate | `submissions/MesQueUnData.csv`, official validation, public-GT correctness check |
| 9b–9c | Final EDA and diagnosis | representativeness of the 5 public minutes, balance, error diagnosis (no tuning) |
| – | Summary of the checks, data scientist's notes, references | |

## Glossary: Indonesian names kept in the code

The notebook was first written in Indonesian. All explanations, comments and printed messages are now in English, but
a few **names that act as data** were deliberately left unchanged, because changing them would change the program:
they are stored in `meta.json` together with the trained models, used as keys or column names, read by the helper
modules, or used as file names. Renaming them could break model loading or the byte-for-byte reproduction of the
submitted CSV. Their English meaning:

| Name in the code | Meaning |
|---|---|
| `celah ala BAR-ATH` | evaluation condition "BAR-ATH-like ball gaps" (main condition) |
| `celah bola hidup BAR-ATH` | evaluation condition "BAR-ATH live-ball gaps" |
| `bola terlihat` | evaluation condition "ball visible" (also the feature "ball visible") |
| `bola halus + celah ala BAR-ATH` | condition "smoothed ball + BAR-ATH-like gaps" (section 6d) |
| `kondisi`, `metode`, `jendela`, `tipe`, `jenis` | condition, method, window, type, kind (table columns) |
| `sendiri` / `lawan` | own team / opponent (feature viewpoint) |
| `sepak pojok`, `tendangan gawang`, `lemparan ke dalam` | corner kick, goal kick, throw-in (restart types, `aturan_keluar.py`) |
| `pengambil`, `sebelum`, `awal D`, `kontradiksi` | restart taker, label before, start of `D`, contradiction (ball-out rule table) |
| `kurung`, `sentuhan` | switch re-timing modes "bracket" and "touch" |
| `broadcast_bola`, `broadcast_kandidat` | touch-evidence sources "broadcast ball" and "broadcast candidate" |
| `latih` / `muat` | accepted aliases for `TRAIN_MODELS`: "train" / "load" |
| `aturan_keluar.py`, `pasang_artefak.py`, `artefak/` | "ball-out rule" helper, "install artefacts" script, artefacts folder |

**Feature names** (`FEATURE_NAMES`, 47 per step; `(sendiri)` = own team, `(lawan)` = opponent):

| Name in the code | Meaning |
|---|---|
| `bola terlihat`, `detik sejak bola terlihat`, `detik sampai bola terlihat` | ball visible, seconds since / until the ball was visible |
| `kecepatan bola`, `bola x ke arah serang`, `bola \|y\| dari tengah` | ball speed, ball x towards the attacking goal, ball \|y\| from the centre line |
| `jarak terdekat 1/2/3` | distance of the 1st/2nd/3rd nearest player to the ball |
| `kec. relatif terdekat–bola`, `kec. mendekat`, `kec. pemain terdekat` | relative speed nearest player–ball, approach speed, speed of the nearest player |
| `n dalam 2/5/10 m` | number of players within 2/5/10 m of the ball |
| `posisi rata-rata`, `lebar`, `panjang`, `sebaran` | mean position, width, length, spread of the team |
| `kec. rata-rata`, `kec. ke gawang`, `n track` | mean speed, speed towards the opponent's goal, number of tracks |
| `bola: kec. ke arah serang`, `bola: kec. ke samping` | ball speed towards the attacking goal, sideways ball speed |
| `bola: percepatan`, `bola: belokan arah` | ball acceleration, ball turn (1 − cos of the angle) |
| `waktu tiba tercepat`, `waktu tiba ke-2`, `porsi kontrol waktu tiba` | fastest / 2nd-fastest arrival time to the ball, arrival-time control share |
| groups `bola`, `lokal bola sendiri/lawan`, `bentuk sendiri/lawan`, `gerak bola & waktu tiba` | ball, local ball (own/opponent), shape (own/opponent), ball motion & arrival time |

Other short Indonesian column names that appear in a few printed tables (e.g. `durasi` = duration, `cocok` = matched,
`porsi` = share, `salah` = wrong, `turun` = drops, `sisipan` = insertions) follow the same rule.

## Parameters

Every stage of the pipeline is controlled from this cell. The values below reproduce the submitted file; change
them only to re-run an experiment.

- `TRAIN_MODELS = "auto"` loads the shipped weights (training only happens if they are missing).
- `*_STAGE = "auto"` reuses a cache when its parameters match; `"off"` switches a stage off.
- The re-timing, smoothed-ball, nearest-player and stacking experiments (`SIM_STAGE`, `SMOOTH_STAGE`, `NEAR_STAGE`, `STACK_STAGE`) are `"off"`: each one was rejected by
  its decision rule.
- `BALLOUT_STAGE = "auto"` validates the ball-out rule on IDSSE and applies it only if it passes; `"force"` applies the
  validated variant without IDSSE.

In [ ]:
TRAIN_MODELS = "auto"     # "auto": train only if weights are missing · True: retrain and overwrite · False: weights must be loaded
RUN_CV = True             # leave-match-out cross-validation + post-processing + error analysis & feature importance (± 25 GPU minutes; only when training)
INFER_DEVICE = "cuda"     # run inference on the GPU if available (falls back to CPU automatically)
TEAM_FILE = "MesQueUnData"
BOOTSTRAP_N = 1000        # official bootstrap interval for the public score (section 9)
# YOLO: ball detector trained on the tactical video (section 2b)
YOLO_STAGE = "auto"       # "auto": reuse cache/yolo if its parameters match · "retrain": relabel, retrain, redetect · "off": tracking ball
YOLO_TEACHER = "yolo26x.pt"   # automatic labeller (teacher): largest YOLO26, COCO-pretrained
YOLO_STUDENT = "yolo26s.pt"   # fine-tuned ball detector (student); "yolo26m.pt" is ± 2× slower and needs batch 8
YOLO_N_LABEL = 5000       # candidate frames for automatic labelling
YOLO_EPOCHS = 25
YOLO_BATCH = 16           # ± 4.5 GB GPU for YOLO26s at 640 px
YOLO_WORKERS = 2          # dataloader workers; with 16 GB RAM keep this ≤ 2
YOLO_INFER_BATCH = 8
YO_INFER_RANGES = None    # None = every graded frame; a list of (start, end) frames for a quick test
YOLO_SHOW_VIDEO = True    # show the review clip in section 9c
YOLO_CLIP = ("09:04", "09:16")   # tactical video time of the review clip (public GT window)
# dead-ball boundaries
D_SNAP = True             # snap D boundaries to the restart kick and the ball crossing a line (IDSSE: dead F1 0.933 → 0.948, 7/7 matches)
SNAP_RECEPTION = False    # delay an H↔A switch made while the ball is in flight until the receiving team touches it;
                          # rejected by external cross-validation (−0.0004), so off by default (kept only for the review clip)
# touch evidence from the video (section 7b and the base decoder)
TOUCH_STAGE = "auto"      # "auto": reuse cache/touch if its parameters match · "retrain": recompute · "off": decoder without touches
TOUCH_BETA = {"tactical": 4.0, "broadcast_bola": 2.0, "broadcast_kandidat": 0.5}   # evidence weight per source (log-odds)
TOUCH_TMAX_S = 1.0        # evidence holds from a touch until the next touch, for at most 1 s
# referee whistle from the tactical video audio (section 8b)
WHISTLE_STAGE = "auto"    # "auto": reuse cache/audio if its parameters match · "off": no whistle rule
WHISTLE_THR = 2.2         # two-tone score threshold (clear whistles: 2.9–3.9), locked before the public check
WHISTLE_MAX_GAP_S = 3.0   # whistle during live play + predicted D start ≤ 3 s later → D starts at the whistle
# re-timing of H↔A switch boundaries (section 8c)
REFINE_MODE = "auto"      # "auto": the method that passes the nested simulation (none passes → "off") ·
                          # "off": base labels · or force "logodds" / "kurung" (bracket) / "sentuhan" (touch)
REFINE_WIN_S = None       # None: window from the simulation · a number (seconds) to force it, e.g. 1.0
SIM_STAGE = "off"         # off (rejected, log #94) · "auto": use cache/retime/cv_oof.npz if present, else train fold models · "off": no simulation
SIM_ARCHS = ["gru_set", "tcn", "tcn_set"]   # models per fold (= base CV, ± 15 models); ["tcn"] for a quick test (± 3× faster)
# smoothed-ball training: training balls smoothed to match the BAR-ATH ball (section 6d)
SMOOTH_STAGE = "off"      # off (rejected, log #96) · "auto": cross-validate base vs smoothed-ball (cached); the smoothed-ball model is used only if it passes the decision rule ·
                          # "off": base weights (result = base) · "force": use the smoothed-ball model regardless of the decision rule
SMOOTH_SIGMA = None       # None: smoothing σ (10 fps steps) matched to the BAR-ATH ball features · a number to force it
SMOOTH_ARCHS = ["gru_set", "tcn", "tcn_set"]   # models per cross-validation fold; ["tcn"] for a quick test (± 3× faster)
# nearest-player: "video method" — team of the player nearest to the ball (section 6e)
NEAR_STAGE = "off"        # off (rejected, log #99) · "auto": simulate on the 23 external matches, use only if it passes · "off" · "force": use NEAR_FORCE_CFG
NEAR_FORCE_CFG = None     # (radius m, β, shape) for "force", e.g. (1.5, 2.0, "tahan 1 s"); None = best configuration from the simulation
# stacking of base embeddings + video-method features → MLP (section 6f)
STACK_STAGE = "off"       # off (rejected, log #102) · "auto": cross-validate (cached), then use only if it passes · "off" · "force": use without the rule
STACK_SUB = 4             # MLP training rows: every 4th step (10 fps) on live frames, both viewpoints
STACK_EPOCHS = 6          # MLP epochs
# last toucher before the ball goes out (section 8d)
BALLOUT_STAGE = "auto"    # "auto": test on the 7 IDSSE matches, use only if it passes · "off" · "force": use the best variant without the rule
BALLOUT_THROWIN = False   # include throw-ins (taker team is not in the BAR-ATH tags → taken from the label after D)


## 0. Environment

**What this step does.** Loads the libraries, finds the repository root and reads the match constants (number of
frames, half boundaries, pitch size 105.3 × 68 m) and the scorer from the official competition repository
(`external/more-than-a-hack-2027`: `games.py`, `possession_metrics.py`). Because the scorer is imported rather than
re-implemented, every validation in this notebook is exactly the organisers' validation.

In [ ]:
import hashlib
import json
import os
import re
import sys
import time
import warnings
import xml.etree.ElementTree as ET
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)


def _find_root(start):
    for folder in [start, *start.parents]:
        if (folder / "external" / "more-than-a-hack-2027" / "games.py").is_file():
            return folder
    raise FileNotFoundError("Repository root (containing external/more-than-a-hack-2027) not found from {}".format(start))


ROOT = _find_root(Path.cwd().resolve())
NB_DIR = ROOT / "notebooks" / "pipeline"
MODEL_DIR = NB_DIR / "models"
OFFICIAL = ROOT / "external" / "more-than-a-hack-2027"
GT_PUBLIC = OFFICIAL / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
CACHE = ROOT / "cache"
SUBMISSIONS = ROOT / "submissions"
EXT_DIR = ROOT / "external_data"
_DEFAULT_DATA = ROOT / "data" if not (ROOT / "tactical").is_dir() and (ROOT / "data" / "tactical").is_dir() else ROOT
DATA_DIR = Path(os.environ.get("MTAH_DATA_DIR") or _DEFAULT_DATA)
for _folder in (CACHE, SUBMISSIONS, MODEL_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

os.environ["MTAH_DATA_DIR"] = str(DATA_DIR)
for _p in (OFFICIAL / "evaluation", OFFICIAL):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
import games                      # noqa: E402  (official)
import possession_metrics as pm   # noqa: E402  (official)

GAME = games.GAME
FPS = games.GAMES[GAME]["fps"]                       # 25
N_FRAMES = games.source_frames("tactical", GAME)     # 199319
PITCH_L, PITCH_W = games.GAMES[GAME]["pitch"]        # 105.3, 68.0
HALVES = {h: (a, b) for h, a, b in games.half_video_blocks(GAME)}
KICKOFF = {1: HALVES[1][0], 2: 113742}
HALF = np.zeros(N_FRAMES, dtype=np.int8)
for _h, (_a, _b) in HALVES.items():
    HALF[_a:_b + 1] = _h
GRADED = HALF > 0
N_GRADED = int(GRADED.sum())
assert N_GRADED == 158857, N_GRADED
print("repo root: {} | data: {} | {} frames, {} graded".format(ROOT, DATA_DIR, N_FRAMES, N_GRADED))

### Utilities

- **`viterbi`**: finds the state path with the largest total score (sum of the emission scores minus a cost for every
  state change; Rabiner, 1989). It is used twice: in the dead-ball detector (2 states) and in the final H/A/D decoder
  (3 states). The switching cost is the direct way of saying that possessions and dead-ball periods **persist** instead
  of flickering frame by frame.
- **`runs`**: consecutive `True` runs; the basis of every segment operation.
- **`check`**: an automatic check; the notebook stops at the end if a critical check failed.

In [ ]:
def asset_path(name, source="tactical"):
    standard = Path(games.asset_paths(GAME, str(DATA_DIR), source)[name])
    if standard.is_file():
        return standard
    suffix = games.SUFFIXES[name]
    matches = sorted(standard.parent.glob(standard.name[: -len(suffix)] + "*" + suffix))
    return matches[0] if len(matches) == 1 else None


def require_asset(name, source="tactical"):
    path = asset_path(name, source)
    if path is None:
        raise FileNotFoundError("Asset {}/{} not found in {}".format(source, name, DATA_DIR / source))
    return path


def runs(mask):
    d = np.diff(np.concatenate(([0], np.asarray(mask).astype(np.int8), [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


@contextmanager
def timer(label):
    t0 = time.time()
    yield
    print("[{}] {:.1f} s".format(label, time.time() - t0))


def viterbi(emis, trans_cost):
    """Best state path. emis (T, K) score per state; trans_cost (K, K) switching cost (zero diagonal).
    Ties are broken towards the lowest state index -> deterministic."""
    E = np.asarray(emis, dtype=np.float64).tolist()
    C = np.asarray(trans_cost, dtype=np.float64).tolist()
    T, K = len(E), len(E[0])
    score = list(E[0])
    back = np.zeros((T, K), dtype=np.int8)
    for t in range(1, T):
        et, new, bt = E[t], [0.0] * K, [0] * K
        for j in range(K):
            best_i, best = 0, score[0] - C[0][j]
            for i in range(1, K):
                v = score[i] - C[i][j]
                if v > best:
                    best_i, best = i, v
            new[j], bt[j] = best + et[j], best_i
        score = new
        back[t] = bt
    path = np.empty(T, dtype=np.int8)
    path[-1] = int(np.argmax(score))
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def auc(pos, neg):
    pos, neg = np.asarray(pos, dtype=float), np.asarray(neg, dtype=float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) < 5 or len(neg) < 5:
        return float("nan")
    ranks = pd.Series(np.concatenate([pos, neg])).rank().to_numpy()
    return float((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


CHECKS = []


def check(name, ok, detail="", critical=True):
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if critical else "WARN")
    CHECKS.append({"cek": name, "status": status, "keterangan": detail})
    print("[{}] {}{}".format(status, name, " — " + detail if detail else ""))
    return ok

### Visual style

Every figure uses one consistent palette: **H (Barcelona) blue**, **A (Athletic) orange**, **D (dead ball) neutral
grey**, and a single-hue blue ramp for magnitudes. Helper lines are kept faint so that the data stands out. Figures
show only **aggregates** or the 5-minute public GT window.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

COL = {"H": "#2a78d6", "A": "#eb6834", "D": "#b4b2a9", "aqua": "#1baf7a", "yellow": "#eda100", "violet": "#6250d6",
       "text": "#0b0b0b", "muted": "#52514e", "grid": "#e4e3df", "surface": "#fcfcfb"}
BLUES = LinearSegmentedColormap.from_list("blues_ramp", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#104281"])
plt.rcParams.update({
    "figure.facecolor": COL["surface"], "axes.facecolor": COL["surface"], "savefig.facecolor": COL["surface"],
    "axes.edgecolor": COL["grid"], "axes.labelcolor": COL["muted"], "axes.titlecolor": COL["text"],
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10,
    "xtick.color": COL["muted"], "ytick.color": COL["muted"], "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.grid": True, "grid.color": COL["grid"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False, "legend.fontsize": 9,
    "font.size": 10, "figure.dpi": 110})


def note(ax, text):
    """small caption under the title, in secondary text colour"""
    ax.text(0, 1.01, text, transform=ax.transAxes, fontsize=9, color=COL["muted"], va="bottom")

## 1. Tactical tracking: metres, halves, attacking direction

**What this step does.** Reads the tactical tracking (ATD) into metres and fixes the attacking direction per half.

- **FIFA/EPTS format.** The XML metadata gives the team of every *track slot* (46 slots; slots, not players) and the
  first-half attacking direction. One data row = `frame:player;…:ball` with coordinates in 0–1. Row *i* is video
  frame *i* (checked).
- **Conversion to metres.** The pitch is not square. Without conversion, distances along the short axis shrink by
  ± 35% and "the player nearest to the ball" can be wrong near the touchline (physical preprocessing as in
  Vidal-Codina et al., 2022).
- **Attacking direction** is flipped in the second half and **verified with the data**: just before kick-off each team
  stands in its own half. A flipped sign would silently invert every "towards the opponent's goal" feature.
- **Forced dead ball:** the second half is graded before its kick-off, so that range is certainly `D` (official FAQ).

In [ ]:
def _local(tag):
    return tag.split("}")[-1]


def _child_text(el, *names):
    for child in el:
        if _local(child.tag).lower() in names:
            return (child.text or "").strip()
    return ""


def parse_atd_meta(source="tactical"):
    path = require_asset("atd_meta", source)
    root = ET.parse(path).getroot()
    teams = [el for el in root.iter() if _local(el.tag) == "Team" and el.get("id")]
    players = [el for el in root.iter() if _local(el.tag) == "Player" and el.get("id") and el.get("teamId")]
    local_ids = [el.get("idLocalTeam") for el in root.iter() if _local(el.tag) == "Score" and el.get("idLocalTeam")]
    home_id = teams[0].get("id") if teams else None
    attack_h1 = None
    for el in (teams[0].iter() if teams else []):
        if _local(el.tag) == "ProviderParameter" and _child_text(el, "name", "n") == "attack_direction_first_half":
            attack_h1 = _child_text(el, "value")
    return {"home_id": home_id, "home_name": _child_text(teams[0], "name", "n") if teams else None,
            "away_name": _child_text(teams[1], "name", "n") if len(teams) > 1 else None,
            "local_team_id": local_ids[0] if local_ids else None,
            "slot_ids": [int(el.get("id")) for el in players],
            "channel_order": [int(i) for i in re.findall(r'playerChannelId="player(\d+)_x"', path.read_text(encoding="utf-8"))],
            "slot_is_home": np.array([el.get("teamId") == home_id for el in players]), "attack_h1": attack_h1}


meta = parse_atd_meta()
SLOT_IS_HOME = meta["slot_is_home"]
N_SLOTS = len(SLOT_IS_HOME)
assert meta["slot_ids"] == meta["channel_order"] and meta["local_team_id"] == meta["home_id"]
H_SIGN_H1 = 1 if meta["attack_h1"] == "left_to_right" else -1
ATTACK_DIR = np.zeros(N_FRAMES, dtype=np.int8)
ATTACK_DIR[HALF == 1] = H_SIGN_H1
ATTACK_DIR[HALF == 2] = -H_SIGN_H1
print("H = {} ({} slot) | A = {} ({} slot)".format(meta["home_name"], int(SLOT_IS_HOME.sum()), meta["away_name"],
                                                  int((~SLOT_IS_HOME).sum())))


def load_atd(path, n_slots):
    ids = np.zeros(N_FRAMES, dtype=np.int64)
    pl = np.full((N_FRAMES, n_slots, 3), np.nan, dtype=np.float32)
    bl = np.full((N_FRAMES, 3), np.nan, dtype=np.float32)
    with open(path, encoding="utf-8") as handle:
        for i, line in enumerate(handle):
            frame, groups, b = line.rstrip("\r\n").split(":")
            ids[i] = int(frame)
            pl[i] = np.array(groups.replace(";", ",").split(","), dtype=np.float32).reshape(n_slots, 3)
            bl[i] = np.array(b.split(","), dtype=np.float32)
    return ids, pl, bl


with timer("read tactical ATD"):
    atd_ids, players, ball = load_atd(require_asset("atd_raw"), N_SLOTS)
check("frame_id = video frame + 1 on every row", (atd_ids == np.arange(1, N_FRAMES + 1)).all())

players[..., 0] *= PITCH_L
players[..., 1] *= PITCH_W
ball_raw = (ball[:, :2] * np.array([PITCH_L, PITCH_W], dtype=np.float32)).astype(np.float32)
finite = np.isfinite(ball_raw[:, 0])

dead_prior = np.zeros(N_FRAMES, dtype=bool)
dead_prior[HALVES[2][0]:KICKOFF[2]] = True
with np.errstate(all="ignore"):
    for h, ko in KICKOFF.items():
        hx = np.nanmean(players[ko - FPS:ko][:, SLOT_IS_HOME, 0])
        ax = np.nanmean(players[ko - FPS:ko][:, ~SLOT_IS_HOME, 0])
        check("attacking direction in half {} is consistent with the positions before kick-off".format(h),
              (hx < ax) if ATTACK_DIR[ko] > 0 else (hx > ax))
print("tactical ball present on {:.1f}% of graded frames".format(100 * finite[GRADED].mean()))

## 2. Ball: filling gaps with the broadcast recording

**What this step does.** Builds the best possible ball trajectory from the two tracking recordings.

The ball is the most informative signal and also the least reliable one: it is missing on ± 41% of the graded frames,
in **long blocks** (median gap ± 11 s), so interpolation alone hardly helps. Most of those blocks are stoppages
(section 4b), but the short gaps during live play still fall at the moments of a challenge. The second recording, the
broadcast, has its own tracking from a different angle and zoom, so a ball missing in one camera is often visible in
the other.

**Alignment without a time table.** The broadcast has its own timeline (cuts, replays, breaks), but the same pitch
coordinates. Broadcast frame *b* and tactical frame *t* are therefore matched through **player positions**: the cost is
the mean distance from every broadcast player to the nearest tactical player of the same team (capped at 20 m).

1. Coarse offset: 40 broadcast frames spread over the match are searched against the whole tactical timeline.
2. Precise offset per half: the mode of the best offset within ± 60 frames over ± 120 samples (the offset turns out to
   be constant per half).
3. Per-frame consistency: a broadcast frame is used only if ≥ 6 of its players match at a mean cost < 2 m. This also
   filters out **replays and close-ups**, whose positions do not match live play. The broadcast view type (`bc_view`)
   is kept: a broadcast showing a replay is strong evidence that play has stopped (*replay grounding*, Deliège et al.,
   2021).
4. The broadcast ball fills the tactical ball gaps; remaining gaps ≤ 1 s are linearly interpolated if the required
   speed is ≤ 40 m/s (above that, the two ends are most likely different detections).
5. **Bridging the gap edges.** The two trackers place the ball slightly differently (median ± 0.8 m), so the ball
   "jumps" when the source switches from tactical to broadcast and back. The offset at both gap edges is removed:
   linearly from the left edge to the right edge for fills ≤ 2 s, and fading with a 0.5 s time constant from each edge
   for longer fills. Fills with an edge jump > 5 m are dropped (the broadcast is probably following another object), and
   the gap is treated like a gap without broadcast.

**How the bridging rule was chosen without labels.** A visible tactical ball is deliberately hidden for L frames, then
filled by each method and compared with its true position (± 3,000 gaps per length, only frames with the ball in both
recordings). No GT is used.

| Gap length | Interpolation | Raw broadcast | Linear bridge | 0.5 s fade |
|---|---|---|---|---|
| 0.4 s | 0.11 m / 0% | 0.73 m / 18% | **0.15 m / 0%** | 0.31 m / 6% |
| 1 s | 0.62 m / 1% | 0.76 m / 16% | **0.30 m / 1%** | 0.34 m / 3% |
| 2 s | 1.79 m / 24% | 0.75 m / 15% | **0.57 m / 6%** | 0.54 m / 9% |
| 5 s | 5.24 m / 81% | 0.68 m / 7% | 0.86 m / 11% | **0.59 m / 6%** |
| 10 s | 11.2 m / 98% | 0.65 m / 4% | 1.03 m / 14% | **0.61 m / 3%** |

(median error per gap / share of gaps with an error > 3 m). The error of a raw broadcast fill grows sharply with the
jump at the gap edge: median 0.4 m when the jump is ≤ 1 m, 4.4 m at 5–10 m, 11 m above 10 m; hence fills with a jump
> 5 m are dropped. The player-matching cost threshold (2 m) is kept: moving it between 0.5 and 4 m does not change the
ball agreement (median ± 0.77 m at every threshold), only the number of filled frames.

Validation: on frames where the ball is visible in both recordings, the distance between the two ball positions must
be small.

In [ ]:
MAX_GAP, MAX_GAP_SPEED = 25, 40.0
BC_COST_MAX, BC_MIN_PLAYERS, BC_SEARCH = 2.0, 6, 60

gap_rows = []
for h, (a, b) in HALVES.items():
    s_, e_ = runs(~finite[a:b + 1])
    for si, ei in zip(s_, e_):
        gap_rows.append({"start": a + si, "end_excl": a + ei, "length": ei - si, "has_left": si > 0,
                         "has_right": a + ei <= b})
gaps = pd.DataFrame(gap_rows)
gaps["dist_m"] = np.nan
both_sides = gaps.has_left & gaps.has_right
gaps.loc[both_sides, "dist_m"] = [float(np.hypot(*(ball_raw[r.end_excl] - ball_raw[r.start - 1])))
                                   for r in gaps[both_sides].itertuples()]
gaps["need_speed"] = gaps.dist_m / ((gaps.length + 1) / FPS)
gaps["interpolated"] = both_sides & (gaps.length <= MAX_GAP) & (gaps.need_speed <= MAX_GAP_SPEED)


def load_atd_xy(path, n_slots):
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            _, groups, b = line.rstrip("\r\n").split(":")
            rows.append((groups, b))
    pl = np.full((len(rows), n_slots, 2), np.nan, dtype=np.float32)
    bl = np.full((len(rows), 2), np.nan, dtype=np.float32)
    for i, (groups, b) in enumerate(rows):
        pl[i] = np.array(groups.replace(";", ",").split(","), dtype=np.float32).reshape(n_slots, 3)[:, :2]
        bl[i] = np.array(b.split(","), dtype=np.float32)[:2]
    scale = np.array([PITCH_L, PITCH_W], dtype=np.float32)
    return pl * scale, bl * scale


ball_bc = np.full((N_FRAMES, 2), np.nan, dtype=np.float32)
bc_cost = np.full(N_FRAMES, np.nan, dtype=np.float32)
bc_view = np.full(N_FRAMES, -2, dtype=np.int8)
bmeta = parse_atd_meta("broadcast")
assert bmeta["home_name"] == meta["home_name"] and bmeta["slot_ids"] == bmeta["channel_order"]
B_HOME = bmeta["slot_is_home"]
with timer("read broadcast ATD"):
    PB, BB = load_atd_xy(require_asset("atd_raw", "broadcast"), len(B_HOME))
NB = len(PB)
nvis_b = np.isfinite(PB[..., 0]).sum(1)
T_TEAM = {True: np.nan_to_num(players[:, SLOT_IS_HOME, :2], nan=1e4),
          False: np.nan_to_num(players[:, ~SLOT_IS_HOME, :2], nan=1e4)}


def match_cost(b, t_idx):
    tot, n = np.zeros(len(t_idx)), 0
    for team in (True, False):
        pb = PB[b, B_HOME == team]
        pb = pb[np.isfinite(pb[:, 0])]
        if len(pb):
            d = np.sqrt(((T_TEAM[team][t_idx][:, None, :, :] - pb[None, :, None, :]) ** 2).sum(-1)).min(-1)
            tot += np.minimum(d, 20.0).sum(1)
            n += len(pb)
    return tot / max(n, 1)


def next_rich(b):
    while b < NB and nvis_b[b] < 10:
        b += 7
    return b


with timer("coarse offset"):
    coarse = []
    cand_t = np.arange(0, N_FRAMES, 5)
    for b in np.linspace(2000, NB - 2000, 40).astype(int):
        b = next_rich(b)
        if b >= NB:
            continue
        c = match_cost(b, cand_t)
        i = int(np.argmin(c))
        if c[i] < 2.0:
            coarse.append((int(HALF[cand_t[i]]), int(cand_t[i]) - b))
coarse = pd.DataFrame(coarse, columns=["half", "offset"])
coarse = coarse[coarse.half > 0]

BC_OFFSET = {}
with timer("precise offset"):
    for h, (a, b_end) in HALVES.items():
        c0 = int(coarse[coarse.half == h].offset.median())
        found = []
        for t_guess in np.linspace(a + 500, b_end - 500, 120).astype(int):
            b = next_rich(t_guess - c0)
            if b >= NB:
                continue
            t_idx = np.arange(b + c0 - BC_SEARCH, b + c0 + BC_SEARCH + 1)
            t_idx = t_idx[(t_idx >= 0) & (t_idx < N_FRAMES)]
            c = match_cost(b, t_idx)
            i = int(np.argmin(c))
            if c[i] < 1.5:
                found.append((b, int(t_idx[i]) - b))
        F_ = np.array(found)
        vals, cnt = np.unique(F_[:, 1], return_counts=True)
        BC_OFFSET[h] = int(vals[np.argmax(cnt)])
print("broadcast → tactical offset per half (frames):", BC_OFFSET)

t_all = np.flatnonzero(GRADED)
b_all = t_all - np.array([BC_OFFSET[h] for h in HALF[t_all]])
ok_ = (b_all >= 0) & (b_all < NB)
t_all, b_all = t_all[ok_], b_all[ok_]
with timer("per-frame consistency"):
    for s0 in range(0, len(t_all), 20000):
        ti, bi = t_all[s0:s0 + 20000], b_all[s0:s0 + 20000]
        tot, cnt = np.zeros(len(ti)), np.zeros(len(ti))
        for team in (True, False):
            pb = PB[bi][:, B_HOME == team]
            fin = np.isfinite(pb[..., 0])
            d = np.sqrt(((np.nan_to_num(pb, nan=-1e4)[:, :, None, :] - T_TEAM[team][ti][:, None, :, :]) ** 2).sum(-1)).min(-1)
            tot += np.where(fin, np.minimum(d, 20.0), 0).sum(1)
            cnt += fin.sum(1)
        bc_cost[ti] = np.where(cnt >= BC_MIN_PLAYERS, tot / np.maximum(cnt, 1), np.nan)
        good = np.isfinite(bc_cost[ti]) & (bc_cost[ti] < BC_COST_MAX)
        ball_bc[ti[good]] = BB[bi[good]]
consistent = np.isfinite(bc_cost) & (bc_cost < BC_COST_MAX)
b_valid = pd.read_csv(require_asset("homography", "broadcast"), usecols=["valid"]).valid.to_numpy() == 1
bc_view[t_all] = np.where(b_valid[b_all], 0, -1)
bc_view[consistent] = 1
both_ball = GRADED & finite & np.isfinite(ball_bc[:, 0])
dist_bb = np.hypot(*(ball_bc[both_ball] - ball_raw[both_ball]).T)
check("broadcast ball = tactical ball (median distance < 1.5 m)", np.median(dist_bb) < 1.5, "{:.2f} m".format(np.median(dist_bb)))

fill = GRADED & ~finite & np.isfinite(ball_bc[:, 0])
ball_filled = np.where(fill[:, None], ball_bc, ball_raw).astype(np.float32)
ball_src = np.where(finite, 1, np.where(fill, 3, 0)).astype(np.int8)

BRIDGE_LINEAR_S, BRIDGE_TAU_S, BRIDGE_MAX_JUMP = 2.0, 0.5, 5.0
bridge_rows = []
edges = np.diff(np.r_[0, fill.astype(np.int8), 0])
for s_, e_ in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
    n_ = e_ - s_
    d_left = ball_raw[s_ - 1] - ball_bc[s_] if s_ > 0 and finite[s_ - 1] else None
    d_right = ball_raw[e_] - ball_bc[e_ - 1] if e_ < N_FRAMES and finite[e_] else None
    jump = max([float(np.hypot(*d)) for d in (d_left, d_right) if d is not None], default=0.0)
    if jump > BRIDGE_MAX_JUMP:
        ball_filled[s_:e_] = np.nan
        ball_src[s_:e_] = 0
        action = "dibuang"
    else:
        i_ = np.arange(n_, dtype=np.float32)[:, None]
        if d_left is not None and d_right is not None and n_ <= BRIDGE_LINEAR_S * FPS:
            corr = d_left + (d_right - d_left) * (i_ + 1) / (n_ + 1)
            action = "linier"
        else:
            corr = np.zeros((n_, 2), np.float32)
            if d_left is not None:
                corr += d_left * np.exp(-i_ / (BRIDGE_TAU_S * FPS))
            if d_right is not None:
                corr += d_right * np.exp(-(n_ - 1 - i_) / (BRIDGE_TAU_S * FPS))
            action = "faded" if (d_left is not None or d_right is not None) else "no edge"
        ball_filled[s_:e_] = ball_bc[s_:e_] + corr
    bridge_rows.append({"panjang_s": n_ / FPS, "loncatan_m": jump, "aksi": action})
bridge = pd.DataFrame(bridge_rows)
print("broadcast fill: {} segments | action: {} | median edge jump {:.2f} m".format(
    len(bridge), bridge.aksi.value_counts().to_dict(), bridge.loncatan_m.median()))
for r in gaps[gaps.interpolated].itertuples():
    seg = slice(r.start, r.end_excl)
    if (ball_src[seg] == 0).all():
        t = (np.arange(1, r.length + 1) / (r.length + 1))[:, None]
        ball_filled[seg] = ball_raw[r.start - 1] + (ball_raw[r.end_excl] - ball_raw[r.start - 1]) * t
        ball_src[seg] = 2
fx, fy = ball_filled[:, 0], ball_filled[:, 1]
dxf = np.maximum.reduce([-fx, fx - PITCH_L, np.zeros_like(fx)])
dyf = np.maximum.reduce([-fy, fy - PITCH_W, np.zeros_like(fy)])
ball_oob_m = np.where(np.isfinite(fx), np.hypot(dxf, dyf), np.nan).astype(np.float32)
print("ball available (graded frames): tactical {:.1f}% → combined {:.1f}% | broadcast replay/close-up {:.1f}%".format(
    100 * finite[GRADED].mean(), 100 * (ball_src[GRADED] > 0).mean(), 100 * (bc_view[GRADED] == -1).mean()))

z = {"players": players, "slot_is_home": SLOT_IS_HOME, "ball_filled": ball_filled, "ball_src": ball_src,
     "ball_oob_m": ball_oob_m, "dead_prior": dead_prior, "attack_dir": ATTACK_DIR, "bc_view": bc_view}

### Figure: ball availability

Left: share of graded frames per half by ball-position source. Right: length distribution of the ball gaps that
**remain** after merging (log scale). Almost all missing frames sit in long gaps, so the possession model must keep
working without the ball and rely on the team shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
src_names = [(1, "tactical", COL["H"]), (3, "broadcast", COL["aqua"]), (2, "interpolation", COL["yellow"]),
             (0, "missing", COL["D"])]
for row, h in enumerate((1, 2)):
    m = GRADED & (HALF == h)
    left = 0.0
    for code_, name, col in src_names:
        share = 100 * float((ball_src[m] == code_).mean())
        ax.barh(row, share, left=left, color=col, height=0.55, edgecolor=COL["surface"], linewidth=2,
                label=name if row == 0 else None)
        if share >= 6:
            ax.text(left + share / 2, row, "{:.0f}%".format(share), ha="center", va="center", fontsize=9,
                    color="white" if code_ != 0 else COL["text"])
        left += share
ax.set_yticks([0, 1], ["half 1", "half 2"])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of graded frames")
ax.grid(axis="y", visible=False)
ax.set_title("Ball position source", pad=18)
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, -0.28))

ax = axes[1]
gap_s = []
for h, (a, b) in HALVES.items():
    s_, e_ = runs(ball_src[a:b + 1] == 0)
    gap_s += list((e_ - s_) / FPS)
gap_s = np.array(gap_s)
bins = np.logspace(np.log10(0.04), np.log10(max(gap_s.max(), 1)), 30)
ax.hist(gap_s, bins=bins, weights=gap_s, color=COL["H"], edgecolor=COL["surface"], linewidth=1.5)
ax.set_xscale("log")
ax.axvline(np.median(gap_s), color=COL["muted"], linewidth=1, linestyle="--")
ax.set_xlabel("gap length (seconds, log scale)")
ax.set_ylabel("seconds of missing ball")
ax.set_title("Remaining ball gaps", pad=18)
note(ax, "{} gaps · {:.0f}% of the missing time lies in gaps > 3 s".format(len(gap_s), 100 * gap_s[gap_s > 3].sum() / gap_s.sum()))
plt.tight_layout()
plt.show()

## 2b. Ball from video: a YOLO detector trained for the tactical camera

**What this step does.** Builds a third, independent ball source from the tactical video, without any hand labels.

The ball tracking we received has two weaknesses that are directly visible in the video (section 6c notes):

- the **ball is missing** on ± 41% of the graded frames (mostly during dead-ball time, but also at the moments of a
  challenge);
- the **ball "floats"**: the tactical tracker sometimes loses the ball and keeps extrapolating it into empty space for
  ± 3 s, while the real ball is clearly visible in the video. Two errors in the public 5 minutes happen exactly here.

CHALLENGE.md allows improving or replacing the tracking (*"Improving the provided data, or replacing it with your own
detection and tracking, is entirely legitimate"*).

| Step | What it does | Why |
|---|---|---|
| 2b.1 Automatic labels | frames where the tactical and broadcast balls agree (< 1 m) → ball position projected to pixels through the homography → **COCO-pretrained YOLO26x** (class *sports ball*) searches for the ball only within 1.5 m of that point; the label is accepted if there is exactly one candidate | the ATD projection alone is off by ± 0.4 m (± 10–30 px, larger than the ± 8 px ball); a COCO detector on the whole frame is too noisy (#41), but in a small window it lands exactly on the ball |
| 2b.2 Train the detector | YOLO26s fine-tuned (1 class: ball) on native-scale 640 px crops from the automatic labels; train/validation split per **60 s block** | a detector specific to this camera, for small 6–12 px balls |
| 2b.3 Detect on the whole video | every graded frame, full 1920 px resolution, up to 8 candidates per frame; saved per chunk so an interrupted run can resume | |
| 2b.4 New tracking | drop candidates off the pitch (spare balls) and at static **hot spots**; choose one track with Viterbi (physical jump ≤ 1.6 m/frame ≈ 40 m/s) | one frame can have several candidates (socks, penalty spot, spare balls) |
| 2b.5 Label-free validation | validation blocks: how often and how close YOLO is to the tactical ball when both trackers agree; confidence threshold chosen for precision ≥ 95%; three-source vote when the trackers disagree | every threshold is set without GT |
| 2b.6 Merging | correct floating tactical balls, fill gaps with the source with the smallest error, bridge the gap edges | |
| 2b.7 Transparency | automatic labels, training curves, example detections, hot-spot map, examples of corrections and fills | |

**The boundary we keep.** The dead-ball detector (section 4) still uses the tracking ball: it uses "ball missing" as a
dead-ball cue, and a YOLO ball during stoppages (ball in a player's hands, spare balls) would change that cue without
validation. The merged ball goes only into the **possession model** (section 5), and it is also used to tidy
the `D` boundaries. The possession model weights do not change.

**Time, memory and cache.** Measured on an RTX 4070 SUPER 12 GB: automatic labels ± 5–10 minutes, YOLO26s training
± 2.5 minutes per epoch (25 epochs ± 1 hour, ± 4.5 GB GPU memory), detection on the whole video ± 85 frames/s (± 31
minutes). YOLO26m on 960 px crops does not fit in 12 GB and is twice as slow. Close other Jupyter kernels before
running this section: the notebook checks the remaining GPU memory and RAM. All results are stored in `cache/yolo/`
(git-ignored, contains crops of the match video: do not share); later runs load them in seconds. `YOLO_STAGE = "off"`
reproduces the tracking-ball pipeline. Ultralytics is licensed under AGPL-3.0.

In [ ]:
import gc
import shutil
import threading
import queue
from contextlib import chdir

import cv2
import torch

YOLO_ON = YOLO_STAGE != "off"
YOLO_DIR = CACHE / "yolo"
YO_W, YO_REP = YOLO_DIR / "weights", YOLO_DIR / "reports"
for _f in (YO_W, YO_REP):
    _f.mkdir(parents=True, exist_ok=True)


def find_video(source):
    try:
        p = asset_path("video", source)
    except (KeyError, TypeError):
        p = None
    if p is not None:
        return p
    for folder in (DATA_DIR / source, DATA_DIR, ROOT / "data", ROOT):
        hits = sorted(folder.glob("*{}*.mp4".format(source))) if folder.is_dir() else []
        if hits:
            return hits[0]
    return None


VIDEO_T = find_video("tactical")
if YOLO_ON and VIDEO_T is None:
    print("tactical video not found in {}: section 2b skipped, tracking ball used".format(DATA_DIR))
    YOLO_ON = False
check("tactical video available for YOLO", VIDEO_T is not None, str(VIDEO_T), critical=False)

hom_t = pd.read_csv(require_asset("homography"))
H_T = hom_t[["h00", "h01", "h02", "h10", "h11", "h12", "h20", "h21", "h22"]].to_numpy(np.float64).reshape(-1, 3, 3)
H_OK = (hom_t["valid"].to_numpy() == 1) & np.isfinite(H_T).all((1, 2))
H_INV = np.full_like(H_T, np.nan)
H_INV[H_OK] = np.linalg.inv(H_T[H_OK])
check("tactical homography: one row per frame", len(H_T) == N_FRAMES, "{} rows, valid on {:.1f}% of graded frames".format(
    len(H_T), 100 * H_OK[GRADED].mean()))


def _apply_h(Hs, pts):
    p = np.einsum("nij,nj->ni", Hs, np.c_[pts, np.ones(len(pts))])
    return p[:, :2] / p[:, 2:3]


def m_to_px(t, xy):
    """pitch metres -> tactical video pixels, per frame t"""
    return _apply_h(H_T[np.asarray(t)], np.asarray(xy, float) / [PITCH_L, PITCH_W])


def px_to_m(t, uv):
    """tactical video pixels -> pitch metres (ground plane), per frame t"""
    return _apply_h(H_INV[np.asarray(t)], np.asarray(uv, float)) * [PITCH_L, PITCH_W]


def px_per_m(t, xy):
    return np.hypot(*(m_to_px(t, np.asarray(xy) + [1.0, 0.0]) - m_to_px(t, xy)).T)


class FrameReader:
    """read video frames in order; short jumps with grab(), long jumps with seek"""

    def __init__(self, path):
        self.cap, self.pos = cv2.VideoCapture(str(path)), 0

    def read(self, t):
        t = int(t)
        if t < self.pos or t - self.pos > 250:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, t)
            self.pos = t
        while self.pos < t:
            self.cap.grab()
            self.pos += 1
        ok_, img = self.cap.read()
        self.pos += 1
        return img if ok_ else None

    def close(self):
        self.cap.release()


def show_image(path, title, width=14):
    img = cv2.imread(str(path))
    if img is None:
        print("image missing:", path)
        return
    h_, w_ = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h_ / w_))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_axis_off()
    ax.set_title(title, pad=10)
    plt.tight_layout()
    plt.show()


def mosaic(tiles, cols, path):
    tiles = [t_ for t_ in tiles if t_ is not None]
    if not tiles:
        return None
    th, tw = tiles[0].shape[:2]
    rows_ = (len(tiles) + cols - 1) // cols
    out = np.full((rows_ * th, cols * tw, 3), 245, np.uint8)
    for k_, t_ in enumerate(tiles):
        out[(k_ // cols) * th:(k_ // cols + 1) * th, (k_ % cols) * tw:(k_ % cols + 1) * tw] = t_
    cv2.imwrite(str(path), out)
    return path


def crop_zoom(img, cx, cy, half=48, zoom=3):
    """square crop around (cx, cy), enlarged; image borders padded with black"""
    pad = cv2.copyMakeBorder(img, half, half, half, half, cv2.BORDER_CONSTANT)
    x0, y0 = int(round(cx)), int(round(cy))
    c_ = pad[y0:y0 + 2 * half, x0:x0 + 2 * half]
    return cv2.resize(c_, (2 * half * zoom, 2 * half * zoom), interpolation=cv2.INTER_NEAREST)


C_T, C_B, C_Y, C_F = (0, 215, 255), (255, 200, 0), (60, 220, 60), (230, 60, 230)   # BGR: tactical, broadcast, YOLO, combined

if YOLO_ON:
    import ultralytics
    from ultralytics import YOLO
    from ultralytics.utils.downloads import attempt_download_asset

    YO_DEV = 0 if torch.cuda.is_available() else "cpu"
    YO_HALF = torch.cuda.is_available()
    _cap = cv2.VideoCapture(str(VIDEO_T))
    VID_W, VID_H, VID_N = int(_cap.get(3)), int(_cap.get(4)), int(_cap.get(7))
    _cap.release()
    check("tactical video: frame count = ATD", VID_N == N_FRAMES, "{} frames, {}×{}".format(VID_N, VID_W, VID_H))

    def yolo_weights(name):
        path_ = YO_W / name
        if not path_.is_file():
            attempt_download_asset(str(path_))
        return path_

    def gpu_free():
        gc.collect()
        if not torch.cuda.is_available():
            return float("nan")
        torch.cuda.empty_cache()
        return torch.cuda.mem_get_info()[0] / 2 ** 30

    print("ultralytics {} | {} | video {} ({}×{}, {} frames)".format(
        ultralytics.__version__, torch.cuda.get_device_name(0) if YO_DEV == 0 else "CPU", VIDEO_T.name, VID_W, VID_H, VID_N))
    try:
        import psutil
        ram_free = psutil.virtual_memory().available / 2 ** 30
    except ImportError:
        ram_free = float("nan")
    print("free memory: GPU {:.1f} GB | RAM {:.1f} GB".format(gpu_free(), ram_free))
    check("free GPU memory ≥ 6 GB to train the detector (close other Jupyter kernels if this fails)",
          not (gpu_free() < 6.0), "{:.1f} GB".format(gpu_free()), critical=False)
    check("free RAM ≥ 4 GB", not (ram_free < 4.0), "{:.1f} GB".format(ram_free), critical=False)

### 2b.1 Automatic labels

Label candidates are graded frames where the ball is present in **both** trackers and they agree within 1 m (two
independent systems agree, so the position can be trusted). That point is projected to pixels, and YOLO26x (COCO)
searches for a *sports ball* only in the 640 px crop around it. The label is accepted if **exactly one** candidate
(confidence ≥ 0.10) lies within 1.5 m; with zero or more than one, the frame is dropped (ambiguous). The YOLO box, not
the ATD projection, becomes the label, so the label sits exactly on the ball pixels.

Every label becomes one **native-scale** 640 × 640 px crop (ball at a random position inside the crop), plus one crop
without a ball for every five frames as a negative example. Frames are split in 60 s blocks: the 3rd block of every 5
is validation, so validation frames are never adjacent to training frames.

In [ ]:
YO_AGREE_M, YO_RADIUS_M, YO_TEACHER_CONF, YO_CROP, YO_NEG_EVERY = 1.0, 1.5, 0.10, 640, 5
YO_DS = YOLO_DIR / "dataset"
YO_LABELS = YOLO_DIR / "labels_auto.csv"
YO_LABEL_KEY = {"n": YOLO_N_LABEL, "teacher": YOLO_TEACHER, "agree_m": YO_AGREE_M, "radius_m": YO_RADIUS_M,
                "conf": YO_TEACHER_CONF, "crop": YO_CROP, "neg_every": YO_NEG_EVERY}


def yo_val_block(t):
    """3rd 60-s block of every 5 = validation"""
    return (np.asarray(t) // (60 * FPS)) % 5 == 2


def _key_ok(path_, key):
    return path_.is_file() and json.loads(path_.read_text(encoding="utf-8")) == key


if YOLO_ON:
    if YOLO_STAGE == "retrain" or not (YO_LABELS.is_file() and _key_ok(YOLO_DIR / "labels_key.json", YO_LABEL_KEY)):
        shutil.rmtree(YO_DS, ignore_errors=True)
        for sub in ("train", "val"):
            (YO_DS / sub / "images").mkdir(parents=True, exist_ok=True)
            (YO_DS / sub / "labels").mkdir(parents=True, exist_ok=True)
        both_ = GRADED & finite & np.isfinite(ball_bc[:, 0]) & H_OK
        both_[both_] = np.hypot(*(ball_raw[both_] - ball_bc[both_]).T) < YO_AGREE_M
        cand_ = np.flatnonzero(both_)
        uv_ = m_to_px(cand_, ball_raw[cand_])
        inside_ = (uv_[:, 0] > 16) & (uv_[:, 0] < VID_W - 16) & (uv_[:, 1] > 16) & (uv_[:, 1] < VID_H - 16)
        cand_ = cand_[inside_]
        rng_ = np.random.default_rng(0)
        pick_ = np.sort(rng_.choice(cand_, min(YOLO_N_LABEL, len(cand_)), replace=False))
        teacher = YOLO(str(yolo_weights(YOLO_TEACHER)))
        reader = FrameReader(VIDEO_T)
        rows_lab, tiles_ok, tiles_bad = [], [], []
        t0_ = time.time()
        for s0 in range(0, len(pick_), 32):
            ts_ = pick_[s0:s0 + 32]
            imgs_, crops_, origin_ = [], [], []
            for t_ in ts_:
                img_ = reader.read(t_)
                u_, v_ = m_to_px([t_], ball_raw[[t_]])[0]
                x0_ = int(np.clip(u_ - 320, 0, VID_W - 640))
                y0_ = int(np.clip(v_ - 320, 0, VID_H - 640))
                imgs_.append(img_)
                crops_.append(img_[y0_:y0_ + 640, x0_:x0_ + 640])
                origin_.append((x0_, y0_, u_, v_))
            preds_ = teacher.predict(crops_, imgsz=640, conf=0.05, classes=[32], half=YO_HALF, device=YO_DEV, verbose=False)
            for t_, img_, (x0_, y0_, u_, v_), r_ in zip(ts_, imgs_, origin_, preds_):
                bx_ = r_.boxes.xyxy.cpu().numpy().astype(float) + [x0_, y0_, x0_, y0_]
                cf_ = r_.boxes.conf.cpu().numpy().astype(float)
                ppm_ = float(px_per_m([t_], ball_raw[[t_]])[0])
                cxy_ = np.c_[(bx_[:, 0] + bx_[:, 2]) / 2, (bx_[:, 1] + bx_[:, 3]) / 2]
                d_ = np.hypot(cxy_[:, 0] - u_, cxy_[:, 1] - v_) / ppm_
                near_ = (d_ < YO_RADIUS_M) & (cf_ >= YO_TEACHER_CONF)
                row_ = {"frame": int(t_), "split": "val" if yo_val_block(t_) else "train", "n_kandidat": int(near_.sum()),
                        "px_per_m": ppm_, "diterima": bool(near_.sum() == 1)}
                if row_["diterima"]:
                    j_ = int(np.flatnonzero(near_)[0])
                    x1_, y1_, x2_, y2_ = bx_[j_]
                    row_.update(conf=cf_[j_], jarak_atd_m=d_[j_], x1=x1_, y1=y1_, x2=x2_, y2=y2_)
                    cx_, cy_ = (x1_ + x2_) / 2, (y1_ + y2_) / 2
                    ox_ = int(np.clip(cx_ - rng_.uniform(64, YO_CROP - 64), 0, VID_W - YO_CROP))
                    oy_ = int(np.clip(cy_ - rng_.uniform(64, YO_CROP - 64), 0, VID_H - YO_CROP))
                    stem_ = "f{:06d}".format(t_)
                    split_dir = YO_DS / row_["split"]
                    cv2.imwrite(str(split_dir / "images" / (stem_ + ".jpg")), img_[oy_:oy_ + YO_CROP, ox_:ox_ + YO_CROP],
                                [cv2.IMWRITE_JPEG_QUALITY, 95])
                    (split_dir / "labels" / (stem_ + ".txt")).write_text("0 {:.6f} {:.6f} {:.6f} {:.6f}\n".format(
                        (cx_ - ox_) / YO_CROP, (cy_ - oy_) / YO_CROP, (x2_ - x1_) / YO_CROP, (y2_ - y1_) / YO_CROP))
                    if len(rows_lab) % YO_NEG_EVERY == 0:
                        nx_ = VID_W - YO_CROP if cx_ < VID_W / 2 else 0
                        if not (nx_ - 16 <= cx_ <= nx_ + YO_CROP + 16):
                            ny_ = int(rng_.integers(0, VID_H - YO_CROP + 1))
                            cv2.imwrite(str(split_dir / "images" / (stem_ + "_neg.jpg")),
                                        img_[ny_:ny_ + YO_CROP, nx_:nx_ + YO_CROP], [cv2.IMWRITE_JPEG_QUALITY, 95])
                            (split_dir / "labels" / (stem_ + "_neg.txt")).write_text("")
                    if len(tiles_ok) < 24 and len(rows_lab) % 7 == 0:
                        tile_ = crop_zoom(img_, cx_, cy_)
                        s_ = 3
                        cv2.rectangle(tile_, (int((x1_ - cx_ + 48) * s_), int((y1_ - cy_ + 48) * s_)),
                                      (int((x2_ - cx_ + 48) * s_), int((y2_ - cy_ + 48) * s_)), C_Y, 2)
                        cv2.drawMarker(tile_, (int((u_ - cx_ + 48) * s_), int((v_ - cy_ + 48) * s_)), C_T, cv2.MARKER_CROSS, 22, 2)
                        cv2.putText(tile_, "f{} conf {:.2f} ATD {:.2f} m".format(t_, cf_[j_], d_[j_]), (4, 14),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
                        tiles_ok.append(tile_)
                elif len(tiles_bad) < 12 and len(rows_lab) % 5 == 0:
                    tile_ = crop_zoom(img_, u_, v_)
                    cv2.drawMarker(tile_, (144, 144), C_T, cv2.MARKER_CROSS, 22, 2)
                    for (x1_, y1_, x2_, y2_), c__ in zip(bx_, cf_):
                        cv2.rectangle(tile_, (int((x1_ - u_ + 48) * 3), int((y1_ - v_ + 48) * 3)),
                                      (int((x2_ - u_ + 48) * 3), int((y2_ - v_ + 48) * 3)), (40, 40, 230), 2)
                    cv2.putText(tile_, "f{} candidates {}".format(t_, int(near_.sum())), (4, 14), cv2.FONT_HERSHEY_SIMPLEX,
                                0.42, (255, 255, 255), 1, cv2.LINE_AA)
                    tiles_bad.append(tile_)
                rows_lab.append(row_)
            if (s0 // 32) % 50 == 0:
                print("  auto labels {}/{} frames | {:.0f} s".format(s0 + len(ts_), len(pick_), time.time() - t0_), flush=True)
        reader.close()
        del teacher
        gpu_free()
        pd.DataFrame(rows_lab).to_csv(YO_LABELS, index=False)
        mosaic(tiles_ok, 6, YO_REP / "label_diterima.jpg")
        mosaic(tiles_bad, 6, YO_REP / "label_ditolak.jpg")
        (YO_DS / "data.yaml").write_text("path: {}\ntrain: train/images\nval: val/images\nnames:\n  0: ball\n".format(
            YO_DS.as_posix()), encoding="utf-8")
        (YOLO_DIR / "labels_key.json").write_text(json.dumps(YO_LABEL_KEY), encoding="utf-8")
        print("automatic labelling finished in {:.0f} s".format(time.time() - t0_))
    LAB_AUTO = pd.read_csv(YO_LABELS)
    acc_ = LAB_AUTO[LAB_AUTO.diterima]
    tab_ = LAB_AUTO.groupby("split").agg(kandidat=("frame", "size"), diterima=("diterima", "sum"))
    tab_["accepted share"] = (tab_.diterima / tab_.kandidat).round(3)
    print(tab_.to_string())
    print("rejected: 0 candidates {:.1%} · ≥ 2 candidates {:.1%} | labels: median confidence {:.2f} · median distance to the ATD projection "
          "{:.2f} m (p90 {:.2f}) · median box size {:.0f}×{:.0f} px".format(
              (LAB_AUTO.n_kandidat == 0).mean(), (LAB_AUTO.n_kandidat >= 2).mean(), acc_.conf.median(),
              acc_.jarak_atd_m.median(), acc_.jarak_atd_m.quantile(0.9), (acc_.x2 - acc_.x1).median(), (acc_.y2 - acc_.y1).median()))
    check("enough automatic labels (≥ 1,000 train, ≥ 200 validation)",
          tab_.loc["train", "diterima"] >= 1000 and tab_.loc["val", "diterima"] >= 200, critical=False)
    show_image(YO_REP / "label_diterima.jpg", "Accepted automatic labels: green box = label (YOLO26x COCO), "
               "yellow cross = projected ATD ball")
    show_image(YO_REP / "label_ditolak.jpg", "Rejected frames (0 or ≥ 2 candidates): red box = COCO candidate", width=12)

### 2b.2 Training the ball detector

The starting point is the COCO-pretrained YOLO26 weights; the model is fine-tuned for **one class (ball)** on native-scale
640 px crops. Because a convolutional network does not depend on the image size, a model trained on these crops can be
used directly on the full 1920 × 1080 frame without shrinking the ball. Augmentation: mosaic, ± 25% scale, translation,
left-right flip, small colour changes. The best weights (validation mAP) are saved together with their SHA-256.

The figures below show the train/validation loss curves, precision, recall and mAP per epoch, the precision-recall
curve on the validation blocks, and example predictions on validation images. If the GPU runs out of memory,
Ultralytics lowers the batch size itself; if it still fails, training is repeated with half the batch and no dataloader
workers.

In [ ]:
YO_RUN = YOLO_DIR / "student"
YO_BEST = YO_RUN / "weights" / "best.pt"
YO_TRAIN_KEY = dict(YO_LABEL_KEY, student=YOLO_STUDENT, epochs=YOLO_EPOCHS, batch=YOLO_BATCH)
if YOLO_ON:
    if YOLO_STAGE == "retrain" or not (YO_BEST.is_file() and _key_ok(YOLO_DIR / "student_key.json", YO_TRAIN_KEY)):
        shutil.rmtree(YO_RUN, ignore_errors=True)
        student = YOLO(str(yolo_weights(YOLO_STUDENT)))
        train_args = dict(data=str(YO_DS / "data.yaml"), imgsz=YO_CROP, epochs=YOLO_EPOCHS, batch=YOLO_BATCH, device=YO_DEV,
                          workers=YOLO_WORKERS, project=str(YOLO_DIR), name="student", exist_ok=True, seed=0,
                          deterministic=False, plots=True, verbose=False, mosaic=1.0, scale=0.25, translate=0.1,
                          degrees=0.0, fliplr=0.5, hsv_h=0.01, close_mosaic=5, patience=20, amp=YO_HALF)
        print("free GPU memory before training: {:.1f} GB".format(gpu_free()))
        with timer("train ball detector"), chdir(YO_W):
            try:
                student.train(**train_args)
            except (RuntimeError, OSError, EOFError, BrokenPipeError, MemoryError) as err_:
                print("training failed (batch {}, workers {}): {} | retrying with batch {} and workers 0".format(
                    YOLO_BATCH, YOLO_WORKERS, str(err_).splitlines()[0], max(YOLO_BATCH // 2, 2)))
                del student
                gpu_free()
                shutil.rmtree(YO_RUN, ignore_errors=True)
                student = YOLO(str(yolo_weights(YOLO_STUDENT)))
                student.train(**dict(train_args, workers=0, batch=max(YOLO_BATCH // 2, 2)))
        del student
        gpu_free()
        (YOLO_DIR / "student_key.json").write_text(json.dumps(YO_TRAIN_KEY), encoding="utf-8")
    YO_BEST_SHA = sha256(YO_BEST)
    print("detector: {} | sha256 {}".format(YO_BEST.relative_to(ROOT), YO_BEST_SHA[:16]))
    hist_ = pd.read_csv(YO_RUN / "results.csv")
    hist_.columns = [c_.strip() for c_ in hist_.columns]
    last_ = hist_.iloc[-1]
    best_ = hist_.loc[hist_["metrics/mAP50-95(B)"].idxmax()]
    print("epoch {} | best at epoch {}: precision {:.3f} · recall {:.3f} · mAP50 {:.3f} · mAP50-95 {:.3f}".format(
        int(last_["epoch"]), int(best_["epoch"]), best_["metrics/precision(B)"], best_["metrics/recall(B)"],
        best_["metrics/mAP50(B)"], best_["metrics/mAP50-95(B)"]))
    check("ball detector: validation recall ≥ 0.80", best_["metrics/recall(B)"] >= 0.80,
          "{:.3f}".format(best_["metrics/recall(B)"]), critical=False)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    loss_cols = [c_ for c_ in hist_.columns if c_.endswith("_loss")]
    for c_ in loss_cols:
        axes[0].plot(hist_["epoch"], hist_[c_], linewidth=1.8, color=COL["H"] if c_.startswith("train") else COL["A"],
                     linestyle="-" if "box" in c_ else ("--" if "cls" in c_ else ":"), label=c_)
    axes[0].set_title("Loss", pad=12)
    axes[0].legend(fontsize=7)
    for c_, col_, nm_ in (("metrics/precision(B)", COL["H"], "presisi"), ("metrics/recall(B)", COL["A"], "recall")):
        axes[1].plot(hist_["epoch"], hist_[c_], color=col_, linewidth=2, label=nm_)
    axes[1].set_ylim(0, 1.02)
    axes[1].set_title("Precision & recall (validation)", pad=12)
    axes[1].legend()
    for c_, col_ in (("metrics/mAP50(B)", COL["aqua"]), ("metrics/mAP50-95(B)", COL["violet"])):
        axes[2].plot(hist_["epoch"], hist_[c_], color=col_, linewidth=2, label=c_.split("/")[1])
    axes[2].set_ylim(0, 1.02)
    axes[2].set_title("mAP (validation)", pad=12)
    axes[2].legend()
    for ax in axes:
        ax.set_xlabel("epoch")
    plt.tight_layout()
    plt.show()
    for name_, title_ in (("BoxPR_curve.png", "Kurva presisi-recall, blok validasi"),
                          ("PR_curve.png", "Kurva presisi-recall, blok validasi"),
                          ("val_batch0_pred.jpg", "Detector predictions on validation images (640 px crops)"),
                          ("val_batch1_pred.jpg", "Detector predictions on validation images (continued)")):
        if (YO_RUN / name_).is_file():
            show_image(YO_RUN / name_, title_, width=11 if "curve" in name_ else 14)

### 2b.3 Ball detection on the whole video

The detector runs on **every graded frame** (158,857 frames) at full resolution, with a low threshold (0.05) and up to
8 candidates per frame: filtering happens in the tracking step, not in the detector. Results are saved per chunk of
5,000 frames in `cache/yolo/det/`; if a run is interrupted, finished chunks are not repeated. If the detector weights
change, the detection cache is discarded automatically.

In [ ]:
YO_INFER_SZ, YO_DET_CONF, YO_K, YO_CHUNK = 1920, 0.05, 8, 5000
YO_DET = YOLO_DIR / "det"
DET_XYXY = np.full((N_FRAMES, YO_K, 4), np.nan, np.float32)
DET_CONF = np.zeros((N_FRAMES, YO_K), np.float32)
if YOLO_ON:
    det_key_ = {"weights": YO_BEST_SHA, "imgsz": YO_INFER_SZ, "conf": YO_DET_CONF, "k": YO_K, "chunk": YO_CHUNK}
    if YOLO_STAGE == "retrain" or not _key_ok(YO_DET / "key.json", det_key_):
        shutil.rmtree(YO_DET, ignore_errors=True)
    YO_DET.mkdir(parents=True, exist_ok=True)
    (YO_DET / "key.json").write_text(json.dumps(det_key_), encoding="utf-8")
    det_frames_ = np.flatnonzero(GRADED & H_OK)
    if YO_INFER_RANGES is not None:
        _m = np.zeros(N_FRAMES, bool)
        for _a, _b in YO_INFER_RANGES:
            _m[_a:_b] = True
        det_frames_ = det_frames_[_m[det_frames_]]
    chunks_ = [det_frames_[i:i + YO_CHUNK] for i in range(0, len(det_frames_), YO_CHUNK)]
    todo_ = [k_ for k_, c_ in enumerate(chunks_) if not (YO_DET / "c{:03d}_{}.npz".format(k_, c_[0])).is_file()]
    print("detection: {} frames in {} chunks | already done {} | to do {}".format(
        len(det_frames_), len(chunks_), len(chunks_) - len(todo_), len(todo_)))
    if todo_:
        detector = YOLO(str(YO_BEST))
        t0_, n_done = time.time(), 0
        for k_ in todo_:
            frames_k = chunks_[k_]
            q_ = queue.Queue(maxsize=64)

            def _produce(frames_list=frames_k, out_q=q_):
                rd_ = FrameReader(VIDEO_T)
                for t__ in frames_list:
                    out_q.put((t__, rd_.read(t__)))
                out_q.put(None)
                rd_.close()

            th_ = threading.Thread(target=_produce, daemon=True)
            th_.start()
            xy_k = np.full((len(frames_k), YO_K, 4), np.nan, np.float32)
            cf_k = np.zeros((len(frames_k), YO_K), np.float32)
            pos_ = {int(t__): i__ for i__, t__ in enumerate(frames_k)}
            batch_, done_ = [], False
            while not done_:
                item_ = q_.get()
                if item_ is not None and item_[1] is not None:
                    batch_.append(item_)
                done_ = item_ is None
                if batch_ and (len(batch_) == YOLO_INFER_BATCH or done_):
                    res_ = detector.predict([im_ for _, im_ in batch_], imgsz=YO_INFER_SZ, conf=YO_DET_CONF, max_det=YO_K,
                                            half=YO_HALF, device=YO_DEV, verbose=False)
                    for (t__, _), r_ in zip(batch_, res_):
                        n_ = len(r_.boxes)
                        if n_:
                            xy_k[pos_[int(t__)], :n_] = r_.boxes.xyxy.cpu().numpy()
                            cf_k[pos_[int(t__)], :n_] = r_.boxes.conf.cpu().numpy()
                    batch_ = []
            th_.join()
            np.savez_compressed(YO_DET / "c{:03d}_{}.npz".format(k_, frames_k[0]), frames=frames_k, xyxy=xy_k, conf=cf_k)
            n_done += len(frames_k)
            el_ = time.time() - t0_
            left_ = sum(len(chunks_[j_]) for j_ in todo_[todo_.index(k_) + 1:])
            print("  chunk {}/{} | {:.0f} frames/s | ± {:.0f} minutes left".format(
                todo_.index(k_) + 1, len(todo_), n_done / el_, left_ / max(n_done / el_, 1e-6) / 60), flush=True)
        del detector
        gpu_free()
    for f_ in sorted(YO_DET.glob("c*.npz")):
        with np.load(f_) as zf_:
            DET_XYXY[zf_["frames"]] = zf_["xyxy"]
            DET_CONF[zf_["frames"]] = zf_["conf"]
    n_cand_ = (DET_CONF > 0).sum(1)
    print("graded frames with ≥ 1 candidate: {:.1%} | candidates per frame (median, p95): {:.0f}, {:.0f}".format(
        (n_cand_[GRADED] > 0).mean(), np.median(n_cand_[GRADED]), np.percentile(n_cand_[GRADED], 95)))

### 2b.4 New tracking: from candidates to a single ball track

The detector produces candidates per frame; the tracking picks **one** ball per frame (or none). Special handling for
this new source:

1. **Ground contact point.** The bottom centre of the box is projected onto the pitch (the homography holds on the
   ground plane, like the ATD coordinates). A ball in the air stays misplaced; this is a limitation of every 2D
   tracking.
2. **Off the pitch** (> 2 m from a line): dropped. This filters spare balls and balls beside the pitch.
3. **Static hot spots.** 1 m × 1 m cells where the detector fires repeatedly while the tactical/broadcast ball is
   clearly > 5 m away (≥ 10 s in total, ≥ 80% of its detections "surely not the ball"). Candidates in such a cell are
   dropped unless they are close to the tracker ball. Examples: the penalty spot, logos, white boots that often stand
   in the same place.
4. **Viterbi data association.** The state per frame = one of the candidates or "no ball". Candidate score =
   log(confidence / 0.25); moving between candidates costs 0.5 per metre and is forbidden above 1.6 m per frame
   (± 40 m/s); starting or ending a track costs 2. As a result, weak detections are used when they continue a track,
   and strong detections that jump on their own are rejected. The penalty is linear (not quadratic) because a fast
   pass really moves 0.5–1.5 m per frame: with a quadratic penalty with σ 0.5 m, tracks break exactly when the ball is
   passed. On the validation blocks (no labels), coverage on the strong reference rises 0.90 → 0.94, and coverage when
   the tactical and broadcast trackers disagree by > 5 m rises 0.59 → 0.78, with almost the same precision
   (0.85 → 0.83).
5. **Short tracks** (< 3 frames) are dropped.
6. **Anchors.** A YOLO track that, in at least one of its frames, agreed (< 1.5 m) with the tactical or broadcast
   ball is called *anchored*. Only anchored tracks may fill gaps (2b.6); the reason is measured in 2b.5.

These parameters are expressed in physical units and are not tuned on the GT.

In [ ]:
YO_PITCH_MARGIN, YO_HOT_CELL, YO_HOT_MIN_S, YO_HOT_SHARE = 2.0, 1.0, 10.0, 0.8
YO_MAX_STEP, YO_STEP_COST, YO_CONF_REF, YO_START_COST, YO_MIN_RUN, YO_ANCHOR_M = 1.6, 0.5, 0.25, 2.0, 3, 1.5
YO_ANCH = np.zeros(N_FRAMES, bool)
YO_BALL = np.full((N_FRAMES, 2), np.nan, np.float32)
YO_BCONF = np.zeros(N_FRAMES, np.float32)
YO_RUNLEN = np.zeros(N_FRAMES, np.int32)
DET_M = np.full((N_FRAMES, YO_K, 2), np.nan, np.float32)


def yo_associate(cm, ok, em):
    """Viterbi over per-frame candidates: cm (T, K, 2) metres, ok (T, K) valid candidates, em (T, K) emission scores, all for
    the same frame range. Returns the chosen candidate index per frame (−1 = no ball)."""
    T_, K_ = ok.shape
    cmn = np.where(ok[..., None], cm, 1e6)
    score = np.r_[np.where(ok[0], em[0], -np.inf), 0.0]
    back = np.zeros((T_, K_ + 1), np.int16)
    for t_ in range(1, T_):
        d_ = np.sqrt(((cmn[t_ - 1][:, None, :] - cmn[t_][None, :, :]) ** 2).sum(-1))
        tr_ = np.where(d_ <= YO_MAX_STEP, -YO_STEP_COST * d_, -np.inf)
        cand_ = score[:K_, None] + tr_
        bp_ = np.argmax(cand_, 0)
        best_ = cand_[bp_, np.arange(K_)]
        from_null = score[K_] - YO_START_COST
        use_null = from_null > best_
        new_real = np.where(use_null, from_null, best_) + np.where(ok[t_], em[t_], -np.inf)
        back[t_, :K_] = np.where(use_null, K_, bp_)
        end_ = score[:K_] - YO_START_COST
        j_ = int(np.argmax(end_))
        if score[K_] >= end_[j_]:
            new_null, back[t_, K_] = score[K_], K_
        else:
            new_null, back[t_, K_] = end_[j_], j_
        score = np.r_[new_real, new_null]
    path_ = np.empty(T_, np.int16)
    path_[-1] = int(np.argmax(score))
    for t_ in range(T_ - 1, 0, -1):
        path_[t_ - 1] = back[t_, path_[t_]]
    return np.where(path_ == K_, -1, path_)


if YOLO_ON:
    has_ = DET_CONF > 0
    fr_, kk_ = np.nonzero(has_)
    bc_xy = np.c_[(DET_XYXY[fr_, kk_, 0] + DET_XYXY[fr_, kk_, 2]) / 2, DET_XYXY[fr_, kk_, 3]]
    DET_M[fr_, kk_] = px_to_m(fr_, bc_xy)
    x_, y_ = DET_M[..., 0], DET_M[..., 1]
    in_pitch = has_ & (x_ > -YO_PITCH_MARGIN) & (x_ < PITCH_L + YO_PITCH_MARGIN) & (y_ > -YO_PITCH_MARGIN) & (y_ < PITCH_W + YO_PITCH_MARGIN)
    ref_ = np.where(finite[:, None], ball_raw, ball_bc)                    # tracker ball (tactical, then broadcast)
    d_ref = np.hypot(x_ - ref_[:, None, 0], y_ - ref_[:, None, 1])
    surely_not = in_pitch & np.isfinite(d_ref) & (d_ref > 5.0)
    nx_, ny_ = int(np.ceil(PITCH_L + 2 * YO_PITCH_MARGIN)), int(np.ceil(PITCH_W + 2 * YO_PITCH_MARGIN))
    cx_ = np.clip(np.floor(np.nan_to_num(x_ + YO_PITCH_MARGIN, nan=-1) / YO_HOT_CELL).astype(int), 0, nx_ - 1)
    cy_ = np.clip(np.floor(np.nan_to_num(y_ + YO_PITCH_MARGIN, nan=-1) / YO_HOT_CELL).astype(int), 0, ny_ - 1)
    all_cnt = np.zeros((nx_, ny_))
    bad_cnt = np.zeros((nx_, ny_))
    np.add.at(all_cnt, (cx_[in_pitch], cy_[in_pitch]), 1)
    np.add.at(bad_cnt, (cx_[surely_not], cy_[surely_not]), 1)
    HOT = (bad_cnt >= YO_HOT_MIN_S * FPS) & (bad_cnt >= YO_HOT_SHARE * np.maximum(all_cnt, 1))
    in_hot = in_pitch & HOT[cx_, cy_] & ~(np.isfinite(d_ref) & (d_ref < 2.0))
    cand_ok = in_pitch & ~in_hot
    YO_EM = np.log(np.maximum(DET_CONF, 1e-6) / YO_CONF_REF)
    sel_ = np.full(N_FRAMES, -1, np.int16)
    with timer("Viterbi association"):
        for _h, (_a, _b) in HALVES.items():
            sl_ = slice(_a, _b + 1)
            sel_[sl_] = yo_associate(DET_M[sl_], cand_ok[sl_], YO_EM[sl_])
    chosen_ = sel_ >= 0
    idx_ = np.flatnonzero(chosen_)
    YO_BALL[idx_] = DET_M[idx_, sel_[idx_]]
    YO_BCONF[idx_] = DET_CONF[idx_, sel_[idx_]]
    for _a, _b in zip(*runs(chosen_)):
        YO_RUNLEN[_a:_b] = _b - _a
    short_ = chosen_ & (YO_RUNLEN < YO_MIN_RUN)
    YO_BALL[short_] = np.nan
    YO_BCONF[short_] = 0
    chosen_ &= ~short_
    near_trk = chosen_ & ((np.nan_to_num(np.hypot(*(YO_BALL - ball_raw).T), nan=99.0) < YO_ANCHOR_M)
                          | (np.nan_to_num(np.hypot(*(YO_BALL - ball_bc).T), nan=99.0) < YO_ANCHOR_M))
    cs_near = np.r_[0, np.cumsum(near_trk)]
    for _a, _b in zip(*runs(chosen_)):
        if cs_near[_b] - cs_near[_a] > 0:
            YO_ANCH[_a:_b] = True
    stage_ = pd.DataFrame([
        {"tahap": "graded frames with ≥ 1 candidate", "frame": int((has_.any(1) & GRADED).sum())},
        {"tahap": "… candidate inside the pitch (± 2 m)", "frame": int((in_pitch.any(1) & GRADED).sum())},
        {"tahap": "… not a hot spot", "frame": int((cand_ok.any(1) & GRADED).sum())},
        {"tahap": "ball chosen by Viterbi (track ≥ 3 frames)", "frame": int((chosen_ & GRADED).sum())},
        {"tahap": "… from tracks anchored to the tracker ball", "frame": int((YO_ANCH & GRADED).sum())}])
    stage_["% of graded frames"] = (100 * stage_.frame / N_GRADED).round(1)
    print(stage_.to_string(index=False))
    rs_, re_ = runs(chosen_)
    print("hot spots: {} cells | candidates dropped as hot spots {:.2%} | tracks: {} · median duration {:.1f} s".format(
        int(HOT.sum()), in_hot.sum() / max(has_.sum(), 1), len(rs_), np.median(re_ - rs_) / FPS if len(rs_) else 0.0))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    ext_ = (-YO_PITCH_MARGIN, nx_ * YO_HOT_CELL - YO_PITCH_MARGIN, ny_ * YO_HOT_CELL - YO_PITCH_MARGIN, -YO_PITCH_MARGIN)
    for ax, cnt_, ttl_ in ((axes[0], all_cnt, "All candidates inside the pitch"),
                           (axes[1], bad_cnt, "Candidates that are surely not the ball (> 5 m from the tracker ball)")):
        ax.imshow(np.log1p(cnt_.T), cmap=BLUES, extent=ext_, aspect="equal")
        ax.plot([0, PITCH_L, PITCH_L, 0, 0], [0, 0, PITCH_W, PITCH_W, 0], color=COL["muted"], linewidth=1)
        ax.axvline(PITCH_L / 2, color=COL["muted"], linewidth=0.8)
        hx_, hy_ = np.nonzero(HOT)
        ax.scatter((hx_ + 0.5) * YO_HOT_CELL - YO_PITCH_MARGIN, (hy_ + 0.5) * YO_HOT_CELL - YO_PITCH_MARGIN, s=80,
                   facecolors="none", edgecolors=COL["A"], linewidths=1.5)
        ax.set_title(ttl_, pad=12)
        ax.grid(False)
        ax.set_xlabel("x (m)")
    note(axes[0], "log scale of frame counts · orange circle = filtered hot spot ({} cells)".format(int(HOT.sum())))
    plt.tight_layout()
    plt.show()

### 2b.5 Label-free validation

There are no hand labels, so correctness is measured by **agreement between independent sources**, only on the
validation blocks (which were not used to train the detector):

1. **Strong reference:** frames where the tactical and broadcast balls agree within 1 m. There we measure *coverage*
   (YOLO finds the ball) and *error* (distance from YOLO to the tactical ball).
2. **Confidence threshold:** YOLO precision (distance < 1.5 m from the reference) is computed per confidence
   threshold; we use the lowest threshold whose precision is within 0.01 of the highest precision. Precision cannot
   approach 100% because the reference itself is noisy: the tactical ball is off by ± 0.4 m (p90 ± 1 m) and sometimes
   floats. If the curve is flat, confidence does not separate right from wrong detections, and every track ball is
   used.
3. **Gap-filling order:** the source with the smaller median error against the tactical ball (YOLO or broadcast) fills
   first.
4. **Three-source vote:** when the tactical ball and YOLO disagree by > 5 m and the broadcast is present, the closest
   pair of sources wins (broadcast–YOLO or broadcast–tactical). If broadcast–YOLO wins in ≥ 80% of the cases, the
   tactical ball may also be corrected by YOLO when the broadcast is absent.
5. **May unanchored tracks fill gaps?** In tactical gaps that still have a broadcast ball, the YOLO ball is
   compared with the broadcast, separately for anchored and unanchored tracks.
6. **What are the YOLO outliers?** On the strong reference, detections > 1.5 m from the tactical ball are compared
   with good detections: does the YOLO ball appear higher in the image? A ball in the air looks right in the image, but
   the homography projects it onto the ground, so its position shifts away from the camera. Examples are shown in 2b.7.

Limitation: reference frames are the "easy" frames (both trackers see the ball), so the error in real gaps can be
larger.

In [ ]:
YO_PREC_SLACK, YO_TRUST_M, YO_DISAGREE_M, YO_VOTE_MIN = 0.01, 1.5, 5.0, 0.80
YO_TRUST_CONF = 1.0
YO_FILL_ORDER = ["broadcast"]
YO_OVERRIDE_SOLO = False
if YOLO_ON:
    valm_ = GRADED & yo_val_block(np.arange(N_FRAMES))
    ref_ok = valm_ & finite & np.isfinite(ball_bc[:, 0])
    ref_ok[ref_ok] = np.hypot(*(ball_raw[ref_ok] - ball_bc[ref_ok]).T) < YO_AGREE_M
    y_ok = np.isfinite(YO_BALL[:, 0])
    err_ = np.hypot(*(YO_BALL - ball_raw).T)
    cov_ = y_ok[ref_ok].mean()
    e_ref = err_[ref_ok & y_ok]
    print("validation blocks, strong reference ({} frames): YOLO coverage {:.1%} | median error {:.2f} m · p90 {:.2f} m · "
          "< 1,5 m {:.1%}".format(int(ref_ok.sum()), cov_, np.median(e_ref), np.percentile(e_ref, 90),
                                  (e_ref < YO_TRUST_M).mean()))
    grid_ = np.round(np.arange(0.05, 0.95, 0.05), 2)
    prec_rows = []
    for c_ in grid_:
        m_ = ref_ok & y_ok & (YO_BCONF >= c_)
        prec_rows.append({"keyakinan ≥": c_, "frame": int(m_.sum()), "presisi": (err_[m_] < YO_TRUST_M).mean() if m_.any() else np.nan,
                          "cakupan": m_.sum() / max(ref_ok.sum(), 1)})
    prec_ = pd.DataFrame(prec_rows)
    big_ = prec_[prec_.frame >= 200]
    if len(big_) and big_.presisi.notna().any():
        YO_TRUST_CONF = float(big_[big_.presisi >= big_.presisi.max() - YO_PREC_SLACK]["keyakinan ≥"].min())
    p_used = float(prec_.loc[prec_["keyakinan ≥"] == YO_TRUST_CONF, "presisi"].iloc[0]) if len(big_) else float("nan")
    check("YOLO on the strong reference: coverage ≥ 0.80 and precision ≥ 0.80", cov_ >= 0.80 and p_used >= 0.80,
          "coverage {:.3f} · precision {:.3f} · confidence threshold {:.2f}".format(cov_, p_used, YO_TRUST_CONF), critical=False)
    YO_TRUST = y_ok & (YO_BCONF >= YO_TRUST_CONF)
    print(prec_.round(3).to_string(index=False))

    three_ = valm_ & finite & np.isfinite(ball_bc[:, 0]) & YO_TRUST
    e_y = np.median(err_[three_]) if three_.any() else np.inf
    e_b = np.median(np.hypot(*(ball_bc - ball_raw).T)[three_]) if three_.any() else np.inf
    YO_FILL_ORDER = ["yolo", "broadcast"] if e_y <= e_b else ["broadcast", "yolo"]
    print("median error against the tactical ball (frames with all three sources, validation blocks): YOLO {:.2f} m · broadcast {:.2f} m "
          "→ gap-filling order: {}".format(e_y, e_b, " → ".join(YO_FILL_ORDER)))

    dis_ = GRADED & finite & YO_TRUST & (err_ > YO_DISAGREE_M) & np.isfinite(ball_bc[:, 0])
    d_bt = np.hypot(*(ball_bc - ball_raw).T)
    d_by = np.hypot(*(ball_bc - YO_BALL).T)
    by_wins = (d_by < d_bt) & (d_by < err_) & (d_by < YO_DISAGREE_M)
    bt_wins = (d_bt < d_by) & (d_bt < err_) & (d_bt < YO_DISAGREE_M)
    vote_ = pd.Series({"broadcast–YOLO closest pair": by_wins[dis_].mean(),
                       "broadcast–tactical closest pair": bt_wins[dis_].mean(),
                       "ketiganya berjauhan (> 5 m)": (~by_wins & ~bt_wins)[dis_].mean()})
    share_y = vote_.iloc[0] / max(vote_.iloc[0] + vote_.iloc[1], 1e-9)
    YO_OVERRIDE_SOLO = bool(share_y >= YO_VOTE_MIN and dis_.sum() >= 250)
    print("tactical vs YOLO disagree by > 5 m while broadcast is present: {} frames (whole match)".format(int(dis_.sum())))
    print(vote_.round(3).to_string())
    print("broadcast sides with YOLO in {:.0%} of the cases where it sides with one → correct tactical without broadcast: {}".format(
        share_y, "ya" if YO_OVERRIDE_SOLO else "no (only when broadcast–YOLO is the closest pair)"))

    gap_b = GRADED & ~finite & np.isfinite(ball_bc[:, 0]) & y_ok
    agree_b = np.hypot(*(YO_BALL - ball_bc).T) < 2.0
    a_in, a_out = gap_b & YO_ANCH, gap_b & ~YO_ANCH
    print("tactical gaps with a broadcast ball: YOLO agrees with broadcast (< 2 m) | anchored tracks "
          "{:.1%} (n {}) · unanchored {:.1%} (n {})".format(agree_b[a_in].mean() if a_in.any() else np.nan, int(a_in.sum()),
                                                              agree_b[a_out].mean() if a_out.any() else np.nan, int(a_out.sum())))
    check("anchored YOLO fills are more reliable than unanchored ones", (not a_out.any()) or
          agree_b[a_in].mean() > agree_b[a_out].mean(), critical=False)
    out_m = ref_ok & y_ok & (err_ > YO_TRUST_M)
    good_m = ref_ok & y_ok & (err_ <= YO_TRUST_M)

    def _up_share(m_):
        t_ = np.flatnonzero(m_)
        if not len(t_):
            return np.nan
        dv_ = m_to_px(t_, YO_BALL[t_])[:, 1] - m_to_px(t_, ball_raw[t_])[:, 1]
        return float((dv_ < -5).mean())

    print("YOLO outliers (> 1.5 m from the strong reference): {:.1%} of frames | appear higher in the image than the tactical ball "
          "(> 5 px): outliers {:.0%} · good detections {:.0%} → most outliers are balls in the air".format(
              out_m.sum() / max((ref_ok & y_ok).sum(), 1), _up_share(out_m), _up_share(good_m)))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    ax = axes[0]
    ax.hist(np.clip(e_ref, 0, 3), bins=np.linspace(0, 3, 31), color=COL["H"], edgecolor=COL["surface"])
    ax.axvline(YO_TRUST_M, color=COL["A"], linestyle="--", linewidth=1)
    ax.set_xlabel("YOLO distance to the tactical ball (m, clipped at 3)")
    ax.set_ylabel("frames")
    ax.set_title("YOLO error on the strong reference", pad=18)
    note(ax, "validation blocks · median {:.2f} m".format(np.median(e_ref)))
    ax = axes[1]
    ax.plot(prec_["keyakinan ≥"], prec_.presisi, color=COL["H"], linewidth=2, marker="o", markersize=5, label="precision")
    ax.plot(prec_["keyakinan ≥"], prec_.cakupan, color=COL["A"], linewidth=2, marker="o", markersize=5, label="coverage")
    ax.axvline(YO_TRUST_CONF, color=COL["aqua"], linestyle="--", linewidth=1.5)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("YOLO confidence threshold")
    ax.set_title("Choosing a threshold without labels", pad=18)
    note(ax, "used: ≥ {:.2f}".format(YO_TRUST_CONF))
    ax.legend(loc="lower left")
    ax = axes[2]
    ax.barh(range(3), 100 * vote_.to_numpy(), color=[COL["aqua"], COL["yellow"], COL["D"]], height=0.6)
    for k_, v_ in enumerate(vote_.to_numpy()):
        ax.text(100 * v_ + 1, k_, "{:.0f}%".format(100 * v_), va="center", fontsize=9)
    ax.set_yticks(range(3), vote_.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 105)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("% of frames")
    ax.set_title("Tactical vs YOLO disagree: whom does broadcast side with?", pad=18)
    plt.tight_layout()
    plt.show()

### 2b.6 Merging for the possession model

Ball source codes: 1 = tactical, 2 = interpolation, 3 = broadcast, **4 = YOLO**, **5 = tactical corrected by YOLO**.

1. **Correct floating tactical balls:** the tactical ball is replaced by the YOLO ball when they disagree by > 5 m and
   broadcast–YOLO is the closest pair (both < 5 m), or when the broadcast is absent and the 2b.5 vote allows it. Only
   runs ≥ 0.4 s are corrected, to avoid flicker.
2. **Fill gaps** in the order from 2b.5; YOLO balls only from anchored tracks.
3. **Bridge the gap edges** for every fill, against the neighbouring tactical ball; broadcast fills with an edge
   jump > 5 m are dropped as in the tracking-ball pipeline.
4. **Interpolate** ≤ 1 s when the required speed is ≤ 40 m/s (as in the tracking-ball pipeline).

In [ ]:
def bridge_fill(xy, src, code, max_jump=None):
    """tracking-ball: remove the offset at the edges of every fill run `code` relative to the neighbouring tactical ball"""
    xy, src = xy.copy(), src.copy()
    rows_ = []
    for s_, e_ in zip(*runs(src == code)):
        n_ = e_ - s_
        d_left = xy[s_ - 1] - xy[s_] if s_ > 0 and src[s_ - 1] in (1, 5) else None
        d_right = xy[e_] - xy[e_ - 1] if e_ < N_FRAMES and src[e_] in (1, 5) else None
        jump_ = max([float(np.hypot(*d)) for d in (d_left, d_right) if d is not None], default=0.0)
        if max_jump is not None and jump_ > max_jump:
            xy[s_:e_] = np.nan
            src[s_:e_] = 0
            act_ = "dibuang"
        else:
            i_ = np.arange(n_, dtype=np.float32)[:, None]
            if d_left is not None and d_right is not None and n_ <= BRIDGE_LINEAR_S * FPS:
                corr = d_left + (d_right - d_left) * (i_ + 1) / (n_ + 1)
                act_ = "linier"
            else:
                corr = np.zeros((n_, 2), np.float32)
                if d_left is not None:
                    corr += d_left * np.exp(-i_ / (BRIDGE_TAU_S * FPS))
                if d_right is not None:
                    corr += d_right * np.exp(-(n_ - 1 - i_) / (BRIDGE_TAU_S * FPS))
                act_ = "faded" if (d_left is not None or d_right is not None) else "no edge"
            xy[s_:e_] = xy[s_:e_] + corr
        rows_.append({"sumber": code, "panjang_s": n_ / FPS, "loncatan_m": jump_, "aksi": act_})
    return xy, src, rows_


SRC_NAMES = {1: "tactical", 5: "tactical corrected by YOLO", 4: "YOLO", 3: "broadcast", 2: "interpolation", 0: "missing"}
if YOLO_ON:
    ball_poss = np.where(finite[:, None], ball_raw, np.nan).astype(np.float32)
    src_poss = np.where(finite, 1, 0).astype(np.int8)
    has_b = np.isfinite(ball_bc[:, 0])
    over_ = GRADED & finite & YO_TRUST & (err_ > YO_DISAGREE_M) & ((has_b & by_wins) | (~has_b & YO_OVERRIDE_SOLO))
    keep_ = np.zeros(N_FRAMES, bool)
    for _a, _b in zip(*runs(over_)):
        if _b - _a >= int(0.4 * FPS):
            keep_[_a:_b] = True
    ball_poss[keep_] = YO_BALL[keep_]
    src_poss[keep_] = 5
    for name_ in YO_FILL_ORDER:
        if name_ == "yolo":
            m_ = GRADED & (src_poss == 0) & YO_TRUST & YO_ANCH
            ball_poss[m_], src_poss[m_] = YO_BALL[m_], 4
        else:
            m_ = GRADED & (src_poss == 0) & has_b
            ball_poss[m_], src_poss[m_] = ball_bc[m_], 3
    bridge_log = []
    ball_poss, src_poss, r_ = bridge_fill(ball_poss, src_poss, 4)
    bridge_log += r_
    ball_poss, src_poss, r_ = bridge_fill(ball_poss, src_poss, 3, max_jump=BRIDGE_MAX_JUMP)
    bridge_log += r_
    fin_p = src_poss > 0
    for _h, (_a, _b) in HALVES.items():
        for s_, e_ in zip(*runs(~fin_p[_a:_b + 1])):
            s_, e_ = _a + s_, _a + e_
            if s_ == _a or e_ > _b or e_ - s_ > MAX_GAP:
                continue
            dist_ = float(np.hypot(*(ball_poss[e_] - ball_poss[s_ - 1])))
            if dist_ / ((e_ - s_ + 1) / FPS) <= MAX_GAP_SPEED:
                tt_ = (np.arange(1, e_ - s_ + 1) / (e_ - s_ + 1))[:, None]
                ball_poss[s_:e_] = ball_poss[s_ - 1] + (ball_poss[e_] - ball_poss[s_ - 1]) * tt_
                src_poss[s_:e_] = 2
    bridge_log = pd.DataFrame(bridge_log)
    print("tactical corrected by YOLO: {} frames in {} runs | bridges: {}".format(
        int(keep_.sum()), len(runs(keep_)[0]),
        bridge_log.groupby(["sumber", "aksi"]).size().rename(index={3: "broadcast", 4: "YOLO"}).to_dict() if len(bridge_log) else {}))
else:
    ball_poss = np.where((ball_src > 0)[:, None], ball_filled, np.nan).astype(np.float32)
    src_poss = ball_src.copy()
    keep_ = np.zeros(N_FRAMES, bool)
    print("YOLO_STAGE = \"off\": the possession model uses the tracking ball")

avail_ = pd.DataFrame({"trk": pd.Series(ball_src[GRADED]).map(SRC_NAMES).value_counts(normalize=True),
                       "yoloball": pd.Series(src_poss[GRADED]).map(SRC_NAMES).value_counts(normalize=True)}).fillna(0)
avail_ = avail_.reindex([n_ for n_ in SRC_NAMES.values() if n_ in avail_.index])
print((100 * avail_).round(1).to_string())
check("YOLO ball available ≥ tracking ball (graded frames)", (src_poss[GRADED] > 0).mean() >= (ball_src[GRADED] > 0).mean() - 1e-3,
      "{:.1%} → {:.1%}".format((ball_src[GRADED] > 0).mean(), (src_poss[GRADED] > 0).mean()), critical=False)
z["ball_poss"], z["src_poss"] = ball_poss, src_poss

fig, ax = plt.subplots(figsize=(12, 2.6))
src_cols = {"tactical": COL["H"], "tactical corrected by YOLO": COL["violet"], "YOLO": COL["aqua"], "broadcast": COL["yellow"],
            "interpolation": "#d9c58f", "missing": COL["D"]}
for row_, ver_ in enumerate(("trk", "yoloball")):
    left_ = 0.0
    for name_ in avail_.index:
        share_ = 100 * float(avail_.loc[name_, ver_])
        ax.barh(row_, share_, left=left_, color=src_cols[name_], height=0.55, edgecolor=COL["surface"], linewidth=2,
                label=name_ if row_ == 1 else None)
        if share_ >= 4:
            ax.text(left_ + share_ / 2, row_, "{:.0f}%".format(share_), ha="center", va="center", fontsize=9,
                    color="white" if name_ != "missing" else COL["text"])
        left_ += share_
ax.set_yticks([0, 1], ["trk", "YOLO-ball (possession model)"])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of graded frames")
ax.grid(axis="y", visible=False)
ax.set_title("Ball position source: tracking-ball vs YOLO-ball", pad=18)
ax.legend(ncol=6, loc="upper left", bbox_to_anchor=(0, -0.42), fontsize=8)
plt.tight_layout()
plt.show()

### 2b.7 Transparency: what the detector sees

Symbols in every image: **yellow cross** = tactical ball (ATD), **light-blue circle** = broadcast ball, **green box** =
selected YOLO detection (number = confidence), **magenta dot** = merged ball that goes into the possession model.

1. Full frames: one example for each case (agreement, tactical corrected, gap filled by YOLO).
2. Mosaics of enlarged crops for each case, picked at random from the whole match (not cherry-picked).
3. Candidates dropped as hot spots.

In [ ]:
def draw_overlay(img, t_, scale=1.0, label=True):
    """draw markers for every ball source on frame t_ (in place); scale = image size factor"""
    def _pt(xy):
        if not np.isfinite(xy).all() or not H_OK[t_]:
            return None
        u_, v_ = m_to_px([t_], np.asarray(xy, float)[None])[0]
        return (int(u_ * scale), int(v_ * scale)) if -50 < u_ < VID_W + 50 and -50 < v_ < VID_H + 50 else None
    p_ = _pt(ball_raw[t_])
    if p_:
        cv2.drawMarker(img, p_, (0, 0, 0), cv2.MARKER_CROSS, 26, 5)
        cv2.drawMarker(img, p_, C_T, cv2.MARKER_CROSS, 26, 2)
    p_ = _pt(ball_bc[t_])
    if p_:
        cv2.circle(img, p_, 15, (0, 0, 0), 4)
        cv2.circle(img, p_, 15, C_B, 2)
    if YOLO_ON and np.isfinite(YO_BALL[t_, 0]):
        d_ = np.hypot(*(DET_M[t_] - YO_BALL[t_]).T)
        d_ = np.where(np.isfinite(d_), d_, np.inf)          # empty candidate slots hold NaN
        j_ = int(np.argmin(d_))
        box_ = DET_XYXY[t_, j_] * scale
        if np.isfinite(d_[j_]) and np.isfinite(box_).all():
            x1_, y1_, x2_, y2_ = (int(round(float(v_))) for v_ in box_)
            cv2.rectangle(img, (x1_ - 4, y1_ - 4), (x2_ + 4, y2_ + 4), C_Y, 2)
            if label:
                cv2.putText(img, "{:.2f}".format(YO_BCONF[t_]), (x2_ + 6, y1_), cv2.FONT_HERSHEY_SIMPLEX, 0.5, C_Y, 1,
                            cv2.LINE_AA)
    p_ = _pt(ball_poss[t_])
    if p_:
        cv2.circle(img, p_, 5, C_F, -1)
    return img


def vtime(t_):
    s_ = t_ / FPS
    return "{:02d}:{:05.2f}".format(int(s_ // 60), s_ % 60)


if YOLO_ON:
    rng_ = np.random.default_rng(1)
    agree_ = GRADED & finite & YO_TRUST & (err_ < 1.0)
    fillY_ = GRADED & (src_poss == 4)
    cases_ = [("Tactical, broadcast and YOLO balls agree", agree_),
              ("Floating tactical ball, corrected by YOLO", keep_),
              ("Tactical ball missing, gap filled by YOLO", fillY_),
              ("YOLO outlier while both trackers agree (mostly balls in the air)", out_m)]
    reader = FrameReader(VIDEO_T)
    for k_case, (ttl_, m_) in enumerate(cases_):
        if not m_.any():
            print("{}: no frames".format(ttl_))
            continue
        t_ = int(rng_.choice(np.flatnonzero(m_)))
        img_ = reader.read(t_)
        if img_ is None:
            continue
        img_ = draw_overlay(img_, t_)
        cv2.putText(img_, "{} | frame {} | video {}".format(ttl_, t_, vtime(t_)), (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255), 2, cv2.LINE_AA)
        out_ = YO_REP / "frame_kasus{}.jpg".format(k_case)
        cv2.imwrite(str(out_), cv2.resize(img_, (1280, 720)), [cv2.IMWRITE_JPEG_QUALITY, 90])
        show_image(out_, "{} (frame {}, video {})".format(ttl_, t_, vtime(t_)), width=13)
    for k_case, (ttl_, m_) in enumerate(cases_):
        pool_ = np.flatnonzero(m_)
        if not len(pool_):
            continue
        tiles_ = []
        for t_ in np.sort(rng_.choice(pool_, min(18, len(pool_)), replace=False)):
            img_ = reader.read(t_)
            if img_ is None:
                continue
            draw_overlay(img_, t_, label=False)
            pts_ = [m_to_px([t_], xy_[[t_]])[0] for xy_ in (ball_poss, YO_BALL) if np.isfinite(xy_[t_, 0])]
            u_, v_ = np.mean(pts_, 0) if pts_ else (VID_W / 2, VID_H / 2)
            half_ = int(np.clip(np.ptp(np.array(pts_), 0).max() / 2 + 40, 64, 160)) if len(pts_) > 1 else 64
            tile_ = cv2.resize(crop_zoom(img_, u_, v_, half=half_, zoom=1), (256, 256))
            cv2.putText(tile_, "{} c{:.2f}".format(vtime(t_), YO_BCONF[t_]), (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                        (255, 255, 255), 1, cv2.LINE_AA)
            tiles_.append(tile_)
        out_ = YO_REP / "mosaik_kasus{}.jpg".format(k_case)
        if mosaic(tiles_, 6, out_):
            show_image(out_, ttl_ + " (18 random frames, enlarged crops)")
    hot_idx = np.argwhere(in_hot & GRADED[:, None])
    if len(hot_idx):
        tiles_ = []
        for t_, k_ in hot_idx[np.sort(rng_.choice(len(hot_idx), min(12, len(hot_idx)), replace=False))]:
            img_ = reader.read(t_)
            if img_ is None:
                continue
            x1_, y1_, x2_, y2_ = DET_XYXY[t_, k_]
            tile_ = crop_zoom(img_, (x1_ + x2_) / 2, (y1_ + y2_) / 2, half=48, zoom=3)
            cv2.rectangle(tile_, (int((x1_ - (x1_ + x2_) / 2 + 48) * 3), int((y1_ - (y1_ + y2_) / 2 + 48) * 3)),
                          (int((x2_ - (x1_ + x2_) / 2 + 48) * 3), int((y2_ - (y1_ + y2_) / 2 + 48) * 3)), (40, 40, 230), 2)
            tiles_.append(tile_)
        out_ = YO_REP / "mosaic_titik_panas.jpg"
        if mosaic(tiles_, 6, out_):
            show_image(out_, "Candidates dropped as static hot spots (red box)", width=12)
    reader.close()

## 3. Smart Tagging

**What this step does.** Converts the automatic tactical annotations (SportsCode XML) to video frames
(`seconds × 25`). Only the **SET PIECES** tags are used, because only their timing is meaningful (the tag window length
is uniform per type):

- start of a set-piece tag → a cue that play has stopped;
- team of a corner kick, goal kick or free kick → the **restart taker team**;
- kick-off after a goal → a certainly dead range.

In [ ]:
def load_smart_tagging(meta_):
    rows = []
    for ins in ET.parse(require_asset("smart_tagging")).getroot().iter("instance"):
        row = {"id": int(ins.findtext("ID")), "code": (ins.findtext("code") or "").strip().upper(),
               "start_s": float(ins.findtext("start")), "end_s": float(ins.findtext("end"))}
        for lab in ins.findall("label"):
            row[(lab.findtext("group") or "").strip()] = (lab.findtext("text") or "").strip()
        rows.append(row)
    tg = pd.DataFrame(rows)
    tg["start_f"] = (tg.start_s * FPS).round().astype(int)
    tg["end_f"] = (tg.end_s * FPS).round().astype(int)
    names = {(meta_["home_name"] or "").lower(): "H", (meta_["away_name"] or "").lower(): "A"}
    team = tg["Team"].fillna("N/A") if "Team" in tg else pd.Series("N/A", index=tg.index)
    tg["side"] = team.str.lower().map(lambda v: names.get(v, "NA" if v in ("n/a", "") else "?"))
    return tg.sort_values(["start_f", "id"]).reset_index(drop=True)


tags = load_smart_tagging(meta).fillna("")        # empty labels as "" (not NaN)
check("every Team value recognised", not (tags.side == "?").any())
print("tags: {} instances | SET PIECES: {}".format(len(tags), int((tags.code == "SET PIECES").sum())))

## 4. Dead-ball detector (`D`)

**What this step does.** Decides, for every graded frame, whether the ball is dead.

A dead ball is rarely visible directly (the tracker often loses the ball exactly as it leaves the pitch), but the
**behaviour of the 22 players** changes clearly: they slow down, stop passing, and walk to their restart positions. This
detector reads that collective behaviour, not only the ball.

**Score per frame** = weighted sum of standardised features (higher = more likely dead):

| Feature | Weight | Meaning |
|---|---|---|
| `calm` | 1.0 | mean player speed (± 3 s) relative to the local tempo (± 2 minutes): calm compared with the surrounding phase |
| `no_pass` | 1.0 | no fast ball (> 8 m/s) inside the pitch within ± 2 s |
| `gap_long` | 0.5 | ball missing for ≥ 3 s |
| `near_line` | 0.5 | ball close to or beyond a line |
| `bc_offpitch` | 0.5 | the broadcast is showing a replay/close-up |
| `ball_slow` | 0.4 | slow ball (median over ± 2 s) |
| `in_sp` | 0.3 | inside a set-piece tag window |

**Anchors** (± 3) for near-certain evidence: ball ≥ 1 m beyond a line for ≥ 1 s → dead; ball missing for ≥ 10 s →
dead; a challenge (players of both teams ≤ 2 m from a moving ball) → live.

A **2-state Viterbi** (switching cost 20; segments < 2 s dropped) turns the score into segments. The score bias is found
by bisection so that the `D` share of graded frames = **0.48**: the effective playing time of elite football is
± 50–56 of ± 95–100 minutes (Altmann et al., 2023), so the dead-ball share comes from domain knowledge, not from the GT.

**Stoppage-cue requirement.** A `D` segment without a single concrete cue (ball gap ≥ 3 s, ball out for ≥ 1 s,
set-piece tag start, replay ≥ 3 s, ball still for ≥ 1 s, or a forced dead frame) is dropped: slow play without a
stoppage (e.g. possession at the back) must not become a spurious `D` that splits a possession.

In [ ]:
WEIGHTS = {"calm": 1.0, "no_pass": 1.0, "gap_long": 0.5, "near_line": 0.5, "bc_offpitch": 0.5, "ball_slow": 0.4,
           "in_sp": 0.3}
SWITCH_COST_D, MIN_SEG, D_SHARE_PRIOR = 20.0, 50, 0.48
GAP_LONG_S, PASS_V, ANCHOR = 3.0, 8.0, 3.0
CONTEST_R, CONTEST_BV = 2.0, 1.0
OOB_ANCHOR_M, OOB_ANCHOR_MIN, GAP_ANCHOR_S = 1.0, 25, 10.0
CUE_STILL_SD, CUE_STILL_MIN, CUE_OFFPITCH_S = 0.3, 25, 3.0

P, HOME = z["players"], z["slot_is_home"]
obs = z["ball_src"] > 0
bx, by = z["ball_filled"][:, 0], z["ball_filled"][:, 1]

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    ps = np.nanmean(P[..., 2], axis=1)
sm = pd.Series(ps).rolling(151, center=True, min_periods=50).mean()
loc = sm.rolling(6001, center=True, min_periods=500).median()
calm = -(np.log(sm + 0.1) - np.log(loc + 0.1)).to_numpy()

gap_len = np.zeros(N_FRAMES)
for h, (a, b) in HALVES.items():
    s_, e_ = runs(~obs[a:b + 1])
    for si, ei in zip(s_, e_):
        gap_len[a + si:a + ei] = (ei - si) / FPS
gap_long = np.where(gap_len >= GAP_LONG_S, np.log1p(gap_len), 0.0)

line = np.where(obs, np.minimum.reduce([bx, PITCH_L - bx, by, PITCH_W - by]), np.nan)
mn = pd.Series(line).rolling(101, center=True, min_periods=10).min().to_numpy()
near_line = -np.clip(np.nan_to_num(mn, nan=20.0), -5.0, 20.0)

hh = 12
ball_v = np.full(N_FRAMES, np.nan)
ball_v[hh:-hh] = np.hypot(bx[2 * hh:] - bx[:-2 * hh], by[2 * hh:] - by[:-2 * hh]) / (2 * hh / FPS)
ball_slow = -np.log1p(pd.Series(ball_v).rolling(101, center=True, min_periods=20).median().to_numpy())

inside = np.nan_to_num(z["ball_oob_m"], nan=1.0) == 0
fast = obs & inside & (np.nan_to_num(ball_v) > PASS_V)
no_pass = -pd.Series(fast.astype(float)).rolling(101, center=True, min_periods=1).mean().to_numpy()
bc_offpitch = (z["bc_view"] == -1).astype(float)

sp = tags[tags.code == "SET PIECES"]
in_sp = np.zeros(N_FRAMES)
for r in sp.itertuples():
    in_sp[max(r.start_f, 0):min(r.end_f, N_FRAMES)] = 1.0

FEATS_D = list(WEIGHTS)
Fd = np.stack([dict(calm=calm, no_pass=no_pass, gap_long=gap_long, near_line=near_line, bc_offpitch=bc_offpitch,
                    ball_slow=ball_slow, in_sp=in_sp)[k] for k in FEATS_D], axis=1)
mu_d = np.nanmean(Fd[GRADED], axis=0)
sd_d = np.nanstd(Fd[GRADED], axis=0) + 1e-9
Zf = (Fd - mu_d) / sd_d
Zf[~np.isfinite(Zf)] = 0.0
feat_score = Zf @ np.array([WEIGHTS[k] for k in FEATS_D])

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    dH = np.where(obs, np.nanmin(np.hypot(P[:, HOME, 0] - bx[:, None], P[:, HOME, 1] - by[:, None]), axis=1), np.nan)
    dA = np.where(obs, np.nanmin(np.hypot(P[:, ~HOME, 0] - bx[:, None], P[:, ~HOME, 1] - by[:, None]), axis=1), np.nan)
contest = obs & (np.nan_to_num(dH, nan=99) <= CONTEST_R) & (np.nan_to_num(dA, nan=99) <= CONTEST_R) \
    & (np.nan_to_num(ball_v) > CONTEST_BV)
oob1 = np.zeros(N_FRAMES, dtype=bool)
for a, b in zip(*runs(np.nan_to_num(z["ball_oob_m"]) > OOB_ANCHOR_M)):
    if b - a >= OOB_ANCHOR_MIN:
        oob1[a:b] = True
gap_anchor = GRADED & (gap_len >= GAP_ANCHOR_S)
anchor_score = ANCHOR * (oob1.astype(float) + gap_anchor.astype(float) - contest.astype(float))
base_score = feat_score + anchor_score

xo_ = pd.Series(np.where(obs, bx, np.nan))
yo_ = pd.Series(np.where(obs, by, np.nan))
ball_sd = np.hypot(xo_.rolling(25, center=True, min_periods=15).std(),
                   yo_.rolling(25, center=True, min_periods=15).std()).to_numpy()
still_cs = np.concatenate(([0], np.cumsum(np.nan_to_num(ball_sd, nan=99.0) < CUE_STILL_SD)))
sp_start = np.zeros(N_FRAMES, dtype=bool)
for r in sp.itertuples():
    sp_start[min(max(int(r.start_f), 0), N_FRAMES - 1)] = True
offpitch_long = np.zeros(N_FRAMES, dtype=bool)
for a, b in zip(*runs(z["bc_view"] == -1)):
    if b - a >= CUE_OFFPITCH_S * FPS:
        offpitch_long[a:b] = True
cue_cs = np.concatenate(([0], np.cumsum((gap_len >= GAP_LONG_S) | oob1 | sp_start | offpitch_long)))

### Forced dead frames & segmentation

The offset between the *Kick Off (Start)* tag and the real kick-off is measured in both halves and then applied to the
*Kick Off (After goal)* tags: from the tag start to the estimated kick-off the ball is certainly dead. These frames are
forced to `D` in the Viterbi.

In [ ]:
ko_tags = sp[sp.Type.str.lower().str.startswith("kick off")]
start_tags = ko_tags[ko_tags.Type.str.lower() == "kick off (start)"].sort_values("start_f")
KO_OFFSET = int(round(np.mean([KICKOFF[h] - int(r.start_f) for h, r in zip((1, 2), start_tags.itertuples())])))
forced = z["dead_prior"].copy()
for r in ko_tags[ko_tags.Type.str.lower() == "kick off (after goal)"].itertuples():
    forced[int(r.start_f):int(r.start_f) + KO_OFFSET] = True
forced &= GRADED
TRANS_D = np.array([[0.0, SWITCH_COST_D], [SWITCH_COST_D, 0.0]])


def has_cue(a, b):
    return bool(cue_cs[b] - cue_cs[a] > 0 or still_cs[b] - still_cs[a] >= CUE_STILL_MIN or forced[a:b].any())


def decode_dead(bias):
    s = base_score + bias
    emis = np.stack([-s / 2, s / 2], axis=1)
    emis[forced, 0], emis[forced, 1] = -1e4, 1e4
    mask = np.zeros(N_FRAMES, dtype=bool)
    for h, (a, b) in HALVES.items():
        mask[a:b + 1] = viterbi(emis[a:b + 1], TRANS_D) == 1
    for a, b in zip(*runs(mask)):
        if b - a < MIN_SEG and not forced[a:b].any():
            mask[a:b] = False
    for a, b in zip(*runs(mask)):
        if not has_cue(a, b):
            mask[a:b] = False
    return mask


def d_share(mask):
    return float(mask[GRADED].mean())


lo, hi = -6.0, 6.0
with timer("D bias bisection"):
    for _ in range(18):
        mid = (lo + hi) / 2
        if d_share(decode_dead(mid)) < D_SHARE_PRIOR:
            lo = mid
        else:
            hi = mid
BIAS_D = round((lo + hi) / 2, 4)
d_mask = decode_dead(BIAS_D)
check("D share hits the prior (± 0.01)", abs(d_share(d_mask) - D_SHARE_PRIOR) <= 0.01, "{:.3f}".format(d_share(d_mask)))
check("every forced frame = D", d_mask[forced].all())
check("D only on graded frames", not d_mask[~GRADED].any())
print("bias {:+.4f} | D segments {} | ball out ≥ 1 s labelled D {:.1f}% | contested balls labelled D {:.1f}%".format(
    BIAS_D, len(runs(d_mask)[0]), 100 * d_mask[oob1 & GRADED].mean(), 100 * d_mask[contest & GRADED].mean()))

### Snapping the `D` boundaries to physical events

The detector score is smoothed over ± 2–3 s, so segment boundaries can shift by a few seconds. Two sharp physical
events are used to tidy them, within a ± 3 s window around the boundary (half the smoothing width):

- **End of `D` → restart kick:** the first frame where the ball moves > 4 m/s for 3 frames, after being still
  (< 1.5 m/s) or invisible for ≥ 0.6 s. This also covers throw-ins (ball held, then thrown).
- **Start of `D` → ball crossing a line:** the first frame where the ball is > 0.3 m beyond a line. Stoppages for a
  whistle have no clear ball event, so their boundary is left as it is.

The ball position used is the merged ball from 2b.6 (with YOLO), because the tactical ball is often lost during
stoppages.

**Validation without the BAR-ATH GT**, on 7 IDSSE matches (Bundesliga, DFL tracking at 25 fps) that carry a
`BallStatus` flag, the same mechanism as the competition GT. The tracking part of this detector is run there:

| | dead-ball F1 | end \|Δ\| ≤ 0.5 s | start \|Δ\| ≤ 0.5 s | wins |
|---|---|---|---|---|
| detector | 0.933 | 44% | 51% | |
| + end → restart kick | 0.944 | **93%** | 51% | 7/7 |
| + start → ball crossing a line | **0.948** | 93% | **69%** | 7/7 |
| + start → players slow down (whistle) | 0.932 | 93% | 49% | 4/7, rejected |

In IDSSE, `BallStatus` switches to live exactly at the restart kick (100% within ± 0.2 s).

In [ ]:
D_SNAP_S, V_KICK, V_STILL, STILL_S, CROSS_M = 3.0, 4.0, 1.5, 0.6, 0.3
d_mask_trk = d_mask.copy()
bxy_ = z["ball_poss"].astype(float)
vb_ = np.full(N_FRAMES, np.nan)
vb_[2:-2] = np.hypot(*(bxy_[4:] - bxy_[:-4]).T) / (4 / FPS)
fast_ = np.nan_to_num(vb_) > V_KICK
still_ = np.nan_to_num(vb_, nan=0.0) < V_STILL          # an unseen ball counts as still: the ball is often lost during stoppages
cs_still = np.r_[0, np.cumsum(still_)]
n_st = int(STILL_S * FPS)
tt_ = np.arange(n_st, N_FRAMES - 3)
KICKS = tt_[fast_[tt_] & fast_[tt_ + 1] & fast_[tt_ + 2] & ~fast_[tt_ - 1] & (cs_still[tt_] - cs_still[tt_ - n_st] >= n_st - 2)]
oob_ = np.maximum.reduce([-bxy_[:, 0], bxy_[:, 0] - PITCH_L, -bxy_[:, 1], bxy_[:, 1] - PITCH_W, np.zeros(N_FRAMES)])
outside_ = np.nan_to_num(oob_) > CROSS_M
CROSSES = np.flatnonzero(outside_[1:] & ~outside_[:-1] & np.isfinite(bxy_[:-1, 0])) + 1


def nearest_event(ev, t, w):
    c_ = ev[(ev >= t - w) & (ev <= t + w)]
    return int(c_[np.argmin(np.abs(c_ - t))]) if len(c_) else None


snap_rows = []
if D_SNAP:
    w_ = int(D_SNAP_S * FPS)
    segs_ = list(zip(*runs(d_mask)))
    new_ = d_mask.copy()
    for i_, (a_, b_) in enumerate(segs_):
        prev_end = segs_[i_ - 1][1] if i_ else 0
        next_start = segs_[i_ + 1][0] if i_ + 1 < len(segs_) else N_FRAMES
        k_ = nearest_event(KICKS, b_, w_)
        if k_ is not None and a_ < k_ < next_start - 1 and HALF[k_] == HALF[b_ - 1]:
            new_[min(b_, k_):max(b_, k_)] = k_ > b_
            snap_rows.append({"batas": "end → restart kick", "geser_s": (k_ - b_) / FPS})
        x_ = nearest_event(CROSSES, a_, w_)
        if x_ is not None and prev_end < x_ < b_ and HALF[x_] == HALF[a_]:
            new_[min(a_, x_):max(a_, x_)] = x_ < a_
            snap_rows.append({"batas": "start → ball crosses a line", "geser_s": (x_ - a_) / FPS})
    new_ |= forced
    new_ &= GRADED
    d_mask = new_
snap_df = pd.DataFrame(snap_rows, columns=["batas", "geser_s"])
n_seg_ = len(runs(d_mask_trk)[0])
print("kicks detected {} · ball crossing a line {} | D segments {}".format(len(KICKS), len(CROSSES), n_seg_))
if len(snap_df):
    print(snap_df.groupby("batas").geser_s.agg(jumlah="size", median="median", lebih_awal=lambda v: (v < 0).mean(),
                                               lebih_lambat=lambda v: (v > 0).mean()).round(2).to_string())
print("D share: {:.3f} → {:.3f} | frames changed {} ({:.2%} of graded frames)".format(
    d_share(d_mask_trk), d_share(d_mask), int((d_mask != d_mask_trk).sum()), (d_mask != d_mask_trk).sum() / N_GRADED))
check("D share after snapping stays close to the prior (± 0.02)", abs(d_share(d_mask) - D_SHARE_PRIOR) <= 0.02,
      "{:.3f}".format(d_share(d_mask)), critical=False)
check("forced dead frames stay D", d_mask[forced].all())
if len(snap_df):
    fig, ax = plt.subplots(figsize=(9, 3.2))
    bins_ = np.arange(-D_SNAP_S, D_SNAP_S + 0.25, 0.25)
    for (name_, col_) in (("end → restart kick", COL["H"]), ("start → ball crosses a line", COL["A"])):
        v_ = snap_df[snap_df.batas == name_].geser_s
        ax.hist(v_, bins=bins_, histtype="step", linewidth=2, color=col_, label="{} ({})".format(name_, len(v_)))
    ax.axvline(0, color=COL["muted"], linewidth=1)
    ax.set_xlabel("D boundary shift (s): negative = earlier, positive = later")
    ax.set_ylabel("segments")
    ax.set_title("D boundary shift after snapping to physical events", pad=18)
    note(ax, "the detector bias shows on whichever side of the histogram is heavier")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

### Figure: dead-ball segments

Duration distribution of the detected `D` segments. Short stoppages (throw-ins, quick free kicks) sit on the left;
substitutions, injuries and goal celebrations form the right tail.

In [ ]:
d_starts, d_ends = runs(d_mask & ~forced)
d_len = (d_ends - d_starts) / FPS
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(d_len, bins=np.arange(0, min(d_len.max(), 150) + 5, 5), color=COL["D"], edgecolor=COL["surface"], linewidth=1.5)
ax.axvline(np.median(d_len), color=COL["text"], linewidth=1, linestyle="--")
ax.text(np.median(d_len), ax.get_ylim()[1] * 0.95, "  median {:.0f} s".format(np.median(d_len)), color=COL["text"],
        fontsize=9, va="top")
ax.set_xlabel("D segment duration (seconds)")
ax.set_ylabel("number of segments")
ax.set_title("Dead-ball duration", pad=18)
note(ax, "{} segments outside the forced-dead ranges · D share {:.1%} of graded frames".format(len(d_len), d_share(d_mask)))
plt.tight_layout()
plt.show()

### Restart taker team

CHALLENGE.md: *"whenever play restarts after a stoppage, there is a clear possessor"*. For every `D` segment, the nearest
set-piece tag (starting ≤ 12 s after the end of the segment) gives the restart type and its team. Only for **corner
kicks, goal kicks and free kicks** is the tag team the taker team (checked against the player geometry); throw-ins and
kick-offs carry no team in the tags, so their owner is left to the model.

In [ ]:
def restart_table(mask):
    rows_ = []
    for i, (a, b) in enumerate(zip(*runs(mask))):
        near_sp = sp[(sp.start_f < b + 300) & (sp.end_f > a)]
        tag_type, tag_side = "NA", "NA"
        if len(near_sp):
            row = near_sp.iloc[(near_sp.start_f - a).abs().argmin()]
            tag_type, tag_side = str(row.Type), str(row.side)
        rows_.append({"seg": i, "start": int(a), "restart_frame": int(b), "tag_type": tag_type,
                      "taker": tag_side if tag_type in ("Corner Kick", "Goal Kick", "Free Kick") and tag_side in ("H", "A")
                      else "NA"})
    return pd.DataFrame(rows_)


restarts = restart_table(d_mask)
restarts_trk = restart_table(d_mask_trk)
print("D segments: {} | restarts with a known taker team: {}".format(len(restarts), int(restarts.taker.isin(["H", "A"]).sum())))

## 4b. EDA before modelling

**What this step does.** Before modelling possession, we check five questions about data quality. The answers decide
what the model may trust and what must be handled in preprocessing or in the decoder.

1. **When and where does the ball go missing in the tactical tracking?** The tactical camera films the whole pitch from
   far away, so the ball looks small.
2. **What does the broadcast camera show?** The broadcast contains replays, close-ups and crowd shots. Such frames must
   not contribute a ball position.
3. **Is the broadcast-to-tactical alignment stable minute by minute?** The offset is searched once per half; that
   assumption is tested in every minute.
4. **What about long balls?** A ball in the air travels fast and is often projected wrongly by the camera.
5. **What does Smart Tagging say?** When set-piece tags happen relative to the dead ball, and how many tags carry a
   team.

Every number in this section is an aggregate.

### 4b.1 When and where the ball goes missing

Left: share of frames with the ball visible, split into live and dead ball (from the section-4 detector). Right:
tactical ball visibility during live play per pitch zone; for frames without a ball, the last visible ball position is
used.

**Finding.** Almost every frame without a ball happens while the ball is dead: the tactical camera keeps recording, but
the tracker does not follow a ball that is held, out of the pitch, or hidden by a crowd of players. During live play the
ball is visible on most frames. Visibility is lowest near both goals (players crowd the penalty box) and along one
touchline, the side far from the camera, where the ball looks smallest. Implication: **a ball gap during dead-ball time
is a dead-ball cue** (used as the `gap_long` feature), while short gaps during live play are handled by the sequence
model through the team shape.

In [ ]:
eda_live = GRADED & ~d_mask
eda_dead = GRADED & d_mask
eda_miss = GRADED & ~finite
print("tactical ball visible: live ball {:.1%} · dead ball {:.1%} | tactical + broadcast combined: live {:.1%} · dead {:.1%}".format(
    finite[eda_live].mean(), finite[eda_dead].mean(), (ball_src[eda_live] > 0).mean(), (ball_src[eda_dead] > 0).mean()))
print("of all graded frames without a tactical ball, {:.1%} occur while the ball is dead".format(d_mask[eda_miss].mean()))

ball_last = pd.DataFrame(np.where(GRADED[:, None], ball_raw, np.nan)).ffill().to_numpy()
ZX, ZY = 6, 4
zx = np.clip(np.floor(ball_last[eda_live, 0] / PITCH_L * ZX), 0, ZX - 1).astype(int)
zy = np.clip(np.floor(ball_last[eda_live, 1] / PITCH_W * ZY), 0, ZY - 1).astype(int)
vis_zone = np.full((ZY, ZX), np.nan)
for i in range(ZY):
    for j in range(ZX):
        m_ = (zy == i) & (zx == j)
        if m_.sum() >= 250:
            vis_zone[i, j] = finite[eda_live][m_].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.5]})
ax = axes[0]
groups = [("live ball", eda_live), ("dead ball", eda_dead)]
for k, (name, m_) in enumerate(groups):
    for off, (lab_, val, col) in zip((-0.18, 0.18), (("tactical", finite[m_].mean(), COL["H"]),
                                                     ("tactical + broadcast", (ball_src[m_] > 0).mean(), COL["aqua"]))):
        ax.bar(k + off, 100 * val, width=0.34, color=col, label=lab_ if k == 0 else None)
        ax.text(k + off, 100 * val + 1.5, "{:.0f}%".format(100 * val), ha="center", fontsize=9, color=COL["text"])
ax.set_xticks([0, 1], [g_[0] for g_ in groups])
ax.set_ylim(0, 110)
ax.set_ylabel("% of frames with a ball")
ax.grid(axis="x", visible=False)
ax.set_title("Ball visible: live vs dead", pad=18)
note(ax, "{:.0%} of frames without a ball occur while the ball is dead".format(d_mask[eda_miss].mean()))
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.1), ncol=2)

ax = axes[1]
im = ax.imshow(vis_zone, cmap=BLUES, vmin=np.nanmin(vis_zone) - 0.05, vmax=1.0, origin="lower",
               extent=[0, PITCH_L, 0, PITCH_W], aspect="auto")
for i in range(ZY):
    for j in range(ZX):
        if np.isfinite(vis_zone[i, j]):
            v = vis_zone[i, j]
            ax.text((j + 0.5) * PITCH_L / ZX, (i + 0.5) * PITCH_W / ZY, "{:.0%}".format(v), ha="center", va="center",
                    fontsize=9, color="white" if v > np.nanmean(vis_zone) else COL["text"])
ax.axvline(PITCH_L / 2, color=COL["surface"], linewidth=1.5)
ax.set_xlabel("pitch x (m)")
ax.set_ylabel("pitch y (m)")
ax.grid(False)
ax.set_title("Tactical ball visibility during live play, per zone", pad=18)
note(ax, "zone = last position where the ball was seen · goals at x = 0 and x = {:.0f}".format(PITCH_L))
plt.tight_layout()
plt.show()

### 4b.2 Broadcast camera content: replays, close-ups, crowd

Every graded frame is classified by its paired broadcast frame (offset from section 2):

- **used**: valid homography and player positions that match the tactical ones (cost < 2 m, ≥ 6 players);
- **pitch, no match**: valid homography but player positions that do not match (zoom too close, few players, or a
  camera transition);
- **replay / close-up / crowd**: invalid official homography, meaning the camera is not showing the pitch live
  (DATA.md: ± 34% of the broadcast frames).

**Zero weight for frames that do not show the pitch.** Broadcast ball positions are taken only from the *used* class;
replay/close-up/crowd frames never contribute a ball position (checked below). When the broadcast is not showing the
pitch, the notebook uses the tactical recording only. Such frames are still useful as a **dead-ball cue**: a broadcast
shows a replay or close-up almost always while play is stopped (*replay grounding*, Deliège et al., 2021), so it is used
as the `bc_offpitch` feature of the section-4 detector.

In [ ]:
bc_cats = [(1, "used", COL["aqua"]), (0, "pitch, no match", COL["yellow"]),
           (-1, "replay / close-up / penonton", COL["D"])]
bc_rows = []
for code_, name, _ in bc_cats:
    m_ = GRADED & (bc_view == code_)
    bc_rows.append({"golongan": name, "share of graded frames": m_.sum() / N_GRADED, "P(D | golongan)": d_mask[m_].mean(),
                    "ball frames from broadcast": int((ball_src[m_] == 3).sum())})
bc_tab = pd.DataFrame(bc_rows)
print(bc_tab.round(3).to_string(index=False))
off_s, off_e = runs(GRADED & (bc_view == -1))
off_len = (off_e - off_s) / FPS
off_dshare = np.array([d_mask[a_:b_].mean() for a_, b_ in zip(off_s, off_e)])
print("replay/close-up/crowd cuts: {} · median {:.1f} s · longest {:.0f} s · {:.0%} of cuts are mostly dead ball".format(
    len(off_len), np.median(off_len), off_len.max(), (off_dshare > 0.5).mean()))
check("broadcast ball never taken from replay/close-up/crowd frames",
      not (ball_src[GRADED & (bc_view == -1)] == 3).any())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
ax = axes[0]
for k, (code_, name, col) in enumerate(bc_cats):
    v = bc_tab.loc[k, "share of graded frames"]
    ax.barh(k, 100 * v, color=col, height=0.6)
    ax.text(100 * v + 1, k, "{:.0f}%".format(100 * v), va="center", fontsize=9, color=COL["text"])
ax.set_yticks(range(len(bc_cats)), [c_[1] for c_ in bc_cats])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of graded frames")
ax.grid(axis="y", visible=False)
ax.set_title("Content of the paired broadcast frame", pad=18)
note(ax, "{} replay/close-up cuts · median {:.0f} s".format(len(off_len), np.median(off_len)))

ax = axes[1]
for k, (code_, name, col) in enumerate(bc_cats):
    v = bc_tab.loc[k, "P(D | golongan)"]
    ax.plot(100 * v, k, marker="o", markersize=11, color=col, markeredgecolor=COL["surface"], markeredgewidth=2,
            linestyle="none")
    ax.text(100 * v + 3, k, "{:.0f}%".format(100 * v), va="center", fontsize=9, color=COL["text"])
ax.axvline(100 * d_share(d_mask), color=COL["muted"], linewidth=1, linestyle="--")
ax.set_yticks(range(len(bc_cats)), [c_[1] for c_ in bc_cats])
ax.invert_yaxis()
ax.set_xlim(0, 105)
ax.set_xlabel("% of frames with a dead ball (detector of section 4)")
ax.grid(axis="y", visible=False)
ax.set_title("A replay almost always means a dead ball", pad=18)
note(ax, "dashed line = whole-match average")
plt.tight_layout()
plt.show()

### 4b.3 Broadcast-to-tactical alignment per minute

The broadcast → tactical offset is searched once per half (section 2). If the broadcast had breaks or cuts that shift
its timeline, the offset would drift within the half and the broadcast ball would be attached to the wrong frame. To
test this, 5 *used* broadcast frames are taken in every minute and their best offset is searched again locally
(± 60 frames). **Residual** = local offset minus the half's offset; a residual ≈ 0 means one offset per half is right.

The bottom panel shows the composition of the broadcast frames per minute: minutes with many replays are minutes with
a stoppage (goal, foul, substitution).

In [ ]:
MIN_F = 60 * FPS
align_rows = []
with timer("per-minute alignment check"):
    for h, (a, b) in HALVES.items():
        for k, m0 in enumerate(range(a, b + 1, MIN_F)):
            seg = np.arange(m0, min(m0 + MIN_F, b + 1))
            row = {"babak": h, "menit": k, "used": (bc_view[seg] == 1).mean(), "no match": (bc_view[seg] == 0).mean(),
                   "replay": (bc_view[seg] == -1).mean(), "residu": np.nan}
            cand = seg[bc_view[seg] == 1]
            if len(cand) >= 50:
                offs = []
                for t in cand[np.linspace(0, len(cand) - 1, 5).astype(int)]:
                    t_idx = np.arange(t - BC_SEARCH, t + BC_SEARCH + 1)
                    t_idx = t_idx[(t_idx >= 0) & (t_idx < N_FRAMES)]
                    offs.append(int(t_idx[np.argmin(match_cost(t - BC_OFFSET[h], t_idx))]) - t)
                row["residu"] = float(np.median(offs))
            align_rows.append(row)
align = pd.DataFrame(align_rows)
res_ok = align.residu.dropna()
print("minutes tested: {} of {} | offset residual: median {:+.0f} frames · |residual| ≤ 1 frame in {:.0%} of minutes · max |residual| {:.0f} frames ({:.2f} s)".format(
    len(res_ok), len(align), res_ok.median(), (res_ok.abs() <= 1).mean(), res_ok.abs().max(), res_ok.abs().max() / FPS))
check("broadcast offset stable per minute (median |residual| ≤ 1 frame)", res_ok.abs().median() <= 1, critical=False)

n1 = int((align.babak == 1).sum())
xpos = np.where(align.babak == 1, align.menit, n1 + 2 + align.menit)
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
ax = axes[0]
ax.axhspan(-1, 1, color=COL["grid"], alpha=0.8, linewidth=0)
ax.plot(xpos, align.residu, linestyle="none", marker="o", markersize=5, color=COL["H"])
ax.axhline(0, color=COL["muted"], linewidth=1)
ax.set_ylabel("residual (frames)")
ax.set_title("Broadcast → tactical offset residual per minute", pad=18)
note(ax, "grey band = ± 1 frame (0.04 s) · empty dot = minute without a matching broadcast frame")
ax = axes[1]
bottom = np.zeros(len(align))
for key, col in (("used", COL["aqua"]), ("no match", COL["yellow"]), ("replay", COL["D"])):
    ax.bar(xpos, 100 * align[key], bottom=bottom, width=0.9, color=col, label=key if key != "replay" else "replay / close-up")
    bottom += 100 * align[key].to_numpy()
ax.set_ylim(0, 100)
ax.set_ylabel("% of frames per minute")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.28), ncol=3)
ticks = [x_ for x_ in range(0, n1, 15)] + [n1 + 2 + x_ for x_ in range(0, len(align) - n1, 15)]
ax.set_xticks(ticks, ["B1 {}'".format(t_) for t_ in range(0, n1, 15)] + ["B2 {}'".format(t_) for t_ in range(0, len(align) - n1, 15)])
ax.set_xlabel("minutes since the start of the half (B1 = half 1, B2 = half 2)")
plt.tight_layout()
plt.show()

### 4b.4 Long balls

Long balls are detected from the ball track: fast ball runs (> 12 m/s) that travel ≥ 30 m during live play. The
question: does the small, distant ball in the tactical camera go missing exactly during long balls? If so, the model
would need another way to follow long-range ball movement.

**Finding.** Long balls almost always stay tracked in the tactical recording; the ball height (`z`) is rarely available
in the ATD, so it is not used. The Smart Tagging `LONG BALL` tags mostly contain a long ball detected in the tracking, so
this detector is consistent with the automatic annotation. What is hard about long balls is not their visibility but
**who wins the ball afterwards**; that is covered in the final EDA (section 9b).

In [ ]:
LB_V, LB_DIST = 12.0, 30.0
ball_z = ball[:, 2]
flight_rows = []
for a_, b_ in zip(*runs(GRADED & np.isfinite(ball_v) & (ball_v > LB_V))):
    a0, b0 = max(a_ - 6, 0), min(b_ + 6, N_FRAMES - 1)
    if np.isfinite(bx[a0]) and np.isfinite(bx[b0]):
        flight_rows.append({"start": a0, "end": b0, "dist": float(np.hypot(bx[b0] - bx[a0], by[b0] - by[a0])),
                            "dur": (b0 - a0) / FPS, "vis_t": float(finite[a0:b0].mean()),
                            "z_ada": float(np.isfinite(ball_z[a0:b0]).mean()), "mati": float(d_mask[a0:b0].mean())})
flights = pd.DataFrame(flight_rows)
long_balls = flights[(flights.dist >= LB_DIST) & (flights.mati < 0.5)].reset_index(drop=True)
in_long = np.zeros(N_FRAMES, dtype=bool)
for r in long_balls.itertuples():
    in_long[r.start:r.end] = True
lb_tags = tags[tags.code == "LONG BALL"]
lb_hit = np.mean([in_long[max(r.start_f, 0):min(r.end_f, N_FRAMES)].any() for r in lb_tags.itertuples()])
print("fast ball runs: {} | long balls (≥ {:.0f} m, live ball): {} · median {:.0f} m in {:.1f} s".format(
    len(flights), LB_DIST, len(long_balls), long_balls.dist.median(), long_balls.dur.median()))
print("tactical ball visible during long balls {:.1%} (all live frames {:.1%}) | height z available on {:.1%} of ball-visible frames".format(
    finite[in_long].mean(), finite[eda_live].mean(), np.isfinite(ball_z[finite & GRADED]).mean()))
print("LONG BALL tags: {} ({} H, {} A) · {:.0%} contain a detected long ball".format(
    len(lb_tags), int((lb_tags.side == "H").sum()), int((lb_tags.side == "A").sum()), lb_hit))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1.4, 1]})
ax = axes[0]
live_fl = flights[flights.mati < 0.5]
bins = np.arange(0, min(live_fl.dist.max(), 80) + 2.5, 2.5)
ax.hist(live_fl.dist[live_fl.dist < LB_DIST], bins=bins, color=COL["D"], edgecolor=COL["surface"], linewidth=1, label="fast pass < 30 m")
ax.hist(live_fl.dist[live_fl.dist >= LB_DIST], bins=bins, color=COL["violet"], edgecolor=COL["surface"], linewidth=1, label="long ball ≥ 30 m")
ax.set_xlabel("ball travel distance (m)")
ax.set_ylabel("number of runs")
ax.set_title("Fast ball runs (> 12 m/s) during live play", pad=18)
note(ax, "{} runs · {} long balls".format(len(live_fl), len(long_balls)))
ax.legend(loc="upper right")
ax = axes[1]
vis_bars = [("during long balls", finite[in_long].mean(), COL["violet"]), ("all live ball frames", finite[eda_live].mean(), COL["H"]),
            ("dead ball", finite[eda_dead].mean(), COL["D"])]
for k, (name, v, col) in enumerate(vis_bars):
    ax.barh(k, 100 * v, color=col, height=0.6)
    ax.text(100 * v + 1, k, "{:.0f}%".format(100 * v), va="center", fontsize=9, color=COL["text"])
ax.set_yticks(range(3), [v_[0] for v_ in vis_bars])
ax.invert_yaxis()
ax.set_xlim(0, 110)
ax.set_xlabel("% of frames with the tactical ball visible")
ax.grid(axis="y", visible=False)
ax.set_title("Long balls stay tracked", pad=18)
plt.tight_layout()
plt.show()

### 4b.5 Smart Tagging

Smart Tagging contains 14 types of automatic tactical tags. Most are **game phases** with wide or fixed time windows
(e.g. *CREATING CHANCES* is always 30 s), so their timing is not precise enough for possession boundaries. Only **SET
PIECES** is used: the restart type and, for corners/goal kicks/free kicks, the taker team.

Right: time difference between the start of a set-piece tag and the end of the nearest dead-ball segment (= the
restart). Set-piece tags **start inside the stoppage**, a few seconds before the ball is played again; the tags are
therefore used as a stoppage cue and as a marker of the restart team, not as the exact restart moment. Throw-ins and
kick-offs carry no team, so their owner is decided by the model.

In [ ]:
tag_counts = tags.code.value_counts()
print(pd.crosstab(sp.Type, sp.side.replace({"NA": "no team"})).to_string())
tag_dur = tags.assign(dur=tags.end_s - tags.start_s).groupby("code").dur.agg(["min", "median", "max"]).round(1)
print("\ntag window duration (s):\n" + tag_dur.to_string())
ds_, de_ = runs(d_mask)
sp_start_f = np.clip(sp.start_f.to_numpy(), 0, N_FRAMES - 1)
sp_rel = np.array([(f - de_[np.argmin(np.abs(de_ - f))]) / FPS for f in sp_start_f])
print("\nset-piece tag start: {:.0%} inside a D segment · {:.0%} while the broadcast shows a replay/close-up · median {:+.1f} s from the restart".format(
    d_mask[sp_start_f].mean(), (bc_view[sp_start_f] == -1).mean(), np.median(sp_rel)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
ax.barh(range(len(tag_counts)), tag_counts.to_numpy(), height=0.65,
        color=[COL["violet"] if c_ == "SET PIECES" else COL["D"] for c_ in tag_counts.index])
for k, v in enumerate(tag_counts.to_numpy()):
    ax.text(v + 1, k, str(v), va="center", fontsize=8, color=COL["text"])
ax.set_yticks(range(len(tag_counts)), [c_.lower() for c_ in tag_counts.index], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("number of instances")
ax.grid(axis="y", visible=False)
ax.set_title("Smart Tagging tags", pad=18)
note(ax, "{} instances · only SET PIECES is used".format(len(tags)))
ax = axes[1]
ax.hist(np.clip(sp_rel, -30, 10), bins=np.arange(-30, 11, 1.5), color=COL["violet"], edgecolor=COL["surface"], linewidth=1)
ax.axvline(0, color=COL["text"], linewidth=1)
ax.text(0.5, ax.get_ylim()[1] * 0.92, "restart", color=COL["text"], fontsize=9)
ax.set_xlabel("tag start − end of the nearest D segment (s, clipped −30…10)")
ax.set_ylabel("set-piece tag")
ax.set_title("Set-piece tags start inside the stoppage", pad=18)
note(ax, "median {:+.1f} s · {:.0%} inside a D segment".format(np.median(sp_rel), d_mask[sp_start_f].mean()))
plt.tight_layout()
plt.show()

## 4c. Data quality: outliers and implausible values

**What this step does.** Uncorrected automatic tracking always contains physically impossible values. Before modelling,
we count how often they occur, with thresholds taken from the physics of the game, not from the data:

| Type | Threshold | Reason |
|---|---|---|
| player step | > 12 m/s; jump > 20 m/s | the fastest elite sprint is ± 10 m/s; above 20 m/s it is almost surely a track identity swap |
| player / ball position | > 3 m beyond a line | players leave the pitch only briefly (throw-ins, celebrations) |
| tracks per team | > 11 or < 9 | > 11 = ghost or wrong-team tracks; < 9 = untracked players |
| duplicate tracks | two tracks of one team < 0.3 m apart | one player detected twice |
| ball step | > 40 m/s; jump > 60 m/s | the hardest shots are ± 35–40 m/s; above that, a false ball detection |

The same numbers are computed on the external training data (6a, shown in 6c). Outliers are not dropped automatically:
what matters is whether the model is **sensitive** to them, and that is tested directly in 6c.

In [ ]:
AUDIT_COLS = ["pemain: langkah > 12 m/s", "pemain: loncatan > 20 m/s", "pemain: > 3 m di luar lapangan",
              "track ganda (< 0,3 m, tim sama)", "tim dengan > 11 track", "tim dengan < 9 track", "bola: langkah > 40 m/s",
              "bola: loncatan > 60 m/s", "bola: > 3 m di luar lapangan", "bola terlihat"]


def audit_tracking(P_, ball_, home, fps, L, W, mask, chunk=20000):
    """% of implausible observations per type (see the table above); P_ (T, slot, 2) and ball_ (T, 2) in metres"""
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore")
        step = np.hypot(*(P_[1:] - P_[:-1]).transpose(2, 0, 1)) * fps
        sp = step[np.isfinite(step) & mask[1:, None]]
        fin = np.isfinite(P_[..., 0]) & mask[:, None]
        x, y = P_[..., 0][fin], P_[..., 1][fin]
        n_tr = [np.isfinite(P_[:, m, 0]).sum(1)[mask] for m in (home, ~home)]
        dup = np.zeros(len(P_), bool)
        for m in (home, ~home):
            iu = np.triu_indices(int(m.sum()), 1)
            for a in range(0, len(P_), chunk):
                X = P_[a:a + chunk, m]
                d = np.hypot(X[:, iu[0], 0] - X[:, iu[1], 0], X[:, iu[0], 1] - X[:, iu[1], 1])
                dup[a:a + chunk] |= (d < 0.3).any(1)
        bs = np.hypot(*(ball_[1:] - ball_[:-1]).T) * fps
        bs = bs[np.isfinite(bs) & mask[1:]]
        fb = np.isfinite(ball_[:, 0]) & mask
        bx, by = ball_[fb, 0], ball_[fb, 1]
        vals = [(sp > 12).mean(), (sp > 20).mean(), ((x < -3) | (x > L + 3) | (y < -3) | (y > W + 3)).mean(),
                dup[mask].mean(), ((n_tr[0] > 11) | (n_tr[1] > 11)).mean(), ((n_tr[0] < 9) | (n_tr[1] < 9)).mean(),
                (bs > 40).mean(), (bs > 60).mean(), ((bx < -3) | (bx > L + 3) | (by < -3) | (by > W + 3)).mean(),
                fb.sum() / mask.sum()]
    return {k: round(100 * float(v), 4) for k, v in zip(AUDIT_COLS, vals)}


AUDIT_BAR = audit_tracking(players[..., :2], ball_raw, SLOT_IS_HOME, FPS, PITCH_L, PITCH_W, GRADED)
print("% of observations on BAR-ATH graded frames:")
print(pd.Series(AUDIT_BAR).to_string())
n_tracks = {t: np.isfinite(players[:, m, 0]).sum(1)[GRADED] for t, m in (("H", SLOT_IS_HOME), ("A", ~SLOT_IS_HOME))}
with np.errstate(all="ignore"):
    p_step = (np.hypot(*(players[1:, :, :2] - players[:-1, :, :2]).transpose(2, 0, 1)) * FPS)[GRADED[1:]]
p_step = p_step[np.isfinite(p_step)]
check("player track jumps > 20 m/s are rare (< 0.1% of steps)", AUDIT_BAR["pemain: loncatan > 20 m/s"] < 0.1,
      "{:.3f}%".format(AUDIT_BAR["pemain: loncatan > 20 m/s"]), critical=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
ax = axes[0]
ks = np.arange(6, 15)
for j, (t, col) in enumerate((("H", COL["H"]), ("A", COL["A"]))):
    share = [100 * (n_tracks[t] == k).mean() for k in ks[:-1]] + [100 * (n_tracks[t] >= ks[-1]).mean()]
    ax.bar(ks + (j - 0.5) * 0.38, share, width=0.36, color=col, label=t)
ax.axvspan(11.5, 14.6, color=COL["yellow"], alpha=0.12, linewidth=0)
ax.set_xticks(ks, [str(k) for k in ks[:-1]] + ["≥ {}".format(ks[-1])])
ax.set_xlabel("number of tracks of one team in one frame")
ax.set_ylabel("% of graded frames")
ax.grid(axis="x", visible=False)
ax.set_title("Tracks per team: there should be 11", pad=18)
note(ax, "> 11 (hatched) = ghost / wrong-team tracks · the external training data never exceeds 11")
ax.legend(loc="upper left")
ax = axes[1]
ax.hist(np.minimum(p_step, 40.5), bins=np.arange(0, 42, 1.0), color=COL["H"], alpha=0.85)
ax.set_yscale("log")
for thr, lab_, yy in ((12, "12 m/s", 0.95), (20, "20 m/s", 0.82)):
    ax.axvline(thr, color=COL["A"], linewidth=1, linestyle="--")
    ax.text(thr + 0.4, yy, lab_, color=COL["A"], fontsize=8, va="top", transform=ax.get_xaxis_transform())
ax.set_xticks(np.arange(0, 45, 5), [str(v) for v in range(0, 40, 5)] + ["≥ 40"])
ax.set_xlabel("player step speed between frames (m/s)")
ax.set_ylabel("number of steps (log)")
ax.set_title("Player speed: a very thin outlier tail", pad=18)
note(ax, "> 12 m/s: {:.3f}% of steps · > 20 m/s: {:.3f}%".format(AUDIT_BAR["pemain: langkah > 12 m/s"],
                                                               AUDIT_BAR["pemain: loncatan > 20 m/s"]))
plt.tight_layout()
plt.show()

**Interpretation.** Speed and position outliers are very rare (≤ 0.1% of steps) and already tamed by the features,
which clip player speed at 12 m/s and ball speed at 40 m/s. What stands out is the **number of tracks**: on ± 13% of
frames one team has more than 11 tracks, mostly not duplicates (the extra track is on average ± 4 m from its nearest
teammate) but ghost or wrong-team tracks. This never happens in the external training data, so it is a **domain
shift**. Instead of guessing which track is wrong, we measure its impact: in 6c, ghost tracks with the same frequency
are injected into the external matches and the model scores are compared.

## 5. Representation for the possession model

**What this step does.** Turns the tracking into per-step features for the possession model.

Who has the ball can be read from two complementary sources:

- **Local ball geometry**: who is close to the ball, moves with it, or approaches it (*possession step*, Vidal-Codina
  et al., 2022). Sharp in time, but available only while the ball is visible, and noisy during pressing.
- **Team shape**: the team in possession spreads out and moves up, the defending team compacts (Fernández & Bornn,
  2018; Ötting et al., 2023). Always available, also when the ball is missing, but it lags ± 1 s behind a change of
  possession.

The sequence model (section 6) gets both and learns by itself when to trust which. Every data source is converted to
the same format (player positions in metres, ball `NaN` when not visible, attacking direction), then to **10 fps**, the
frame rate of the external training data.

**YOLO:** the BAR-ATH ball position is the merged ball from 2b.6 (tactical + YOLO corrections/fills + broadcast). The same
features are also computed with the tracking ball so that the tracking-ball and YOLO-ball predictions can be compared frame by frame in 9c.

| Group | Features |
|---|---|
| ball | visible?, seconds since/until the ball was visible (≤ 5 s), speed, position towards the attacking goal, distance to the halfway line |
| local ball (per team) | distance of the 3 nearest players, relative speed nearest player–ball, approach speed, player speed, number of players within 2/5/10 m (zero when the ball is not visible) |
| shape (per team) | mean position towards the attacking goal, width, length, spread, mean speed, speed towards the opponent's goal, number of tracked players |
| player set | the 8 nearest players per team relative to the ball (or to the team centroid when the ball is missing): dx, dy, vx, vy, distance, valid |
| **ball motion & arrival time** | ball speed towards the attacking goal and sideways, ball acceleration and turn (the sign of a touch); arrival time of each team's 2 fastest players to the ball and the control share |

**Why ball motion & arrival time were added.** The error analysis (6c) shows errors piling up within ± 1 s of a
change of possession, especially during passes and when the opponent is already closer to the ball than the owner. The
earlier features gave only the *magnitude* of the ball speed, without direction, so the model did not know where a pass
was going. The new group answers two questions: where is the ball going (a pass usually goes forward, towards the
attacking goal of the team in possession), and who will get there first. Arrival time uses the simple motion model of
Spearman (2018): a player keeps running in the same direction for 0.7 s (reaction time), then runs straight to the ball
at 5 m/s; control share = logistic of the difference in arrival time between the two teams (scale 0.45 s).

**H↔A symmetry.** Features are computed from the viewpoint of one team ("own", `sendiri`) against the opponent
(`lawan`). Swapping the viewpoint flips the label, so the model cannot learn "the home team has the ball more often"; it
learns only geometry.

*Note:* the feature names in the code (`FEATURE_NAMES`) are kept in their original Indonesian form because they are
stored with the trained models; see the glossary at the top of the notebook for their English meaning.

In [ ]:
FPS10 = 10
SET_K = 8


def resample10(s):
    T = len(s["label"])
    step = int(s["fps"]) / FPS10
    idx = np.arange(T) if step == 1 else np.round(np.arange(0, T, step)).astype(int)
    idx = idx[idx < T]
    out = {k: (v[idx] if isinstance(v, np.ndarray) and v.ndim >= 1 and len(v) == T else v) for k, v in s.items()}
    out["idx"] = idx
    return out


def vel10(X, k=2):
    V = np.full(X.shape, np.nan, np.float32)
    V[k:-k] = (X[2 * k:] - X[:-2 * k]) / (2 * k / FPS10)
    return V


def ball_motion(vb, a_own):
    """ball speed towards the attacking goal & sideways, acceleration, turn (1 − cos of the angle, only when the ball moves > 2 m/s)"""
    T = len(vb)
    acc, turn = np.full(T, np.nan, np.float32), np.full(T, np.nan, np.float32)
    acc[1:-1] = np.hypot(*(vb[2:] - vb[:-2]).T) / 0.2
    cos = np.einsum("ij,ij->i", vb[2:], vb[:-2]) / (np.hypot(*vb[2:].T) * np.hypot(*vb[:-2].T))
    turn[1:-1] = np.where(np.hypot(*vb[2:].T) > 2, 1 - cos, 0)
    return np.stack([np.clip(vb[:, 0] * a_own, -30, 30) / 10, np.clip(np.abs(vb[:, 1]), 0, 30) / 10,
                     np.clip(acc, 0, 100) / 30, np.clip(turn, 0, 2)], 1)


def arrival(P_, VP, ball_, own_m, opp_m, react=0.7, vmax=5.0):
    """arrival time of each team's 2 fastest players to the ball (Spearman 2018) and the control share; zero when the ball is not visible"""
    out = []
    for m in (own_m, opp_m):
        q = P_[:, m] + np.nan_to_num(VP[:, m]) * react
        t = react + np.hypot(q[..., 0] - ball_[:, None, 0], q[..., 1] - ball_[:, None, 1]) / vmax
        ts = np.sort(np.where(np.isfinite(t), t, np.inf), 1)
        out += [np.minimum(ts[:, 0], 6.0), np.minimum(ts[:, 1], 6.0)]
    ctrl = 1 / (1 + np.exp(-(out[2] - out[0]) / 0.45))
    F = np.stack([o / 3 for o in out] + [ctrl - 0.5], 1)
    F[~np.isfinite(ball_[:, 0])] = 0
    return F


def features(s, swap=False):
    P_, home, ball_ = s["P"].astype(np.float32), s["home"].astype(bool), s["ball"].astype(np.float32)
    L, W = float(s["L"]), float(s["W"])
    att = s["att"].astype(np.float32)
    lab = s["label"].astype(np.int8).copy()
    own_m, opp_m, a_own = (home, ~home, att) if not swap else (~home, home, -att)
    if swap:
        lab = np.where(lab >= 0, 1 - lab, -1).astype(np.int8)
    T = len(lab)
    seen = np.isfinite(ball_[:, 0])
    vb = vel10(ball_)
    VP = vel10(P_)
    cols = []
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore")
        ts, tu = np.zeros(T, np.float32), np.zeros(T, np.float32)
        c = 5.0
        for t in range(T):
            c = 0.0 if seen[t] else min(c + 0.1, 5.0)
            ts[t] = c
        c = 5.0
        for t in range(T - 1, -1, -1):
            c = 0.0 if seen[t] else min(c + 0.1, 5.0)
            tu[t] = c
        cols += [seen.astype(np.float32), ts, tu, np.nan_to_num(np.clip(np.hypot(*vb.T), 0, 40) / 10),
                 np.nan_to_num((ball_[:, 0] - L / 2) * a_own / (L / 2)), np.nan_to_num(np.abs(ball_[:, 1] - W / 2) / (W / 2))]
        for m, a in ((own_m, a_own), (opp_m, -a_own)):
            X, V = P_[:, m], VP[:, m]
            D = np.hypot(X[..., 0] - ball_[:, None, 0], X[..., 1] - ball_[:, None, 1])
            Dn = np.where(np.isfinite(D), D, np.inf)
            order = np.argsort(Dn, 1)
            ds = np.take_along_axis(Dn, order[:, :3], 1)
            ds = np.where(np.isfinite(ds), np.minimum(ds, 30.0), 30.0)
            vel_near = V[np.arange(T), order[:, 0]]
            u = (ball_ - X[np.arange(T), order[:, 0]]) / np.maximum(ds[:, :1], 0.3)
            local = [ds[:, 0] / 10, ds[:, 1] / 10, ds[:, 2] / 10, np.nan_to_num(np.clip(np.hypot(*(vb - vel_near).T), 0, 30) / 10),
                     np.nan_to_num(np.clip(np.einsum("ij,ij->i", vel_near - vb, u), -15, 15) / 5),
                     np.nan_to_num(np.clip(np.hypot(*vel_near.T), 0, 12) / 5)]
            local += [np.sum(Dn < r, 1).astype(np.float32) for r in (2.0, 5.0, 10.0)]
            cols += [np.where(seen, v, 0.0) for v in local]
            xs, ys = X[..., 0], X[..., 1]
            spd = np.hypot(V[..., 0], V[..., 1])
            shape = [(np.nanmean(xs, 1) - L / 2) * a / (L / 2), (np.nanmax(ys, 1) - np.nanmin(ys, 1)) / W,
                     (np.nanmax(xs, 1) - np.nanmin(xs, 1)) / L, np.sqrt(np.nanvar(xs, 1) + np.nanvar(ys, 1)) / 20,
                     np.nanmean(np.clip(spd, 0, 12), 1) / 5, np.nanmean(np.clip(V[..., 0], -12, 12), 1) * a / 5]
            cols += [np.nan_to_num(v) for v in shape] + [np.isfinite(xs).sum(1).astype(np.float32) / 11]
        extra = np.concatenate([ball_motion(vb, a_own), arrival(P_, VP, ball_, own_m, opp_m)], 1)
    Fm = np.stack(cols, 1).astype(np.float32)
    Fm[~np.isfinite(Fm)] = 0
    Fm = np.concatenate([Fm, np.clip(np.nan_to_num(extra.astype(np.float32)), -10, 10)], 1)
    return Fm, lab


def set_features(s, swap=False, k=SET_K):
    P_, home, ball_ = s["P"].astype(np.float32), s["home"].astype(bool), s["ball"].astype(np.float32)
    att = s["att"].astype(np.float32)
    own_m, opp_m, a_own = (home, ~home, att) if not swap else (~home, home, -att)
    T = len(P_)
    VP = vel10(P_)
    seen = np.isfinite(ball_[:, 0])
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore")
        cen = np.nanmean(P_[:, own_m], 1)
    ref = np.where(seen[:, None], ball_, cen)
    out = np.zeros((T, 2 * k, 6), np.float16)
    for t_i, m in enumerate((own_m, opp_m)):
        rel = P_[:, m] - ref[:, None]
        Dn = np.hypot(rel[..., 0], rel[..., 1])
        Dn = np.where(np.isfinite(Dn), Dn, np.inf)
        order = np.argsort(Dn, 1)[:, :k]
        r = np.take_along_axis(rel, order[..., None], 1)
        v = np.take_along_axis(VP[:, m], order[..., None], 1)
        d = np.take_along_axis(Dn, order, 1)
        ok = np.isfinite(d) & np.isfinite(r[..., 0])
        a = a_own[:, None]
        blk = np.stack([r[..., 0] * a / 10, r[..., 1] * a / 10, np.nan_to_num(v[..., 0]) * a / 5,
                        np.nan_to_num(v[..., 1]) * a / 5, np.minimum(d, 50) / 10, ok.astype(np.float32)], -1)
        blk[~ok] = 0
        out[:, t_i * k:(t_i + 1) * k] = np.clip(np.nan_to_num(blk), -8, 8).astype(np.float16)
    return out


ball_bar = z["ball_poss"].astype(np.float32).copy()
ball_bar[z["src_poss"] == 0] = np.nan
BAR = dict(P=z["players"][..., :2].astype(np.float32), home=z["slot_is_home"].astype(bool), ball=ball_bar,
           att=z["attack_dir"].astype(np.int8), label=np.full(N_FRAMES, -1, np.int8), fps=25, L=PITCH_L, W=PITCH_W)
bar10 = resample10(BAR)
with timer("BAR-ATH features"):
    XB_H = (features(bar10)[0], set_features(bar10))
    XB_A = (features(bar10, swap=True)[0], set_features(bar10, swap=True))
if YOLO_ON:
    ball_bar_trk = z["ball_filled"].astype(np.float32).copy()
    ball_bar_trk[z["ball_src"] == 0] = np.nan
    bar10_trk = resample10(dict(BAR, ball=ball_bar_trk))
    with timer("BAR-ATH features with the tracking ball (for comparison)"):
        XB_TRK_H = (features(bar10_trk)[0], set_features(bar10_trk))
        XB_TRK_A = (features(bar10_trk, swap=True)[0], set_features(bar10_trk, swap=True))
N_FEAT = XB_H[0].shape[1]
FEATURE_NAMES = ["bola terlihat", "detik sejak bola terlihat", "detik sampai bola terlihat", "kecepatan bola",
                 "bola x ke arah serang", "bola |y| dari tengah"]
for _side in ("sendiri", "lawan"):
    FEATURE_NAMES += ["{} ({})".format(c_, _side) for c_ in (
        "jarak terdekat 1", "jarak terdekat 2", "jarak terdekat 3", "kec. relatif terdekat–bola", "kec. mendekat",
        "kec. pemain terdekat", "n dalam 2 m", "n dalam 5 m", "n dalam 10 m", "posisi rata-rata", "lebar", "panjang",
        "sebaran", "kec. rata-rata", "kec. ke gawang", "n track")]
FEATURE_NAMES += ["bola: kec. ke arah serang", "bola: kec. ke samping", "bola: percepatan", "bola: belokan arah",
                  "waktu tiba tercepat (sendiri)", "waktu tiba ke-2 (sendiri)", "waktu tiba tercepat (lawan)",
                  "waktu tiba ke-2 (lawan)", "porsi kontrol waktu tiba"]
FEATURE_GROUPS = {"bola": list(range(0, 6)), "lokal bola sendiri": list(range(6, 15)), "bentuk sendiri": list(range(15, 22)),
                  "lokal bola lawan": list(range(22, 31)), "bentuk lawan": list(range(31, 38)),
                  "gerak bola & waktu tiba": list(range(38, 47))}
assert len(FEATURE_NAMES) == N_FEAT, (len(FEATURE_NAMES), N_FEAT)
feat_df = pd.DataFrame(XB_H[0], columns=FEATURE_NAMES)
const = [c_ for c_ in FEATURE_NAMES if feat_df[c_].std() < 1e-6]
corr_ = np.nan_to_num(feat_df.corr().abs().to_numpy())
iu_ = np.triu_indices(N_FEAT, 1)
top_ = np.argsort(-corr_[iu_])[:6]
print("constant features on BAR-ATH:", ", ".join(const) or "none")
print("most redundant feature pairs on BAR-ATH (|r|):")
for k in top_:
    print("  {:.3f}  {} ~ {}".format(corr_[iu_][k], FEATURE_NAMES[iu_[0][k]], FEATURE_NAMES[iu_[1][k]]))
print("features per 10 fps step: {} + player set {}×{} | steps: {}".format(N_FEAT, *XB_H[1].shape[1:], len(bar10["idx"])))

## 6. Possession sequence model

**What this step does.** Trains (or loads) the six sequence models that output the H vs A log-odds for every step.

### Why learn from other matches

Hand-written rules ("nearest player", "control ≥ 0.5 s without an opponent nearby") fail exactly in the cases that cost
the most spellF1: in automatic tracking, a **pressing opponent** looks almost identical to the **ball owner**, and short
possessions (a brief interception) cannot be told apart from noise. What is needed is a large number of labelled
examples, and those are openly available:

| Source | Content | Labels |
|---|---|---|
| **SkillCorner Open Data** (20 A-League 2024/25 matches) | **automatic** broadcast tracking at 10 fps, with noise similar to our data | `player_possession` events + game interruptions |
| **Metrica Sports sample data** (3 matches) | 25 fps tracking from the same provider & format as the competition data | PASS / CARRY / RECOVERY / BALL LOST / SET PIECE / BALL OUT events |

Label per frame = **the last team that had the ball** (CHALLENGE.md: *"possession stays with the last team that had
it"*); dead balls and data gaps are ignored during training. That gives ± 3,700 labelled changes of possession, two
hundred times more than the public GT.

**External data is explicitly allowed.** RULES.md: *"Within that, nothing is off the table. Any language, any
framework. Pre-trained models, commercial or open. LLMs, hosted or local. **External data.**"* CHALLENGE.md and
OVERVIEW.md say the same (*"any framework, model or external data"*). What is forbidden is hand-made labels for this
match and the private GT; the external data here are public releases under open licences, from other matches, with
their sources cited.

### Architecture

- **BiGRU**, 3 layers (96 units per direction), with two-sided context over a 40 s window.
- **Non-causal TCN**: 8 residual blocks of dilated convolutions 1…128 (± 25 s). TCNs were designed for action
  segmentation and are evaluated with *segmental F1 at IoU*, the metric spellF1 is built on (Lea et al., 2017).
- **Player-set encoder** (`*_set`): a shared per-player MLP followed by mean + max pooling per team, so it does not
  depend on the order or identity of the slots (Deep Sets, Zaheer et al., 2017). Per-object embeddings with
  permutation-invariant pooling are also used by Borghesi et al. (2023) for possession and dead balls from tracking.
- **Ensemble of 6 models**: 2 × BiGRU-set + 2 × TCN + 2 × TCN-set (seeds 0 and 1). Different architectures make
  different errors; mixing models with and without the set encoder proved better than either alone.

### Training

- **Domain augmentation.** The external data are cleaner than our tracking, so every training copy is "damaged" to look
  like it: the ball-gap pattern is taken from **this match's real ball-gap pattern** (randomly shifted; half of the
  copies without gaps so the model is also strong when the ball is visible), small position noise, brief ball jumps
  (projections of a ball in the air), and lost player tracks.
- **Possession-weighted loss.** Frame weight ∝ (1 / possession duration)^0.5: spellF1 counts every possession once,
  while a plain loss is dominated by long possessions and ignores short ones.
- AdamW (Loshchilov & Hutter, 2019) + OneCycle schedule (Smith & Topin, 2019), 12 epochs, 40 s window.

### Validation

Every choice above is tested with **leave-match-out cross-validation** on the 23 matches (each match is predicted by a
model that never saw it; score = official spellF1; three conditions: this match's ball-gap pattern, its live-ball gap
pattern only, and ball visible).

**Why split per match and not per time chunk.** Neighbouring frames are almost identical, so a random split per frame
or per minute leaks information and gives an over-optimistic score. With leave-match-out, every test match is scored
**whole, ± 90 minutes**, including the start and end of each half, tired phases and dead balls, exactly as BAR-ATH will
be scored. An extra analysis in 6b compares the score of the first 5 minutes with the score of the whole match on the
same match.

| Method | BAR-ATH-like ball gaps | ball visible |
|---|---|---|
| team-shape + nearest-player rule (no labels) | ± 0.61 | ± 0.63 |
| BiGRU / TCN | ± 0.79 | ± 0.82 |
| ensemble without ball motion | ± 0.81 | ± 0.84 |
| **final ensemble** (+ ball motion & arrival time) | **± 0.81** (+0.002) | **± 0.84** (+0.005) |

The third condition, **BAR-ATH live-ball gaps** (`celah bola hidup BAR-ATH` in the code), was added together with the ball-motion features: the gap
pattern is taken only from live-ball frames (according to the `D` detector, section 4), where the ball is missing on
± 4% of frames instead of ± 37% as in the pattern over all graded frames. This condition is closest to live play in
BAR-ATH, but it is used only for **evaluation**, not as a training augmentation (training with this pattern makes the
model fragile to long gaps: −0.03 in the gap condition, see 6c).

Every one of the 23 matches improves over the rule-based method (≥ +0.10).

In [ ]:
import torch
import torch.nn as nn

torch.backends.cudnn.benchmark = False      # (cudnn.deterministic=True makes the TCN ± 55× slower — not used)
TRAIN_DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = [("gru_set", 0), ("gru_set", 1), ("tcn", 0), ("tcn", 1), ("tcn_set", 0), ("tcn_set", 1)]
CV_ARCHS = ["gru_set", "tcn", "tcn_set"]
N_AUG, MIX, SPELL_W = 4, True, 0.5
EPOCHS, WIN, BATCH = 12, 400, 32
COST = 10.0                                 # H↔A switching cost (½ log-odds units), from external cross-validation
GAP_UNKNOWN_S = 6.0
N_FOLDS, SEED = 5, 0
FEATURE_VERSION = "f47"


class SetEnc(nn.Module):
    def __init__(self, d_in=6, d=48, k=SET_K):
        super().__init__()
        self.k = k
        self.phi = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d), nn.GELU())
        self.out_dim = 4 * d

    def forward(self, S):
        h, m = self.phi(S), S[..., 5:6]
        outs = []
        for t in range(2):
            ht, mt = h[:, :, t * self.k:(t + 1) * self.k], m[:, :, t * self.k:(t + 1) * self.k]
            outs.append((ht * mt).sum(2) / mt.sum(2).clamp(min=1))
            outs.append((ht * mt - 1e4 * (1 - mt)).max(2).values.clamp(min=-10))
        return torch.cat(outs, -1)


class BiGRU(nn.Module):
    def __init__(self, F, use_set, H=96, layers=3):
        super().__init__()
        self.set = SetEnc() if use_set else None
        self.inp = nn.Sequential(nn.Linear(F + (self.set.out_dim if use_set else 0), H), nn.GELU())
        self.gru = nn.GRU(H, H, num_layers=layers, batch_first=True, bidirectional=True, dropout=0.1)
        self.out = nn.Linear(2 * H, 1)

    def forward(self, X, S):
        x = torch.cat([X, self.set(S)], -1) if self.set is not None else X
        h, _ = self.gru(self.inp(x))
        return self.out(h).squeeze(-1)


class TCN(nn.Module):
    def __init__(self, F, use_set, H=128, n_blocks=8):
        super().__init__()
        self.set = SetEnc() if use_set else None
        self.inp = nn.Conv1d(F + (self.set.out_dim if use_set else 0), H, 1)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Conv1d(H, H, 3, padding=2 ** i, dilation=2 ** i), nn.GELU(),
                                                   nn.Dropout(0.1), nn.Conv1d(H, H, 1)) for i in range(n_blocks)])
        self.out = nn.Conv1d(H, 1, 1)

    def forward(self, X, S):
        x = torch.cat([X, self.set(S)], -1) if self.set is not None else X
        h = self.inp(x.transpose(1, 2))
        for b in self.blocks:
            h = h + b(h)
        return self.out(h)[:, 0]


def build_net(arch):
    use_set = arch.endswith("_set")
    return BiGRU(N_FEAT, use_set) if arch.startswith("gru") else TCN(N_FEAT, use_set)


@torch.no_grad()
def predict(net, XS, device, chunk=3000, overlap=300):
    X, S = XS
    net = net.to(device).eval()
    T, out, a = len(X), np.zeros(len(X), np.float32), 0
    while a < T:
        lo_, hi_, b = max(0, a - overlap), min(T, a + chunk + overlap), min(T, a + chunk)
        o = net(torch.from_numpy(np.ascontiguousarray(X[lo_:hi_])[None]).to(device),
                torch.from_numpy(np.ascontiguousarray(S[lo_:hi_]).astype(np.float32)[None]).to(device))[0].float().cpu().numpy()
        out[a:b] = o[a - lo_:b - lo_]
        a = b
    return out


def weight_path(arch, seed):
    return MODEL_DIR / "{}_s{}.pt".format(arch, seed)


def train_mode(value):
    """TRAIN_MODELS → "auto" / "train" / "load"; accepts a bool or text ("True", "false", "auto", ...)."""
    v = str(value).strip().lower()
    if v == "auto":
        return "auto"
    if v in ("true", "1", "yes", "ya", "latih", "train", "always"):
        return "train"
    if v in ("false", "0", "no", "tidak", "muat", "load", "never"):
        return "load"
    raise ValueError("TRAIN_MODELS = {!r} not recognised; use \"auto\", True or False".format(value))


META_PATH = MODEL_DIR / "meta.json"
MODE = train_mode(TRAIN_MODELS)
have_all = META_PATH.is_file() and all(weight_path(a, s).is_file() for a, s in MODELS)
DO_TRAIN = MODE == "train" or (MODE == "auto" and not have_all)
if MODE == "load" and not have_all:
    raise FileNotFoundError("TRAIN_MODELS = False (load) but the weights/meta.json are incomplete in {}. "
                            "Switch to \"auto\" or True to train the models.".format(MODEL_DIR))
print("TRAIN_MODELS = {!r} → mode {} | weights in {}: {} → {}".format(
    TRAIN_MODELS, MODE, MODEL_DIR.relative_to(ROOT), "complete" if have_all else "missing",
    "TRAIN from external data (" + TRAIN_DEV + ")" if DO_TRAIN else "LOAD"))

### 6a. Training from external data

Runs only if the weights are missing (or `TRAIN_MODELS = True`). Needs `external_data/skillcorner/` and
`external_data/metrica/` (see the README) and, in practice, a CUDA GPU. **The submitted file does not run this cell:
the six trained models are shipped with the repository.**

- **SkillCorner:** coordinates centred on the pitch centre → shifted; the ball only when it was really detected
  (extrapolated positions are dropped to look like our tracking); dead ball from a possession that ends with an
  interruption until the next possession.
- **Metrica:** 0–1 coordinates (y pointing down) → metres; dead ball from BALL OUT / FAULT RECEIVED / shots out until
  the next SET PIECE; Game 3 (EPTS format) is read with `kloppy`.
- No possession event for > 6 s (e.g. a replay without tracking) → label unknown, ignored.

In [ ]:
EXT_CACHE = CACHE / "external"


def ffill_labels(T, starts, ends, teams, dead_from, dead_to, fps):
    lab = np.full(T, -1, dtype=np.int8)
    order = np.argsort(starts, kind="stable")
    starts, ends, teams = starts[order], ends[order], teams[order]
    for i, (a, t) in enumerate(zip(starts, teams)):
        b = starts[i + 1] if i + 1 < len(starts) else min(T, ends[i] + 1)
        lab[a:b] = t
        gap_end = ends[i] + int(GAP_UNKNOWN_S * fps)
        if b > gap_end:
            lab[gap_end:b] = -1
    for a, b in zip(dead_from, dead_to):
        lab[a:b] = -1
    return lab


def _cached(name, build):
    EXT_CACHE.mkdir(exist_ok=True)
    path = EXT_CACHE / (name + ".npz")
    if not path.is_file():
        np.savez_compressed(path, **build())
    with np.load(path) as zf:
        return {k: zf[k] for k in zf.files}


def load_skillcorner(mid):
    def build():
        d = EXT_DIR / "skillcorner" / "data" / "matches" / mid
        meta_ = json.load(open(d / "{}_match.json".format(mid), encoding="utf-8"))
        L, W = float(meta_["pitch_length"]), float(meta_["pitch_width"])
        home_id = meta_["home_team"]["id"]
        col = {p["id"]: i for i, p in enumerate(meta_["players"])}
        home = np.array([p["team_id"] == home_id for p in meta_["players"]])
        rows = [json.loads(line) for line in open(d / "{}_tracking_extrapolated.jsonl".format(mid), encoding="utf-8")]
        T = len(rows)
        P_ = np.full((T, len(col), 2), np.nan, np.float32)
        bl = np.full((T, 2), np.nan, np.float32)
        period = np.zeros(T, np.int8)
        for r in rows:
            if r["period"] is None:
                continue
            f = r["frame"]
            period[f] = r["period"]
            for pl in r["player_data"]:
                j = col.get(pl["player_id"])
                if j is not None:
                    P_[f, j] = (pl["x"] + L / 2, pl["y"] + W / 2)
            bd = r["ball_data"]
            if bd["is_detected"] and bd["x"] is not None:
                bl[f] = (bd["x"] + L / 2, bd["y"] + W / 2)
        att = np.zeros(T, np.int8)
        for k, side in enumerate(meta_["home_team_side"], start=1):
            att[period == k] = 1 if side == "left_to_right" else -1
        e = pd.read_csv(d / "{}_dynamic_events.csv".format(mid), low_memory=False)
        pp = e[e.event_type == "player_possession"].sort_values("frame_start")
        st, en = pp.frame_start.to_numpy().astype(int), pp.frame_end.to_numpy().astype(int)
        after = pp.game_interruption_after.notna().to_numpy()
        dead = [(en[i] + 1, st[i + 1]) for i in np.flatnonzero(after) if i + 1 < len(st)]
        lab = ffill_labels(T, st, en, (pp.team_id.to_numpy() == home_id).astype(np.int8),
                           [a for a, _ in dead], [b for _, b in dead], 10)
        lab[period == 0] = -1
        return dict(P=P_, home=home, ball=bl, att=att, label=lab, fps=10, L=L, W=W)
    return _cached("sc_" + mid, build)


METRICA_POSS = {"PASS", "SET PIECE", "SHOT", "RECOVERY", "BALL LOST"}


def _metrica_dead(stops, set_pieces):
    spa = np.array(sorted(set_pieces))
    dead = [(a, spa[spa > a][0]) for a in stops if (spa > a).any()]
    return [a for a, _ in dead], [b for _, b in dead]


def _metrica_att(P_, hmask, period, L):
    att = np.zeros(len(P_), np.int8)
    for k in (1, 2):
        m = period == k
        att[m] = 1 if np.nanmean(P_[np.flatnonzero(m)[:250]][:, hmask, 0]) < L / 2 else -1
    return att


def load_metrica_csv(g):
    def build():
        d = EXT_DIR / "metrica" / "data" / "Sample_Game_{}".format(g)
        L, W = 105.0, 68.0
        blocks, home = [], []
        for side in ("Home", "Away"):
            raw = pd.read_csv(d / "Sample_Game_{}_RawTrackingData_{}_Team.csv".format(g, side), skiprows=2)
            cols = list(raw.columns)
            names = [c for c in cols if c.startswith("Player")]
            blocks.append(np.stack([raw[[n, cols[cols.index(n) + 1]]].to_numpy(float) for n in names], 1))
            home += [side == "Home"] * len(names)
            bi = cols.index("Ball")
            bl = raw.iloc[:, [bi, bi + 1]].to_numpy(float)
            period = raw["Period"].to_numpy().astype(np.int8)
        P_ = np.concatenate(blocks, 1)
        P_ = np.stack([P_[..., 0] * L, (1 - P_[..., 1]) * W], -1).astype(np.float32)
        bl = np.stack([bl[:, 0] * L, (1 - bl[:, 1]) * W], -1).astype(np.float32)
        hmask = np.array(home)
        ev = pd.read_csv(d / "Sample_Game_{}_RawEventsData.csv".format(g))
        pe = ev[ev.Type.isin(METRICA_POSS)]
        st = pe["Start Frame"].to_numpy().astype(int) - 1
        en = np.maximum(pe["End Frame"].to_numpy().astype(int) - 1, st)
        stops_ev = ev[ev.Type.isin(["BALL OUT", "FAULT RECEIVED"]) | ev.Subtype.fillna("").str.contains("-OUT|GOAL")]
        stops = [int(r["Start Frame"]) - 1 if r.Type == "FAULT RECEIVED" else int(max(r["End Frame"], r["Start Frame"])) - 1
                 for _, r in stops_ev.iterrows()]
        df_, dt_ = _metrica_dead(stops, ev[ev.Type == "SET PIECE"]["Start Frame"].to_numpy().astype(int) - 1)
        lab = ffill_labels(len(P_), st, en, (pe.Team.to_numpy() == "Home").astype(np.int8), df_, dt_, 25)
        return dict(P=P_, home=hmask, ball=bl, att=_metrica_att(P_, hmask, period, L), label=lab, fps=25, L=L, W=W)
    return _cached("me_{}".format(g), build)


def load_metrica3():
    def build():
        from kloppy import metrica
        d = EXT_DIR / "metrica" / "data" / "Sample_Game_3"
        ds = metrica.load_tracking_epts(meta_data=str(d / "Sample_Game_3_metadata.xml"),
                                        raw_data=str(d / "Sample_Game_3_tracking.txt"), coordinates="metrica")
        df = ds.to_df()
        L, W = 105.0, 68.0
        home_team = ds.metadata.teams[0]
        home_ids = {p.player_id for p in home_team.players}
        pids = sorted({c[:-2] for c in df.columns if c.endswith("_x") and c != "ball_x"})
        P_ = np.stack([df[[p + "_x", p + "_y"]].to_numpy(float) for p in pids], 1)
        P_ = np.stack([P_[..., 0] * L, (1 - P_[..., 1]) * W], -1).astype(np.float32)
        bl = np.stack([df.ball_x.to_numpy(float) * L, (1 - df.ball_y.to_numpy(float)) * W], -1).astype(np.float32)
        period = df.period_id.to_numpy().astype(np.int8)
        hmask = np.array([p in home_ids for p in pids])
        fid0 = int(df.frame_id.iloc[0])
        st, en, teams, stops, sps = [], [], [], [], []
        for e in json.load(open(d / "Sample_Game_3_events.json", encoding="utf-8"))["data"]:
            typ = e["type"]["name"]
            sub = e.get("subtypes") or {}
            sub = sub.get("name", "") if isinstance(sub, dict) else " ".join(x["name"] for x in sub)
            a, b = int(e["start"]["frame"]) - fid0, int(e["end"]["frame"]) - fid0
            if typ in METRICA_POSS | {"CARRY"}:
                st.append(a); en.append(max(a, b)); teams.append(int(e["team"]["id"] == home_team.team_id))
            if typ == "SET PIECE":
                sps.append(a)
            if typ in ("BALL OUT", "FAULT RECEIVED") or "-OUT" in sub or "GOAL" in sub:
                stops.append(max(a, b) if typ != "FAULT RECEIVED" else a)
        df_, dt_ = _metrica_dead(stops, sps)
        lab = ffill_labels(len(P_), np.array(st), np.array(en), np.array(teams, np.int8), df_, dt_, 25)
        return dict(P=P_, home=hmask, ball=bl, att=_metrica_att(P_, hmask, period, L), label=lab, fps=25, L=L, W=W)
    return _cached("me_3", build)


def smooth_noise(rng_, shape, sigma, width=5):
    n = rng_.normal(0, sigma, shape).astype(np.float32)
    k = np.ones(width, np.float32) / width
    return np.apply_along_axis(lambda v: np.convolve(v, k, "same"), 0, n) * np.sqrt(width)


def augment(s, rng_, bank):
    s = dict(s)
    T = len(s["label"])
    P_ = s["P"] + smooth_noise(rng_, s["P"].shape, 0.2)
    bl = s["ball"] + smooth_noise(rng_, s["ball"].shape, 0.4)
    for _ in range(rng_.poisson(T / 300)):
        a = rng_.integers(0, T)
        bl[a:min(T, a + rng_.integers(5, 25))] += rng_.normal(0, 2.5, 2).astype(np.float32)
    if bank is not None:
        m = np.resize(np.roll(bank, rng_.integers(0, len(bank))), T)
        bl[~m] = np.nan
    for j in range(P_.shape[1]):
        for _ in range(rng_.poisson(T * 0.06 / 50)):
            a = rng_.integers(0, T)
            P_[a:a + rng_.integers(10, 90), j] = np.nan
    s["P"], s["ball"] = P_, bl
    return s


def make_copy(s, rng_, bank, swap=False):
    sa = augment(s, rng_, bank)
    X, lab = features(sa, swap=swap)
    return (X, set_features(sa, swap=swap)), lab


def spell_weight(y):
    w = np.ones(len(y), np.float32)
    a = 0
    for t in range(1, len(y) + 1):
        if t == len(y) or y[t] != y[a]:
            if y[a] >= 0:
                w[a:t] = (1.0 / max((t - a) / FPS10, 0.5)) ** SPELL_W
            a = t
    return w / w[y >= 0].mean()


def train_model(arch, seed, train_names):
    torch.manual_seed(seed)
    r = np.random.default_rng(seed)
    seqs = [(XS, y, spell_weight(y)) for n in train_names for (XS, y) in DATA[n]]
    net = build_net(arch).to(TRAIN_DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    lens = np.array([len(y) for _, y, _ in seqs], float)
    steps = int(EPOCHS * lens.sum() // (WIN * BATCH))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        xb, sb, yb, wb = [], [], [], []
        for si in r.choice(len(seqs), BATCH, p=lens / lens.sum()):
            (X, S), y, w = seqs[si]
            a = r.integers(0, len(y) - WIN)
            xb.append(X[a:a + WIN]); sb.append(S[a:a + WIN]); yb.append(y[a:a + WIN]); wb.append(w[a:a + WIN])
        x = torch.from_numpy(np.stack(xb).astype(np.float32, copy=False)).to(TRAIN_DEV)
        s_ = torch.from_numpy(np.stack(sb).astype(np.float32)).to(TRAIN_DEV)
        y = torch.from_numpy(np.stack(yb).astype(np.float32)).to(TRAIN_DEV)
        w = torch.from_numpy(np.stack(wb)).to(TRAIN_DEV)
        m = y >= 0
        loss = nn.functional.binary_cross_entropy_with_logits(net(x, s_)[m], y[m], weight=w[m])
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
    return net.eval()


def decode_ext(lg, lab, cost):
    emis = np.zeros((len(lab), 3))
    emis[:, 0], emis[:, 1] = 0.5 * lg, -0.5 * lg
    dead = lab < 0
    emis[dead, :2] = -1e4
    emis[dead, 2], emis[~dead, 2] = 1e4, -1e4
    return np.array(["H", "A", "D"])[viterbi(emis, np.array([[0, cost, 0], [cost, 0, 0], [0, 0, 0]], float))]


def spell_ext(pred, lab):
    gtl = np.where(lab == 1, "H", np.where(lab == 0, "A", "D")).astype(object)
    return pm.evaluate(pm.label_codes(gtl), pm.label_codes(np.asarray(pred, dtype=object)))["spell_f1_mean"]


COND_MAIN = "celah ala BAR-ATH"
POST_COSTS = [0.0, 2.0, 5.0, 10.0, 20.0, 40.0]
POST_MEDIAN_S = [1.0, 3.0, 5.0, 9.0]
POST_MERGE_S = [0.5, 1.0, 2.0]


def score_ext(pred, lab, keep=None):
    """official (spellF1, macroF1); keep = graded frames (the rest are X)"""
    gtl = np.where(lab == 1, "H", np.where(lab == 0, "A", "D")).astype(object)
    pr = np.asarray(pred, dtype=object).copy()
    if keep is not None:
        gtl[~keep], pr[~keep] = "X", "X"
    r = pm.evaluate(pm.label_codes(gtl), pm.label_codes(pr))
    return float(r["spell_f1_mean"]), float(r["macro_f1"])


def median_smooth(lg, lab, win_s):
    """baseline: per-frame labels from the log-odds sign, then a median filter (no duration model)"""
    w = max(1, int(round(win_s * FPS10))) | 1
    sgn = pd.Series(np.sign(lg)).rolling(w, center=True, min_periods=1).median().to_numpy()
    out = np.where(sgn >= 0, "H", "A").astype(object)
    out[lab < 0] = "D"
    return out


def merge_short(pred, min_s):
    """possessions shorter than min_s that sit between two possessions of the same opposing team are merged into that team"""
    p = np.asarray(pred, dtype=object).copy()
    chg_ = np.flatnonzero(p[1:] != p[:-1]) + 1
    st_, en_ = np.r_[0, chg_], np.r_[chg_, len(p)]
    for i in range(1, len(st_) - 1):
        prev_, next_ = p[st_[i - 1]], pred[st_[i + 1]]
        if pred[st_[i]] in ("H", "A") and en_[i] - st_[i] < min_s * FPS10 and prev_ == next_ and prev_ in ("H", "A"):
            p[st_[i]:en_[i]] = prev_
    return p


def post_analysis(oof):
    """On the out-of-fold predictions: (1) post-processing variants, (2) score of the first 5 minutes vs the whole match and per
    15-minute block, (3) possession balance per match. Main condition (BAR-ATH-like gaps) only."""
    post, window, match = [], [], []
    for (n, cond), (lg, lab) in oof.items():
        if cond != COND_MAIN:
            continue
        for c in POST_COSTS:
            pred = decode_ext(lg, lab, c)
            sf, mf = score_ext(pred, lab)
            post.append({"match": n, "metode": "Viterbi" if c > 0 else "per frame (tanpa smoothing)", "param": c,
                         "spell_f1": sf, "macro_f1": mf})
            if c == COST:
                base = pred
        for w in POST_MEDIAN_S:
            sf, mf = score_ext(median_smooth(lg, lab, w), lab)
            post.append({"match": n, "metode": "filter median", "param": w, "spell_f1": sf, "macro_f1": mf})
        for k in POST_MERGE_S:
            sf, mf = score_ext(merge_short(base, k), lab)
            post.append({"match": n, "metode": "Viterbi + gabung pendek", "param": k, "spell_f1": sf, "macro_f1": mf})
        idx = np.flatnonzero(lab >= 0)
        a0, a1 = int(idx[0]), int(idx[-1]) + 1
        keep = np.zeros(len(lab), bool)
        keep[a0:a0 + 5 * 60 * FPS10] = True
        window.append({"match": n, "jendela": "5 menit pertama", "spell_f1": score_ext(base, lab, keep)[0]})
        sf_all, mf_all = score_ext(base, lab)
        window.append({"match": n, "jendela": "seluruh match", "spell_f1": sf_all})
        edges = np.linspace(a0, a1, 7).astype(int)
        for k in range(6):
            keep = np.zeros(len(lab), bool)
            keep[edges[k]:edges[k + 1]] = True
            window.append({"match": n, "jendela": "blok {}".format(k + 1), "spell_f1": score_ext(base, lab, keep)[0]})
        y, pr = lab[lab >= 0], base[lab >= 0]
        dom = 1 if (y == 1).mean() >= 0.5 else 0
        rec = {t: float((pr[y == t] == ("H" if t == 1 else "A")).mean()) for t in (0, 1)}
        n_sp = int(((y[1:] != y[:-1])).sum() + 1)
        match.append({"match": n, "porsi_home": float((y == 1).mean()), "spell_f1": sf_all, "macro_f1": mf_all,
                      "recall_dominan": rec[dom], "recall_minor": rec[1 - dom], "possession": n_sp})
    return post, window, match


GHOST_SHARE = 0.08                          # share of frames with extra tracks per team, as in BAR-ATH (4c)
ERR_SINCE_BINS = [0, 0.5, 1, 2, 3, 5, 10, np.inf]
ERR_DUR_BINS = [0, 1, 2, 3, 5, 10, 20, np.inf]


def records(df):
    """DataFrame → JSON-safe list of dicts (NaN → null)"""
    return json.loads(df.to_json(orient="records", force_ascii=False))


def bin_labels(edges):
    return ["{:g}–{:g} s".format(a, b) if np.isfinite(b) else "> {:g} s".format(a) for a, b in zip(edges[:-1], edges[1:])]


def add_ghost_tracks(s, rng_, share=GHOST_SHARE):
    """extra tracks per team as in BAR-ATH: opponent player positions copied into empty slots for 2–20 s"""
    s = dict(s)
    P_ = s["P"].copy()
    home = s["home"].astype(bool)
    T = len(P_)
    for m in (home, ~home):
        covered, tries = 0, 0
        while covered < share * T and tries < 5000:
            tries += 1
            a, L = int(rng_.integers(0, T - 200)), int(rng_.integers(20, 200))
            seg = slice(a, a + L)
            free = [j for j in np.flatnonzero(m) if np.isnan(P_[seg, j, 0]).all()]
            src_ = [j for j in np.flatnonzero(~m) if np.isfinite(P_[seg, j, 0]).all()]
            if free and src_:
                P_[seg, free[0]] = P_[seg, rng_.choice(src_)] + rng_.normal(0, 0.7, 2).astype(np.float32)
                covered += L
    s["P"] = P_
    return s


def perm_importance(nets, XS, lab, seed):
    """Δlogloss (& ΔspellF1 per group) when one column / group is circularly shifted by a random offset: its distribution and
    autocorrelation are kept, its alignment with the other features and the labels is broken"""
    X, S = XS
    rng_ = np.random.default_rng(seed)
    m = lab >= 0
    y = (lab[m] == 1).astype(float)

    def run(X_, S_, spell):
        lg = np.mean([predict(net, (X_, S_), TRAIN_DEV) for net in nets], 0)
        p = 1 / (1 + np.exp(-np.clip(lg[m], -30, 30)))
        ll = float(-(y * np.log(np.clip(p, 1e-6, 1)) + (1 - y) * np.log(np.clip(1 - p, 1e-6, 1))).mean())
        return ll, (spell_ext(decode_ext(lg, lab, COST), lab) if spell else None)

    def shift():
        return int(rng_.integers(len(X) // 4, 3 * len(X) // 4))

    ll0, sf0 = run(X, S, True)
    rows = []
    for g, idx in list(FEATURE_GROUPS.items()) + [("himpunan pemain", None)]:
        X_, S_ = X, S
        if idx is None:
            S_ = np.roll(S, shift(), 0)
        else:
            X_ = X.copy()
            for j in idx:
                X_[:, j] = np.roll(X_[:, j], shift())
        ll, sf = run(X_, S_, True)
        rows.append({"jenis": "kelompok", "nama": g, "d_logloss": ll - ll0, "d_spell_f1": sf - sf0})
    for j, name in enumerate(FEATURE_NAMES):
        X_ = X.copy()
        X_[:, j] = np.roll(X_[:, j], shift())
        rows.append({"jenis": "fitur", "nama": name, "d_logloss": run(X_, S, False)[0] - ll0, "d_spell_f1": None})
    return rows


def error_context(s10, rng_, bank):
    """per-step context from exactly the same evaluation copy (same seed as make_copy)"""
    sa = augment(s10, rng_, bank)
    P_, bl, home = sa["P"], sa["ball"], sa["home"].astype(bool)
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore")
        dmin = [np.nanmin(np.where(np.isfinite(d), d, np.inf), 1) for d in
                (np.hypot(P_[:, m, 0] - bl[:, None, 0], P_[:, m, 1] - bl[:, None, 1]) for m in (home, ~home))]
    return {"vis": np.isfinite(bl[:, 0]), "dH": dmin[0], "dA": dmin[1]}


def error_analysis(oof, ctx):
    """Detailed evaluation & error analysis on the out-of-fold predictions (main condition): where the model is wrong, which
    possessions are missed, and what kind of spurious switches occur"""
    fr, gs, ps, tr, piou = [], [], [], [], []
    for (n, cond), (lg, lab) in oof.items():
        if cond != COND_MAIN:
            continue
        pred = decode_ext(lg, lab, COST)
        gc = pm.label_codes(np.where(lab == 1, "H", np.where(lab == 0, "A", "D")).astype(object))
        pc = pm.label_codes(np.asarray(pred, dtype=object))
        runs_ = pm.valid_runs(gc)
        for thr, v in pm.spell_scores(gc, pc, runs_)["per_iou"].items():
            piou.append({"iou": float(thr), "precision": v["precision"], "recall": v["recall"], "f1": v["f1"]})
        gsp, psp = pm.possession_spells(gc, runs_), pm.possession_spells(pc, runs_)
        c, T = ctx[n], len(lab)
        since, until = np.full(T, np.nan), np.full(T, np.nan)
        for a, b, _ in gsp:
            since[a:b] = np.arange(b - a) / FPS10
            until[a:b] = np.arange(b - a)[::-1] / FPS10
        live = lab >= 0
        own_d, opp_d = np.where(lab == 1, c["dH"], c["dA"]), np.where(lab == 1, c["dA"], c["dH"])
        sit = np.where(~c["vis"], "bola hilang", np.where((own_d < 2) & (opp_d < 2), "duel: keduanya < 2 m",
                       np.where(own_d < 2, "pemilik < 2 m dari bola", np.where(opp_d < 2, "lawan < 2 m, pemilik tidak",
                                                                                   "bola bebas (tak ada < 2 m)"))))
        fr.append(pd.DataFrame({"salah": (pred != np.where(lab == 1, "H", "A"))[live], "sejak": since[live],
                                "sampai": until[live], "situasi": sit[live],
                                "p": 1 / (1 + np.exp(-np.clip(lg[live], -30, 30))), "y": (lab[live] == 1).astype(float)}))
        best_g, n_same = np.zeros(len(gsp)), np.zeros(len(gsp), int)
        g_ok, p_ok = np.zeros(len(gsp), bool), np.zeros(len(psp), bool)
        for iou, gi, pj in pm.spell_overlaps(gsp, psp):           # decreasing IoU → the official greedy matching
            best_g[gi] = max(best_g[gi], iou)
            n_same[gi] += 1
            if iou >= 0.5 and not g_ok[gi] and not p_ok[pj]:
                g_ok[gi], p_ok[pj] = True, True
        for gi, (a, b, _) in enumerate(gsp):
            gs.append({"durasi": (b - a) / FPS10, "cocok": bool(g_ok[gi]), "best_iou": best_g[gi], "n_pred": int(n_same[gi]),
                       "bola_terlihat": float(c["vis"][a:b].mean()), "dari_bola_mati": bool(a > 0 and lab[a - 1] < 0)})
        for pj, (a, b, _) in enumerate(psp):
            team = 1 if pred[a] == "H" else 0
            ps.append({"durasi": (b - a) / FPS10, "cocok": bool(p_ok[pj]), "porsi_tim_benar": float((lab[a:b] == team).mean())})
        gl = np.where(lab >= 0, lab, -1)
        pl = np.where(pred == "H", 1, np.where(pred == "A", 0, -1))
        pch = np.flatnonzero((pl[1:] >= 0) & (pl[:-1] >= 0) & (pl[1:] != pl[:-1])) + 1
        for t in np.flatnonzero((gl[1:] >= 0) & (gl[:-1] >= 0) & (gl[1:] != gl[:-1])) + 1:
            cand = pch[(np.abs(pch - t) <= 3 * FPS10) & (pl[pch] == gl[t])]
            tr.append({"tertangkap": len(cand) > 0,
                       "lag_s": float(cand[np.argmin(np.abs(cand - t))] - t) / FPS10 if len(cand) else None,
                       "bola_terlihat": bool(c["vis"][t])})
    F, Gs, Ps, Tr = pd.concat(fr), pd.DataFrame(gs), pd.DataFrame(ps), pd.DataFrame(tr)

    def by(key, edges=None):
        g = pd.cut(F[key], edges, labels=bin_labels(edges), right=False) if edges else F[key]
        t = F.groupby(g, observed=True).salah.agg(["mean", "size", "sum"]).reset_index()
        t.columns = ["kelas", "error", "n", "n_salah"]
        t["porsi_error"] = t.n_salah / F.salah.sum()
        return records(t.drop(columns="n_salah"))

    Gs["kelas"] = pd.cut(Gs.durasi, ERR_DUR_BINS, labels=bin_labels(ERR_DUR_BINS)).astype(str)
    Ps["kelas"] = pd.cut(Ps.durasi, ERR_DUR_BINS, labels=bin_labels(ERR_DUR_BINS)).astype(str)
    Gs["tipe"] = np.where(Gs.cocok, "cocok", np.where(Gs.best_iou == 0, "terlewat total (tak ada prediksi tim itu)",
                          np.where(Gs.n_pred > 1, "terpecah (≥ 2 prediksi)", "batas meleset / tergabung")))
    Ps["tipe"] = np.where(Ps.cocok, "cocok", np.where(Ps.porsi_tim_benar < 0.5, "tim salah (pergantian palsu)",
                          "batas meleset / tergabung"))
    F["kal"] = pd.cut(F.p, np.linspace(0, 1, 11), include_lowest=True)
    cal = F.groupby("kal", observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size")).reset_index(drop=True)
    lag = Tr.lag_s.dropna()
    return {
        "ringkas": {"error_frame": float(F.salah.mean()), "brier": float(((F.p - F.y) ** 2).mean()),
                    "n_possession_gt": int(len(Gs)), "n_possession_pred": int(len(Ps)), "n_frame_hidup": int(len(F))},
        "per_iou": records(pd.DataFrame(piou).groupby("iou").mean().reset_index()),
        "sejak_awal": by("sejak", ERR_SINCE_BINS), "sampai_akhir": by("sampai", ERR_SINCE_BINS), "situasi": by("situasi"),
        "recall_durasi": records(Gs.groupby("kelas", sort=False).agg(n=("cocok", "size"), recall=("cocok", "mean")).reset_index()),
        "presisi_durasi": records(Ps.groupby("kelas", sort=False).agg(n=("cocok", "size"), presisi=("cocok", "mean")).reset_index()),
        "tipe_gt": records(Gs.groupby("tipe").agg(porsi=("durasi", "size"), durasi_median=("durasi", "median"),
                                                 bola_terlihat=("bola_terlihat", "median")).reset_index()
                           .assign(porsi=lambda d: d.porsi / len(Gs))),
        "tipe_pred": records(Ps.groupby("tipe").agg(porsi=("durasi", "size"), durasi_median=("durasi", "median")).reset_index()
                             .assign(porsi=lambda d: d.porsi / len(Ps))),
        "asal": records(Gs.groupby("dari_bola_mati").agg(n=("cocok", "size"), recall=("cocok", "mean")).reset_index()),
        "kalibrasi": records(cal),
        "transisi": {"tertangkap_3s": float(Tr.tertangkap.mean()), "lag_median_s": float(lag.median()),
                     "lag_q25_s": float(lag.quantile(0.25)), "lag_q75_s": float(lag.quantile(0.75)),
                     "tertangkap_bola_terlihat": float(Tr[Tr.bola_terlihat].tertangkap.mean()),
                     "tertangkap_bola_hilang": float(Tr[~Tr.bola_terlihat].tertangkap.mean())},
    }


cv_rows, cv_post, cv_window, cv_match = [], [], [], []
cv_err, cv_perm, cv_ghost, data_audit = {}, [], [], {}
CV_OOF = {}
if DO_TRAIN:
    if not (EXT_DIR / "skillcorner" / "data" / "matches").is_dir() or not (EXT_DIR / "metrica" / "data").is_dir():
        raise FileNotFoundError("External data not found in {} — see the README (clone SkillCorner/opendata & "
                                "metrica-sports/sample-data)".format(EXT_DIR))
    with timer("load external data"):
        sc_ids = sorted(p.name for p in (EXT_DIR / "skillcorner" / "data" / "matches").iterdir() if p.is_dir())
        EXT = {"sc" + m: load_skillcorner(m) for m in sc_ids}
        EXT.update({"me1": load_metrica_csv(1), "me2": load_metrica_csv(2), "me3": load_metrica3()})
    NAMES = sorted(EXT)
    with timer("external data quality audit"):
        audit_ext = pd.DataFrame({n: audit_tracking(v["P"], v["ball"], v["home"].astype(bool), int(v["fps"]), float(v["L"]),
                                                    float(v["W"]), np.ones(len(v["label"]), bool)) for n, v in EXT.items()}).T
    data_audit = {"BAR-ATH (frame dinilai)": AUDIT_BAR,
                  "SkillCorner (median 20 match)": audit_ext[audit_ext.index.str.startswith("sc")].median().round(4).to_dict(),
                  "Metrica (median 3 match)": audit_ext[audit_ext.index.str.startswith("me")].median().round(4).to_dict()}
    print("external matches: {} | labelled H↔A switches: {}".format(len(NAMES), sum(
        int(((v["label"][1:] != v["label"][:-1]) & (v["label"][1:] >= 0) & (v["label"][:-1] >= 0)).sum()) for v in EXT.values())))
    g_idx = np.flatnonzero(GRADED)
    BALL_BANK = (z["ball_src"] > 0)[g_idx[np.round(np.arange(0, len(g_idx) - 1, 2.5)).astype(int)]]
    rng = np.random.default_rng(SEED)
    EXT10 = {n: resample10(EXT[n]) for n in NAMES}
    with timer("features + external augmentation"):
        DATA = {n: [make_copy(EXT10[n], rng, None if (MIX and (k // 2) % 2 == 1) else BALL_BANK, swap=k % 2 == 1)
                    for k in range(N_AUG)] for n in NAMES}

    if RUN_CV:
        EVAL = {n: make_copy(EXT10[n], np.random.default_rng(1000 + i), BALL_BANK) for i, n in enumerate(NAMES)}
        EVAL_NG = {n: make_copy(EXT10[n], np.random.default_rng(2000 + i), None) for i, n in enumerate(NAMES)}
        l_idx = np.flatnonzero(GRADED & ~d_mask)
        BANK_LIVE = (z["ball_src"] > 0)[l_idx[np.round(np.arange(0, len(l_idx) - 1, 2.5)).astype(int)]]
        EVAL_LIVE = {n: make_copy(EXT10[n], np.random.default_rng(3000 + i), BANK_LIVE) for i, n in enumerate(NAMES)}
        with timer("cross-validation"):
            for k, held in enumerate([NAMES[i::N_FOLDS] for i in range(N_FOLDS)]):
                nets = [train_model(arch, SEED, [n for n in NAMES if n not in held]) for arch in CV_ARCHS]
                for n in held:
                    for cond, (XS, lab) in (("celah ala BAR-ATH", EVAL[n]), ("celah bola hidup BAR-ATH", EVAL_LIVE[n]),
                                         ("bola terlihat", EVAL_NG[n])):
                        lgs = [predict(net, XS, TRAIN_DEV) for net in nets]
                        row = {"match": n, "kondisi": cond}
                        for arch, lg_ in zip(CV_ARCHS, lgs):
                            row[arch] = spell_ext(decode_ext(lg_, lab, COST), lab)
                        row["ensembel"] = spell_ext(decode_ext(np.mean(lgs, 0), lab, COST), lab)
                        cv_rows.append(row)
                        CV_OOF[(n, cond)] = (np.mean(lgs, 0).astype(np.float32), lab)
                    i_n = NAMES.index(n)
                    for r_ in perm_importance(nets, EVAL[n][0], EVAL[n][1], seed=4000 + i_n):
                        cv_perm.append(dict(r_, match=n))
                    sg = add_ghost_tracks(augment(EXT10[n], np.random.default_rng(1000 + i_n), BALL_BANK),
                                          np.random.default_rng(7000 + i_n))
                    Xg, lab_g = features(sg)
                    lg_g = np.mean([predict(net, (Xg, set_features(sg)), TRAIN_DEV) for net in nets], 0)
                    cv_ghost.append({"match": n, "bersih": spell_ext(decode_ext(CV_OOF[(n, COND_MAIN)][0], lab_g, COST), lab_g),
                                     "track hantu": spell_ext(decode_ext(lg_g, lab_g, COST), lab_g)})
                print("fold {}/{} done".format(k + 1, N_FOLDS), flush=True)
        cv = pd.DataFrame(cv_rows)
        print("external leave-match-out spellF1 ({} matches):".format(len(NAMES)))
        print(cv.groupby("kondisi")[CV_ARCHS + ["ensembel"]].mean().round(3).to_string())
        with timer("post-processing & window analysis (out-of-fold)"):
            cv_post, cv_window, cv_match = post_analysis(CV_OOF)
        with timer("error analysis (out-of-fold)"):
            cv_err = error_analysis(CV_OOF, {n: error_context(EXT10[n], np.random.default_rng(1000 + i), BALL_BANK)
                                             for i, n in enumerate(NAMES)})

    with timer("train the 6 final models"):
        for arch, seed in MODELS:
            net = train_model(arch, seed, NAMES)
            torch.save({k: v.cpu() for k, v in net.state_dict().items()}, weight_path(arch, seed))
    META_PATH.write_text(json.dumps({
        "version": "final", "feature_version": FEATURE_VERSION, "n_feat": N_FEAT, "set_k": SET_K,
        "models": [{"arch": a, "seed": s, "file": weight_path(a, s).name, "sha256": sha256(weight_path(a, s))}
                   for a, s in MODELS],
        "training": {"matches": NAMES, "n_aug": N_AUG, "mix": MIX, "spell_w": SPELL_W, "epochs": EPOCHS, "win": WIN,
                     "batch": BATCH, "seed": SEED, "device": TRAIN_DEV, "torch": torch.__version__,
                     "sources": ["SkillCorner Open Data (github.com/SkillCorner/opendata)",
                                 "Metrica Sports sample data (github.com/metrica-sports/sample-data)"]},
        "decoder_cost": COST, "cv": cv_rows, "cv_post": cv_post, "cv_window": cv_window, "cv_match": cv_match,
        "cv_err": cv_err, "cv_perm": cv_perm, "cv_ghost": cv_ghost, "data_audit": data_audit},
        indent=2), encoding="utf-8")
    print("weights & meta.json saved in", MODEL_DIR.relative_to(ROOT))

### 6b. Load the weights

Every weight file is verified against its SHA-256 in `meta.json`, and so is the feature configuration, so that what is
loaded is exactly the model that was trained.

In [ ]:
if not META_PATH.is_file():
    raise FileNotFoundError("{} missing: section 6a did not train the models. Re-run from the parameter cell with "
                            "TRAIN_MODELS = \"auto\" or True.".format(META_PATH))
MODEL_META = json.loads(META_PATH.read_text(encoding="utf-8"))
check("weight feature configuration = notebook", MODEL_META["n_feat"] == N_FEAT and MODEL_META["set_k"] == SET_K
      and MODEL_META["feature_version"] == FEATURE_VERSION)
NETS = []
for m_ in MODEL_META["models"]:
    path = MODEL_DIR / m_["file"]
    check("sha256 {}".format(m_["file"]), sha256(path) == m_["sha256"])
    net = build_net(m_["arch"])
    net.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    NETS.append(net.eval())
print("{} models loaded | trained on {} external matches ({}) | COST {}".format(
    len(NETS), len(MODEL_META["training"]["matches"]), MODEL_META["training"]["device"], MODEL_META["decoder_cost"]))

### 6d. BAR-ATH-style ball — *rejected, off by default*

> This experiment did not pass its decision rule (log #96). With `SMOOTH_STAGE = "off"` (the default) the cells below only
> print that they are skipped, and the base weights are used. The section is kept for the record.

**Problem.** The model learns to recognise a touch from the **ball motion**: the ball suddenly turns or accelerates. In
the 23 training matches that motion is sharp. In BAR-ATH the tracking ball turns out to be **smoothed**: even without
labels, the features `bola: belokan arah` (ball: turn) and `bola: percepatan` (ball: acceleration) are much smaller
(diagnosis before this notebook: turn p90 0.06 vs 0.25; at 25 fps acceleration p99 33 m/s² vs 287 m/s² in Metrica). The
tracking ball also does **not lag** behind the YOLO ball (the speed correlation peaks at a 0-frame shift), so the problem
is the shape of the motion, not its timing. Brief touches (which form *short possessions*) suffer the most.

**Method.**

| Step | What it does | Uses labels? |
|---|---|---|
| 6d.1 | measure the ball-motion signature (speed, acceleration, turn) on BAR-ATH live frames with the ball visible; find the Gaussian smoothing σ that gives the external training balls the same distribution (normalised Wasserstein distance) | no |
| 6d.2 | 5-fold cross-validation on the 23 external matches: **base** (old augmentation) vs **smoothed-ball** (old augmentation + ball smoothed with σ ~ U(0; 1.5 σ\*) per copy). Both use the same random draws, so the only difference is the smoothing | external labels |
| 6d.3 | if it passes: train the 6 final smoothed-ball models (2 seeds × 3 architectures, as in base) and use them for BAR-ATH | external labels |

**Evaluation conditions** (every held-out match): `bola halus + celah ala BAR-ATH` (smooth ball + BAR-ATH-like gaps,
σ\*, the main condition: closest to BAR-ATH), `celah ala BAR-ATH` (BAR-ATH-like gaps), `celah bola hidup BAR-ATH`
(BAR-ATH live-ball gaps), `bola terlihat` (ball visible) — the three old conditions with a sharp ball.

**Decision rule (locked before the run).** The smoothed-ball model is used if (1) the mean per-match Δ spellF1 (smoothed-ball − base) in the
main condition is > 0 **and** the 5% bootstrap lower bound (2,000 resamples over matches) is > 0, and (2) no other
condition has a mean Δ < −0.005 (the model must not break on a sharp ball). *Short-possession recall* (< 2 s, IoU ≥ 0.3,
same team) is also reported but not used for the decision. The public GT is not used. If σ\* = 0 (no shift), this
section stops and the base weights are used.

Cost: cross-validation ± 2 × 65 minutes and final models ± 30 minutes on an RTX 3050 Ti (once; results cached in
`cache/smooth/` and `notebooks/pipeline/models_smooth/`).

In [ ]:
# 6d.1  ball-motion signature: BAR-ATH vs the 23 training matches, and the smoothing σ that matches them (no labels)
import gc

import psutil
from scipy.ndimage import gaussian_filter1d
from scipy.stats import wasserstein_distance

SMOOTH_DIR = CACHE / "smooth"
SMOOTH_DIR.mkdir(parents=True, exist_ok=True)
SMOOTH_MODE = str(SMOOTH_STAGE).strip().lower()
assert SMOOTH_MODE in ("auto", "off", "force"), SMOOTH_STAGE
NAMES_CV = list(MODEL_META["training"]["matches"])
SIG_FEATS = ["kecepatan bola", "bola: percepatan", "bola: belokan arah"]
SIG_IDX = [FEATURE_NAMES.index(c_) for c_ in SIG_FEATS]
SMOOTH_SIGMAS = [0.0, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]       # in 10 fps steps (1 = 0.1 s)


def smooth_ball(bl, sigma):
    """Gaussian smoothing per run of visible ball; NaN stays NaN, run edges are not pulled towards zero"""
    if sigma <= 0:
        return bl
    out = bl.copy()
    for a_, b_ in zip(*runs(np.isfinite(bl[:, 0]))):
        if b_ - a_ >= 3:
            out[a_:b_] = gaussian_filter1d(bl[a_:b_].astype(np.float64), sigma, axis=0, mode="nearest")
    return out.astype(np.float32)


def augment_smooth(s, rng_, bank, sigma):
    """= augment (section 6a) with the same random sequence, plus ball smoothing before the gaps are applied"""
    s = dict(s)
    T = len(s["label"])
    P_ = s["P"] + smooth_noise(rng_, s["P"].shape, 0.2)
    bl = s["ball"] + smooth_noise(rng_, s["ball"].shape, 0.4)
    for _ in range(rng_.poisson(T / 300)):
        a = rng_.integers(0, T)
        bl[a:min(T, a + rng_.integers(5, 25))] += rng_.normal(0, 2.5, 2).astype(np.float32)
    bl = smooth_ball(bl, sigma)
    if bank is not None:
        m = np.resize(np.roll(bank, rng_.integers(0, len(bank))), T)
        bl[~m] = np.nan
    for j in range(P_.shape[1]):
        for _ in range(rng_.poisson(T * 0.06 / 50)):
            a = rng_.integers(0, T)
            P_[a:a + rng_.integers(10, 90), j] = np.nan
    s["P"], s["ball"] = P_, bl
    return s


def make_copy_smooth(s, rng_, bank, sigma, swap=False):
    sa = augment_smooth(s, rng_, bank, sigma)
    X, lab = features(sa, swap=swap)
    return (X, set_features(sa, swap=swap)), lab


def _lean(copy_):
    (X_, S_), lab_ = copy_
    return (X_.astype(np.float16), S_), lab_


def _load_ext_cv(n):
    if n.startswith("sc"):
        return load_skillcorner(n[2:])
    return load_metrica3() if n == "me3" else load_metrica_csv(int(n[2:]))


SIGMA_SMOOTH = 0.0
SMOOTH_PASS = False
MODEL_USED = "base"
if SMOOTH_MODE != "off":
    g_idx = np.flatnonzero(GRADED)
    BALL_BANK = (z["ball_src"] > 0)[g_idx[np.round(np.arange(0, len(g_idx) - 1, 2.5)).astype(int)]]
    l_idx = np.flatnonzero(GRADED & ~d_mask)
    BANK_LIVE = (z["ball_src"] > 0)[l_idx[np.round(np.arange(0, len(l_idx) - 1, 2.5)).astype(int)]]
    with timer("load external data (cache/external)"):
        EXT10_CV = {}
        for n in NAMES_CV:
            EXT10_CV[n] = resample10(_load_ext_cv(n))
            gc.collect()
    # sanity: σ = 0 must equal the old augmentation (so that base vs smoothed-ball stay paired)
    _a = augment(EXT10_CV[NAMES_CV[0]], np.random.default_rng(5), BALL_BANK)
    _b = augment_smooth(EXT10_CV[NAMES_CV[0]], np.random.default_rng(5), BALL_BANK, 0.0)
    check("augment_smooth(σ = 0) identical to the base augment", np.array_equal(_a["ball"], _b["ball"], equal_nan=True)
          and np.array_equal(_a["P"], _b["P"], equal_nan=True))
    del _a, _b

    live10_ = GRADED[bar10["idx"]] & ~d_mask[bar10["idx"]]
    XBl_ = XB_H[0][live10_]
    SIG_BAR = XBl_[XBl_[:, 0] > 0.5][:, SIG_IDX].astype(np.float64)
    bar_sd = SIG_BAR.std(0) + 1e-9
    SIG_EXT, sig_rows = {}, []
    with timer("match the smoothing σ ({} values × {} matches, features only)".format(len(SMOOTH_SIGMAS), len(NAMES_CV))):
        for sg_ in SMOOTH_SIGMAS:
            parts_, src_ = [], []
            for i_, n in enumerate(NAMES_CV):
                sa_ = augment_smooth(EXT10_CV[n], np.random.default_rng(1000 + i_), BALL_BANK, sg_)
                X_, lab_ = features(sa_)
                keep_ = (lab_ >= 0) & (X_[:, 0] > 0.5)
                parts_.append(X_[keep_][:, SIG_IDX])
                src_.append(np.full(int(keep_.sum()), n[:2]))
            E_ = np.concatenate(parts_).astype(np.float64)
            src_ = np.concatenate(src_)
            SIG_EXT[sg_] = (E_, src_)
            w_ = [wasserstein_distance(SIG_BAR[:, j], E_[:, j]) / bar_sd[j] for j in range(len(SIG_IDX))]
            sig_rows.append(dict({"σ (langkah 10 fps)": sg_, "σ (s)": sg_ / FPS10}, **{"W " + f_: w_[j] for j, f_ in
                                                                                       enumerate(SIG_FEATS)}, total=sum(w_)))
    SIG_TAB = pd.DataFrame(sig_rows)
    print("ball-motion distribution distance BAR-ATH vs external (Wasserstein / BAR-ATH sd; small = similar):")
    print(SIG_TAB.round(3).to_string(index=False))
    sig_fit = float(SIG_TAB.loc[SIG_TAB.total.idxmin(), "σ (langkah 10 fps)"])
    SIGMA_SMOOTH = float(SMOOTH_SIGMA) if SMOOTH_SIGMA is not None else sig_fit
    q_ = [10, 50, 90, 99]
    pq_ = pd.DataFrame({"BAR-ATH": [np.percentile(SIG_BAR[:, j], q_) for j in range(len(SIG_IDX))],
                        "eksternal σ = 0": [np.percentile(SIG_EXT[0.0][0][:, j], q_) for j in range(len(SIG_IDX))],
                        "eksternal σ*": [np.percentile(SIG_EXT[sig_fit][0][:, j], q_) for j in range(len(SIG_IDX))]},
                       index=SIG_FEATS).map(lambda v: "/".join("{:.2f}".format(x) for x in v))
    print("\npercentiles p10/p50/p90/p99 (model features, live frames, ball visible):")
    print(pq_.to_string())
    for s_ in ("sc", "me"):
        m_ = SIG_EXT[0.0][1] == s_
        print("  external σ = 0, {}: turn p90 {:.3f} · acceleration p50 {:.3f}".format(
            "SkillCorner" if s_ == "sc" else "Metrica", np.percentile(SIG_EXT[0.0][0][m_, 2], 90),
            np.percentile(SIG_EXT[0.0][0][m_, 1], 50)))
    print("σ* (label-free match) = {:.2f} steps = {:.3f} s{}".format(
        sig_fit, sig_fit / FPS10, "" if SMOOTH_SIGMA is None else " · forced SMOOTH_SIGMA = {}".format(SMOOTH_SIGMA)))
    SIG_EXT = {k_: (v_[0].astype(np.float32), v_[1]) for k_, v_ in SIG_EXT.items() if k_ in (0.0, sig_fit)}   # save RAM
    del XBl_, E_, parts_, src_
    gc.collect()
    if SIGMA_SMOOTH <= 0:
        print("σ = 0: no smoothing makes the training ball closer to BAR-ATH → section 6d stops, base weights")
        SMOOTH_MODE = "off"

In [ ]:
# 6d.2  5-fold × 23-match cross-validation: base vs smoothed-ball augmentation (paired), 4 evaluation conditions
COND_CV = {"bola halus + celah ala BAR-ATH": (1000, "bank", True), "celah ala BAR-ATH": (1000, "bank", False),
          "celah bola hidup BAR-ATH": (3000, "live", False), "bola terlihat": (2000, None, False)}
COND_CV_MAIN = "bola halus + celah ala BAR-ATH"
VARIANTS_SMOOTH = ["base", "smooth"]
CV_SMOOTH_P, CV_SMOOTH_KEY_P = SMOOTH_DIR / "cv_smooth.npz", SMOOTH_DIR / "cv_smooth_key.json"
CV_SMOOTH_KEY = {"archs": list(SMOOTH_ARCHS), "folds": N_FOLDS, "seed": SEED, "feature_version": FEATURE_VERSION,
            "epochs": EPOCHS, "win": WIN, "batch": BATCH, "n_aug": N_AUG, "mix": MIX, "spell_w": SPELL_W,
            "sigma": SIGMA_SMOOTH, "conds": list(COND_CV), "names": NAMES_CV, "x_dtype": "float16", "versi": 1,
            "ball": hashlib.sha256(np.packbits(z["ball_src"] > 0).tobytes()).hexdigest()[:16],
            "dead": hashlib.sha256(np.packbits(d_mask).tobytes()).hexdigest()[:16]}
OOF_CV = {}


def sigma_copies(variant, rs_):
    """σ for every augmented copy: base = 0; smoothed-ball = U(0; 1.5 σ*) to be robust to a misestimated σ*"""
    return [0.0 if variant == "base" else float(rs_.uniform(0, 1.5 * SIGMA_SMOOTH)) for _ in range(N_AUG)]


def build_data_cv(variant, names):
    rng_ = np.random.default_rng(SEED)
    rs_ = np.random.default_rng(SEED + 14)
    out = {}
    for n in names:
        sg_ = sigma_copies(variant, rs_)
        out[n] = [_lean(make_copy_smooth(EXT10_CV[n], rng_, None if (MIX and (k // 2) % 2 == 1) else BALL_BANK, sg_[k],
                                      swap=k % 2 == 1)) for k in range(N_AUG)]
    return out


def eval_copy_cv(n, cond):
    seed_, bank_, smooth_ = COND_CV[cond]
    bank_ = BALL_BANK if bank_ == "bank" else (BANK_LIVE if bank_ == "live" else None)
    return make_copy_smooth(EXT10_CV[n], np.random.default_rng(seed_ + NAMES_CV.index(n)), bank_, SIGMA_SMOOTH if smooth_ else 0.0)


if SMOOTH_MODE == "auto" and CV_SMOOTH_P.is_file() and CV_SMOOTH_KEY_P.is_file() and json.loads(CV_SMOOTH_KEY_P.read_text()) == CV_SMOOTH_KEY:
    with np.load(CV_SMOOTH_P) as zf:
        for v_ in VARIANTS_SMOOTH:
            for n in NAMES_CV:
                for ci_, c_ in enumerate(COND_CV):
                    OOF_CV[(v_, n, c_)] = (zf["lg_{}_{}_{}".format(v_, n, ci_)], zf["lab_{}_{}".format(n, ci_)])
    print("smoothed-ball cross-validation loaded from cache: {} variants × {} matches × {} conditions".format(
        len(VARIANTS_SMOOTH), len(NAMES_CV), len(COND_CV)))
elif SMOOTH_MODE == "auto":
    print("smoothed-ball cross-validation cache missing / parameters changed → train {} variants × {} folds × {} on {}".format(
        len(VARIANTS_SMOOTH), N_FOLDS, "+".join(SMOOTH_ARCHS), TRAIN_DEV))
    folds_cv = [NAMES_CV[i::N_FOLDS] for i in range(N_FOLDS)]
    for v_ in VARIANTS_SMOOTH:
        globals().pop("DATA", None)
        gc.collect()
        with timer("features + external augmentation ({})".format(v_)):
            DATA = build_data_cv(v_, NAMES_CV)
            gc.collect()
        print("kernel process RAM: {:.1f} GB · free system RAM: {:.1f} GB".format(
            psutil.Process().memory_info().rss / 1e9, psutil.virtual_memory().available / 1e9))
        with timer("cross-validation {}".format(v_)):
            for k, held in enumerate(folds_cv):
                t0_ = time.time()
                nets_ = [train_model(arch, SEED, [n for n in NAMES_CV if n not in held]) for arch in SMOOTH_ARCHS]
                for n in held:
                    for c_ in COND_CV:
                        XS_, lab_ = eval_copy_cv(n, c_)
                        lg_ = np.mean([predict(net, XS_, TRAIN_DEV) for net in nets_], 0)
                        OOF_CV[(v_, n, c_)] = (lg_.astype(np.float32), np.asarray(lab_).astype(np.int8))
                        del XS_
                del nets_
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print("  {} fold {}/{} done ({:.0f} s)".format(v_, k + 1, N_FOLDS, time.time() - t0_), flush=True)
    del DATA
    gc.collect()
    np.savez_compressed(CV_SMOOTH_P, **{"lg_{}_{}_{}".format(v_, n, list(COND_CV).index(c_)): val_[0]
                                   for (v_, n, c_), val_ in OOF_CV.items()},
                        **{"lab_{}_{}".format(n, list(COND_CV).index(c_)): val_[1]
                           for (v_, n, c_), val_ in OOF_CV.items() if v_ == "base"})
    CV_SMOOTH_KEY_P.write_text(json.dumps(CV_SMOOTH_KEY))
    print("saved in", CV_SMOOTH_P.relative_to(ROOT))


def short_recall(pred, lab, max_s=2.0, iou=0.3):
    """share of GT possessions shorter than max_s that have a same-team predicted match with time IoU ≥ iou"""
    gl_ = np.where(lab == 1, "H", np.where(lab == 0, "A", "D"))
    hit_, tot_ = 0, 0
    for t_ in ("H", "A"):
        gs_, ge_ = runs(gl_ == t_)
        ps_, pe_ = runs(np.asarray(pred) == t_)
        for a_, b_ in zip(gs_, ge_):
            if (b_ - a_) / FPS10 >= max_s:
                continue
            tot_ += 1
            j_ = np.flatnonzero((ps_ < b_) & (pe_ > a_))
            if len(j_):
                inter_ = np.minimum(pe_[j_], b_) - np.maximum(ps_[j_], a_)
                union_ = np.maximum(pe_[j_], b_) - np.minimum(ps_[j_], a_)
                hit_ += int((inter_ / union_ >= iou).any())
    return hit_, tot_


if SMOOTH_MODE == "auto" and len(OOF_CV) == len(VARIANTS_SMOOTH) * len(NAMES_CV) * len(COND_CV):
    rows_smooth = []
    for (v_, n, c_), (lg_, lab_) in OOF_CV.items():
        pred_ = decode_ext(lg_, lab_, COST)
        h_, t_ = short_recall(pred_, lab_)
        rows_smooth.append({"varian": v_, "match": n, "kondisi": c_, "spellF1": spell_ext(pred_, lab_), "pendek_hit": h_,
                       "pendek_n": t_})
    CV_SMOOTH = pd.DataFrame(rows_smooth)
    piv_ = CV_SMOOTH.pivot_table(index=["kondisi", "match"], columns="varian", values="spellF1")
    piv_["Δ"] = piv_["smooth"] - piv_["base"]
    rb_ = np.random.default_rng(14)
    sum_rows = []
    for c_ in COND_CV:
        d_ = piv_.loc[c_, "Δ"].to_numpy()
        boot_ = np.array([rb_.choice(d_, len(d_)).mean() for _ in range(2000)])
        sh_ = CV_SMOOTH[CV_SMOOTH.kondisi == c_].groupby("varian")[["pendek_hit", "pendek_n"]].sum()
        sum_rows.append({"kondisi": c_, "base": piv_.loc[c_, "base"].mean(), "smooth": piv_.loc[c_, "smooth"].mean(),
                         "Δ": d_.mean(), "5%": np.percentile(boot_, 5), "95%": np.percentile(boot_, 95),
                         "matches won": "{}/{}".format(int((d_ > 0).sum()), len(d_)),
                         "short-possession recall base": sh_.loc["base", "pendek_hit"] / max(sh_.loc["base", "pendek_n"], 1),
                         "short-possession recall smoothed-ball": sh_.loc["smooth", "pendek_hit"] / max(sh_.loc["smooth", "pendek_n"], 1)})
    SUM_SMOOTH = pd.DataFrame(sum_rows)
    print("external leave-match-out spellF1 (mean of 23 matches), Δ = smoothed-ball − base, bootstrap interval over matches:")
    print(SUM_SMOOTH.round(4).to_string(index=False))
    main_ = SUM_SMOOTH.set_index("kondisi").loc[COND_CV_MAIN]
    worst_other = SUM_SMOOTH[SUM_SMOOTH.kondisi != COND_CV_MAIN]["Δ"].min()
    SMOOTH_PASS = bool(main_["Δ"] > 0 and main_["5%"] > 0 and worst_other >= -0.005)
    print("\ndecision rule: main Δ {:+.4f} [{:+.4f}; {:+.4f}] · worst Δ in the other conditions {:+.4f} → {}".format(
        main_["Δ"], main_["5%"], main_["95%"], worst_other, "PASS" if SMOOTH_PASS else "does not pass"))

#### Figure: ball motion and cross-validation result

Left: distribution of the `bola: belokan arah` feature (ball turn, the sign of a touch) on BAR-ATH and on the training
balls before and after smoothing with σ\*. Right: Δ spellF1 per match in the main condition (dots right of zero = smoothed-ball is
better on that match).

In [ ]:
if SMOOTH_MODE != "off" and "SIG_EXT" in globals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    ax = axes[0]
    j_ = SIG_FEATS.index("bola: belokan arah")
    xs_ = np.linspace(0, 0.6, 200)
    for lab_, vals_, col_, lw_ in (("BAR-ATH", SIG_BAR[:, j_], COL["text"], 2.2),
                                   ("training σ = 0", SIG_EXT[0.0][0][:, j_], COL["D"], 2.0),
                                   ("training σ* = {:.2f}".format(sig_fit), SIG_EXT[sig_fit][0][:, j_], COL["violet"], 2.0)):
        ax.plot(xs_, np.searchsorted(np.sort(vals_), xs_, side="right") / len(vals_), color=col_, linewidth=lw_, label=lab_)
    ax.set_xlabel("ball: turn (1 − cos of the angle)")
    ax.set_ylabel("share of frames ≤ x")
    ax.set_title("The BAR-ATH ball is smoother", pad=10)
    ax.legend(loc="lower right")
    ax = axes[1]
    if "piv_" in globals():
        d_ = piv_.loc[COND_CV_MAIN, "Δ"].sort_values()
        ax.scatter(d_.to_numpy(), np.arange(len(d_)), s=36, color=COL["aqua"], zorder=3)
        ax.axvline(0, color=COL["muted"], linewidth=1)
        ax.set_yticks(np.arange(len(d_)))
        ax.set_yticklabels(d_.index, fontsize=7)
        ax.set_xlabel("Δ spellF1 (smoothed-ball − base)")
        ax.set_title("Per match, main condition", pad=10)
    else:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

#### 6d.3 The model that is used

The smoothed-ball model is trained on the 23 matches with smoothed training balls and used in sections 7 to 9 **only if 6d.2
passes** (or when forced with `SMOOTH_STAGE = "force"`). The base weights stay stored and are used for comparison.

In [ ]:
# 6d.3  final smoothed-ball model (2 seeds × 3 architectures, 23 matches) — only if it passes (or SMOOTH_STAGE = "force")
MODEL_DIR_SMOOTH = NB_DIR / "models_smooth"
META_SMOOTH_P = MODEL_DIR_SMOOTH / "meta.json"
USE_SMOOTH = SMOOTH_MODE == "force" or (SMOOTH_MODE == "auto" and SMOOTH_PASS)
NETS_BASE = NETS
if USE_SMOOTH:
    key_smooth_ = {"sigma": SIGMA_SMOOTH, "n_aug": N_AUG, "mix": MIX, "spell_w": SPELL_W, "epochs": EPOCHS, "win": WIN, "batch": BATCH,
              "seed": SEED, "models": [list(m_) for m_ in MODELS], "matches": NAMES_CV, "feature_version": FEATURE_VERSION,
              "x_dtype": "float16", "versi": 1}
    meta_smooth_ = json.loads(META_SMOOTH_P.read_text(encoding="utf-8")) if META_SMOOTH_P.is_file() else {}
    if meta_smooth_.get("key") != key_smooth_ or not all((MODEL_DIR_SMOOTH / m_["file"]).is_file() for m_ in meta_smooth_.get("models", [])):
        MODEL_DIR_SMOOTH.mkdir(parents=True, exist_ok=True)
        globals().pop("DATA", None)
        gc.collect()
        with timer("features + smoothed-ball augmentation (23 matches)"):
            DATA = build_data_cv("smooth", NAMES_CV)
        models_ = []
        with timer("train the 6 final smoothed-ball models"):
            for arch, seed in MODELS:
                net = train_model(arch, seed, NAMES_CV)
                p_ = MODEL_DIR_SMOOTH / "{}_s{}.pt".format(arch, seed)
                torch.save({k: v.cpu() for k, v in net.state_dict().items()}, p_)
                models_.append({"arch": arch, "seed": seed, "file": p_.name, "sha256": sha256(p_)})
                del net
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        del DATA
        gc.collect()
        meta_smooth_ = {"version": "smooth", "key": key_smooth_, "models": models_, "feature_version": FEATURE_VERSION, "n_feat": N_FEAT,
                   "set_k": SET_K, "decoder_cost": COST, "device": TRAIN_DEV, "torch": torch.__version__,
                   "cv_pass": SMOOTH_PASS, "cv": SUM_SMOOTH.to_dict("records") if "SUM14" in globals() else None}
        META_SMOOTH_P.write_text(json.dumps(meta_smooth_, indent=2, ensure_ascii=False), encoding="utf-8")
        print("smoothed-ball weights saved in", MODEL_DIR_SMOOTH.relative_to(ROOT))
    NETS = []
    for m_ in meta_smooth_["models"]:
        p_ = MODEL_DIR_SMOOTH / m_["file"]
        check("sha256 smoothed-ball {}".format(m_["file"]), sha256(p_) == m_["sha256"])
        net = build_net(m_["arch"])
        net.load_state_dict(torch.load(p_, map_location="cpu", weights_only=True))
        NETS.append(net.eval())
    MODEL_USED = "smooth"
    dev_ = INFER_DEVICE if (INFER_DEVICE == "cpu" or torch.cuda.is_available()) else "cpu"
    lg_base_ = np.mean([0.5 * (predict(n_, XB_H, dev_) - predict(n_, XB_A, dev_)) for n_ in NETS_BASE], 0)
    lg_smooth_ = np.mean([0.5 * (predict(n_, XB_H, dev_) - predict(n_, XB_A, dev_)) for n_ in NETS], 0)
    live10_ = GRADED[bar10["idx"]] & ~d_mask[bar10["idx"]]
    print("BAR-ATH live frames (10 fps): log-odds sign differs base → smoothed-ball {:.2%} · median |log-odds| {:.2f} → {:.2f}".format(
        float((np.sign(lg_base_[live10_]) != np.sign(lg_smooth_[live10_])).mean()), float(np.median(np.abs(lg_base_[live10_]))),
        float(np.median(np.abs(lg_smooth_[live10_])))))
print("possession model used for BAR-ATH: {} ({} models)".format(MODEL_USED, len(NETS)))

### 6e. "Video method": team of the player nearest to the ball — *rejected, off by default*

> This experiment did not pass its decision rule (log #99). With `NEAR_STAGE = "off"` (the default) it is skipped.

**Origin.** A video the team reviewed (an Arsenal–Chelsea broadcast): YOLO detects players, goalkeepers, referees and
the ball; pitch keypoints give a homography to a 2D 105 × 68 m tactical map; teams come from kit colours; then a *convex
hull* and team-shape metrics (length, width, defensive line, block height) plus **possession percentage: the team of
the player nearest to the ball**. In BAR-ATH the player and ball positions in metres already exist from the tracking
(and our own YOLO ball, section 2b), so what is tested here is the **possession rule** itself.

| Variant | Rule |
|---|---|
| pure | the player nearest to the ball within radius R (and ≥ 0.5 m closer than the nearest opponent) → their team has the ball; no claim → the last team is kept |
| evidence | model log-odds + β × (+1 / −1 / 0 from that rule), either *instantaneous* or *held ≤ 1 s* until the next claim, then the usual Viterbi with cost 10 |

**Validation.** The 23 external matches, the 4 ball conditions from section 6d, the **out-of-fold** log-odds of the base
model (`cache/smooth/`), with augmented ball & players *identical* to what the model saw (same random seeds). Grid
R ∈ {1; 1.5; 2; 3} m × β ∈ {0.5; 1; 2; 4} × 2 shapes = 32 configurations, chosen **nested** (5 folds over matches) in
the main condition.

**Decision rule (locked before the run).** Used for BAR-ATH if the nested Δ (evidence − base) in the main condition is
> 0 with a 5% bootstrap lower bound > 0, and no other condition has a mean nested Δ < −0.002. The pure variant is only
reported (as a comparison). The public GT is not used.

Related history: the "possession zone" / nearest-player rule had already been tested as a replacement for the model
(log #13, #14: team accuracy ± 0.80–0.88 vs 0.92 for the team-shape model); what is new here is its use as **evidence
on top of the sequence model** and the nested test with out-of-fold log-odds.

In [ ]:
# 6e.1  nearest-player rule on the 23 external matches: alone, and as evidence on top of the base OOF log-odds
NEAR_MODE = str(NEAR_STAGE).strip().lower()
assert NEAR_MODE in ("auto", "off", "force"), NEAR_STAGE
NEAR_RS = [1.0, 1.5, 2.0, 3.0]
NEAR_BETAS = [0.5, 1.0, 2.0, 4.0]
NEAR_FORMS = {"sesaat": 0, "tahan 1 s": 10}          # 10 fps steps
NEAR_MARGIN_M = 0.5
NEAR_CFGS = [(r_, b_, f_) for r_ in NEAR_RS for b_ in NEAR_BETAS for f_ in NEAR_FORMS]
NEAR_PASS, NEAR_CFG = False, None


def cfg_key(c_):
    return "R {} · β {} · {}".format(*c_)


def nearest_team(P_, home_, ball_, r_m, margin=NEAR_MARGIN_M):
    """+1 = home player nearest within r_m (and closer than away by ≥ margin), −1 = away, 0 = no claim"""
    d_ = np.linalg.norm(P_ - ball_[:, None, :], axis=-1)
    d_ = np.where(np.isfinite(d_), d_, np.inf)
    dh_, da_ = d_[:, home_].min(1), d_[:, ~home_].min(1)
    s_ = np.zeros(len(d_), np.int8)
    s_[(dh_ <= r_m) & (dh_ + margin < da_)] = 1
    s_[(da_ <= r_m) & (da_ + margin < dh_)] = -1
    s_[~np.isfinite(ball_[:, 0])] = 0
    return s_


def hold_ev(s_, n_):
    """evidence held until the next claim, for at most n_ steps"""
    if n_ <= 0:
        return s_.astype(np.float32)
    idx_ = np.maximum.accumulate(np.where(s_ != 0, np.arange(len(s_)), -1))
    age_ = np.arange(len(s_)) - idx_
    return np.where((idx_ >= 0) & (age_ <= n_), s_[np.maximum(idx_, 0)], 0).astype(np.float32)


def video_method_labels(s_, lab_):
    """pure video method: the last team claimed by the nearest player; dead ball taken from the labels (same as the decoder)"""
    idx_ = np.maximum.accumulate(np.where(s_ != 0, np.arange(len(s_)), -1))
    team_ = np.where(idx_ >= 0, s_[np.maximum(idx_, 0)], 1)
    out_ = np.where(team_ > 0, "H", "A").astype(object)
    out_[lab_ < 0] = "D"
    return out_


have_oof_near = "OOF14" in globals() and len(OOF_CV) == len(VARIANTS_SMOOTH) * len(NAMES_CV) * len(COND_CV)
if NEAR_MODE == "auto" and not have_oof_near:
    print("OOF log-odds of section 6d not available (SMOOTH_STAGE = \"off\"?) → 6e simulation skipped, video method not used")
if NEAR_MODE == "auto" and have_oof_near:
    rows_near, pure_near, mism_ = [], [], 0
    with timer("video-method simulation ({} conditions × {} matches × {} configurations)".format(len(COND_CV), len(NAMES_CV),
                                                                                       len(NEAR_CFGS))):
        for c_, (seed_, bank_, smooth_) in COND_CV.items():
            bank_ = BALL_BANK if bank_ == "bank" else (BANK_LIVE if bank_ == "live" else None)
            for i_, n in enumerate(NAMES_CV):
                sa_ = augment_smooth(EXT10_CV[n], np.random.default_rng(seed_ + i_), bank_, SIGMA_SMOOTH if smooth_ else 0.0)
                lg_, lab_ = OOF_CV[("base", n, c_)]
                if not np.array_equal(np.asarray(lab_, np.int8), sa_["label"].astype(np.int8)):
                    mism_ += 1
                    continue
                home_ = sa_["home"].astype(bool)
                live_ = lab_ >= 0
                row_ = {"kondisi": c_, "match": n, "base": spell_ext(decode_ext(lg_, lab_, COST), lab_)}
                for r_ in NEAR_RS:
                    s_ = nearest_team(sa_["P"], home_, sa_["ball"], r_)
                    said_ = (s_ != 0) & live_
                    pure_near.append({"kondisi": c_, "match": n, "R (m)": r_,
                                   "spellF1 murni": spell_ext(video_method_labels(s_, lab_), lab_),
                                   "cakupan": float(said_.mean() / max(live_.mean(), 1e-9)),
                                   "accuracy when claiming": float(((s_ == 1) == (lab_ == 1))[said_].mean())
                                   if said_.any() else np.nan})
                    for f_, nh_ in NEAR_FORMS.items():
                        ev_ = hold_ev(s_, nh_)
                        for b_ in NEAR_BETAS:
                            row_[cfg_key((r_, b_, f_))] = spell_ext(decode_ext(lg_ + b_ * ev_, lab_, COST), lab_)
                rows_near.append(row_)
    check("6e augmentation seeds = 6d OOF log-odds (identical labels)", mism_ == 0, "{} mismatched pairs".format(mism_))
    SIM_NEAR = pd.DataFrame(rows_near)
    PURE_NEAR = pd.DataFrame(pure_near)
    print("PURE video method (no model), mean of 23 matches:")
    pt_ = PURE_NEAR.groupby(["kondisi", "R (m)"])[["spellF1 murni", "cakupan", "accuracy when claiming"]].mean().unstack(0)
    print(pt_["spellF1 murni"].round(3).to_string())
    print("base (model) as the reference:", SIM_NEAR.groupby("kondisi")["base"].mean().round(3).to_dict())
    print("coverage (share of live frames with a claim) / team accuracy when claiming, main condition:")
    print(PURE_NEAR[PURE_NEAR.kondisi == COND_CV_MAIN].groupby("R (m)")[["cakupan", "accuracy when claiming"]].mean()
          .round(3).to_string())

    dlt_ = {c_: (SIM_NEAR[cfg_key(c_)] - SIM_NEAR["base"]).to_numpy() for c_ in NEAR_CFGS}
    main_m = (SIM_NEAR.kondisi == COND_CV_MAIN).to_numpy()
    insample_ = pd.DataFrame({"R": [c_[0] for c_ in NEAR_CFGS], "β": [c_[1] for c_ in NEAR_CFGS],
                              "bentuk": [c_[2] for c_ in NEAR_CFGS]})
    for k_ in COND_CV:
        m_ = (SIM_NEAR.kondisi == k_).to_numpy()
        insample_[k_[:24]] = [dlt_[c_][m_].mean() for c_ in NEAR_CFGS]
    insample_ = insample_.sort_values(COND_CV_MAIN[:24], ascending=False)
    print("\nin-sample Δ spellF1 (evidence − base), top 8 configurations in the main condition:")
    print(insample_.head(8).round(4).to_string(index=False))

    nest_ = {k_: [] for k_ in COND_CV}
    picks_ = []
    for held in [NAMES_CV[i::N_FOLDS] for i in range(N_FOLDS)]:
        tr_ = main_m & ~SIM_NEAR.match.isin(held).to_numpy()
        best_ = max(NEAR_CFGS, key=lambda c_: dlt_[c_][tr_].mean())
        picks_.append(best_)
        for k_ in COND_CV:
            m_ = ((SIM_NEAR.kondisi == k_) & SIM_NEAR.match.isin(held)).to_numpy()
            nest_[k_] += list(dlt_[best_][m_])
    rb_ = np.random.default_rng(15)
    nsum_ = []
    for k_ in COND_CV:
        d_ = np.asarray(nest_[k_])
        bt_ = np.array([rb_.choice(d_, len(d_)).mean() for _ in range(2000)])
        nsum_.append({"kondisi": k_, "Δ bersarang": d_.mean(), "5%": np.percentile(bt_, 5), "95%": np.percentile(bt_, 95),
                      "matches won": "{}/{}".format(int((d_ > 0).sum()), len(d_))})
    NEST_NEAR = pd.DataFrame(nsum_)
    print("\nchoice per fold:", [("R {} · β {} · {}".format(*p_)) for p_ in picks_])
    print(NEST_NEAR.round(4).to_string(index=False))
    r0_ = NEST_NEAR.set_index("kondisi").loc[COND_CV_MAIN]
    worst_near = NEST_NEAR[NEST_NEAR.kondisi != COND_CV_MAIN]["Δ bersarang"].min()
    NEAR_PASS = bool(r0_["Δ bersarang"] > 0 and r0_["5%"] > 0 and worst_near >= -0.002)
    NEAR_CFG = max(NEAR_CFGS, key=lambda c_: dlt_[c_][main_m].mean())
    print("\ndecision rule: main nested Δ {:+.4f} [{:+.4f}; {:+.4f}] · worst in the other conditions {:+.4f} → {} "
          "(configuration over all 23 matches: R {} m · β {} · {})".format(r0_["Δ bersarang"], r0_["5%"], r0_["95%"], worst_near,
                                                                  "PASS" if NEAR_PASS else "does not pass", *NEAR_CFG))

#### Figure: pure video method vs model

spellF1 of the pure nearest-player method for each radius, compared with the base model, in the main condition (mean of
23 matches).

In [ ]:
if "PURE15" in globals() and len(PURE_NEAR):
    fig, ax = plt.subplots(figsize=(7, 3.4))
    pm_ = PURE_NEAR[PURE_NEAR.kondisi == COND_CV_MAIN].groupby("R (m)")["spellF1 murni"].mean()
    ax.plot(pm_.index, pm_.values, marker="o", markersize=7, color=COL["violet"], linewidth=2, label="nearest player (pure)")
    base_mean_ = SIM_NEAR[SIM_NEAR.kondisi == COND_CV_MAIN]["base"].mean()
    ax.axhline(base_mean_, color=COL["text"], linewidth=1.5, linestyle="--", label="base model = {:.3f}".format(base_mean_))
    ax.set_xlabel("ball-holder radius R (m)")
    ax.set_ylabel("spellF1 (23 matches)")
    ax.set_ylim(0, 1)
    ax.set_title("Nearest-player rule vs model", pad=10)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 6e.2  BAR-ATH: nearest-player evidence (10 fps) + video-style team-shape metrics (descriptive)
NEAR_USE = NEAR_MODE == "force" or (NEAR_MODE == "auto" and NEAR_PASS)
if NEAR_MODE == "force":
    NEAR_CFG = tuple(NEAR_FORCE_CFG) if NEAR_FORCE_CFG is not None else (NEAR_CFG or (1.5, 2.0, "tahan 1 s"))
live10_ = GRADED[bar10["idx"]] & ~d_mask[bar10["idx"]]
home10_ = bar10["home"].astype(bool)
s_bar = nearest_team(bar10["P"], home10_, bar10["ball"], NEAR_CFG[0] if NEAR_CFG else 1.5)
print("BAR-ATH live frames at 10 fps: the nearest player claims a team on {:.1%} of frames (R {} m) · H {:.1%} / A {:.1%} of claims".format(
    float((s_bar != 0)[live10_].mean()), NEAR_CFG[0] if NEAR_CFG else 1.5,
    float((s_bar == 1)[live10_ & (s_bar != 0)].mean()), float((s_bar == -1)[live10_ & (s_bar != 0)].mean())))
EVID10_BAR = np.zeros(len(s_bar), np.float32)
if NEAR_USE:
    EVID10_BAR = NEAR_CFG[1] * hold_ev(s_bar, NEAR_FORMS[NEAR_CFG[2]])
print("video method used for BAR-ATH: {}{}".format("YA" if NEAR_USE else "no",
      " (R {} m · β {} · {})".format(*NEAR_CFG) if NEAR_USE else ""))


def team_shape(P_, m_, a_):
    """length, width, defensive line, block height (m) as in a video overlay; the goalkeeper (deepest player) is dropped"""
    x_, y_ = P_[:, m_, 0], P_[:, m_, 1]
    xg_ = np.where(a_[:, None] > 0, x_, PITCH_L - x_)          # distance from the own goal
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        srt_ = np.sort(np.where(np.isfinite(xg_), xg_, np.nan), 1)
        out_ = srt_[:, 1:]                                        # drop the goalkeeper
        yv_ = np.where(np.isfinite(xg_) & (xg_ > srt_[:, :1]), y_, np.nan)
        return {"panjang": np.nanmax(out_, 1) - np.nanmin(out_, 1), "lebar": np.nanmax(yv_, 1) - np.nanmin(yv_, 1),
                "garis pertahanan": np.nanmin(out_, 1), "tinggi blok": np.nanmax(out_, 1)}


att10_ = bar10["att"].astype(float)
shape_rows_ = []
for t_, m_, a_ in (("H", home10_, att10_), ("A", ~home10_, -att10_)):
    sh_ = team_shape(bar10["P"], m_, a_)
    shape_rows_.append({"tim": t_, **{k_: float(np.nanmedian(v_[live10_])) for k_, v_ in sh_.items()}})
print("\nvideo-style team-shape metrics, median over live frames of the whole match (m; descriptive, already covered by the model's 'shape' features):")
print(pd.DataFrame(shape_rows_).round(1).to_string(index=False))

### 6f. Stacking: base embeddings + video-method features → MLP — *rejected, off by default*

> This experiment did not pass its decision rule (log #102: −0.018 [−0.024; −0.013] against soft voting). With
> `STACK_STAGE = "off"` (the default) it is skipped.

**Idea.** The base output is one number per frame (log-odds). Inside every model there is an **embedding**: the vector
before the *classification head* (gru_set: 192-dimensional BiGRU output; tcn and tcn_set: 128-dimensional TCN block
output). The embeddings of the three architectures are concatenated (448 dimensions) with **15 "video method"
features** from the own team's viewpoint, and an **MLP** (464 → 128 → 64 → 1) is trained to predict possession.

| Video-method feature (per 10 fps step) | |
|---|---|
| distance of the nearest player to the ball: own team, opponent, difference | 3 |
| ball holder (nearest ≤ 1.5 m and ≥ 0.5 m closer than the opponent): +1 / −1 / 0, instantaneous and held ≤ 1 s | 2 |
| ball visible | 1 |
| length, width, defensive line, block height: own team and opponent | 8 |
| *convex hull* area own − opponent | 1 |

**Keeping it honest (no leakage).**

1. *Level 1:* 5 folds over matches. For each fold, the three architectures are trained without the held-out matches
   (augmentation = base). The held-out embeddings come from that model, so they are **out-of-fold**. Fold weights are
   cached.
2. *Level 2:* the MLP of fold k is trained on the out-of-fold embeddings of the matches outside fold k, then scored on
   fold k. The output is H↔A symmetric: z = ½ (MLP(H view) − MLP(A view)).
3. The scale λ (log-odds = λ·z, so that the Viterbi cost of 10 and the base touch evidence keep their meaning) is chosen
   **nested**.
4. Baseline: the 3-architecture base ensemble from the same fold models, with identical decoding. Ablation: the MLP
   **without** the video features.

**Decision rule (locked before the run).** Used for BAR-ATH if the nested Δ (combined MLP − base) in `celah ala BAR-ATH`
(BAR-ATH-like gaps) is > 0 with a 5% bootstrap lower bound > 0, and the nested Δ in `bola terlihat` (ball visible) is
≥ −0.002. The public GT is not used. Cost the first time: ± 65–75 GPU minutes (training 15 fold models); afterwards a
few minutes.

In [ ]:
# 6f.1  out-of-fold embeddings (level 1) + video-method features
import gc

import psutil

STACK_MODE = str(STACK_STAGE).strip().lower()
assert STACK_MODE in ("auto", "off", "force"), STACK_STAGE
STACK_DIR = CACHE / "stack"
STACK_DIR.mkdir(parents=True, exist_ok=True)
STACK_ARCHS = ["gru_set", "tcn", "tcn_set"]
STACK_CONDS = {"celah ala BAR-ATH": (1000, "bank"), "bola terlihat": (2000, None)}
STACK_MAIN = "celah ala BAR-ATH"
STACK_LAMBDAS = [0.5, 1.0, 2.0, 4.0, 8.0]
STACK_R_M = 1.5
NAMES_STACK = list(MODEL_META["training"]["matches"])
FOLDS_STACK = [NAMES_STACK[i::N_FOLDS] for i in range(N_FOLDS)]
STACK_PASS, STACK_USE, STACK_LAMBDA = False, False, None
VF_NAMES = ["jarak terdekat sendiri", "jarak terdekat lawan", "selisih jarak", "pemegang bola", "pemegang (tahan 1 s)",
            "bola terlihat", "panjang sendiri", "lebar sendiri", "garis pert. sendiri", "blok sendiri", "panjang lawan",
            "lebar lawan", "garis pert. lawan", "blok lawan", "luas hull sendiri − lawan"]


@torch.no_grad()
def embed_predict(net, XS, device, chunk=3000, overlap=300):
    """(log-odds, embedding before the classification head) per step; same chunking as predict()"""
    X, S = XS
    net = net.to(device).eval()
    box_ = {}
    hk_ = net.out.register_forward_hook(lambda m_, i_, o_: box_.__setitem__("h", i_[0]))
    T = len(X)
    lg, emb, a = np.zeros(T, np.float32), None, 0
    try:
        while a < T:
            lo_, hi_, b = max(0, a - overlap), min(T, a + chunk + overlap), min(T, a + chunk)
            o = net(torch.from_numpy(np.ascontiguousarray(X[lo_:hi_]).astype(np.float32)[None]).to(device),
                    torch.from_numpy(np.ascontiguousarray(S[lo_:hi_]).astype(np.float32)[None]).to(device))[0].float().cpu().numpy()
            h = box_["h"][0].float()
            h = (h.transpose(0, 1) if isinstance(net, TCN) else h).cpu().numpy()
            if emb is None:
                emb = np.zeros((T, h.shape[1]), np.float16)
            lg[a:b] = o[a - lo_:b - lo_]
            emb[a:b] = h[a - lo_:b - lo_]
            a = b
    finally:
        hk_.remove()
    return lg, emb


def _shape_team(P_, m_, a_, L_):
    x_, y_ = P_[:, m_, 0], P_[:, m_, 1]
    xg_ = np.where(a_[:, None] > 0, x_, L_ - x_)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        srt_ = np.sort(np.where(np.isfinite(xg_), xg_, np.nan), 1)[:, 1:]       # drop the deepest player (goalkeeper)
        yv_ = np.where(np.isfinite(xg_) & (xg_ > np.nanmin(xg_, 1, keepdims=True)), y_, np.nan)
        out_ = np.stack([np.nanmax(srt_, 1) - np.nanmin(srt_, 1), np.nanmax(yv_, 1) - np.nanmin(yv_, 1),
                         np.nanmin(srt_, 1), np.nanmax(srt_, 1)], 1)
    return np.nan_to_num(out_ / np.array([L_, 68.0, L_, L_]), nan=0.0).astype(np.float32)


def _hull_area(P_, m_):
    out_ = np.zeros(len(P_), np.float32)
    for t_ in range(len(P_)):
        q_ = P_[t_, m_]
        q_ = q_[np.isfinite(q_).all(1)]
        if len(q_) >= 3:
            out_[t_] = cv2.contourArea(cv2.convexHull(q_.astype(np.float32)))
    return out_


def video_base(sa):
    """per-team quantities (home/away) from the augmented copy (10 fps); combined per viewpoint in video_view"""
    P_, home_, b_ = sa["P"].astype(np.float32), sa["home"].astype(bool), sa["ball"].astype(np.float32)
    L_ = float(sa.get("L", PITCH_L))
    a_ = sa["att"].astype(float)
    d_ = np.linalg.norm(P_ - b_[:, None, :], axis=-1)
    d_ = np.where(np.isfinite(d_), d_, np.inf)
    return {"dh": d_[:, home_].min(1), "da": d_[:, ~home_].min(1), "vis": np.isfinite(b_[:, 0]),
            "sh": _shape_team(P_, home_, a_, L_), "sa": _shape_team(P_, ~home_, -a_, L_),
            "ah": _hull_area(P_, home_), "aa": _hull_area(P_, ~home_)}


def video_view(vb, swap=False):
    do_, dp_ = (vb["da"], vb["dh"]) if swap else (vb["dh"], vb["da"])
    so_, sp_ = (vb["sa"], vb["sh"]) if swap else (vb["sh"], vb["sa"])
    ao_, ap_ = (vb["aa"], vb["ah"]) if swap else (vb["ah"], vb["aa"])
    s_ = np.where((do_ <= STACK_R_M) & (do_ + 0.5 < dp_), 1, np.where((dp_ <= STACK_R_M) & (dp_ + 0.5 < do_), -1, 0))
    s_ = np.where(vb["vis"], s_, 0).astype(np.int8)
    cols_ = [np.clip(np.nan_to_num(do_, posinf=30), 0, 30) / 10, np.clip(np.nan_to_num(dp_, posinf=30), 0, 30) / 10,
             np.clip(np.nan_to_num(do_ - dp_, posinf=0, neginf=0), -30, 30) / 10, s_.astype(np.float32), hold_ev(s_, 10),
             vb["vis"].astype(np.float32)]
    out_ = np.concatenate([np.stack(cols_, 1), so_, sp_, ((ao_ - ap_) / 1000.0)[:, None]], 1)
    return np.nan_to_num(out_).astype(np.float32)


def views_stack(nets_, sa):
    """embeddings & log-odds of the three architectures from both viewpoints + video features, for one copy"""
    XH_, lab_ = features(sa)
    XA_, _ = features(sa, swap=True)
    SH_, SA_ = set_features(sa), set_features(sa, swap=True)
    eh_, ea_, lgs_ = [], [], []
    for arch in STACK_ARCHS:
        lh_, e1_ = embed_predict(nets_[arch], (XH_, SH_), TRAIN_DEV)
        la_, e2_ = embed_predict(nets_[arch], (XA_, SA_), TRAIN_DEV)
        eh_.append(e1_), ea_.append(e2_), lgs_.append(0.5 * (lh_ - la_))
    vb_ = video_base(sa)
    return {"eH": np.concatenate(eh_, 1), "eA": np.concatenate(ea_, 1), "vH": video_view(vb_), "vA": video_view(vb_, True),
            "lg12": np.mean(lgs_, 0).astype(np.float32), "lab": lab_}


def _eval_sa_stack(n, cond):
    seed_, bank_ = STACK_CONDS[cond]
    return augment_smooth(EXT10_STACK[n], np.random.default_rng(seed_ + NAMES_STACK.index(n)), BALL_BANK_STACK if bank_ else None, 0.0)


def _fold_path(k, arch):
    return STACK_DIR / "f{}_{}.pt".format(k, arch)


def load_fold_nets(k):
    out_ = {}
    for arch in STACK_ARCHS:
        net_ = build_net(arch)
        net_.load_state_dict(torch.load(_fold_path(k, arch), map_location="cpu", weights_only=True))
        out_[arch] = net_.eval()
    return out_


CV_STACK_P, CV_STACK_KEY_P = STACK_DIR / "rows_stack.npz", STACK_DIR / "rows_stack_key.json"
CV_STACK_KEY = {"archs": STACK_ARCHS, "folds": N_FOLDS, "seed": SEED, "feature_version": FEATURE_VERSION, "epochs": EPOCHS,
            "win": WIN, "batch": BATCH, "n_aug": N_AUG, "mix": MIX, "spell_w": SPELL_W, "names": NAMES_STACK,
            "conds": list(STACK_CONDS), "sub": STACK_SUB, "r_m": STACK_R_M, "vf": VF_NAMES, "versi": 1,
            "ball": hashlib.sha256(np.packbits(z["ball_src"] > 0).tobytes()).hexdigest()[:16]}
ROWS_STACK = None
if STACK_MODE != "off":
    g_idx = np.flatnonzero(GRADED)
    BALL_BANK_STACK = (z["ball_src"] > 0)[g_idx[np.round(np.arange(0, len(g_idx) - 1, 2.5)).astype(int)]]
    with timer("load external data (cache/external)"):
        EXT10_STACK = {}
        for n in NAMES_STACK:
            EXT10_STACK[n] = resample10(_load_ext_cv(n))
            gc.collect()
    have_ = (CV_STACK_P.is_file() and CV_STACK_KEY_P.is_file() and json.loads(CV_STACK_KEY_P.read_text()) == CV_STACK_KEY
             and all(_fold_path(k, a).is_file() for k in range(N_FOLDS) for a in STACK_ARCHS))
    if have_:
        with np.load(CV_STACK_P) as zf:
            ROWS_STACK = {k_: zf[k_] for k_ in zf.files}
        print("stacking out-of-fold embeddings loaded from cache: {} rows".format(len(ROWS_STACK["y"])))
    else:
        print("stacking cache missing / parameters changed → train {} folds × {} on {}, then extract embeddings".format(
            N_FOLDS, "+".join(STACK_ARCHS), TRAIN_DEV))
        globals().pop("DATA", None)
        gc.collect()
        with timer("features + external augmentation (= base)"):
            rng_ = np.random.default_rng(SEED)
            DATA = {n: [_lean(make_copy_smooth(EXT10_STACK[n], rng_, None if (MIX and (k // 2) % 2 == 1) else BALL_BANK_STACK, 0.0,
                                            swap=k % 2 == 1)) for k in range(N_AUG)] for n in NAMES_STACK}
            gc.collect()
        print("kernel process RAM: {:.1f} GB · free system RAM: {:.1f} GB".format(
            psutil.Process().memory_info().rss / 1e9, psutil.virtual_memory().available / 1e9))
        parts_ = {"e": [], "v": [], "y": [], "m": []}
        with timer("level 1: folds + out-of-fold embeddings"):
            for k, held in enumerate(FOLDS_STACK):
                t0_ = time.time()
                nets_ = {}
                for arch in STACK_ARCHS:
                    nets_[arch] = train_model(arch, SEED, [n for n in NAMES_STACK if n not in held])
                    torch.save({kk: v.cpu() for kk, v in nets_[arch].state_dict().items()}, _fold_path(k, arch))
                for n in held:
                    vw_ = views_stack(nets_, _eval_sa_stack(n, STACK_MAIN))
                    idx_ = np.flatnonzero(vw_["lab"] >= 0)[::STACK_SUB]
                    for e_, v_, y_ in ((vw_["eH"], vw_["vH"], vw_["lab"][idx_]), (vw_["eA"], vw_["vA"], 1 - vw_["lab"][idx_])):
                        parts_["e"].append(e_[idx_]), parts_["v"].append(v_[idx_].astype(np.float16))
                        parts_["y"].append(y_.astype(np.int8)), parts_["m"].append(np.full(len(idx_), NAMES_STACK.index(n), np.int16))
                    del vw_
                del nets_
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print("  fold {}/{} done ({:.0f} s)".format(k + 1, N_FOLDS, time.time() - t0_), flush=True)
        del DATA
        gc.collect()
        ROWS_STACK = {k_: np.concatenate(v_) for k_, v_ in parts_.items()}
        del parts_
        np.savez_compressed(CV_STACK_P, **ROWS_STACK)
        CV_STACK_KEY_P.write_text(json.dumps(CV_STACK_KEY))
        print("saved in {} ({:.0f} MB)".format(CV_STACK_P.relative_to(ROOT), CV_STACK_P.stat().st_size / 1e6))
    print("MLP training rows: {} · embedding dimension {} + video features {}".format(
        len(ROWS_STACK["y"]), ROWS_STACK["e"].shape[1], ROWS_STACK["v"].shape[1]))

In [ ]:
# 6f.2  level 2: MLP per fold (combined & without video features), scored nested against base
class FuseMLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d_in, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 64), nn.GELU(),
                               nn.Linear(64, 1))

    def forward(self, x):
        return self.f(x).squeeze(-1)


def fit_mlp(X_, y_, seed=0, epochs=None, bs=2048):
    torch.manual_seed(seed)
    r_ = np.random.default_rng(seed)
    mu_ = X_.astype(np.float32).mean(0)
    sd_ = X_.astype(np.float32).std(0) + 1e-3
    net_ = FuseMLP(X_.shape[1]).to(TRAIN_DEV)
    opt_ = torch.optim.AdamW(net_.parameters(), lr=1e-3, weight_decay=1e-4)
    mu_t, sd_t = torch.from_numpy(mu_).to(TRAIN_DEV), torch.from_numpy(sd_).to(TRAIN_DEV)
    net_.train()
    for _ in range(epochs or STACK_EPOCHS):
        perm_ = r_.permutation(len(y_))
        for b0_ in range(0, len(perm_), bs):
            ix_ = perm_[b0_:b0_ + bs]
            xb_ = (torch.from_numpy(X_[ix_].astype(np.float32)).to(TRAIN_DEV) - mu_t) / sd_t
            yb_ = torch.from_numpy(y_[ix_].astype(np.float32)).to(TRAIN_DEV)
            loss_ = nn.functional.binary_cross_entropy_with_logits(net_(xb_), yb_)
            opt_.zero_grad()
            loss_.backward()
            opt_.step()
    return net_.eval(), mu_t, sd_t


@torch.no_grad()
def mlp_logit(model_, X_, bs=8192):
    net_, mu_t, sd_t = model_
    out_ = np.zeros(len(X_), np.float32)
    for b0_ in range(0, len(X_), bs):
        xb_ = (torch.from_numpy(X_[b0_:b0_ + bs].astype(np.float32)).to(TRAIN_DEV) - mu_t) / sd_t
        out_[b0_:b0_ + bs] = net_(xb_).float().cpu().numpy()
    return out_


def _design(e_, v_, with_video):
    return np.concatenate([e_, v_.astype(np.float16)], 1) if with_video else e_


RES_STACK, NEST_STACK = None, None
if STACK_MODE == "auto" and ROWS_STACK is not None:
    res_ = []
    with timer("level 2: MLP per fold + decoding ({} conditions × {} matches × {} λ × 2 variants)".format(
            len(STACK_CONDS), len(NAMES_STACK), len(STACK_LAMBDAS))):
        for k, held in enumerate(FOLDS_STACK):
            tr_ = ~np.isin(ROWS_STACK["m"], [NAMES_STACK.index(n) for n in held])
            mlps_ = {wv_: fit_mlp(_design(ROWS_STACK["e"][tr_], ROWS_STACK["v"][tr_], wv_), ROWS_STACK["y"][tr_], seed=k)
                     for wv_ in (True, False)}
            nets_ = load_fold_nets(k)
            for n in held:
                for c_ in STACK_CONDS:
                    vw_ = views_stack(nets_, _eval_sa_stack(n, c_))
                    lab_ = vw_["lab"]
                    row_ = {"match": n, "kondisi": c_, "base": spell_ext(decode_ext(vw_["lg12"], lab_, COST), lab_)}
                    for wv_, nm_ in ((True, "gabungan"), (False, "without video")):
                        zz_ = 0.5 * (mlp_logit(mlps_[wv_], _design(vw_["eH"], vw_["vH"], wv_))
                                     - mlp_logit(mlps_[wv_], _design(vw_["eA"], vw_["vA"], wv_)))
                        for lam_ in STACK_LAMBDAS:
                            row_["{} λ{}".format(nm_, lam_)] = spell_ext(decode_ext(lam_ * zz_, lab_, COST), lab_)
                    res_.append(row_)
                    del vw_
            del nets_, mlps_
            gc.collect()
            print("  fold {}/{} done".format(k + 1, N_FOLDS), flush=True)
    RES_STACK = pd.DataFrame(res_)
    rb_ = np.random.default_rng(16)
    nrows_ = []
    for nm_ in ("gabungan", "without video"):
        cols_ = ["{} λ{}".format(nm_, l_) for l_ in STACK_LAMBDAS]
        nd_ = {c_: [] for c_ in STACK_CONDS}
        picks_ = []
        for held in FOLDS_STACK:
            trm_ = (RES_STACK.kondisi == STACK_MAIN) & ~RES_STACK.match.isin(held)
            best_ = max(cols_, key=lambda c_: (RES_STACK.loc[trm_, c_] - RES_STACK.loc[trm_, "base"]).mean())
            picks_.append(best_.split("λ")[1])
            for c_ in STACK_CONDS:
                m_ = (RES_STACK.kondisi == c_) & RES_STACK.match.isin(held)
                nd_[c_] += list((RES_STACK.loc[m_, best_] - RES_STACK.loc[m_, "base"]).to_numpy())
        for c_ in STACK_CONDS:
            d_ = np.asarray(nd_[c_])
            bt_ = np.array([rb_.choice(d_, len(d_)).mean() for _ in range(2000)])
            nrows_.append({"varian": nm_, "kondisi": c_, "base": RES_STACK.loc[RES_STACK.kondisi == c_, "base"].mean(),
                           "Δ bersarang": d_.mean(), "5%": np.percentile(bt_, 5), "95%": np.percentile(bt_, 95),
                           "matches won": "{}/{}".format(int((d_ > 0).sum()), len(d_)), "λ per fold": "/".join(picks_)})
    NEST_STACK = pd.DataFrame(nrows_)
    print("in-sample spellF1 (mean of 23 matches) per λ:")
    print(RES_STACK.groupby("kondisi").mean(numeric_only=True).T.round(4).to_string())
    print("\nnested Δ against base (λ chosen on the other folds):")
    print(NEST_STACK.round(4).to_string(index=False))
    g_ = NEST_STACK[NEST_STACK.varian == "gabungan"].set_index("kondisi")
    STACK_PASS = bool(g_.loc[STACK_MAIN, "Δ bersarang"] > 0 and g_.loc[STACK_MAIN, "5%"] > 0
                    and g_.loc["bola terlihat", "Δ bersarang"] >= -0.002)
    mm_ = RES_STACK.kondisi == STACK_MAIN
    STACK_LAMBDA = max(STACK_LAMBDAS, key=lambda l_: (RES_STACK.loc[mm_, "gabungan λ{}".format(l_)] - RES_STACK.loc[mm_, "base"]).mean())
    print("\ndecision rule: main nested Δ {:+.4f} [{:+.4f}; {:+.4f}] · ball visible {:+.4f} → {} (λ over all 23 matches: {})".format(
        g_.loc[STACK_MAIN, "Δ bersarang"], g_.loc[STACK_MAIN, "5%"], g_.loc[STACK_MAIN, "95%"], g_.loc["bola terlihat", "Δ bersarang"],
        "PASS" if STACK_PASS else "does not pass", STACK_LAMBDA))

In [ ]:
# 6f.3  BAR-ATH: final base model embeddings (seed 0 of each architecture) + video features → final MLP → log-odds
STACK_USE = STACK_MODE == "force" or (STACK_MODE == "auto" and STACK_PASS)
if STACK_USE and ROWS_STACK is not None:
    STACK_LAMBDA = STACK_LAMBDA or 2.0
    with timer("final MLP (all out-of-fold rows)"):
        mlp_fin_ = fit_mlp(_design(ROWS_STACK["e"], ROWS_STACK["v"], True), ROWS_STACK["y"], seed=99)
    fin_ = {}
    for m_, net_ in zip(MODEL_META["models"], NETS):
        if m_["seed"] == 0 and m_["arch"] in STACK_ARCHS:
            fin_[m_["arch"]] = net_
    eh_, ea_ = [], []
    for arch in STACK_ARCHS:
        eh_.append(embed_predict(fin_[arch], XB_H, TRAIN_DEV)[1])
        ea_.append(embed_predict(fin_[arch], XB_A, TRAIN_DEV)[1])
    vb_ = video_base(dict(bar10, L=PITCH_L))
    zz_ = 0.5 * (mlp_logit(mlp_fin_, _design(np.concatenate(eh_, 1), video_view(vb_), True))
                 - mlp_logit(mlp_fin_, _design(np.concatenate(ea_, 1), video_view(vb_, True), True)))
    LG10_STACK = (STACK_LAMBDA * zz_).astype(np.float32)
    live10_ = GRADED[bar10["idx"]] & ~d_mask[bar10["idx"]]
    lg_base_ = np.mean([0.5 * (predict(n_, XB_H, TRAIN_DEV) - predict(n_, XB_A, TRAIN_DEV)) for n_ in NETS], 0)
    print("BAR-ATH live frames: log-odds sign differs base → stacking {:.2%} · median |log-odds| {:.2f} → {:.2f}".format(
        float((np.sign(lg_base_[live10_]) != np.sign(LG10_STACK[live10_])).mean()), float(np.median(np.abs(lg_base_[live10_]))),
        float(np.median(np.abs(LG10_STACK[live10_])))))
if "EXT10_16" in globals():
    del EXT10_STACK
    gc.collect()
print("log-odds for BAR-ATH: {}".format("stacking (combined MLP, λ {})".format(STACK_LAMBDA) if STACK_USE else "base"))

### Figure: external cross-validation

Leave-match-out spellF1 per architecture on the 23 external matches, for the three ball conditions. The figure appears
if cross-validation was run during training (`RUN_CV = True`); its results are stored in `meta.json`.

In [ ]:
cv_saved = pd.DataFrame(MODEL_META.get("cv", []))
if cv_saved.empty:
    print("cross-validation not stored in meta.json (retrain with RUN_CV = True to show it)")
else:
    methods = [c for c in ("gru_set", "tcn", "tcn_set", "ensembel") if c in cv_saved]
    tab = cv_saved.groupby("kondisi")[methods].mean()
    conds = list(tab.index)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for k, cond in enumerate(conds):
        ys = tab.loc[cond, methods].to_numpy(float)
        ax.plot(np.arange(len(methods)), ys, linestyle="none", marker="o", markersize=10, color=[COL["H"], COL["aqua"], COL["A"]][k % 3],
                markeredgecolor=COL["surface"], markeredgewidth=2, label=cond)
        for xi, v in enumerate(ys):
            ax.text(xi + 0.08, v, "{:.3f}".format(v), ha="left", va="center", fontsize=8, color=COL["text"])
    ax.set_xticks(np.arange(len(methods)), methods)
    ax.set_xlim(-0.4, len(methods) - 0.4)
    ax.set_ylim(float(tab.values.min()) - 0.02, float(tab.values.max()) + 0.02)
    ax.set_ylabel("spellF1 (mean of 23 matches)")
    ax.set_title("External cross-validation", pad=18)
    note(ax, "every match is predicted by a model that never saw it")
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.12), ncol=2)
    plt.tight_layout()
    plt.show()

### Figure: post-processing, representativeness of 5 minutes, and imbalance

Three analyses on the **out-of-fold** cross-validation predictions (BAR-ATH-like ball gaps). All three use the external
matches, whose labels cover the full 90 minutes, because the BAR-ATH GT covers only 5 minutes.

1. **Which post-processing raises spellF1?** Compared: per-frame labels without smoothing, a median filter with several
   window widths, Viterbi with several switching costs, and Viterbi plus merging of short possessions. spellF1 and
   macroF1 are shown side by side: macroF1 counts frames, so it barely moves; spellF1 counts possessions, so a single
   spurious switch is punished immediately.
2. **Do the first 5 minutes represent 90 minutes?** For every external match, the spellF1 of the first 5 minutes is
   compared with the spellF1 of the whole match and of each ± 15-minute block.
3. **Does a dominant team make the model one-sided?** Each match's home possession share is compared with its spellF1,
   and the recall of the dominant team with the recall of the minor team.

In [ ]:
cv_post_df = pd.DataFrame(MODEL_META.get("cv_post", []))
cv_win_df = pd.DataFrame(MODEL_META.get("cv_window", []))
cv_match_df = pd.DataFrame(MODEL_META.get("cv_match", []))
if cv_post_df.empty:
    print("post-processing analysis not stored in meta.json (retrain with RUN_CV = True to show it)")
else:
    post_tab = cv_post_df.groupby(["metode", "param"])[["spell_f1", "macro_f1"]].mean().reset_index()
    print(post_tab.round(3).to_string(index=False))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), gridspec_kw={"width_ratios": [1.1, 1]})
    ax = axes[0]
    vit = post_tab[post_tab.metode.isin(["Viterbi", "per frame (tanpa smoothing)"])].sort_values("param")
    ax.plot(vit.param, vit.spell_f1, color=COL["aqua"], linewidth=2, marker="o", markersize=7,
            markeredgecolor=COL["surface"], markeredgewidth=2, label="spellF1")
    ax.plot(vit.param, vit.macro_f1, color=COL["muted"], linewidth=1.5, linestyle="--", marker="o", markersize=5,
            label="macroF1 (H/A per frame)")
    ax.axvline(COST, color=COL["H"], linewidth=1, linestyle=":")
    ax.text(COST, 0.5, " used: {:g}".format(COST), color=COL["H"], fontsize=9, va="center",
            transform=ax.get_xaxis_transform())
    ax.set_xscale("symlog", linthresh=2)
    ax.set_xticks(POST_COSTS, ["{:g}".format(c_) for c_ in POST_COSTS])
    ax.set_xlabel("H↔A switching cost in the Viterbi (0 = per-frame labels)")
    ax.set_ylabel("score (mean of 23 matches)")
    ax.set_title("Smoothing: spellF1 is sensitive, macroF1 is almost flat", pad=18)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.16), ncol=2)

    ax = axes[1]
    best = []
    for metode, col in (("per frame (tanpa smoothing)", COL["D"]), ("filter median", COL["yellow"]), ("Viterbi", COL["aqua"]),
                        ("Viterbi + gabung pendek", COL["violet"])):
        sub = post_tab[post_tab.metode == metode]
        if metode == "Viterbi":
            sub = sub[sub.param == COST]
        r_ = sub.loc[sub.spell_f1.idxmax()]
        unit = {"filter median": " s", "Viterbi + gabung pendek": " s"}.get(metode, "")
        best.append(("{}\n({}{:g}{})".format(metode, "biaya " if metode == "Viterbi" else "", r_.param, unit)
                     if metode != "per frame (tanpa smoothing)" else metode, r_.spell_f1, col))
    for k, (name, v, col) in enumerate(best):
        ax.plot(v, k, marker="o", markersize=11, color=col, markeredgecolor=COL["surface"], markeredgewidth=2, linestyle="none")
        ax.text(v + 0.005, k, "{:.3f}".format(v), va="center", fontsize=9, color=COL["text"])
    ax.set_yticks(range(len(best)), [b_[0] for b_ in best], fontsize=8)
    ax.invert_yaxis()
    lo_ = min(b_[1] for b_ in best)
    ax.set_xlim(lo_ - 0.03, max(b_[1] for b_ in best) + 0.04)
    ax.set_xlabel("spellF1 (mean of 23 matches)")
    ax.grid(axis="y", visible=False)
    ax.set_title("Best post-processing variant per family", pad=18)
    note(ax, "the best parameter of each family is chosen in cross-validation")
    plt.tight_layout()
    plt.show()

if not cv_win_df.empty:
    wide = cv_win_df.pivot(index="match", columns="jendela", values="spell_f1")
    r5 = float(np.corrcoef(wide["5 menit pertama"], wide["seluruh match"])[0, 1])
    gap5 = (wide["5 menit pertama"] - wide["seluruh match"]).abs()
    print("first 5 minutes vs whole match: correlation {:.2f} · median absolute difference {:.3f} · max {:.3f}".format(
        r5, gap5.median(), gap5.max()))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
    ax = axes[0]
    ax.plot([0, 1], [0, 1], color=COL["grid"], linewidth=1.5)
    ax.plot(wide["seluruh match"], wide["5 menit pertama"], linestyle="none", marker="o", markersize=7, color=COL["H"],
            markeredgecolor=COL["surface"], markeredgewidth=1.5)
    lim = (min(wide[["5 menit pertama", "seluruh match"]].min()) - 0.05, 1.0)
    ax.set_xlim(*lim)
    ax.set_ylim(*lim)
    ax.set_xlabel("spellF1 over the whole match")
    ax.set_ylabel("spellF1 of the first 5 minutes")
    ax.set_title("First 5 minutes vs 90 minutes, per match", pad=18)
    note(ax, "line = identical · correlation {:.2f} · median difference {:.3f}".format(r5, gap5.median()))
    ax = axes[1]
    blocks = ["blok {}".format(k) for k in range(1, 7)]
    data_b = [wide[b_].dropna().to_numpy() for b_ in blocks]
    bp = ax.boxplot(data_b, widths=0.5, patch_artist=True, showfliers=False,
                    medianprops={"color": COL["text"], "linewidth": 1.5})
    for patch in bp["boxes"]:
        patch.set(facecolor="#cde2fb", edgecolor=COL["H"])
    for k, d_ in enumerate(data_b, start=1):
        ax.plot(np.full(len(d_), k) + np.linspace(-0.12, 0.12, len(d_)), d_, linestyle="none", marker="o", markersize=3,
                color=COL["H"], alpha=0.6)
    ax.set_xticks(range(1, 7), ["0–15'", "15–30'", "30–45'", "45–60'", "60–75'", "75–90'"])
    ax.set_xlabel("time block (± 15 minutes)")
    ax.set_ylabel("spellF1")
    ax.grid(axis="x", visible=False)
    ax.set_title("Score per time block, 23 matches", pad=18)
    note(ax, "spellF1 of the first 5 minutes differs from the 90-minute score; even the per-block scores vary")
    plt.tight_layout()
    plt.show()

if not cv_match_df.empty:
    rho = float(np.corrcoef(np.abs(cv_match_df.porsi_home - 0.5), cv_match_df.spell_f1)[0, 1])
    print("correlation |home share − 0.5| vs spellF1: {:.2f} | recall of the dominant team {:.3f} vs the minor team {:.3f}".format(
        rho, cv_match_df.recall_dominan.mean(), cv_match_df.recall_minor.mean()))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    ax = axes[0]
    ax.plot(100 * cv_match_df.porsi_home, cv_match_df.spell_f1, linestyle="none", marker="o", markersize=7, color=COL["H"],
            markeredgecolor=COL["surface"], markeredgewidth=1.5)
    ax.axvline(50, color=COL["muted"], linewidth=1, linestyle="--")
    ax.set_xlabel("% of live frames held by the home team")
    ax.set_ylabel("spellF1 over the whole match")
    ax.set_title("Possession dominance vs score", pad=18)
    note(ax, "correlation |share − 50%| vs spellF1: {:.2f}".format(rho))
    ax = axes[1]
    for k, r_ in enumerate(cv_match_df.sort_values("recall_minor").itertuples()):
        ax.plot([r_.recall_minor, r_.recall_dominan], [k, k], color=COL["grid"], linewidth=2)
        ax.plot(r_.recall_dominan, k, marker="o", markersize=6, color=COL["H"], linestyle="none",
                label="dominant team" if k == 0 else None)
        ax.plot(r_.recall_minor, k, marker="o", markersize=6, color=COL["A"], linestyle="none",
                label="minor team" if k == 0 else None)
    ax.set_yticks([])
    ax.set_xlabel("per-frame recall (H/A)")
    ax.set_ylabel("match (sorted)")
    ax.set_title("Recall of the dominant vs the minor team", pad=18)
    note(ax, "mean {:.3f} vs {:.3f}".format(cv_match_df.recall_dominan.mean(), cv_match_df.recall_minor.mean()))
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

### 6c. Detailed evaluation and error analysis

An average spellF1 hides *where* the model is wrong. This section dissects the out-of-fold predictions (23 external
matches, main condition) from four angles:

1. **Detailed metrics**: possession precision and recall at every IoU threshold, switch timing, and probability
   calibration.
2. **Per-frame errors by context**: distance from the start/end of a possession, local ball situation (duel, loose
   ball, ball missing).
3. **Per-possession errors**: which possessions are missed (and why), and what kind of spurious predictions occur.
4. **Feature importance and data quality**: permutation importance on held-out folds, sensitivity to ghost tracks, and
   BAR-ATH outliers vs the training data.

Everything comes from labelled external data; the public GT is not used here.

In [ ]:
ERR = MODEL_META.get("cv_err") or {}
if not ERR:
    print("error analysis not stored in meta.json (retrain with RUN_CV = True to show it)")
else:
    rk = ERR["ringkas"]
    print("graded live frames: {:,} · per-frame error {:.1%} · Brier {:.3f} · GT possessions {:,} · predicted {:,}".format(
        rk["n_frame_hidup"], rk["error_frame"], rk["brier"], rk["n_possession_gt"], rk["n_possession_pred"]))
    print("\nPossession precision / recall per IoU threshold (mean of 23 matches):")
    print(pd.DataFrame(ERR["per_iou"]).set_index("iou").round(3).to_string())
    tr_ = ERR["transisi"]
    print("\nTrue H↔A switches followed by a same-direction predicted switch within ± 3 s: {:.1%} (ball visible {:.1%}, "
          "ball missing {:.1%}) · median time difference {:+.2f} s [IQR {:+.2f} … {:+.2f}]".format(
              tr_["tertangkap_3s"], tr_["tertangkap_bola_terlihat"], tr_["tertangkap_bola_hilang"], tr_["lag_median_s"],
              tr_["lag_q25_s"], tr_["lag_q75_s"]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.9), gridspec_kw={"width_ratios": [1.15, 1.15, 0.8]})
    ax = axes[0]
    for key, col, lab_ in (("sejak_awal", COL["H"], "since the possession start"), ("sampai_akhir", COL["A"], "until the possession end")):
        t_ = pd.DataFrame(ERR[key])
        ax.plot(np.arange(len(t_)), 100 * t_.error, color=col, linewidth=2, marker="o", markersize=6,
                markeredgecolor=COL["surface"], markeredgewidth=1.5, label=lab_)
    ax.set_xticks(np.arange(len(t_)), t_.kelas, fontsize=8)
    ax.set_xlabel("time distance from the true possession boundary")
    ax.set_ylabel("% of frames with the wrong team")
    ax.set_title("Errors pile up at possession boundaries", pad=18)
    note(ax, "in the middle of long possessions the error is ± 2–3%")
    ax.legend(loc="upper right")
    ax = axes[1]
    rd, pdur = pd.DataFrame(ERR["recall_durasi"]), pd.DataFrame(ERR["presisi_durasi"])
    order_ = bin_labels(ERR_DUR_BINS)
    rd = rd.set_index("kelas").reindex(order_)
    pdur = pdur.set_index("kelas").reindex(order_)
    xk = np.arange(len(order_))
    ax.bar(xk - 0.19, 100 * rd.recall, width=0.38, color=COL["H"], label="recall (true possessions)")
    ax.bar(xk + 0.19, 100 * pdur.presisi, width=0.38, color=COL["aqua"], label="precision (predicted possessions)")
    for k, n_ in enumerate(rd.n):
        ax.text(k - 0.19, 2, "{:.0f}".format(n_), ha="center", fontsize=7, color=COL["surface"], rotation=90)
    ax.set_xticks(xk, order_, fontsize=8)
    ax.set_xlabel("possession duration")
    ax.set_ylabel("% (IoU ≥ 0.5)")
    ax.set_ylim(0, 105)
    ax.grid(axis="x", visible=False)
    ax.set_title("Short possessions are missed most often", pad=18)
    note(ax, "number on the bar = count of true possessions")
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.16), ncol=2)
    ax = axes[2]
    cal_ = pd.DataFrame(ERR["kalibrasi"])
    ax.plot([0, 1], [0, 1], color=COL["grid"], linewidth=1.5)
    ax.plot(cal_.p, cal_.y, color=COL["violet"], linewidth=2, marker="o", markersize=6, markeredgecolor=COL["surface"],
            markeredgewidth=1.5)
    ax.set_xlabel("predicted P(H)")
    ax.set_ylabel("true share of H")
    ax.set_title("Calibration", pad=18)
    note(ax, "Brier {:.3f} · line = perfect calibration".format(rk["brier"]))
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.9), gridspec_kw={"width_ratios": [1, 1.1]})
    ax = axes[0]
    sit_ = pd.DataFrame(ERR["situasi"]).sort_values("error")
    ax.barh(np.arange(len(sit_)), 100 * sit_.error, color=[COL["A"] if e > 0.1 else COL["H"] for e in sit_.error], height=0.6)
    for k, r_ in enumerate(sit_.itertuples()):
        ax.text(100 * r_.error + 0.4, k, "{:.1f}% · {:.0%} of all errors · {:.0%} of frames".format(
            100 * r_.error, r_.porsi_error, r_.n / sit_.n.sum()), va="center", fontsize=8, color=COL["text"])
    ax.set_yticks(np.arange(len(sit_)), sit_.kelas, fontsize=9)
    ax.set_xlim(0, 100 * sit_.error.max() * 2.3)
    ax.set_xlabel("% of frames with the wrong team")
    ax.grid(axis="y", visible=False)
    ax.set_title("Error by the local ball situation", pad=18)
    note(ax, "owner = team in possession according to the label; distance = nearest player to the ball")
    ax = axes[1]
    tg, tp = pd.DataFrame(ERR["tipe_gt"]), pd.DataFrame(ERR["tipe_pred"])
    rows_ = [("true possessions", tg), ("predicted possessions", tp)]
    palette = {"cocok": COL["aqua"], "terlewat total (tak ada prediksi tim itu)": COL["A"], "batas meleset / tergabung": COL["yellow"],
               "terpecah (≥ 2 prediksi)": COL["violet"], "tim salah (pergantian palsu)": "#c0392b"}
    for k, (name, t_) in enumerate(rows_):
        left = 0.0
        for tipe in palette:
            v = float(t_.loc[t_.tipe == tipe, "porsi"].sum())
            if v <= 0:
                continue
            ax.barh(k, 100 * v, left=left, color=palette[tipe], height=0.5,
                    label=tipe if k == 0 or tipe == "tim salah (pergantian palsu)" else None)
            if v > 0.04:
                ax.text(left + 50 * v, k, "{:.0f}%".format(100 * v), ha="center", va="center", fontsize=8, color=COL["surface"])
            left += 100 * v
    ax.set_yticks([0, 1], [r_[0] for r_ in rows_])
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("% of possessions (IoU 0.5)")
    ax.grid(axis="y", visible=False)
    ax.set_title("Error type per possession", pad=18)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.18), ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()
    print("True possessions by type (median duration & ball-visible share):")
    print(tg.round(3).to_string(index=False))
    print("Recall by possession origin:", {("after a dead ball" if r_["dari_bola_mati"] else "after a contest"):
                                            round(r_["recall"], 3) for r_ in ERR["asal"]})

#### Feature importance, uninformative features, and sensitivity to outliers

**Permutation importance** (Breiman, 2001; Fisher et al., 2019) on held-out folds: one feature (or one group) is
circularly shifted by a random offset within the match, so its distribution and autocorrelation are kept but its
alignment with the other features and with the labels is broken. The rise in logloss / drop in spellF1 = how much the
model **depends** on that feature. Features with Δ ≈ 0 are candidates for "no information" for this model; those
candidates were then tested by retraining without them, because Δ ≈ 0 can also mean the information is duplicated in
other features.

In [ ]:
PERM = pd.DataFrame(MODEL_META.get("cv_perm", []))
if PERM.empty:
    print("permutation importance not stored in meta.json (retrain with RUN_CV = True)")
else:
    grp = PERM[PERM.jenis == "kelompok"].groupby("nama").agg(d_spell=("d_spell_f1", "mean"), d_ll=("d_logloss", "mean"),
                                                             turun=("d_spell_f1", lambda v: int((v < 0).sum())))
    grp = grp.sort_values("d_spell")
    fe = PERM[PERM.jenis == "fitur"].groupby("nama").d_logloss.agg(["mean", "std", "size", lambda v: int((v > 0).sum())])
    fe.columns = ["d_ll", "sd", "n", "naik"]
    fe["se"] = fe.sd / np.sqrt(fe.n)
    fe = fe.sort_values("d_ll", ascending=False)
    print("Feature groups (ΔspellF1 when shuffled; 'turun' = matches whose score drops, out of {}):".format(int(PERM.match.nunique())))
    print(grp.round(4).to_string())
    weak = fe[(fe.d_ll < 2 * fe.se) | (fe.naik <= fe.n / 2 + 1)]
    print("\nFeatures whose Δlogloss is not different from zero (uninformative candidates):", ", ".join(weak.index) or "-")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [0.9, 1.3]})
    ax = axes[0]
    ax.barh(np.arange(len(grp)), grp.d_spell, color=COL["H"], height=0.6)
    for k, v in enumerate(grp.d_spell):
        ax.text(v - 0.002, k, "{:+.3f}".format(v), va="center", ha="right", fontsize=8, color=COL["text"])
    ax.set_yticks(np.arange(len(grp)), grp.index)
    ax.set_xlim(grp.d_spell.min() * 1.35, 0.005)
    ax.set_xlabel("ΔspellF1 when the group is shuffled")
    ax.grid(axis="y", visible=False)
    ax.set_title("Feature groups the model relies on", pad=18)
    note(ax, "more negative = more important")
    ax = axes[1]
    show = pd.concat([fe.head(10), fe.tail(8)])
    yk = np.arange(len(show))
    ax.barh(yk, show.d_ll, xerr=1.96 * show.se, color=[COL["D"] if nm in weak.index else COL["aqua"] for nm in show.index],
            height=0.6, error_kw={"elinewidth": 0.8, "ecolor": COL["muted"]})
    ax.set_yticks(yk, show.index, fontsize=8)
    ax.invert_yaxis()
    ax.axhline(9.5, color=COL["grid"], linewidth=1, linestyle="--")
    ax.set_xlabel("Δlogloss when the feature is shuffled (± 95%)")
    ax.grid(axis="y", visible=False)
    ax.set_title("10 most important & 8 weakest features", pad=18)
    note(ax, "grey = not different from zero")
    plt.tight_layout()
    plt.show()

GHOST = pd.DataFrame(MODEL_META.get("cv_ghost", []))
AUD = MODEL_META.get("data_audit") or {}
if AUD:
    print("Outliers & tracking quality (% of observations):")
    print(pd.DataFrame(AUD).round(3).to_string())
if not GHOST.empty:
    d_ = GHOST["track hantu"] - GHOST["bersih"]
    print("\nSensitivity to ghost tracks ({:.0%} of frames per team, as in BAR-ATH): spellF1 {:.4f} → {:.4f} · Δ {:+.4f} "
          "(± {:.4f}) · drops in {}/{} matches".format(GHOST_SHARE, GHOST["bersih"].mean(), GHOST["track hantu"].mean(), d_.mean(),
                                                  1.96 * d_.std(ddof=1) / np.sqrt(len(d_)), int((d_ < 0).sum()), len(d_)))

#### From error analysis to action

Every finding above was turned into a testable hypothesis and tested with the same protocol: leave-match-out
cross-validation on the 23 matches, a 3-architecture ensemble, and a **paired per-match** comparison with a 95%
bootstrap interval. Noise reference: retraining the baseline with another seed shifts the mean spellF1 by ± 0.0014
(interval ± 0.005).

| Finding | Proposed fix | Result (ensemble Δ spellF1) | Decision |
|---|---|---|---|
| Errors pile up within ± 1 s of possession boundaries; switches after a pass and when the opponent is closer to the ball are often late or missed | ball motion direction + each team's arrival time to the ball (section 5) | 2 seeds: +0.002 (BAR-ATH-like gaps) · +0.003 (live-ball gaps) · **+0.005 [+0.002; +0.009]** (ball visible); all six estimates ≥ 0; switches caught with the ball visible 0.778 → 0.795, recall of 1–2 s possessions 0.43 → 0.45 | **adopted** |
| Short possessions (< 2 s) are missed most often | per-team touch memory: seconds since the last touch (univariate AUC 0.88, the strongest single feature) | −0.001 | rejected: the sequence model already infers it from ± 20 s of context |
| Opponent close to the ball but not yet in possession | interception: when each team's players can cut the ball path | −0.002 | rejected |
| Duels and pressing | pressure on the nearest player (number of opponents within 3/6 m) | −0.001 / −0.005 | rejected |
| `n track` is uninformative (permutation ≈ 0) and hit by BAR-ATH outliers | drop the feature | −0.004 [−0.009; +0.001] | kept: uninformative, but dropping it does not help |
| > 11 tracks per team on ± 13% of BAR-ATH frames (4c), absent from the training data | inject ghost tracks with the same frequency into the external matches | see the sensitivity numbers above (≈ 0) | no cleaning needed: the model is robust |
| CV gap pattern: ± 40% of live frames without a ball, while BAR-ATH live play has only ± 4% | train with the live-ball gap pattern | −0.030 (BAR-ATH-like gaps), −0.005 (live-ball gaps) | rejected as augmentation; **added as an evaluation condition** |
| Spurious switches and short flickers remain after Viterbi | extra smoothing: Gaussian-smoothed log-odds (σ 0.2–1 s) before Viterbi, confidence clipping (± 4–12), minimum possession duration 0.3–1 s (semi-Markov), switching cost 5–15; 32 variants, chosen nested (chosen on 18 matches, scored on the other 5) | best on the same data only +0.0002; nested **−0.002** [−0.004; 0.000] (gaps) · −0.001 (live-ball gaps); Gaussian σ 1 s −0.02 | rejected: a Viterbi with cost 10 is already at the optimum, picking another variant only chases noise |
| Possessions < 1.5 s inside an opponent sequence are missed (public GT 10:13 and 10:30: tackles that knock the ball loose) | **last-touch detector** (LightGBM, 70 features of distance/relative speed/ball turn, ± 0.3 s) trained on the DFL tracking `BallPossession` flag in 7 IDSSE matches, which switches exactly when a player touches the ball; its output becomes an extra feature channel | detector: AUC 0.85 (ball visible **0.94**) vs 0.64 for the nearest player, leave-one-match-out; sequence model: **−0.002** [−0.006; +0.003] (gaps) · −0.001 (visible) · −0.001 (live-ball gaps); recall of possessions ≤ 1 s 0.283 → 0.293 | rejected: the sequence model already infers who touched last from context |
| Limited training data (23 matches) | + 7 IDSSE matches (optical tracking at 25 fps) as training data only; label = team of the last ball event (same definition as Metrica), switch times tidied to the first touch according to the tracking flag, dead ball from `BallStatus` | −0.0005 [−0.005; +0.004] (gaps) · −0.004 [−0.009; +0.001] (visible) · +0.002 [−0.004; +0.008] (live-ball gaps) | rejected: inconsistent across conditions |

The main lesson: features with a strong univariate signal (touch memory, a trained touch detector) do not necessarily
add anything to a sequence model that already sees ± 20 s of context. What helped was information that **cannot** be
inferred from the old features, namely the ball direction. The gain is small (± 0.002–0.005), in line with the error
analysis: the largest remaining errors are possessions < 2 s while the ball is not visible, which carry no information
in the tracking.

#### The label definition, and what happens in the public-GT errors

CHALLENGE.md says the GT comes from *corrected tracking*: a frame is `D` when the tracking marks the ball as dead, and
otherwise the team the tracking names as owner. The IDSSE data has flags with the same mechanism (`BallPossession`,
`BallStatus`). Are those flags suitable as labels? We compare the rate of direct H↔A switches per 5 minutes of live play:

| Label source | H↔A switches / 5 live minutes | Switches |
|---|---|---|
| BAR-ATH public GT | 14.9 | 12 |
| SkillCorner (events, used) | 15.8 | 2,996 |
| Metrica (events, used) | 23.1 | 683 |
| IDSSE events + flag | 23.3 | 1,594 |
| IDSSE raw flag | 31.9 | 2,351 |

The raw flag switches at every touch, including touches without control that the competition rules do not count, so it
is not used as a possession label. The event labels we already use are in the same range as the GT.

We inspected the errors in the public 5 minutes directly in the tactical and broadcast video (both trackers' ball
positions drawn on the image). This is diagnosis, not tuning:

- *Two short H possessions (0.8 s and 1.3 s)*: a tackle or duel that knocks the ball loose, after which Athletic
  regains it. The ball is visible and both trackers agree (0.2 m); what is missing is information about control, not
  ball data.
- *Late switch at 09:11 and spurious A at 11:33*: the tactical tracker loses the ball and keeps extrapolating it into
  empty space for ± 3 s (its steps slow down smoothly from ± 26 m/s to ± 0), while the broadcast ball stays on the real
  ball (10–18 m apart). Two label-free rules for choosing the camera when they disagree (an unnatural slow-down without
  players, a still ball far from any player) do not separate these cases: those signs are equally rare in both sources.
- *`D` at 08:34*: a whistle while a player is still dribbling; three of the four `D` starts in the public GT happen
  without the ball going out.

## 7. H vs A log-odds for BAR-ATH

**What this step does.** Runs the six loaded models on BAR-ATH. Each model predicts from both viewpoints (home as
"own", and away as "own" with the sign flipped), and the two are averaged, so the prediction is exactly H↔A symmetric;
then the 6 models are averaged. The 10 fps log-odds are linearly interpolated to 25 fps.

In [ ]:
if INFER_DEVICE != "cpu" and not torch.cuda.is_available():
    INFER_DEVICE = "cpu"
with timer("inference ({})".format(INFER_DEVICE)):
    lg10 = np.mean([0.5 * (predict(net, XB_H, INFER_DEVICE) - predict(net, XB_A, INFER_DEVICE)) for net in NETS], 0)
if NEAR_USE:
    lg10 = lg10 + EVID10_BAR      # nearest-player evidence (section 6e)
if STACK_USE:
    lg10 = LG10_STACK.copy()      # log-odds of the combined MLP (section 6f)
L_seq = np.interp(np.arange(N_FRAMES), bar10["idx"], lg10).astype(np.float32)
if YOLO_ON:
    with timer("inference with the tracking ball (for comparison)"):
        lg10_trk = np.mean([0.5 * (predict(net, XB_TRK_H, INFER_DEVICE) - predict(net, XB_TRK_A, INFER_DEVICE)) for net in NETS], 0)
    L_seq_trk = np.interp(np.arange(N_FRAMES), bar10["idx"], lg10_trk).astype(np.float32)
else:
    L_seq_trk = L_seq
live = GRADED & ~d_mask
check("log-odds finite on every frame", np.isfinite(L_seq).all())
print("live frames: share of L_seq > 0 (leaning H) {:.3f} | median |L_seq| {:.2f}".format(
    float((L_seq[live] > 0).mean()), float(np.median(np.abs(L_seq[live])))))

### Figure: model confidence

Distribution of the H vs A log-odds on the live frames of the whole match. Two peaks far from zero mean the model is
confident on most frames; frames around zero are ambiguous moments (challenges, a ball in the air) that the decoder
resolves by looking at the context.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
vals = np.clip(L_seq[live], -16, 16)
bins = np.linspace(-16, 16, 65)
ax.hist(vals[vals < 0], bins=bins, color=COL["A"], edgecolor=COL["surface"], linewidth=1, label="leaning A (< 0)")
ax.hist(vals[vals >= 0], bins=bins, color=COL["H"], edgecolor=COL["surface"], linewidth=1, label="leaning H (≥ 0)")
ax.axvline(0, color=COL["text"], linewidth=1)
ax.set_xlabel("log-odds H vs A (clipped at ± 16)")
ax.set_ylabel("live frames")
ax.set_title("Possession model confidence", pad=18)
note(ax, "{:.1%} of live frames with |log-odds| < 2 (ambiguous)".format(float((np.abs(L_seq[live]) < 2).mean())))
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.18), ncol=2)
plt.tight_layout()
plt.show()

## 7b. Touch evidence from the video

**What this step does.** Finds, in the video, the moments when a player touches the ball and which team that player
belongs to, so the decoder can use them as extra evidence.

**Why.** The error analysis (experiment log #86) shows that ± 80% of the spellF1 loss, both in 30 simulated matches and
in the public window, comes from **missed team switches**: a short possession (< 2–5 s) is lost, and the two
possessions on either side merge into one. The possession model sees only the tracking positions; the tracking ball is
heavily smoothed (p99 speed jump 3.8 m/s vs 19 m/s for the video ball), so the *moment* of a touch is blurred. The video
shows directly **who** touches the ball and **when**.

**How.**

1. **Touch candidates (tactical):** peaks in the speed change of the YOLO video ball (section 2b) at 25 fps:
   least-squares lines 4 frames before/after, |Δv| > 4 m/s, local peak within ± 4 frames, only anchored balls during
   live play.
2. **Contact in the image:** in frames t−3 … t+3, YOLO11x person boxes; contact if the ball is closer than 0.25 box
   heights. Touch time = the nearest contact frame (GT definition: the switch happens exactly at the touch).
3. **Toucher team:** a 3-class kit-colour classifier (H / A / other = referee), LightGBM on torso colour features,
   trained from the **tracking** team labels (no hand labels, no GT). 'Other' boxes are not touchers; a team is given
   only when the confident boxes agree.
4. **Broadcast** for the touches the tactical camera misses: COCO ball in a 640 px crop around the projected merged
   ball, candidates from the broadcast ball and from tactical candidates without contact; contact and team as above
   with the broadcast kit classifier.

**Validation (without the public GT).** Visual check outside the public window: contacts 7/7 real, rejections 7/7
correct (#81); 'other' samples 58/60 referees (#84). Simulation on 30 labelled matches (#78, #83, #87): touch evidence
with exact timing raises spellF1 (external +0.01, IDSSE +0.05 … +0.08); the weight per source follows its accuracy,
estimated without GT (tactical ± 0.9 → β 4; broadcast from the broadcast ball ± 0.77 → β 2; broadcast from tactical
candidates ± 0.56 → β 0.5). Tried and **rejected**: snapping switches to touches (#83), a duel rule (#89), a timing
delay/smoother (#90), ball-impulse direction for players standing close together (#91).

Results are stored in `cache/touch/` (positions and teams per frame, kit-colour models). Without the cache: ± 1 GPU
hour.

In [ ]:
import lightgbm as lgb
from scipy.optimize import linear_sum_assignment

TOUCH_ON = TOUCH_STAGE != "off" and YOLO_ON
VIDEO_B = find_video("broadcast")
check("broadcast video available for touches", VIDEO_B is not None, str(VIDEO_B), critical=False)
TOUCH_DIR = CACHE / "touch"
TOUCH_DIR.mkdir(parents=True, exist_ok=True)
CONTACT = 0.25
LIVE = GRADED & ~d_mask
_pub = np.flatnonzero(pm.label_codes(pm.read_label_csv(str(GT_PUBLIC))) != pm.X)
OUT_PUB = np.ones(N_FRAMES, bool)                 # only used to pick visual-check examples outside the public window
OUT_PUB[_pub.min() - 100:_pub.max() + 101] = False

hom_b = pd.read_csv(require_asset("homography", "broadcast"))
H_B = hom_b[["h00", "h01", "h02", "h10", "h11", "h12", "h20", "h21", "h22"]].to_numpy(np.float64).reshape(-1, 3, 3)
HB_OK = (hom_b["valid"].to_numpy() == 1) & np.isfinite(H_B).all((1, 2))


def bframe(t):
    h_ = int(HALF[int(t)])
    return int(t) - BC_OFFSET[h_] if h_ in BC_OFFSET else -1


def m_to_px_b(b, xy):
    p_ = np.c_[np.asarray(xy, float) / [PITCH_L, PITCH_W], np.ones(len(xy))] @ H_B[b].T
    return p_[:, :2] / p_[:, 2:3]


def px_to_m_b(b, uv):
    p_ = np.c_[np.asarray(uv, float), np.ones(len(uv))] @ np.linalg.inv(H_B[b]).T
    return p_[:, :2] / p_[:, 2:3] * [PITCH_L, PITCH_W]


def m_to_px_t(t, xy):
    xy = np.asarray(xy, float)
    return _apply_h(np.repeat(H_T[t][None], len(xy), 0), xy / [PITCH_L, PITCH_W])


def px_to_m_t(t, uv):
    uv = np.asarray(uv, float)
    return _apply_h(np.repeat(H_INV[t][None], len(uv), 0), uv) * [PITCH_L, PITCH_W]


def torso_feat(img, box):
    """torso colour features (rows 15–55%, columns 25–75% of the box), grass pixels removed"""
    x0, y0, x1, y1 = box
    h_, w_ = y1 - y0, x1 - x0
    c_ = img[int(y0 + 0.15 * h_):int(y0 + 0.55 * h_) + 1, int(x0 + 0.25 * w_):int(x0 + 0.75 * w_) + 1]
    if c_.size == 0:
        return None
    hsv = cv2.cvtColor(c_, cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(np.float32)
    lab_ = cv2.cvtColor(c_, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    grass = (hsv[:, 0] > 30) & (hsv[:, 0] < 90) & (hsv[:, 1] > 50)
    keep = ~grass
    if keep.sum() < 4:
        return None
    hs, lb = hsv[keep], lab_[keep]
    hist = np.histogram2d(hs[:, 0], hs[:, 1], bins=[12, 3], range=[[0, 180], [0, 256]])[0].ravel() / keep.sum()
    dark = (hs[:, 2] < 60).mean()
    white = ((hs[:, 1] < 40) & (hs[:, 2] > 170)).mean()
    return np.r_[hist, lb.mean(0), lb.std(0), np.percentile(lb, [25, 75], 0).ravel(), dark, white, grass.mean()].astype(np.float32)


def contact_nd(boxes, uv):
    """distance from the ball to a person box, in units of box height (0 = ball inside the box)"""
    x0, y0, x1, y1 = boxes.T
    return np.hypot(np.maximum(0, np.maximum(x0 - uv[0], uv[0] - x1)), np.maximum(0, np.maximum(y0 - uv[1], uv[1] - y1))) / (y1 - y0)


def frame_contact(img, boxes, uv, kit):
    """(distance to the nearest non-referee box, toucher team 0 H / 1 A / -1, number of touching referee boxes)"""
    if not len(boxes):
        return np.nan, -1, 0
    nd = contact_nd(boxes, uv)
    drop, teams = set(), []
    for k_ in np.flatnonzero(nd < CONTACT):
        ft = torso_feat(img, boxes[k_])
        if ft is None:
            continue
        p_ = kit.predict(ft[None])[0]
        if p_[2] > 0.5:
            drop.add(k_)
            continue
        pH = p_[0] / max(p_[0] + p_[1], 1e-9)
        if abs(pH - 0.5) >= 0.3:
            teams.append(0 if pH > 0.5 else 1)
    rest = [k_ for k_ in range(len(boxes)) if k_ not in drop]
    return (float(nd[rest].min()) if rest else np.nan), (teams[0] if teams and len(set(teams)) == 1 else -1), len(drop)


def touch_key_ok(name, key):
    p_ = TOUCH_DIR / "{}_key.json".format(name)
    return TOUCH_STAGE != "retrain" and p_.is_file() and json.loads(p_.read_text(encoding="utf-8")) == key


def touch_key_save(name, key):
    (TOUCH_DIR / "{}_key.json".format(name)).write_text(json.dumps(key), encoding="utf-8")


if TOUCH_ON:
    PERSON_W = ROOT / "notebooks" / "yolo11x.pt"
    if not PERSON_W.is_file():
        PERSON_W = yolo_weights("yolo11x.pt")
    PERSON_SHA = sha256(PERSON_W)
    person_model = YOLO(str(PERSON_W))
    print("COCO person & ball detector: {} (sha256 {}…)".format(PERSON_W.name, PERSON_SHA[:12]))

### 7b.1 Three-class kit-colour classifier (H / A / other)

Samples are taken every 80 frames (tactical) / 40 frames (broadcast). Person boxes are matched to the tracking players
projected through the homography (Hungarian matching, only unambiguous pairs) → H/A label from the tracking. Boxes whose
feet fall inside the pitch but > 3 m from every tracked player → 'other' (referees, assistant referees). Accuracy is
measured on held-out 5-minute blocks; a mosaic of 'other' boxes is shown for a visual check.

In [ ]:
KIT_CFG = {"tactical": dict(step=80, imgsz=1920, conf=0.25, half=True), "broadcast": dict(step=40, imgsz=1280, conf=0.3, half=False)}
KIT_PRM = dict(objective="multiclass", num_class=3, learning_rate=0.05, num_leaves=31, min_data_in_leaf=30, verbose=-1, num_threads=4)


def match_tactical(t, boxes):
    """person box → tracking player slot (feet = bottom centre of the box); only unambiguous pairs"""
    PL_ = z["players"][t, :, :2]
    slots = np.flatnonzero(np.isfinite(PL_[:, 0]))
    if not len(slots) or not len(boxes):
        return {}
    pp = m_to_px_t(t, PL_[slots])
    feet = np.c_[(boxes[:, 0] + boxes[:, 2]) / 2, boxes[:, 3]]
    d_ = np.hypot(feet[:, None, 0] - pp[None, :, 0], feet[:, None, 1] - pp[None, :, 1])
    ppm = np.hypot(*(m_to_px_t(t, PL_[slots] + [0.0, 1.0]) - pp).T)
    cost = np.where(d_ < np.maximum(0.6 * ppm, 6.0)[None, :], d_, 1e6)
    r_, c_ = linear_sum_assignment(cost)
    out = {}
    for i_, j_ in zip(r_, c_):
        others = np.delete(d_[i_], j_)
        if cost[i_, j_] < 1e6 and (not len(others) or others.min() > 1.5 * d_[i_, j_] + 3):
            out[i_] = 0 if SLOT_IS_HOME[slots[j_]] else 1
    return out


def kit_samples(source):
    cfg = KIT_CFG[source]
    X_, y_, fr_, tiles = [], [], [], []
    if source == "tactical":
        frames_ = np.flatnonzero(GRADED & H_OK)[::cfg["step"]]
        rd_ = FrameReader(VIDEO_T)
    else:
        frames_ = np.flatnonzero(GRADED & (bc_view == 1))[::cfg["step"]]
        rd_ = FrameReader(VIDEO_B)
    for n_, t in enumerate(frames_):
        if source == "tactical":
            img = rd_.read(t)
        else:
            b = bframe(t)
            if b < 0 or b >= len(PB) or not HB_OK[b]:
                continue
            img = rd_.read(b)
        if img is None:
            continue
        boxes = person_model.predict(img, imgsz=cfg["imgsz"], classes=[0], conf=cfg["conf"], half=cfg["half"] and YO_HALF,
                                     device=YO_DEV, verbose=False)[0].boxes.xyxy.cpu().numpy()
        if not len(boxes):
            continue
        feet = np.c_[(boxes[:, 0] + boxes[:, 2]) / 2, boxes[:, 3]]
        if source == "tactical":
            lab_ = match_tactical(t, boxes)
            fm = px_to_m_t(t, feet)
            PL_ = z["players"][t, :, :2]
            used = set(lab_)
        else:
            slots = np.flatnonzero(np.isfinite(PB[b, :, 0]))
            lab_, used = {}, set()
            if len(slots):
                pp = m_to_px_b(b, PB[b, slots])
                d_ = np.hypot(feet[:, None, 0] - pp[None, :, 0], feet[:, None, 1] - pp[None, :, 1])
                cost = np.where(d_ < 0.5 * (boxes[:, 3] - boxes[:, 1])[:, None], d_, 1e6)
                r_, c_ = linear_sum_assignment(cost)
                for i_, j_ in zip(r_, c_):
                    if cost[i_, j_] < 1e6:
                        used.add(i_)
                        others = np.delete(d_[i_], j_)
                        if not len(others) or others.min() >= 1.5 * d_[i_, j_] + 5:
                            lab_[i_] = 0 if B_HOME[slots[j_]] else 1
            fm = px_to_m_b(b, feet)
            PL_ = PB[b]
        with np.errstate(invalid="ignore"):
            dmin = np.nanmin(np.where(np.isfinite(PL_[None, :, 0]), np.hypot(fm[:, None, 0] - PL_[None, :, 0],
                                                                              fm[:, None, 1] - PL_[None, :, 1]), np.inf), 1)
        inside = (fm[:, 0] > -1) & (fm[:, 0] < PITCH_L + 1) & (fm[:, 1] > -1) & (fm[:, 1] < PITCH_W + 1)
        for i_ in range(len(boxes)):
            if i_ in lab_:
                l_ = lab_[i_]
            elif i_ not in used and inside[i_] and dmin[i_] > 3.0:
                l_ = 2
            else:
                continue
            f_ = torso_feat(img, boxes[i_])
            if f_ is None:
                continue
            X_.append(f_); y_.append(l_); fr_.append(t)
            if l_ == 2 and len(tiles) < 36 and len(tiles) <= n_ // 4:
                x0, y0, x1, y1 = boxes[i_].astype(int)
                tiles.append(cv2.resize(img[max(y0, 0):y1, max(x0, 0):x1], (48, 96)))
    rd_.close()
    return np.array(X_), np.array(y_), np.array(fr_), tiles


def kit_model(source):
    cfg = KIT_CFG[source]
    key = dict(cfg, source=source, person=PERSON_SHA, prm=KIT_PRM, version=1)
    path_, rep_ = TOUCH_DIR / "kit3_{}.txt".format(source), TOUCH_DIR / "kit3_{}.json".format(source)
    if touch_key_ok("kit3_" + source, key) and path_.is_file() and rep_.is_file():
        return lgb.Booster(model_file=str(path_)), json.loads(rep_.read_text(encoding="utf-8"))
    with timer("kit-colour samples {}".format(source)):
        X_, y_, fr_, tiles = kit_samples(source)
    blk = fr_ // (FPS * 300)
    ub = np.unique(blk)
    Pr = np.zeros((len(y_), 3))
    for k_ in range(5):
        te = np.isin(blk, ub[ub % 5 == k_])
        Pr[te] = lgb.train(KIT_PRM, lgb.Dataset(X_[~te], y_[~te]), 300).predict(X_[te])
    cm = np.zeros((3, 3), int)
    np.add.at(cm, (y_, Pr.argmax(1)), 1)
    booster = lgb.train(KIT_PRM, lgb.Dataset(X_, y_), 300)
    booster.save_model(str(path_))
    rep = {"n": np.bincount(y_, minlength=3).tolist(), "acc_blok": float((Pr.argmax(1) == y_).mean()), "cm": cm.tolist()}
    rep_.write_text(json.dumps(rep), encoding="utf-8")
    if len(tiles) >= 12:
        mosaic(tiles[:36], 12, TOUCH_DIR / "kit3_{}_lain.png".format(source))
    touch_key_save("kit3_" + source, key)
    return booster, rep


KIT = {}
if TOUCH_ON:
    for _s in ("tactical", "broadcast") if VIDEO_B is not None else ("tactical",):
        KIT[_s], _rep = kit_model(_s)
        print("{}: samples H {} · A {} · other {} · accuracy on held-out 5-minute blocks {:.3f}".format(_s, *_rep["n"], _rep["acc_blok"]))
        print("  matrix (rows true H/A/other, columns predicted):", _rep["cm"])
        check("kit classifier {}: held-out block accuracy ≥ 0.9".format(_s), _rep["acc_blok"] >= 0.9, "{:.3f}".format(_rep["acc_blok"]))
        if (TOUCH_DIR / "kit3_{}_lain.png".format(_s)).is_file():
            show_image(TOUCH_DIR / "kit3_{}_lain.png".format(_s), "Examples of 'other' boxes ({}): should be referees / assistant referees".format(_s), width=12)

### 7b.2 Touches in the tactical video

Candidates = |Δv| peaks of the anchored video ball (section 2b) during live play. Every frame t−3 … t+3 that has a YOLO
ball is read once (frames shared by several candidates are not read again).

In [ ]:
DV_N, DV_THR, DV_NMS = 4, 4.0, 4


def dv_fit(x, n):
    """|v after − v before| per frame; least-squares lines on [t−n, t] and [t, t+n] (NaN if a point is missing)"""
    W_ = np.lib.stride_tricks.sliding_window_view(x, n + 1, axis=0)
    tt = (np.arange(n + 1) - n / 2) / FPS
    slope = (W_ * tt).sum(-1) / (tt ** 2).sum()
    out = np.full(len(x), np.nan)
    out[n:len(x) - n] = np.hypot(*(slope[n:] - slope[:len(x) - 2 * n]).T)
    return out


def dv_peaks(dv, mask):
    x_ = np.where(np.isfinite(dv), dv, -1.0)
    pad = np.r_[np.full(DV_NMS, -1.0), x_, np.full(DV_NMS, -1.0)]
    peak = x_ >= np.lib.stride_tricks.sliding_window_view(pad, 2 * DV_NMS + 1).max(1)
    return np.flatnonzero(peak & (x_ > DV_THR) & mask)


def fill_short(x, ok, gap, live=None):
    x = x.copy()
    d_ = np.diff(np.r_[1, ok.astype(np.int8), 1])
    for a_, b_ in zip(np.flatnonzero(d_ == -1), np.flatnonzero(d_ == 1)):
        if 0 < a_ and b_ < len(x) and b_ - a_ <= gap and (live is None or live[a_ - 1:b_ + 1].all()):
            x[a_:b_] = x[a_ - 1] + (x[b_] - x[a_ - 1]) * (np.arange(1, b_ - a_ + 1) / (b_ - a_ + 1))[:, None]
    return x


def touch_table(cand, frames_info):
    """frames_info: frame → (distance, team); per 7-frame candidate → team vote, nearest contact"""
    ndf = np.full((len(cand), 7), np.nan, np.float32)
    tmf = np.full((len(cand), 7), -2, np.int8)
    for i_, t in enumerate(cand):
        for j_ in range(7):
            v_ = frames_info.get(int(t + j_ - 3))
            if v_ is not None:
                ndf[i_, j_], tmf[i_, j_] = v_
    c_ = ndf < CONTACT
    votes = np.stack([((tmf == 0) & c_).sum(1), ((tmf == 1) & c_).sum(1)], 1)
    obs = np.where(votes.sum(1) == 0, -1, np.where(votes[:, 0] > votes[:, 1], 0, np.where(votes[:, 1] > votes[:, 0], 1, -1)))
    ndw = np.where(np.isfinite(ndf), ndf, np.inf)
    return dict(cand=cand, frame=cand + np.argmin(ndw, 1) - 3, obs=obs.astype(np.int8), ndmin=ndw.min(1).astype(np.float32),
                ndf=ndf, tmf=tmf)


def read_contacts(frames_, video, to_video, ball_px, kit, imgsz, conf, label):
    info, n_ref, t0_ = {}, 0, time.time()
    rd_ = FrameReader(video)
    batch_ = []

    def _flush():
        nonlocal n_ref
        res_ = person_model.predict([im_ for _, im_ in batch_], imgsz=imgsz, classes=[0], conf=conf, half=False,
                                    device=YO_DEV, verbose=False)
        for (g_, im_), r_ in zip(batch_, res_):
            nd_, tm_, nr_ = frame_contact(im_, r_.boxes.xyxy.cpu().numpy(), ball_px[g_], kit)
            info[int(g_)] = (nd_, tm_)
            n_ref += nr_
        batch_.clear()

    for n_, g in enumerate(frames_):
        img = rd_.read(to_video(g))
        if img is not None:
            batch_.append((g, img))
        if len(batch_) == 8:
            _flush()
        if n_ % 5000 == 0:
            print("  {} {}/{} frames · {:.0f} s".format(label, n_, len(frames_), time.time() - t0_), flush=True)
    if batch_:
        _flush()
    rd_.close()
    return info, n_ref


# tactical video ball: anchored, live ball, holes ≤ 2 frames filled linearly
VB = np.where((YO_ANCH & LIVE)[:, None], YO_BALL.astype(np.float64), np.nan)
VB = fill_short(VB, np.isfinite(VB[:, 0]), 2, LIVE)
YO_PX = np.full((N_FRAMES, 2), np.nan)
_f = np.flatnonzero(np.isfinite(YO_BALL[:, 0]) & (sel_ >= 0))
_bx = DET_XYXY[_f, sel_[_f]]
YO_PX[_f] = np.c_[(_bx[:, 0] + _bx[:, 2]) / 2, (_bx[:, 1] + _bx[:, 3]) / 2]
_P = z["players"][..., :2].astype(np.float64)
_dm = np.full(N_FRAMES, np.inf)
for _a in range(0, N_FRAMES, 20000):
    with np.errstate(invalid="ignore"):
        _d = np.hypot(_P[_a:_a + 20000, :, 0] - VB[_a:_a + 20000, None, 0], _P[_a:_a + 20000, :, 1] - VB[_a:_a + 20000, None, 1])
    _dm[_a:_a + 20000] = np.where(np.isfinite(_d), _d, np.inf).min(1)
_dm2 = np.lib.stride_tricks.sliding_window_view(np.r_[np.full(2, np.inf), _dm, np.full(2, np.inf)], 5).min(1)
cand_t = dv_peaks(dv_fit(VB, DV_N), LIVE & np.isfinite(_dm2))
print("tactical touch candidates: {} ({:.1f} per live-ball minute)".format(len(cand_t), len(cand_t) / (LIVE.sum() / FPS / 60)))

TT = None
if TOUCH_ON:
    key_t = {"cand": hashlib.sha256(cand_t.tobytes()).hexdigest(), "ball": hashlib.sha256(np.nan_to_num(YO_PX).tobytes()).hexdigest(),
             "kit": hashlib.sha256((TOUCH_DIR / "kit3_tactical.txt").read_bytes()).hexdigest(), "imgsz": 1920, "conf": 0.25}
    if touch_key_ok("tactical", key_t) and (TOUCH_DIR / "tactical.npz").is_file():
        TT = dict(np.load(TOUCH_DIR / "tactical.npz"))
    else:
        _fr = np.unique((cand_t[:, None] + np.arange(-3, 4)[None]).ravel())
        _fr = _fr[(_fr >= 0) & (_fr < N_FRAMES)]
        _fr = _fr[np.isfinite(YO_PX[_fr, 0])]
        with timer("tactical contact ({} frames)".format(len(_fr))):
            _info, _nref = read_contacts(_fr, VIDEO_T, lambda g: g, YO_PX, KIT["tactical"], 1920, 0.25, "tactical")
        TT = touch_table(cand_t, _info)
        TT["n_ref"] = np.array(_nref)
        np.savez_compressed(TOUCH_DIR / "tactical.npz", **TT)
        touch_key_save("tactical", key_t)
    _c = TT["ndmin"] < CONTACT
    print("tactical: contact {} of {} candidates ({:.0%}) · clear team {} · touching referee boxes (dropped) {}".format(
        _c.sum(), len(_c), _c.mean(), (_c & (TT["obs"] >= 0)).sum(), int(TT["n_ref"])))

### 7b.3 Touches in the broadcast video

Only on frames whose broadcast is aligned with the tactical recording (`bc_view == 1`, section 2).

1. **Broadcast ball:** on frames where the tactical YOLO ball is missing (± 12 frames) and around tactical candidates
   without contact, YOLO11x COCO 'sports ball' in a 640 px crop around the projected merged ball; nearest detection
   ≤ 120 px.
2. **Candidates:** |Δv| peaks of the broadcast ball (metres, broadcast homography) that are not within ± 4 frames of a
   tactical touch, plus tactical candidates without contact.
3. Contact and team as for the tactical video, with the broadcast kit classifier.

In [ ]:
BC_CROP, BC_MAXD, BC_BALL_CONF = 640, 120.0, 0.1
TB = None
if TOUCH_ON and VIDEO_B is not None and TT is not None:
    _tac_contact = TT["frame"][TT["ndmin"] < CONTACT]
    _tac_nocontact = TT["cand"][~(TT["ndmin"] < CONTACT)]
    # (1) broadcast ball
    _miss = LIVE & ~np.isfinite(VB[:, 0])
    _near_miss = np.convolve(_miss.astype(float), np.ones(25), "same") > 0
    _win = np.zeros(N_FRAMES, bool)
    for _t in _tac_nocontact:
        _win[max(_t - 7, 0):_t + 8] = True
    _B = np.array([bframe(t) for t in range(N_FRAMES)])
    _ok = LIVE & (bc_view == 1) & (_B >= 0) & (_B < len(HB_OK))
    _ok[_ok] &= HB_OK[_B[_ok]]
    _bm = z["ball_poss"].astype(float)
    _ok &= np.isfinite(_bm[:, 0])
    F_bc = np.flatnonzero(_ok & (_near_miss | _win))
    key_bb = {"frames": hashlib.sha256(F_bc.tobytes()).hexdigest(), "person": PERSON_SHA, "crop": BC_CROP, "maxd": BC_MAXD,
              "conf": BC_BALL_CONF}
    if touch_key_ok("bc_ball", key_bb) and (TOUCH_DIR / "bc_ball.npz").is_file():
        BCB = dict(np.load(TOUCH_DIR / "bc_ball.npz"))
    else:
        out_ = np.full((len(F_bc), 6), np.nan, np.float32)
        rd_ = FrameReader(VIDEO_B)
        batch_ = []

        def _flush_ball():
            res_ = person_model.predict([c_ for c_, _ in batch_], imgsz=BC_CROP, classes=[32], conf=BC_BALL_CONF,
                                        half=YO_HALF, device=YO_DEV, verbose=False)
            for (_, (k_, x0, y0, uv)), r_ in zip(batch_, res_):
                bx = r_.boxes.xyxy.cpu().numpy()
                if len(bx):
                    c_ = np.c_[(bx[:, 0] + bx[:, 2]) / 2 + x0, (bx[:, 1] + bx[:, 3]) / 2 + y0]
                    d_ = np.hypot(*(c_ - uv).T)
                    j_ = int(np.argmin(d_))
                    if d_[j_] <= BC_MAXD:
                        out_[k_] = [c_[j_, 0], c_[j_, 1], bx[j_, 2] - bx[j_, 0], bx[j_, 3] - bx[j_, 1], r_.boxes.conf[j_].item(), d_[j_]]
            batch_.clear()

        with timer("broadcast ball ({} frames)".format(len(F_bc))):
            for k_, t in enumerate(F_bc):
                b = int(_B[t])
                img = rd_.read(b)
                if img is None:
                    continue
                uv = m_to_px_b(b, _bm[t][None])[0]
                if not (-50 <= uv[0] < img.shape[1] + 50 and -50 <= uv[1] < img.shape[0] + 50):
                    continue
                x0 = int(np.clip(uv[0] - BC_CROP / 2, 0, img.shape[1] - BC_CROP))
                y0 = int(np.clip(uv[1] - BC_CROP / 2, 0, img.shape[0] - BC_CROP))
                batch_.append((img[y0:y0 + BC_CROP, x0:x0 + BC_CROP], (k_, x0, y0, uv)))
                if len(batch_) == 32:
                    _flush_ball()
            if batch_:
                _flush_ball()
        rd_.close()
        m_ = np.full((len(F_bc), 2), np.nan, np.float32)
        for k_ in np.flatnonzero(np.isfinite(out_[:, 0])):
            m_[k_] = px_to_m_b(int(_B[F_bc[k_]]), out_[k_, :2][None])[0]
        BCB = dict(frame=F_bc, px=out_, m=m_)
        np.savez_compressed(TOUCH_DIR / "bc_ball.npz", **BCB)
        touch_key_save("bc_ball", key_bb)
    _det_ok = np.isfinite(BCB["px"][:, 0])
    print("broadcast ball: searched in {} frames · detected {:.0%} · median size {:.0f} px".format(
        len(BCB["frame"]), _det_ok.mean(), np.nanmedian(BCB["px"][:, 3])))

    # (2) broadcast candidates
    BM = np.full((N_FRAMES, 2), np.nan)
    BPX = np.full((N_FRAMES, 2), np.nan)
    BM[BCB["frame"]] = BCB["m"]
    BPX[BCB["frame"]] = BCB["px"][:, :2]
    _det = np.isfinite(BPX[:, 0])
    _near_tac = np.zeros(N_FRAMES, bool)
    for _t in _tac_contact:
        _near_tac[max(_t - 4, 0):_t + 5] = True
    _ca = dv_peaks(dv_fit(fill_short(BM, _det, 3), DV_N), LIVE)
    _ca = _ca[~_near_tac[_ca]]
    cand_b = np.r_[_ca, _tac_nocontact]
    src_b = np.r_[np.zeros(len(_ca), np.int8), np.ones(len(_tac_nocontact), np.int8)]
    _o = np.argsort(cand_b, kind="stable")
    cand_b, src_b = cand_b[_o], src_b[_o]
    print("broadcast candidates: from the broadcast ball {} · from tactical candidates without contact {}".format(
        (src_b == 0).sum(), (src_b == 1).sum()))

    # (3) broadcast contact
    key_tb = {"cand": hashlib.sha256(cand_b.tobytes()).hexdigest(), "ball": hashlib.sha256(np.nan_to_num(BPX).tobytes()).hexdigest(),
              "kit": hashlib.sha256((TOUCH_DIR / "kit3_broadcast.txt").read_bytes()).hexdigest(), "imgsz": 1280, "conf": 0.3}
    if touch_key_ok("broadcast", key_tb) and (TOUCH_DIR / "broadcast.npz").is_file():
        TB = dict(np.load(TOUCH_DIR / "broadcast.npz"))
    else:
        _fr = np.unique((cand_b[:, None] + np.arange(-3, 4)[None]).ravel())
        _fr = _fr[(_fr >= 0) & (_fr < N_FRAMES)]
        _fr = _fr[_det[_fr] & (bc_view[_fr] == 1)]
        with timer("broadcast contact ({} frames)".format(len(_fr))):
            _info, _nref = read_contacts(_fr, VIDEO_B, bframe, BPX, KIT["broadcast"], 1280, 0.3, "broadcast")
        TB = touch_table(cand_b, _info)
        TB["src"], TB["n_ref"] = src_b, np.array(_nref)
        np.savez_compressed(TOUCH_DIR / "broadcast.npz", **TB)
        touch_key_save("broadcast", key_tb)
    _c = TB["ndmin"] < CONTACT
    print("broadcast: contact {} of {} candidates · clear team {} · touching referee boxes (dropped) {}".format(
        _c.sum(), len(_c), (_c & (TB["obs"] >= 0)).sum(), int(TB["n_ref"])))

### 7b.4 Touch table and visual check

One row per touch with contact: frame, team (0 H / 1 A / −1 unclear), source, evidence weight. Mosaic: random tactical
touches **outside** the public window, frames t−4, t, t+4 around the ball (red circle = the ball at the touch).

In [ ]:
TOUCH_F = np.zeros(0, np.int64)
TOUCH_TEAM = np.zeros(0, np.int8)
TOUCH_SRC = np.zeros(0, np.int8)
TOUCH_W = np.zeros(0)
SRC_LABEL = {0: "tactical", 1: "broadcast_bola", 2: "broadcast_kandidat"}
if TT is not None:
    _ct = TT["ndmin"] < CONTACT
    parts = [(TT["frame"][_ct], TT["obs"][_ct], np.zeros(_ct.sum(), np.int8))]
    if TB is not None:
        _cb = TB["ndmin"] < CONTACT
        parts.append((TB["frame"][_cb], TB["obs"][_cb], (1 + TB["src"][_cb]).astype(np.int8)))
    TOUCH_F = np.concatenate([p_[0] for p_ in parts]).astype(np.int64)
    TOUCH_TEAM = np.concatenate([p_[1] for p_ in parts]).astype(np.int8)
    TOUCH_SRC = np.concatenate([p_[2] for p_ in parts])
    _o = np.argsort(TOUCH_F, kind="stable")
    TOUCH_F, TOUCH_TEAM, TOUCH_SRC = TOUCH_F[_o], TOUCH_TEAM[_o], TOUCH_SRC[_o]
    TOUCH_W = np.array([TOUCH_BETA[SRC_LABEL[s_]] for s_ in TOUCH_SRC])
    touch_tab = pd.DataFrame({"sentuhan": pd.Series(TOUCH_SRC).map(SRC_LABEL).value_counts(),
                              "clear team": pd.Series(TOUCH_SRC[TOUCH_TEAM >= 0]).map(SRC_LABEL).value_counts()})
    touch_tab["weight β"] = [TOUCH_BETA[k_] for k_ in touch_tab.index]
    print(touch_tab.to_string())
    print("touches per live-ball minute: {:.1f}".format(len(TOUCH_F) / (LIVE.sum() / FPS / 60)))
    check("video touches present (≥ 1,000 with contact)", len(TOUCH_F) >= 1000, str(len(TOUCH_F)))

    _pool = np.flatnonzero((TOUCH_SRC == 0) & OUT_PUB[TOUCH_F])
    _pool = _pool[[np.isfinite(YO_PX[[t - 4, t, t + 4], 0]).all() for t in TOUCH_F[_pool]]]
    _pick = np.sort(np.random.default_rng(0).choice(_pool, min(8, len(_pool)), replace=False))
    tiles_, rd_ = [], FrameReader(VIDEO_T)
    for i_ in _pick:
        t = int(TOUCH_F[i_])
        cu, cv_ = YO_PX[t]
        for h_ in (-4, 0, 4):
            img = rd_.read(t + h_)
            tl = crop_zoom(img, cu, cv_, half=48, zoom=2)
            bu, bv = YO_PX[t + h_]
            cv2.circle(tl, (int((bu - cu + 48) * 2), int((bv - cv_ + 48) * 2)), 10, (0, 0, 255) if h_ == 0 else (0, 255, 255), 1)
            cv2.putText(tl, "{} {}".format(vtime(t) if h_ == -4 else "{:+d}".format(h_), "HA?"[TOUCH_TEAM[i_]] if h_ == -4 else ""),
                        (4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            tiles_.append(tl)
    rd_.close()
    if tiles_:
        mosaic(tiles_, 6, TOUCH_DIR / "sentuhan_contoh.png")
        show_image(TOUCH_DIR / "sentuhan_contoh.png", "Random tactical touches outside the public window (t−4 · t · t+4), two per row", width=13)

## 8. H/A/D decoder

**What this step does.** Turns the per-frame log-odds and the dead-ball segments into the final labels.

The final labels are chosen by a 3-state Viterbi per half (Rabiner, 1989):

| State | Emission on live frames | Emission on `D` frames (section 4) |
|---|---|---|
| `H` | +½ · log-odds | −10⁴ |
| `A` | −½ · log-odds | −10⁴ |
| `D` | −10⁴ | +10⁴ |

- **H↔A switching cost = 10** (½ log-odds units): a switch happens only when the evidence for the new team is strong
  enough for long enough. This encodes the official definition that a touch without control is not a switch. The value
  was chosen on external cross-validation (the curve is flat over 5–20).
- Entering/leaving `D` is free because `D` was already decided in section 4.
- **Restart prior:** the first 2 s after a corner/goal kick/free kick = the taker team.
- Frames outside the halves = `X` (not graded).
- **`SNAP_RECEPTION` option (off by default).** An H↔A switch made while the ball is in flight (> 2 m from every
  player) is delayed until the receiving team touches the ball (≤ 1.5 m and closer than the opponent for 0.2 s, at
  most 2 s). The idea came from the 09:04–09:16 clip, but on external cross-validation only 2–5 switches per match
  happen while the ball is in flight and the score drops slightly (−0.0004; two-way snapping −0.002), so it is not used
  for the submission. The early switch in that clip was caused by a floating tactical ball, which 2b handles.

In [ ]:
RESTART_S = 2.0
STATES = np.array(["H", "A", "D"])


def restart_prior(L, rs, secs, strength=1e3):
    L = np.array(L, dtype=float, copy=True)
    rs = rs[rs.taker.isin(["H", "A"])]
    for b, team in zip(rs.restart_frame.to_numpy(), rs.taker.to_numpy()):
        L[b:b + int(secs * FPS)] += strength if team == "H" else -strength
    return L, len(rs)


def decode_possession(L, dmask, switch_cost):
    emis = np.zeros((N_FRAMES, 3))
    emis[:, 0], emis[:, 1] = 0.5 * L, -0.5 * L
    emis[dmask, 0] = emis[dmask, 1] = -1e4
    emis[dmask, 2] = 1e4
    emis[~dmask, 2] = -1e4
    trans = np.array([[0.0, switch_cost, 0.0], [switch_cost, 0.0, 0.0], [0.0, 0.0, 0.0]])
    out = np.full(N_FRAMES, "X", dtype="<U1")
    for _h, (_a, _b) in HALVES.items():
        out[_a:_b + 1] = STATES[viterbi(emis[_a:_b + 1], trans)]
    return out


L_dec, n_restart = restart_prior(L_seq.astype(np.float64), restarts, RESTART_S)
with timer("viterbi H/A/D"):
    labels = decode_possession(L_dec, d_mask, float(MODEL_META["decoder_cost"]))
    labels_trk = decode_possession(restart_prior(L_seq_trk.astype(np.float64), restarts_trk, RESTART_S)[0], d_mask_trk,
                                  float(MODEL_META["decoder_cost"]))


def snap_reception(lab_, ball_xy, free_m=2.0, R=1.5, w_s=2.0, hold=5):
    """delay an H↔A switch made while the ball is in flight until the receiving team's first touch"""
    out_ = lab_.copy()
    Pz = z["players"][..., :2]
    with np.errstate(all="ignore"):
        dH_ = np.nanmin(np.hypot(Pz[:, SLOT_IS_HOME, 0] - ball_xy[:, None, 0], Pz[:, SLOT_IS_HOME, 1] - ball_xy[:, None, 1]), 1)
        dA_ = np.nanmin(np.hypot(Pz[:, ~SLOT_IS_HOME, 0] - ball_xy[:, None, 0], Pz[:, ~SLOT_IS_HOME, 1] - ball_xy[:, None, 1]), 1)
    chg_ = np.flatnonzero(lab_[1:] != lab_[:-1]) + 1
    n_ = 0
    for i_, s_ in enumerate(chg_):
        if lab_[s_] not in ("H", "A") or lab_[s_ - 1] not in ("H", "A"):
            continue
        if not (np.isfinite(dH_[s_]) and min(dH_[s_], dA_[s_]) > free_m):
            continue
        dY, dX = (dH_, dA_) if lab_[s_] == "H" else (dA_, dH_)
        hi_ = min(s_ + int(w_s * FPS), (chg_[i_ + 1] if i_ + 1 < len(chg_) else N_FRAMES) - 1)
        ctrl_ = np.where(np.isfinite(dY) & np.isfinite(dX), (dY <= R) & (dY < dX), False)
        for t_ in range(s_, hi_ + 1):
            if ctrl_[t_:t_ + hold].all():
                if t_ > s_:
                    out_[s_:t_] = lab_[s_ - 1]
                    n_ += 1
                break
    return out_, n_


if SNAP_RECEPTION:
    labels, n_snap = snap_reception(labels, z["ball_poss"].astype(float))
    print("SNAP_RECEPTION on: {} switches delayed until the ball is received".format(n_snap))
g = labels[GRADED]
chg = np.flatnonzero(g[1:] != g[:-1]) + 1
starts, ends = np.r_[0, chg], np.r_[chg, len(g)]
spell_len = (ends - starts)[np.isin(g[starts], ["H", "A"])] / FPS
print("restart prior: {} restarts | possessions over the whole match: {} | mean {:.1f} s | < 2 s {:.1f}%".format(
    n_restart, len(spell_len), spell_len.mean(), 100 * (spell_len < 2).mean()))

### base decoder: log-odds + video touch evidence (10 fps)

The decoder without touches above is still computed for comparison (`labels_yolo`). The base decoder works at **10 fps** (the model's
resolution; the same as in cross-validation and simulation) on the same log-odds (with the restart prior):

- Every touch with a clear team adds **±β** to the log-odds from the touch step until the next touch, for at most
  `TOUCH_TMAX_S` (1 s). β per source: tactical 4, broadcast ball 2, broadcast candidate 0.5.
- A 2-state H/A Viterbi per live-ball run (from the section-4 `D` detector), with the same switching cost (10).
- Back to 25 fps through the nearest 10 fps step; `D` and `X` are unchanged from the decoder without touches.

Every parameter was locked from simulation and visual checks before looking at the public GT (experiment log #78–#87).

In [ ]:
labels_yolo = labels.copy()


def decode_touch(L, f_touch, team, w, tmax_s, cost):
    idx10 = bar10["idx"]
    e10 = L[idx10].astype(np.float64)
    T10 = len(idx10)
    ev = np.zeros(T10)
    if len(f_touch):
        s_ = np.clip(np.searchsorted(idx10, f_touch), 0, T10 - 1)
        keep = np.r_[True, s_[1:] != s_[:-1]]
        s_, tm_, w_ = s_[keep], team[keep], w[keep]
        tmax = int(round(tmax_s * FPS10))
        for a_, n_, t_, ww in zip(s_, np.r_[s_[1:], T10], tm_, w_):
            if t_ >= 0:
                ev[a_:min(n_, a_ + tmax)] = ww if t_ == 0 else -ww
    dead10 = (d_mask | ~GRADED)[idx10]
    p10 = np.full(T10, "D", dtype="<U1")
    trans = np.array([[0.0, cost], [cost, 0.0]])
    for a_, b_ in zip(*runs(~dead10)):
        x_ = e10[a_:b_] + ev[a_:b_]
        p10[a_:b_] = np.array(["H", "A"])[viterbi(np.c_[0.5 * x_, -0.5 * x_], trans)]
    f_ = np.arange(N_FRAMES)
    j_ = np.clip(np.searchsorted(idx10, f_), 1, T10 - 1)
    j_ = np.where(np.abs(idx10[j_ - 1] - f_) <= np.abs(idx10[j_] - f_), j_ - 1, j_)
    out = p10[j_].astype("<U1")
    bad = np.flatnonzero(GRADED & ~d_mask & (out == "D"))             # live 25 fps frames whose nearest 10 fps step is dead
    for t_ in bad:
        jj = j_[t_] + np.array([1, -1, 2, -2])
        jj = jj[(jj >= 0) & (jj < T10)]
        cand_ = [p10[x] for x in jj if p10[x] in ("H", "A")]
        out[t_] = cand_[0] if cand_ else "H"
    out[GRADED & d_mask] = "D"
    out[~GRADED] = "X"
    return out


if TOUCH_ON and len(TOUCH_F):
    with timer("base decoder"):
        labels = decode_touch(L_dec, TOUCH_F, TOUCH_TEAM, TOUCH_W, TOUCH_TMAX_S, float(MODEL_META["decoder_cost"]))
else:
    print("TOUCH_STAGE off or no touches: labels = decoder without touches")


def spell_stats(lab_):
    g_ = lab_[GRADED]
    c_ = np.flatnonzero(g_[1:] != g_[:-1]) + 1
    s_, e_ = np.r_[0, c_], np.r_[c_, len(g_)]
    return (e_ - s_)[np.isin(g_[s_], ["H", "A"])] / FPS


def agree_touch(lab_, src_set):
    """share of live frames (outside the public window, ≤ 2 s after a touch) whose label = team of the last toucher"""
    m_ = np.isin(TOUCH_SRC, list(src_set)) & (TOUCH_TEAM >= 0)
    f_, t_ = TOUCH_F[m_], TOUCH_TEAM[m_]
    last = np.full(N_FRAMES, -1, np.int8)
    for a_, n_, tm_ in zip(f_, np.r_[f_[1:], N_FRAMES], t_):
        last[a_:min(n_, a_ + 2 * FPS)] = tm_
    sel = LIVE & OUT_PUB & (last >= 0)
    return float((lab_[sel] == np.where(last[sel] == 0, "H", "A")).mean()) if sel.any() else float("nan")


cmp_rows = []
for name_, lab_ in (("yoloball", labels_yolo), ("base", labels)):
    d_ = spell_stats(lab_)
    cmp_rows.append({"decoder": name_, "possession": len(d_), "mean s": round(d_.mean(), 1), "< 2 s": round((d_ < 2).mean(), 3),
                     "cocok penyentuh tactical": round(agree_touch(lab_, {0}), 3),
                     "cocok penyentuh broadcast": round(agree_touch(lab_, {1, 2}), 3) if TB is not None else np.nan})
print(pd.DataFrame(cmp_rows).to_string(index=False))
print("frames that differ YOLO-ball → base: {} ({:.2%} of graded frames)".format(int((labels != labels_yolo).sum()),
                                                                     (labels != labels_yolo)[GRADED].mean()))
g = labels[GRADED]
chg = np.flatnonzero(g[1:] != g[:-1]) + 1
starts, ends = np.r_[0, chg], np.r_[chg, len(g)]
spell_len = (ends - starts)[np.isin(g[starts], ["H", "A"])] / FPS

### 8b. Referee whistle: the dead ball starts at the whistle

"The referee determines when the ball is in play" (CHALLENGE.md). The referee's whistle is clearly audible in the
**tactical video audio** (the same file as the images, so audio time = frame / 25). Its fingerprint was read from
certain whistles (kick-off, final whistle), without labels: **two flat parallel tones** at ± 3.75–3.9 kHz with a
frequency ratio of 1.107–1.119 (the pitch follows how hard the whistle is blown, the ratio stays fixed) plus harmonics
at ± 7.5–8.6 kHz. Crowd noise is broadband and human voices are curved lines, so neither passes.

- Spectrogram at 22.05 kHz, window 2048, hop 882 samples (= 1 video frame). Prominence = log power − the mean over
  80–250 Hz on both sides; score = min(prominence of the lower tone at 3.2–4.6 kHz, largest prominence at
  1.09–1.135 × the lower tone).
- Whistle = score ≥ `WHISTLE_THR` for ≥ 0.12 s with a stable tone; blows ≤ 0.4 s apart are merged.
- **Rule (locked before the public check):** a whistle while the ball is predicted live, with a predicted `D` start
  ≤ 3 s later → `D` starts at the whistle. Whistles without a nearby `D` (advantage, offside not given) change nothing.

Evidence without GT (log #95, #97): 5 of 6 certain whistles are at the ≥ 99.9th percentile; a mosaic of 24 random
whistles shows the two-tone pattern almost everywhere; in the rule cases, the mean player speed drops from 2.3 to
1.1 m/s within 2 s after the whistle (11 of 13), so play really stops at the whistle, while the predicted `D` only
starts ± 1 s later.

In [ ]:
import subprocess
import json as _json

WHISTLE_ON = WHISTLE_STAGE != "off" and VIDEO_T is not None
labels_pre_whistle = labels.copy()
AUDIO_DIR = CACHE / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
SR_W, NFFT_W, HOP_W = 22050, 2048, 882          # step = 1 video frame


def whistle_score(x):
    """two-tone score per video frame and the frequency of the lower tone"""
    n = len(x) // HOP_W
    df = SR_W / NFFT_W
    lo_b, hi_b, f_lo, f_hi = int(2800 / df), int(5600 / df), int(3200 / df), int(4600 / df)
    k_in, k_out = int(80 / df), int(250 / df)
    ker = np.zeros(2 * k_out + 1, np.float32)
    ker[:k_out - k_in + 1] = 1
    ker[-(k_out - k_in + 1):] = 1
    ker /= ker.sum()
    win = np.hanning(NFFT_W).astype(np.float32)
    xp = np.r_[np.zeros(NFFT_W // 2, np.float32), x, np.zeros(NFFT_W, np.float32)]
    score, f1 = np.zeros(n, np.float32), np.zeros(n, np.float32)
    for a in range(0, n, 2048):
        b = min(n, a + 2048)
        idx = (np.arange(a, b) * HOP_W)[:, None] + np.arange(NFFT_W)[None]
        lS = np.log10(np.abs(np.fft.rfft(xp[idx] * win, axis=1)[:, lo_b - k_out:hi_b + k_out]) ** 2 + 1e-12)
        nb = np.apply_along_axis(lambda r: np.convolve(r, ker, "same"), 1, lS)
        P = (lS - nb)[:, k_out:-k_out]
        P3 = np.c_[P[:, :1], np.maximum(np.maximum(P[:, :-2], P[:, 1:-1]), P[:, 2:]), P[:, -1:]]
        best, bf = np.full(b - a, -9.0, np.float32), np.zeros(b - a, np.float32)
        for k in range(f_lo, f_hi + 1):
            up = P3[:, int(np.ceil(1.09 * k)) - lo_b:int(np.floor(1.135 * k)) - lo_b + 1].max(1)
            s_ = np.minimum(P3[:, k - lo_b], up)
            m_ = s_ > best
            best[m_], bf[m_] = s_[m_], k * df
        score[a:b], f1[a:b] = best, bf
    return score, f1


def whistle_events(score, f1, thr, min_len=3, merge=10):
    on = score >= thr
    d_ = np.diff(np.r_[0, on.astype(np.int8), 0])
    ev = []
    for a, b in zip(np.flatnonzero(d_ == 1), np.flatnonzero(d_ == -1)):
        if b - a >= min_len and np.ptp(f1[a:b]) <= 120:
            if ev and a - ev[-1][1] <= merge:
                ev[-1][1] = b
            else:
                ev.append([a, b])
    return np.array(ev, int).reshape(-1, 2)


if WHISTLE_ON:
    key = {"video": VIDEO_T.name, "bytes": VIDEO_T.stat().st_size, "sr": SR_W, "nfft": NFFT_W, "hop": HOP_W, "versi": 1}
    feat_p, key_p = AUDIO_DIR / "whistle_feat.npz", AUDIO_DIR / "whistle_key.json"
    if feat_p.is_file() and key_p.is_file() and _json.loads(key_p.read_text()) == key:
        _w = np.load(feat_p)
        W_SCORE, W_F1 = _w["score"], _w["f1"]
        print("whistle score loaded from cache")
    else:
        try:
            import imageio_ffmpeg
            FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
        except ImportError:
            FFMPEG = shutil.which("ffmpeg")
        if FFMPEG is None:
            print("ffmpeg not found (pip install imageio-ffmpeg): section 8b skipped")
            WHISTLE_ON = False
        else:
            with timer("tactical audio → whistle score"):
                raw_p = AUDIO_DIR / "tactical_22k.raw"
                subprocess.run([FFMPEG, "-hide_banner", "-loglevel", "error", "-y", "-i", str(VIDEO_T), "-vn", "-ac", "1",
                                "-ar", str(SR_W), "-f", "s16le", str(raw_p)], check=True)
                W_SCORE, W_F1 = whistle_score(np.fromfile(raw_p, np.int16).astype(np.float32) / 32768)
                raw_p.unlink()                      # raw audio is not stored, only the per-frame score
                np.savez_compressed(feat_p, score=W_SCORE, f1=W_F1)
                key_p.write_text(_json.dumps(key))

if WHISTLE_ON:
    check("tactical audio aligned with the video (± 2 frames)", abs(len(W_SCORE) - N_FRAMES) <= 2,
          "{} audio frames vs {} video frames".format(len(W_SCORE), N_FRAMES))
    _g0 = int(np.flatnonzero(GRADED)[0])
    _k = _g0 - 125 + int(np.argmax(W_SCORE[_g0 - 125:_g0 + 25]))
    check("half-1 kick-off whistle detected (score ≥ threshold)", W_SCORE[_k] >= WHISTLE_THR,
          "score {:.2f} at {:+.2f} s from the first graded frame · percentile {:.4f}".format(
              W_SCORE[_k], (_k - _g0) / FPS, (W_SCORE < W_SCORE[_k]).mean()), critical=False)
    W_EV = whistle_events(W_SCORE, W_F1, WHISTLE_THR)
    gap = int(round(WHISTLE_MAX_GAP_S * FPS))
    cases = []
    for a, b in W_EV:
        if a >= N_FRAMES or not GRADED[a] or labels[a] not in ("H", "A"):
            continue
        n_ = a
        while n_ < N_FRAMES and n_ - a <= gap and labels[n_] in ("H", "A"):
            n_ += 1
        if n_ - a <= gap and n_ < N_FRAMES and labels[n_] == "D":
            cases.append((a, n_))
            labels[a:n_] = np.where(GRADED[a:n_], "D", labels[a:n_])
    _in_g = GRADED[np.minimum(W_EV[:, 0], N_FRAMES - 1)]
    print("whistles: {} ({} on graded frames, {:.2f}/minute) · inside predicted D {:.0%} · rule cases {} · frames changed {} · "
          "D start moved earlier by a median {:.2f} s".format(len(W_EV), _in_g.sum(), _in_g.sum() / (GRADED.sum() / FPS / 60),
                                               d_mask[W_EV[_in_g, 0]].mean(), len(cases), int((labels != labels_pre_whistle).sum()),
                                               np.median([(n_ - a) / FPS for a, n_ in cases]) if cases else 0.0))
    try:                                         # check without GT: do the players slow down after the whistle?
        _v = np.linalg.norm(np.diff(z["players"][:, :, :2].astype(np.float32), axis=0), axis=2) * FPS
        _v[_v > 12] = np.nan
        _sp = np.convolve(np.nan_to_num(np.nanmean(_v, 1), nan=2.0), np.ones(13) / 13, "same")
        _prof = np.array([[_sp[min(a + o, len(_sp) - 1)] for o in (-25, 0, 25, 50)] for a, _ in cases])
        if len(_prof):
            print("mean player speed in the rule cases (m/s), −1 s / whistle / +1 s / +2 s: " +
                  " / ".join("{:.2f}".format(x) for x in _prof.mean(0)) +
                  " · slowing down within 2 s: {} of {}".format(int((_prof[:, 3] < _prof[:, 1]).sum()), len(_prof)))
    except Exception as e:
        print("player-speed check skipped:", e)
else:
    print("WHISTLE_STAGE off: no whistle rule")

g = labels[GRADED]
chg = np.flatnonzero(g[1:] != g[:-1]) + 1
starts, ends = np.r_[0, chg], np.r_[chg, len(g)]
spell_len = (ends - starts)[np.isin(g[starts], ["H", "A"])] / FPS

### 8c. Re-timing the H↔A switch boundaries — *rejected, off by default*

> The nested simulation did not pass the decision rule (log #94), so `REFINE_MODE` resolves to `"off"` and
> `SIM_STAGE = "off"` skips the simulation. The labels stay equal to base. The section is kept for the record.

**Problem.** Diagnosis on the public window (diagnosis only; no parameter was tuned there): base catches 10 of the 12 GT
switches (without touch evidence: 7), but the caught switches are shifted by **± 0.79 s** on average (without touch evidence ± 0.55 s), in both directions
(−1.3 … +1.4 s). If every caught switch were placed exactly at the GT time, the public spellF1 would rise
0.741 → 0.765 (switches ≤ 1 s) / 0.778 (≤ 2 s). A plausible cause: the base touch evidence (±β for ≤ 1 s) pulls the
boundary towards a touch of the wrong team or towards the new team's second touch, and the decoder works at 10 fps.

Because the error goes both ways, a global shift or smoothing does not help (log #90). So re-timing **separates two
decisions**: base still decides *whether* there is a switch and the order of the teams; this section only decides
*when*.

**Method.** For every base H↔A switch at frame `s` (old team → new team), look for a new position `t` in the window
`[s − W, s + W]`, limited by the neighbouring boundaries (remaining possession ≥ 0.2 s), so that **the number of
possessions does not change**:

| mode | position `t` |
|---|---|
| `logodds` | the best single change point according to the **base log-odds** (model + restart prior, *without* touch evidence): maximise Σ support for the old team before `t` + Σ support for the new team after `t` |
| `kurung` (bracket) | like `logodds`, but `t` is limited to lie between the **last clear touch of the old team** and the **first clear touch of the new team** after it (GT definition: owner = last toucher, log #73) |
| `sentuhan` (touch) | `t` = the first clear touch of the new team after the last clear touch of the old team; if there is none, `logodds` with the same limits |

Clear touch = a video touch with a team and weight β ≥ 2 (tactical, broadcast ball), section 7b.

**Validation (without the public GT).** A simulation on the 23 labelled external matches using **out-of-fold**
log-odds (models trained without that match). A simulated touch observer reports touches by the new team at part of the
true switches (coverage 0.5, time ± jitter) and random touches by the owning team inside possessions at the same rate as
in BAR-ATH, with team accuracy 0.8 / 0.9 (BAR-ATH without GT: tactical ± 0.86, broadcast ± 0.75; log #86). Then the base
decoder (±β 4, ≤ 1 s, cost 10) and the three modes × windows of 0.5 / 1 / 2 s. Mode and window are chosen **nested**
(chosen on 4/5 of the matches, scored on 1/5).

**Decision rule, locked before seeing the results.** `REFINE_MODE = "auto"` uses the best configuration over the 23
matches only if, in the main condition (BAR-ATH-like ball gaps), the mean nested Δ spellF1 is > 0 with the lower bound of
the 90% bootstrap interval > 0, **and** no condition × scenario combination has a mean Δ < −0.002. Otherwise `off`
(labels = base).

In [ ]:
# 8c.1  out-of-fold log-odds on the 23 external matches (simulation material). Computed once, then cached.
import gc

import psutil

RETIME_DIR = CACHE / "retime"
RETIME_DIR.mkdir(parents=True, exist_ok=True)
SIM_CONDS = ["celah ala BAR-ATH", "celah bola hidup BAR-ATH", "bola terlihat"]
NAMES_S = list(MODEL_META["training"]["matches"])
OOF_KEY = {"archs": list(SIM_ARCHS), "folds": N_FOLDS, "seed": SEED, "feature_version": FEATURE_VERSION,
           "epochs": EPOCHS, "win": WIN, "batch": BATCH, "n_aug": N_AUG, "mix": MIX, "spell_w": SPELL_W, "x_dtype": "float16",
           "conds": SIM_CONDS, "names": NAMES_S,
           "ball": hashlib.sha256(np.packbits(z["ball_src"] > 0).tobytes()).hexdigest()[:16],
           "dead": hashlib.sha256(np.packbits(d_mask).tobytes()).hexdigest()[:16]}
OOF_P, OOF_KEY_P = RETIME_DIR / "cv_oof.npz", RETIME_DIR / "cv_oof_key.json"
SIM_OOF = {}      # (match, condition) → (10 fps OOF log-odds, 10 fps labels: 1 H · 0 A · −1 dead / unknown)
SIM_ON = str(SIM_STAGE).strip().lower() != "off"


def _load_ext(n):
    if n.startswith("sc"):
        return load_skillcorner(n[2:])
    return load_metrica3() if n == "me3" else load_metrica_csv(int(n[2:]))


if SIM_ON and OOF_P.is_file() and OOF_KEY_P.is_file() and json.loads(OOF_KEY_P.read_text()) == OOF_KEY:
    with np.load(OOF_P) as zf:
        for n_ in NAMES_S:
            for ci_, c_ in enumerate(SIM_CONDS):
                SIM_OOF[(n_, c_)] = (zf["lg_{}_{}".format(n_, ci_)], zf["lab_{}_{}".format(n_, ci_)])
    print("OOF log-odds loaded from cache: {} matches × {} conditions".format(len(NAMES_S), len(SIM_CONDS)))
elif SIM_ON:
    print("OOF cache missing / parameters changed → train {} folds × {} on {} (only once; a 4 GB GPU is enough)".format(
        N_FOLDS, "+".join(SIM_ARCHS), TRAIN_DEV))
    # save RAM (16 GB laptop): drop leftover section-6 training data if present, drop raw data after resampling,
    # store features as float16, and build the evaluation copies per fold only for the held-out matches
    for _n in ("DATA", "EVAL", "EVAL_NG", "EVAL_LIVE", "EXT", "EXT10"):
        globals().pop(_n, None)
    gc.collect()
    with timer("load external data (cache/external, or external_data if not cached yet)"):
        EXT10_S = {}
        for n in NAMES_S:
            EXT10_S[n] = resample10(_load_ext(n))
            gc.collect()
    g_idx = np.flatnonzero(GRADED)
    BALL_BANK = (z["ball_src"] > 0)[g_idx[np.round(np.arange(0, len(g_idx) - 1, 2.5)).astype(int)]]
    l_idx = np.flatnonzero(GRADED & ~d_mask)
    BANK_LIVE = (z["ball_src"] > 0)[l_idx[np.round(np.arange(0, len(l_idx) - 1, 2.5)).astype(int)]]
    EVAL_SPEC = {SIM_CONDS[0]: (1000, BALL_BANK), SIM_CONDS[1]: (3000, BANK_LIVE), SIM_CONDS[2]: (2000, None)}
    rng = np.random.default_rng(SEED)

    def _lean(copy_):
        (X_, S_), lab_ = copy_
        return (X_.astype(np.float16), S_), lab_

    with timer("features + external augmentation"):
        DATA = {n: [_lean(make_copy(EXT10_S[n], rng, None if (MIX and (k // 2) % 2 == 1) else BALL_BANK, swap=k % 2 == 1))
                    for k in range(N_AUG)] for n in NAMES_S}
        gc.collect()
    print("kernel process RAM: {:.1f} GB · free system RAM: {:.1f} GB".format(
        psutil.Process().memory_info().rss / 1e9, psutil.virtual_memory().available / 1e9))
    with timer("cross-validation for the simulation"):
        for k, held in enumerate([NAMES_S[i::N_FOLDS] for i in range(N_FOLDS)]):
            t0_ = time.time()
            nets = [train_model(arch, SEED, [n for n in NAMES_S if n not in held]) for arch in SIM_ARCHS]
            for n_ in held:
                i_ = NAMES_S.index(n_)
                for c_ in SIM_CONDS:
                    seed_, bank_ = EVAL_SPEC[c_]
                    XS_, lab_ = make_copy(EXT10_S[n_], np.random.default_rng(seed_ + i_), bank_)
                    lg_ = np.mean([predict(net, XS_, TRAIN_DEV) for net in nets], 0)
                    SIM_OOF[(n_, c_)] = (lg_.astype(np.float32), np.asarray(lab_).astype(np.int8))
                    del XS_
            del nets
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print("fold {}/{} done ({:.0f} s)".format(k + 1, N_FOLDS, time.time() - t0_), flush=True)
    np.savez_compressed(OOF_P, **{"lg_{}_{}".format(n_, SIM_CONDS.index(c_)): v_[0] for (n_, c_), v_ in SIM_OOF.items()},
                        **{"lab_{}_{}".format(n_, SIM_CONDS.index(c_)): v_[1] for (n_, c_), v_ in SIM_OOF.items()})
    OOF_KEY_P.write_text(json.dumps(OOF_KEY))
    del DATA, EXT10_S
    gc.collect()
    print("OOF log-odds saved in", OOF_P.relative_to(ROOT))
else:
    print("SIM_STAGE = off: simulation skipped")
if SIM_ON:
    check("OOF log-odds complete for the simulation (23 matches × 3 conditions)", len(SIM_OOF) == len(NAMES_S) * len(SIM_CONDS),
          str(len(SIM_OOF)))

In [ ]:
# 8c.2  re-timing + simulated touch observer on the 23 external matches
REFINE_MODES = ["logodds", "kurung", "sentuhan"]
REFINE_WINS = [0.5, 1.0, 2.0]
REFINE_CFGS = ["{}@{}".format(m_, w_) for m_ in REFINE_MODES for w_ in REFINE_WINS]
REFINE_GAP_S = 0.2                         # minimum possession left on either side of a moved boundary
CLEAR_W = 2.0                              # clear touch = weight β ≥ 2 (tactical, broadcast ball)
SIM_BETA, SIM_COVER = 4.0, 0.5
SIM_SCEN = [(0.8, 0), (0.8, 2), (0.9, 0), (0.9, 2)]   # (observer team accuracy, time jitter ± 10 fps steps)

_clear = (TOUCH_TEAM >= 0) & (TOUCH_W >= CLEAR_W) if len(TOUCH_F) else np.zeros(0, bool)
TF_CLEAR, TT_CLEAR = TOUCH_F[_clear], TOUCH_TEAM[_clear]
_live_min = (GRADED & ~d_mask).sum() / FPS / 60
R_TOUCH_MIN = len(TF_CLEAR) / _live_min if len(TF_CLEAR) else 14.0
print("clear BAR-ATH touches: {} ({:.1f} per live-ball minute) → touch rate of the simulated observer".format(
    len(TF_CLEAR), R_TOUCH_MIN))


def refine_switches(lab_, L_, tf, tt, fps, mode, win_s, gap_s=REFINE_GAP_S):
    """Move every H↔A switch within a ± win_s window without adding/removing possessions.
    lab_: H/A/D/X labels · L_: base log-odds (+ = H) without touch evidence · tf, tt: frame & team (0 H / 1 A) of the
    clear touches, sorted. Returns (new labels, shift of each switch in frames)."""
    out = lab_.copy()
    if mode == "off":
        return out, np.zeros(0, int)
    isha = np.isin(lab_, ["H", "A"])
    chg = np.flatnonzero(lab_[1:] != lab_[:-1]) + 1
    W, gap = int(round(win_s * fps)), max(1, int(round(gap_s * fps)))
    tf = np.asarray(tf, np.int64)
    tt = np.asarray(tt)
    moved, last = [], -10 ** 9
    for i, s in enumerate(chg):
        if not (isha[s] and isha[s - 1]):
            last = s
            continue
        new = lab_[s]
        prev_b = chg[i - 1] if i > 0 else 0
        next_b = chg[i + 1] if i + 1 < len(chg) else len(lab_)
        lo, hi = max(s - W, prev_b + gap, last + gap), min(s + W, next_b - gap)
        if not (lo <= s <= hi):                       # neighbouring possession too short: leave it as it is
            last = s
            continue
        sg = (1.0 if new == "H" else -1.0) * np.asarray(L_[lo:hi], np.float64)
        cs = np.r_[0.0, np.cumsum(sg)]
        score = cs[-1] - 2.0 * cs                     # score when the first frame of the new team = lo + j, j = 0 … hi − lo
        cand = np.arange(lo, hi + 1)
        pick = None
        if mode in ("kurung", "sentuhan") and len(tf):
            new_code = 0 if new == "H" else 1
            a_, b_ = np.searchsorted(tf, lo), np.searchsorted(tf, hi, side="right")
            f_, t_ = tf[a_:b_], tt[a_:b_]
            old_f, new_f = f_[t_ != new_code], f_[t_ == new_code]
            last_old = int(old_f.max()) if len(old_f) else None
            after_ = new_f if last_old is None else new_f[new_f > last_old]
            first_new = int(after_.min()) if len(after_) else None
            keep = np.ones(len(cand), bool)
            if last_old is not None:
                keep &= cand > last_old
            if first_new is not None:
                keep &= cand <= first_new
            if mode == "sentuhan" and first_new is not None:
                pick = int(first_new)
            elif keep.any():
                cand = cand[keep]
        if pick is None:
            sc_ = score[cand - lo] - 1e-9 * np.abs(cand - s)   # ties → the one closest to the old position
            pick = int(cand[np.argmax(sc_)])
        if pick > s:
            out[s:pick] = lab_[s - 1]
        elif pick < s:
            out[pick:s] = new
        moved.append(pick - s)
        last = pick
    return out, np.asarray(moved, int)


def sim_touches(lab, rng_, acc, jit, rate_min, cover=SIM_COVER):
    """Simulated touch observer (10 fps): the new team at part of the true switches + random touches by the owning team."""
    T, live = len(lab), lab >= 0
    sw = np.flatnonzero((lab[1:] != lab[:-1]) & (lab[1:] >= 0) & (lab[:-1] >= 0)) + 1
    sw = sw[rng_.random(len(sw)) < cover]
    f1 = np.clip(sw + (rng_.integers(-jit, jit + 1, len(sw)) if jit else 0), 0, T - 1)
    true1 = np.where(lab[sw] == 1, 0, 1)
    lf = np.flatnonzero(live)
    k = min(int(rng_.poisson(rate_min / 60.0 / FPS10 * len(lf))), len(lf))
    f2 = np.sort(rng_.choice(lf, k, replace=False))
    true2 = np.where(lab[f2] == 1, 0, 1)
    f_ = np.r_[f1, f2]
    t_ = np.r_[true1, true2]
    ok = live[f_]
    f_, t_ = f_[ok], t_[ok]
    t_ = np.where(rng_.random(len(t_)) < acc, t_, 1 - t_)
    o = np.argsort(f_, kind="stable")
    return f_[o].astype(np.int64), t_[o].astype(np.int8)


def decode_touch_ext(lg, lab, tf, tt, beta=SIM_BETA, tmax_s=None, cost=None):
    """base decoder (log-odds + ±β until the next touch, ≤ tmax) on 10 fps external data"""
    tmax_s = TOUCH_TMAX_S if tmax_s is None else tmax_s
    cost = float(MODEL_META["decoder_cost"]) if cost is None else cost
    T = len(lab)
    ev = np.zeros(T)
    keep = np.r_[True, tf[1:] != tf[:-1]] if len(tf) else np.zeros(0, bool)
    tf, tt = tf[keep], tt[keep]
    tmax = int(round(tmax_s * FPS10))
    for a_, n_, t_ in zip(tf, np.r_[tf[1:], T], tt):
        ev[a_:min(n_, a_ + tmax)] = beta if t_ == 0 else -beta
    out = np.full(T, "D", dtype="<U1")
    trans = np.array([[0.0, cost], [cost, 0.0]])
    for a_, b_ in zip(*runs(lab >= 0)):
        x_ = np.asarray(lg[a_:b_], np.float64) + ev[a_:b_]
        out[a_:b_] = np.array(["H", "A"])[viterbi(np.c_[0.5 * x_, -0.5 * x_], trans)]
    return out, tf, tt


def spell_fast(pred, lab):
    gtc = pm.label_codes(np.where(lab == 1, "H", np.where(lab == 0, "A", "D")).astype(object))
    prc = pm.label_codes(np.asarray(pred, dtype=object))
    return float(pm.spell_scores(gtc, prc, pm.valid_runs(gtc))["mean_f1"])


def switch_dt(pred, ref, tol):
    """|difference| (frames) from each ref H↔A switch to the nearest same-direction pred switch within ± tol; NaN if missed"""
    def _sw(x):
        c_ = np.flatnonzero((x[1:] != x[:-1]) & np.isin(x[1:], ["H", "A"]) & np.isin(x[:-1], ["H", "A"])) + 1
        return c_, x[c_]
    rs, rt = _sw(ref)
    ps, pt = _sw(pred)
    out = np.full(len(rs), np.nan)
    for j_, (f_, t_) in enumerate(zip(rs, rt)):
        q = ps[(pt == t_) & (np.abs(ps - f_) <= tol)]
        if len(q):
            out[j_] = np.abs(q - f_).min()
    return out


def _lab_str(lab):
    return np.where(lab == 1, "H", np.where(lab == 0, "A", "D"))


SIM = pd.DataFrame()
SIM_PASS, SIM_BEST = False, None
if SIM_ON and len(SIM_OOF):
    rows_ = []
    with timer("re-timing simulation ({} matches × {} conditions × {} scenarios × {} configurations)".format(
            len(NAMES_S), len(SIM_CONDS), len(SIM_SCEN), len(REFINE_CFGS) + 1)):
        for i_, n_ in enumerate(NAMES_S):
            for ci_, c_ in enumerate(SIM_CONDS):
                lg_, lab_ = SIM_OOF[(n_, c_)]
                yolo_ = decode_ext(lg_, lab_, float(MODEL_META["decoder_cost"]))
                ref_ = _lab_str(lab_)
                for si_, (acc_, jit_) in enumerate(SIM_SCEN):
                    rng_ = np.random.default_rng(50000 + 100 * i_ + 10 * ci_ + si_)
                    tf_, tt_ = sim_touches(lab_, rng_, acc_, jit_, R_TOUCH_MIN)
                    base_, tf_, tt_ = decode_touch_ext(lg_, lab_, tf_, tt_)
                    row = {"match": n_, "kondisi": c_, "skenario": si_, "akurasi": acc_, "jitter": jit_,
                           "yoloball": spell_fast(yolo_, lab_), "off": spell_fast(base_, lab_)}
                    dt0 = switch_dt(base_, ref_, 2 * FPS10)
                    row["dt off"] = np.nanmean(dt0) / FPS10
                    row["tangkap"] = np.isfinite(dt0).mean() if len(dt0) else np.nan
                    for cfg_ in REFINE_CFGS:
                        m_, w_ = cfg_.split("@")
                        pr_, _ = refine_switches(base_, lg_, tf_, tt_, FPS10, m_, float(w_))
                        row[cfg_] = spell_fast(pr_, lab_)
                        row["dt " + cfg_] = np.nanmean(switch_dt(pr_, ref_, 2 * FPS10)) / FPS10
                    rows_.append(row)
    SIM = pd.DataFrame(rows_)

    # in-sample summary (all matches): Δ against the simulated base decoder
    summ = pd.DataFrame({c_: (SIM[c_] - SIM["off"]).groupby(SIM.kondisi).mean() for c_ in ["yoloball"] + REFINE_CFGS}).T
    summ = summ[SIM_CONDS]
    summ["|dt| s (mean over conditions)"] = [np.nan if c_ == "yoloball" else SIM["dt " + c_].mean() for c_ in summ.index]
    print("simulated base decoder: spellF1", SIM.groupby("kondisi")["off"].mean().round(4).to_dict(),
          "| mean |dt| {:.2f} s | switches caught {:.1%}".format(SIM["dt off"].mean(), SIM["tangkap"].mean()))
    print("\nin-sample Δ spellF1 against the simulated base decoder (row YOLO-ball = no touch evidence, as a reference):")
    print(summ.round(4).to_string())

    # nested selection: configuration chosen on 4/5 of the matches (mean over all conditions & scenarios), scored on 1/5
    nest_ = []
    for held in [NAMES_S[i::N_FOLDS] for i in range(N_FOLDS)]:
        tr_ = SIM[~SIM.match.isin(held)]
        best_ = max(REFINE_CFGS, key=lambda c_: (tr_[c_] - tr_["off"]).mean())
        te_ = SIM[SIM.match.isin(held)]
        nest_.append(te_[["match", "kondisi", "skenario"]].assign(pilihan=best_, delta=(te_[best_] - te_["off"]).to_numpy()))
    NEST = pd.concat(nest_, ignore_index=True)
    boot_rng = np.random.default_rng(0)
    rows_ = []
    for c_ in SIM_CONDS:
        pm_ = NEST[NEST.kondisi == c_].groupby("match").delta.mean().to_numpy()
        bs_ = boot_rng.choice(pm_, (10000, len(pm_))).mean(1)
        rows_.append({"kondisi": c_, "Δ bersarang": pm_.mean(), "5%": np.percentile(bs_, 5), "95%": np.percentile(bs_, 95),
                      "matches won": "{}/{}".format(int((pm_ > 0).sum()), len(pm_))})
    NEST_SUM = pd.DataFrame(rows_)
    print("\nchoice per fold:", NEST.groupby("pilihan").match.nunique().to_dict())
    print(NEST_SUM.round(4).to_string(index=False))
    worst_ = NEST.groupby(["kondisi", "skenario"]).delta.mean().min()
    main_ = NEST_SUM.iloc[0]
    SIM_BEST = max(REFINE_CFGS, key=lambda c_: (SIM[c_] - SIM["off"]).mean())
    SIM_PASS = bool(main_["Δ bersarang"] > 0 and main_["5%"] > 0 and worst_ > -0.002)
    print("\nbest over 23 matches: {} | nested Δ main condition {:+.4f} [{:+.4f}; {:+.4f}] | worst Δ condition × scenario {:+.4f}"
          " → decision rule: {}".format(SIM_BEST, main_["Δ bersarang"], main_["5%"], main_["95%"], worst_,
                                           "PASS" if SIM_PASS else "does not pass"))

#### Figure: simulation results

Left: nested Δ spellF1 per ball condition (dot = mean, line = 90% bootstrap interval). Right: mean |time difference|
of the caught switches against the true labels, per configuration; dashed line = the simulated base decoder.

In [ ]:
if len(SIM):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    ax = axes[0]
    for k_, r_ in NEST_SUM.reset_index(drop=True).iterrows():
        ax.errorbar(r_["Δ bersarang"], k_, xerr=[[r_["Δ bersarang"] - r_["5%"]], [r_["95%"] - r_["Δ bersarang"]]],
                    fmt="o", color=COL["H"], capsize=4)
    ax.axvline(0, color=COL["D"], lw=1)
    ax.set_yticks(range(len(NEST_SUM)))
    ax.set_yticklabels(NEST_SUM.kondisi)
    ax.set_xlabel("nested Δ spellF1 vs the base decoder")
    ax.set_title("Re-timing: nested simulation", pad=12)
    ax = axes[1]
    dts_ = [SIM["dt " + c_].mean() for c_ in REFINE_CFGS]
    ax.barh(range(len(REFINE_CFGS)), dts_, color=COL["A"])
    ax.axvline(SIM["dt off"].mean(), color=COL["D"], ls="--", lw=1)
    ax.set_yticks(range(len(REFINE_CFGS)))
    ax.set_yticklabels(REFINE_CFGS)
    ax.set_xlabel("|time difference| of caught switches (s)")
    ax.set_title("Timing accuracy", pad=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# 8c.3  apply to BAR-ATH
labels_base = labels.copy()
mode_ = str(REFINE_MODE).strip().lower()
if mode_ == "auto":
    if SIM_PASS:
        mode_, win_ = SIM_BEST.split("@")[0], float(SIM_BEST.split("@")[1])
    else:
        mode_, win_ = "off", 0.0
        print("REFINE_MODE auto: no configuration passes the decision rule (or the simulation is off) → base labels")
else:
    assert mode_ in ["off"] + REFINE_MODES, REFINE_MODE
    win_ = 1.0
    if len(SIM):
        cfgs_ = [c_ for c_ in REFINE_CFGS if c_.startswith(mode_ + "@")]
        if cfgs_:
            win_ = float(max(cfgs_, key=lambda c_: (SIM[c_] - SIM["off"]).mean()).split("@")[1])
if REFINE_WIN_S is not None:
    win_ = float(REFINE_WIN_S)
REFINE_USED = "off" if mode_ == "off" else "{}@{}".format(mode_, win_)
labels, MOVED = refine_switches(labels_base, L_dec, TF_CLEAR, TT_CLEAR, FPS, mode_, win_)
labels[~GRADED] = "X"

check("8c: D and X unchanged", ((labels == "D") == (labels_base == "D")).all() and ((labels == "X") == (labels_base == "X")).all())
check("8c: number of possessions unchanged", len(spell_stats(labels)) == len(spell_stats(labels_base)),
      "{} → {}".format(len(spell_stats(labels_base)), len(spell_stats(labels))))
n_sw = len(MOVED)
print("re-timing: {} | H↔A switches {} | moved {} ({:.0%}) | median |shift| {:.2f} s, p90 {:.2f} s | frames changed {}"
      .format(REFINE_USED, n_sw, int((MOVED != 0).sum()), (MOVED != 0).mean() if n_sw else 0,
              np.median(np.abs(MOVED[MOVED != 0])) / FPS if (MOVED != 0).any() else 0,
              np.percentile(np.abs(MOVED[MOVED != 0]), 90) / FPS if (MOVED != 0).any() else 0,
              int((labels != labels_base).sum())))

# GT-free proxy (outside the public window): distance from a switch to the nearest clear touch of the new team.
# Note: the "sentuhan" (touch) mode snaps to that very touch, so this proxy is biased towards it — descriptive only.
def touch_proxy(lab_):
    c_ = np.flatnonzero((lab_[1:] != lab_[:-1]) & np.isin(lab_[1:], ["H", "A"]) & np.isin(lab_[:-1], ["H", "A"])) + 1
    c_ = c_[OUT_PUB[c_]]
    d_ = []
    for s_ in c_:
        code_ = 0 if lab_[s_] == "H" else 1
        q_ = TF_CLEAR[(TT_CLEAR == code_) & (np.abs(TF_CLEAR - s_) <= FPS)]
        d_.append(np.abs(q_ - s_).min() if len(q_) else np.nan)
    d_ = np.asarray(d_, float)
    return {"switches": len(c_), "new-team touch within ± 1 s": np.isfinite(d_).mean(),
            "≤ 2 frames": (d_ <= 2).mean(), "≤ 5 frames": (d_ <= 5).mean(), "median |Δ| (frames)": np.nanmedian(d_)}


if len(TF_CLEAR):
    print(pd.DataFrame({"base": touch_proxy(labels_base), "retime": touch_proxy(labels)}).round(3).to_string())

g = labels[GRADED]
chg = np.flatnonzero(g[1:] != g[:-1]) + 1
starts, ends = np.r_[0, chg], np.r_[chg, len(g)]
spell_len = (ends - starts)[np.isin(g[starts], ["H", "A"])] / FPS

### 8d. Last toucher before the ball goes out — *the final rule*

**Logic.** Corners and goal kicks (and throw-ins) are awarded to the **opponents** of the team that touched the ball
last. So at the moment the ball goes out, possession belongs to the team that is *not* the restart taker (¬R). On
BAR-ATH, the base prediction gives possession to the restart taker right up to the moment the ball goes out on 17 of 20
corners (diagnosis log #103; example: the public error clip at 11:33).

**Two ways to fix it** (only for restarts whose team is known from Smart Tagging: corners and goal kicks):

| Variant | What it does |
|---|---|
| terminal prior | the log-odds are forced towards ¬R in the last 0.2 s before the ball goes out (the mirror of the restart prior); Viterbi + the model log-odds choose when the switch to ¬R happens |
| proximity insertion | if the label right before the ball goes out = R, the label is replaced by ¬R from the last ¬R touch (nearest ¬R player ≤ 1.5 m and closer than any R player, ≤ 5 s before the ball goes out) until the ball goes out |

**Validation on IDSSE** (7 Bundesliga matches, DFL tracking at 25 fps): the `BallPossession` flag switches at every touch
(touch-semantics labels), `BallStatus` gives the dead ball, and the restart type & team come from the ball position and
the flag after the restart. Log-odds from the **final base model** (it never saw IDSSE), decoded exactly as for BAR-ATH
(cost 10, restart prior). The helper code lives in `idsse_loader.py` and `aturan_keluar.py` ("ball-out rule").

**Decision rule (locked).** The variant with the best mean Δ spellF1 is used for BAR-ATH if the mean Δ over the
7 matches is > 0 **and** the 5% bootstrap lower bound (over matches) is > 0. The public GT is not used.
Result (log #104–#105): proximity insertion **+0.0065 [+0.004; +0.009], 7/7 matches** → used; terminal prior −0.0045.

In [ ]:
# 8d.1  test the rule on the 7 IDSSE matches (touch-flag labels), final base model log-odds
import gc
import importlib

BALLOUT_MODE = str(BALLOUT_STAGE).strip().lower()
assert BALLOUT_MODE in ("auto", "off", "force"), BALLOUT_STAGE
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
import aturan_keluar as AK      # noqa: E402
import idsse_loader             # noqa: E402
importlib.reload(AK)
BALLOUT_JENIS = ("sepak pojok", "tendangan gawang") + (("lemparan ke dalam",) if BALLOUT_THROWIN else ())
BALLOUT_R_M = 1.5
BALLOUT_PASS, BALLOUT_BEST, BALLOUT_USE = False, None, False
SUM_BALLOUT = None
if BALLOUT_MODE != "off":
    with timer("load IDSSE (cache/external/id_*.npz; first parse ± 2 minutes)"):
        IDS = idsse_loader.load_idsse(EXT_DIR / "idsse", CACHE / "external")
    if not IDS and BALLOUT_MODE == "auto":
        raise FileNotFoundError("IDSSE data not found in {} (README: download huggingface.co/datasets/pysport/idsse-data), "
                                "or set BALLOUT_STAGE = \"force\" to use the already validated variant".format(EXT_DIR / "idsse"))
    dev_ = INFER_DEVICE if (INFER_DEVICE == "cpu" or torch.cuda.is_available()) else "cpu"
    rows_ballout, evs_ballout = [], []
    with timer("base inference + decoding ({} matches)".format(len(IDS))):
        for mid, d in IDS.items():
            lab25_ = np.where(d["status"] == 1, np.where(d["poss"] == 1, 1, np.where(d["poss"] == 0, 0, -1)), -1).astype(np.int8)
            s_ = dict(P=d["P"], home=d["home"].astype(bool), ball=d["ball"], att=AK.att_dir(d), label=lab25_, fps=25,
                      L=float(d["L"]), W=float(d["W"]))
            s10_ = resample10(s_)
            XH_ = (features(s10_)[0], set_features(s10_))
            XA_ = (features(s10_, swap=True)[0], set_features(s10_, swap=True))
            lg10_ = np.mean([0.5 * (predict(n_, XH_, dev_) - predict(n_, XA_, dev_)) for n_ in NETS], 0)
            T_ = len(lab25_)
            lg_ = np.interp(np.arange(T_), s10_["idx"], lg10_)
            dead_ = d["status"] == 0
            gt_ = AK.flag_labels(d)
            ev_ = AK.idsse_restarts(d)
            pr_, pt_ = np.zeros(T_), np.zeros(T_)
            for e in ev_:
                sg_ = 1.0 if e["R"] == 1 else -1.0
                pr_[e["r"]:e["r"] + 2 * FPS] += 1e3 * sg_
                if e["jenis"] in BALLOUT_JENIS:
                    pt_[max(0, e["o"] - 5):e["o"]] += -1e3 * sg_
            base_ = AK.decode3(lg_, dead_, pr_, viterbi)
            term_ = AK.decode3(lg_, dead_, pr_ + pt_, viterbi)
            ins_ = base_.copy()
            dh_, da_ = AK.nearest_dist(d["P"], d["home"].astype(bool), d["ball"])
            n_ins_ = 0
            for e in ev_:
                if e["jenis"] in BALLOUT_JENIS:
                    d_other_, d_r_ = (da_, dh_) if e["R"] == 1 else (dh_, da_)
                    n_ins_ += AK.insert_last_touch(ins_, e["o"], e["R"], d_other_, d_r_, BALLOUT_R_M)
                    nr_ = "H" if e["R"] == 0 else "A"
                    evs_ballout.append({"jenis": e["jenis"], "premis": e["terakhir"] != e["R"], "base already ¬R when the ball goes out": base_[e["o"] - 1] == nr_,
                                 "durasi spell ¬R (flag, s)": (e["o"] - e["awal_spell"]) / FPS})
            gc_ = pm.label_codes(gt_)
            r_ = {"match": mid, "restarts used": sum(e["jenis"] in BALLOUT_JENIS for e in ev_), "sisipan": n_ins_}
            for nm_, p_ in (("base", base_), ("prior terminal", term_), ("sisipan kedekatan", ins_)):
                res_ = pm.evaluate(gc_, pm.label_codes(p_))
                r_[nm_] = float(res_["spell_f1_mean"])
                r_[nm_ + " (macroF1)"] = float(res_["macro_f1"])
            rows_ballout.append(r_)
            del s_, s10_, XH_, XA_, lg_, gt_, base_, term_, ins_, dh_, da_
            gc.collect()
    R_BALLOUT = pd.DataFrame(rows_ballout)
    E_BALLOUT = pd.DataFrame(evs_ballout)
    print("Laws-of-the-Game premise in IDSSE (last toucher ≠ taker team) and the base prediction when the ball goes out:")
    print(E_BALLOUT.groupby("jenis").agg(n=("premis", "size"), premis=("premis", "mean"),
                                   base_benar=("base already ¬R when the ball goes out", "mean"),
                                   durasi_median_s=("durasi spell ¬R (flag, s)", "median")).round(3).to_string())
    print("\nspellF1 per match (IDSSE touch-flag labels):")
    print(R_BALLOUT.round(4).to_string(index=False))
    rb_ = np.random.default_rng(17)
    sum_ = []
    for nm_ in ("prior terminal", "sisipan kedekatan"):
        d_ = (R_BALLOUT[nm_] - R_BALLOUT["base"]).to_numpy()
        bt_ = np.array([rb_.choice(d_, len(d_)).mean() for _ in range(2000)])
        sum_.append({"varian": nm_, "Δ spellF1": d_.mean(), "5%": np.percentile(bt_, 5), "95%": np.percentile(bt_, 95),
                     "matches won": "{}/{}".format(int((d_ > 0).sum()), len(d_)),
                     "Δ macroF1": (R_BALLOUT[nm_ + " (macroF1)"] - R_BALLOUT["base (macroF1)"]).mean()})
    SUM_BALLOUT = pd.DataFrame(sum_)
    print("\nΔ against base (bootstrap over matches):")
    print(SUM_BALLOUT.round(4).to_string(index=False))
    best_ = SUM_BALLOUT.sort_values("Δ spellF1", ascending=False).iloc[0]
    BALLOUT_BEST = best_["varian"]
    BALLOUT_PASS = bool(best_["Δ spellF1"] > 0 and best_["5%"] > 0)
    print("\ndecision rule: best variant '{}' Δ {:+.4f} [{:+.4f}; {:+.4f}] → {}".format(
        BALLOUT_BEST, best_["Δ spellF1"], best_["5%"], best_["95%"], "PASS" if BALLOUT_PASS else "does not pass"))
    del IDS
    gc.collect()

#### 8d.2 Apply the rule to BAR-ATH

The cell below first lists, for every corner and goal kick, whether the base label right before the ball goes out
contradicts the Laws of the Game (label = the taker team). If the rule passed in 8d.1 (or `BALLOUT_STAGE = "force"`), it then
inserts the opponents' last-touch possession and re-checks that `D` and `X` are unchanged. The resulting labels are the
final labels written in section 9.

In [ ]:
# 8d.2  BAR-ATH: contradictions with the Laws of the Game (descriptive) and application if it passes
JENIS_TAG = {"Corner Kick": "sepak pojok", "Goal Kick": "tendangan gawang", "Throw-in": "lemparan ke dalam"}
BALLOUT_USE = BALLOUT_MODE == "force" or (BALLOUT_MODE == "auto" and BALLOUT_PASS)
labels_base_final = labels.copy()
segs_ballout = []
for a_, b_ in zip(*runs((labels == "D") & GRADED)):
    near_ = restarts[(restarts.restart_frame - b_).abs() <= 2 * FPS]
    if not len(near_):
        continue
    rw_ = near_.iloc[(near_.restart_frame - b_).abs().argmin()]
    jn_ = JENIS_TAG.get(str(rw_.tag_type))
    if jn_ is None or jn_ not in BALLOUT_JENIS:
        continue
    if jn_ == "lemparan ke dalam":
        aft_ = labels[b_:b_ + 2 * FPS]
        aft_ = aft_[np.isin(aft_, ["H", "A"])]
        team_ = aft_[0] if len(aft_) else None
    else:
        team_ = rw_.taker if rw_.taker in ("H", "A") else None
    if team_ is None:
        continue
    bef_ = labels[max(0, a_ - 10 * FPS):a_]
    bef_ = bef_[np.isin(bef_, ["H", "A"])]
    segs_ballout.append({"awal D": int(a_), "jenis": jn_, "pengambil": team_, "sebelum": bef_[-1] if len(bef_) else None})
S_BALLOUT = pd.DataFrame(segs_ballout)
if len(S_BALLOUT):
    S_BALLOUT["kontradiksi"] = S_BALLOUT.sebelum == S_BALLOUT.pengambil
    print("BAR-ATH: label before the ball goes out = restart taker team (contradicts the Laws of the Game):")
    print(S_BALLOUT.groupby("jenis").kontradiksi.agg(["size", "sum"]).rename(columns={"size": "restart", "sum": "kontradiksi"}).to_string())
n_fix_ballout = 0
if BALLOUT_USE and len(S_BALLOUT):
    best_ballout = BALLOUT_BEST or "sisipan kedekatan"     # "force" without IDSSE: the variant that passed validation (log #104)
    K_BALLOUT = S_BALLOUT[S_BALLOUT.kontradiksi]
    if best_ballout == "prior terminal":
        Lt_ = np.array(L_dec, dtype=float, copy=True)
        for _, r_ in K_BALLOUT.iterrows():
            Lt_[max(0, r_["awal D"] - 5):r_["awal D"]] += -1e3 if r_.pengambil == "H" else 1e3
        if TOUCH_ON and len(TOUCH_F):
            red_ = decode_touch(Lt_, TOUCH_F, TOUCH_TEAM, TOUCH_W, TOUCH_TMAX_S, float(MODEL_META["decoder_cost"]))
        else:
            red_ = decode_possession(Lt_, d_mask, float(MODEL_META["decoder_cost"]))
        m_ = np.isin(labels, ["H", "A"]) & np.isin(red_, ["H", "A"])
        labels = np.where(m_, red_, labels).astype(labels.dtype)
    else:
        P_ = z["players"][..., :2]
        B_ = np.where((z["src_poss"] > 0)[:, None], z["ball_poss"], np.nan)
        dh_, da_ = AK.nearest_dist(P_, SLOT_IS_HOME, B_)
        for _, r_ in K_BALLOUT.iterrows():
            Rh_ = r_.pengambil == "H"
            n_fix_ballout += AK.insert_last_touch(labels, r_["awal D"], 1 if Rh_ else 0, da_ if Rh_ else dh_, dh_ if Rh_ else da_, BALLOUT_R_M)
    check("8d: D and X unchanged", ((labels == "D") == (labels_base_final == "D")).all()
          and ((labels == "X") == (labels_base_final == "X")).all())
print("ball-out rule used for BAR-ATH: {} · frames changed {} ({:.2%} of graded frames)".format(
    "YA ({})".format(BALLOUT_BEST or "sisipan kedekatan") if BALLOUT_USE else "no", int((labels != labels_base_final).sum()),
    float((labels != labels_base_final)[GRADED].mean())))
g = labels[GRADED]
chg = np.flatnonzero(g[1:] != g[:-1]) + 1
starts, ends = np.r_[0, chg], np.r_[chg, len(g)]
spell_len = (ends - starts)[np.isin(g[starts], ["H", "A"])] / FPS

### Figure: possessions over the whole match

Duration distribution of the produced possessions, per team. spellF1 counts every possession once, so the number and
duration of possessions matter as much as per-frame accuracy.

In [ ]:
spell_team = g[starts][np.isin(g[starts], ["H", "A"])]
fig, ax = plt.subplots(figsize=(8, 3.4))
bins = np.arange(0, min(spell_len.max(), 90) + 2, 2)
for team, lab in (("H", "Barcelona (H)"), ("A", "Athletic (A)")):
    ax.hist(spell_len[spell_team == team], bins=bins, histtype="step", color=COL[team], linewidth=2,
            label="{} · {} possessions".format(lab, int((spell_team == team).sum())))
ax.set_xlabel("possession duration (seconds)")
ax.set_ylabel("number of possessions")
ax.set_title("Possession duration over the whole match", pad=18)
note(ax, "{} possessions · mean {:.1f} s · median {:.1f} s".format(len(spell_len), spell_len.mean(), np.median(spell_len)))
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 9. Write, validate officially, check correctness

**What this step does.** Writes the submission file, validates it with the official validator and prints the
correctness check against the public GT.

The file is named after the team (SUBMISSION.md) and checked with the **official validator**, the same function the
organisers' scorer uses.

**Which frames are in the CSV.** Row *i* is frame *i* of the **tactical video** (0 … 199,318); the broadcast is used
only as an extra input and mapped back to tactical frames (section 2). The 5-minute public GT is also indexed on the
tactical video: frames 11,185 … 18,684, the first 5 minutes of play in the first half. Frames before kick-off, during
half-time and after the final whistle get `X` and are not graded. The score against the public GT (5 minutes, 17
possessions) is printed with the official bootstrap interval as a **correctness check**: no parameter is tuned on it.

In [ ]:
# back up the base submission (without the ball-out rule) before it is overwritten (same sha256 on two machines, log #94)
BASE_SHA = "5f3c8a6eca0890bb171d8b57628555889ccd36b08f4fd66d8e73ae4c1097e183"
BASE_CSV = SUBMISSIONS / "{}_base.csv".format(TEAM_FILE)
_cur = SUBMISSIONS / "{}.csv".format(TEAM_FILE)
if not BASE_CSV.is_file() and _cur.is_file() and sha256(_cur) == BASE_SHA:
    shutil.copy2(_cur, BASE_CSV)
check("base submission available for comparison", BASE_CSV.is_file() and sha256(BASE_CSV) == BASE_SHA,
      str(BASE_CSV.relative_to(ROOT)), critical=False)

### 9a. Write the submission and validate it

Writes one `frame,label` row per tactical frame, prints the file's SHA-256, runs the official validator, and prints the
public-GT score with its bootstrap interval (a correctness check only). The decoder without touches and the version without the
whistle rule are printed next to it for comparison; they are not used for selection.

In [ ]:
path = SUBMISSIONS / "{}.csv".format(TEAM_FILE)
with open(path, "w", newline="\n", encoding="utf-8") as handle:
    handle.write("frame,label\n")
    handle.writelines("{},{}\n".format(i, lab) for i, lab in enumerate(labels))
table = pm.read_label_csv(str(path))
reason = pm.validate_submission(table, N_FRAMES)
check("official validation", not reason, reason or "")
codes = pm.label_codes(table)
counts = {lab: int(np.sum(codes == i)) for i, lab in enumerate(pm.LABELS)}
DIGEST = sha256(path)
print("file: {} | sha256 {} | {}".format(path.relative_to(ROOT), DIGEST, counts))

gt_codes = pm.label_codes(pm.read_label_csv(str(GT_PUBLIC)))
res = pm.evaluate(gt_codes, codes, bootstrap_n=BOOTSTRAP_N)
public = {k: round(float(res[k]), 4) for k in ("spell_f1_mean", "spell_bootstrap_low", "spell_bootstrap_high", "macro_f1",
                                                "alive_dead_f1", "team_acc_gt_alive")}
public.update({"n_pred_spells": int(res["n_pred_spells"]), "n_gt_spells": int(res["n_gt_spells"])})
print("Public GT (correctness check):", public)
res_yolo = pm.evaluate(gt_codes, pm.label_codes(labels_yolo.astype(object)), bootstrap_n=BOOTSTRAP_N)
print("decoder without touches comparison (not for selection): spellF1 {:.4f} [{:.4f}–{:.4f}] · predicted possessions {} / GT {}".format(
    res_yolo["spell_f1_mean"], res_yolo["spell_bootstrap_low"], res_yolo["spell_bootstrap_high"], res_yolo["n_pred_spells"],
    res_yolo["n_gt_spells"]))
if (labels_pre_whistle != labels).any():
    res_pw = pm.evaluate(gt_codes, pm.label_codes(labels_pre_whistle.astype(object)))
    print("comparison without the whistle rule (not for selection): spellF1 {:.4f} · macroF1 {:.4f} · alive/dead F1 {:.4f}".format(
        res_pw["spell_f1_mean"], res_pw["macro_f1"], res_pw["alive_dead_f1"]))


### 9a.2 Final file vs the base file

Compares the new labels with the backed-up base submission: score on the public GT (correctness check, NOT for selection),
the number of frames that differ over the whole match, and the timing of each public-GT switch.

In [ ]:
# final labels: comparison with the base submission (file) — correctness check on the public GT, NOT for selection
print("possession model: {} · σ = {:.2f} · video method: {}".format(MODEL_USED, SIGMA_SMOOTH, "R {} m · β {} · {}".format(*NEAR_CFG) if NEAR_USE else "not used") + " · stacking: {}".format("used (λ {})".format(STACK_LAMBDA) if STACK_USE else "not used") + " · ball-out rule: {}".format("used ({})".format(BALLOUT_BEST) if BALLOUT_USE else "not used"))
if BASE_CSV.is_file():
    codes_base = pm.label_codes(pm.read_label_csv(str(BASE_CSV)))
    lab_base = np.array(pm.LABELS, dtype=object)[codes_base]
    res_base = pm.evaluate(gt_codes, codes_base, bootstrap_n=BOOTSTRAP_N)
    print("base (file): spellF1 {:.4f} [{:.4f}–{:.4f}] · macroF1 {:.4f} · possessions {} / GT {}".format(
        res_base["spell_f1_mean"], res_base["spell_bootstrap_low"], res_base["spell_bootstrap_high"], res_base["macro_f1"],
        res_base["n_pred_spells"], res_base["n_gt_spells"]))
    print("final:      spellF1 {:.4f} [{:.4f}–{:.4f}] · macroF1 {:.4f} · possessions {} / GT {}".format(
        public["spell_f1_mean"], public["spell_bootstrap_low"], public["spell_bootstrap_high"], public["macro_f1"],
        public["n_pred_spells"], public["n_gt_spells"]))

    def _poss(lab_):
        out_ = []
        for t_ in ("H", "A"):
            s_, e_ = runs(np.asarray(lab_) == t_)
            out_ += list((e_ - s_) / FPS)
        return np.asarray(out_)
    p_base_, p_smooth_ = _poss(lab_base), _poss(labels)
    print("whole match: frames that differ {} ({:.2%} of graded frames) · possessions {} → {} · < 2 s {:.1%} → {:.1%}".format(
        int((lab_base != labels).sum()), float((lab_base != labels)[GRADED].mean()), len(p_base_), len(p_smooth_),
        float((p_base_ < 2).mean()), float((p_smooth_ < 2).mean())))
    gl_ = np.array(pm.LABELS)[gt_codes]
    gsw_ = np.flatnonzero((gl_[1:] != gl_[:-1]) & np.isin(gl_[1:], ["H", "A"]) & np.isin(gl_[:-1], ["H", "A"])) + 1

    def _signed(pred_, tol=2 * FPS):
        pred_ = np.asarray(pred_)
        ps_ = np.flatnonzero((pred_[1:] != pred_[:-1]) & np.isin(pred_[1:], ["H", "A"]) & np.isin(pred_[:-1], ["H", "A"])) + 1
        out_ = []
        for f_ in gsw_:
            q_ = ps_[(pred_[ps_] == gl_[f_]) & (np.abs(ps_ - f_) <= tol)]
            out_.append((q_[np.argmin(np.abs(q_ - f_))] - f_) / FPS if len(q_) else np.nan)
        return np.asarray(out_)
    tab_ = pd.DataFrame({"video time": [vtime(f_) for f_ in gsw_], "ke": gl_[gsw_],
                         "Δ base (s)": _signed(lab_base), "Δ final (s)": _signed(labels)})
    print("\ntime difference of each public-GT switch (+ = prediction late, NaN = not caught within ± 2 s):")
    print(tab_.round(2).to_string(index=False))
    print("caught: base {}/{} · final {}/{}".format(int(tab_["Δ base (s)"].notna().sum()), len(tab_),
                                                    int(tab_["Δ final (s)"].notna().sum()), len(tab_)))


### Figure: metrics on the public GT

1. **Timeline** of the public 5 minutes: GT on top, prediction below. Missed or split possessions are immediately
   visible as unpaired blocks.
2. Per-frame **confusion matrix** (H/A/D), normalised per GT row.
3. **spellF1 per IoU threshold**: precision, recall and F1 of the possession matching at IoU 0.3–0.7; spellF1 is the
   mean F1 over the five thresholds.

In [ ]:
gt_lab = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
win = np.flatnonzero(gt_lab != "X")
w0, w1 = int(win[0]), int(win[-1]) + 1

fig, ax = plt.subplots(figsize=(12, 2.4))
for row, (name, arr) in enumerate((("GT publik", gt_lab[w0:w1]), ("Prediksi base", labels[w0:w1]))):
    chg_ = np.flatnonzero(arr[1:] != arr[:-1]) + 1
    a_, b_ = np.r_[0, chg_], np.r_[chg_, len(arr)]
    for a, b in zip(a_, b_):
        ax.broken_barh([(a / FPS, (b - a) / FPS)], (row - 0.32, 0.64), facecolors=COL.get(arr[a], COL["D"]),
                       edgecolor=COL["surface"], linewidth=1.2)
ax.set_yticks([0, 1], ["public GT", "base prediction"])
ax.invert_yaxis()
ax.set_xlim(0, (w1 - w0) / FPS)
ax.set_xlabel("seconds since the start of the public window")
ax.grid(axis="y", visible=False)
ax.set_title("Possession timeline, 5-minute public window", pad=18)
note(ax, "GT {} possessions · predicted {} possessions".format(public["n_gt_spells"], public["n_pred_spells"]))
ax.legend(handles=[Patch(color=COL["H"], label="H Barcelona"), Patch(color=COL["A"], label="A Athletic"),
                   Patch(color=COL["D"], label="D dead ball")], loc="upper left", bbox_to_anchor=(0, -0.38), ncol=3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.3]})
ax = axes[0]
conf = np.array(res["confusion"])[:3, :3]
rown = conf / conf.sum(1, keepdims=True)
ax.imshow(rown, cmap=BLUES, vmin=0, vmax=1)
cls = ["H", "A", "D"]
for i in range(3):
    for j in range(3):
        ax.text(j, i, "{:.0%}\n{:,}".format(rown[i, j], conf[i, j]).replace(",", "."), ha="center", va="center",
                fontsize=9, color="white" if rown[i, j] > 0.55 else COL["text"])
ax.set_xticks(range(3), cls)
ax.set_yticks(range(3), cls)
ax.set_xlabel("predicted")
ax.set_ylabel("GT")
ax.grid(False)
ax.set_title("Per-frame confusion matrix", pad=18)
note(ax, "accuracy {:.3f} · macroF1 {:.3f}".format(res["accuracy"], res["macro_f1"]))

ax = axes[1]
thr = sorted(res["spell_f1"])
for key, col, name in (("precision", COL["H"], "presisi"), ("recall", COL["A"], "recall"), ("f1", COL["aqua"], "F1")):
    ys = [res["spell_f1"][t][key] for t in thr]
    ax.plot(thr, ys, color=col, linewidth=2, marker="o", markersize=7, markeredgecolor=COL["surface"],
            markeredgewidth=2, label=name)
ax.axhline(res["spell_f1_mean"], color=COL["muted"], linewidth=1, linestyle="--")
ax.text(thr[-1], res["spell_f1_mean"], " spellF1 {:.3f}".format(res["spell_f1_mean"]), color=COL["muted"],
        fontsize=9, va="bottom", ha="right")
ax.set_xticks(thr)
ax.set_ylim(0, 1.02)
ax.set_xlabel("IoU threshold for possession matching")
ax.set_ylabel("score")
ax.set_title("Possession matching per IoU threshold", pad=18)
note(ax, "spellF1 bootstrap interval [{:.3f} – {:.3f}]".format(public["spell_bootstrap_low"], public["spell_bootstrap_high"]))
ax.legend(loc="lower left", ncol=3)
plt.tight_layout()
plt.show()

## 9b. Final EDA: representativeness, balance, and material for post-processing

### 9b.1 Do the public 5 minutes represent 90 minutes?

Profile per 5-minute block over the whole match: dead-ball share, ball availability, and Barcelona's possession share
on live frames (from the predictions). The hatched block is the public GT window.

**Finding.** The first 5 minutes **do not represent** the rest of the match: there is much less dead ball and the ball
is visible much more often than on average, because the start of a half runs without many stoppages. The public score
therefore tends to be higher than the graded 90-minute score, and with only 17 possessions its interval is wide. This
is consistent with the official FAQ and with the 5-minute vs 90-minute analysis on the 23 external matches (section
6b). The design decisions in this notebook are therefore not taken from the public GT.

In [ ]:
BLK_F = 5 * 60 * FPS
blk_rows = []
for h, (a, b) in HALVES.items():
    for k, m0 in enumerate(range(a, b + 1, BLK_F)):
        seg = slice(m0, min(m0 + BLK_F, b + 1))
        lv = ~d_mask[seg]
        lab_seg = labels[seg]
        blk_rows.append({"babak": h, "blok": k, "D": d_mask[seg].mean(), "bola": (ball_src[seg] > 0).mean(),
                         "replay": (bc_view[seg] == -1).mean(),
                         "H_hidup": (lab_seg[lv] == "H").mean() if lv.any() else np.nan, "publik": m0 == w0})
blk = pd.DataFrame(blk_rows)
pub = slice(w0, w1)
rest = GRADED.copy()
rest[pub] = False


def profile(mask_, lab_arr):
    lv = mask_ & ~d_mask
    g_ = lab_arr[mask_]
    c_ = np.flatnonzero(g_[1:] != g_[:-1]) + 1
    st_ = np.r_[0, c_]
    n_sp = int(np.isin(g_[st_], ["H", "A"]).sum())
    return {"D share": d_mask[mask_].mean(), "ball available": (ball_src[mask_] > 0).mean(),
            "replay/close-up": (bc_view[mask_] == -1).mean(), "H on live frames": (lab_arr[lv] == "H").mean(),
            "possessions per minute": n_sp / (mask_.sum() / FPS / 60)}


pub_mask = np.zeros(N_FRAMES, dtype=bool)
pub_mask[pub] = True
prof = pd.DataFrame({"public 5 minutes (prediction)": profile(pub_mask, labels),
                     "public 5 minutes (GT)": {"D share": (gt_lab[pub] == "D").mean(), "ball available": np.nan,
                                             "replay/close-up": np.nan,
                                             "H on live frames": (gt_lab[pub][gt_lab[pub] != "D"] == "H").mean(),
                                             "possessions per minute": public["n_gt_spells"] / 5},
                     "sisa match (prediksi)": profile(rest, labels)})
print(prof.round(3).to_string())

xb_ = np.where(blk.babak == 1, blk.blok, blk.blok + (blk.babak == 1).sum() + 1)
fig, axes = plt.subplots(3, 1, figsize=(12, 6.2), sharex=True)
for ax, key, col, ylab in ((axes[0], "D", COL["D"], "% dead ball"), (axes[1], "bola", COL["aqua"], "% ball available"),
                           (axes[2], "H_hidup", COL["H"], "% of live time held by H")):
    ax.bar(xb_, 100 * blk[key], width=0.8, color=col)
    ax.axvspan(xb_[blk.publik].min() - 0.5, xb_[blk.publik].max() + 0.5, color=COL["yellow"], alpha=0.18, linewidth=0)
    ax.axhline(100 * np.nanmean(blk[key]), color=COL["muted"], linewidth=1, linestyle="--")
    ax.set_ylabel(ylab)
    ax.set_ylim(0, 100)
    ax.grid(axis="x", visible=False)
axes[2].axhline(50, color=COL["text"], linewidth=0.8)
axes[0].set_title("Profile per 5 minutes · hatched = public GT window", pad=18)
note(axes[0], "dashed line = block mean · predictions of this notebook (H on live frames) and the section-4 detector (D)")
axes[2].set_xticks(xb_, ["B{} {}'".format(r_.babak, 5 * r_.blok) for r_ in blk.itertuples()], fontsize=7, rotation=90)
plt.tight_layout()
plt.show()

### 9b.2 H/A possession balance

Barcelona keeps the ball longer, especially at the start of each half. Is this an "imbalanced classes" problem?

- **For spellF1, not really.** H and A possessions **alternate**: every Barcelona possession ends with an Athletic
  possession or a dead ball. The number of possessions is therefore almost the same for both teams, even though their
  durations differ. The imbalance is in **duration**, not in the number of scored objects.
- **The model cannot learn "Barcelona has the ball more often".** The features are H↔A symmetric and the predictions
  are averaged over both viewpoints (section 7); the model sees only geometry. On the 23 external matches (6b), the
  per-frame recall of the dominant and the minor team differs by only ± 1 point (± 0.95 vs ± 0.94), and more lopsided
  matches do not get a lower spellF1.
- **The possession-weighted loss** (section 6) gives short possessions a larger weight, so the short possessions of the
  team with less of the ball are not drowned by the long possessions of the dominant team.

In [ ]:
spell_tab = pd.DataFrame({"possession": pd.Series(spell_team).value_counts(),
                          "total duration (minutes)": pd.Series(spell_len).groupby(spell_team).sum() / 60,
                          "median duration (s)": pd.Series(spell_len).groupby(spell_team).median()}).loc[["H", "A"]]
spell_tab["time share"] = spell_tab["total duration (minutes)"] / spell_tab["total duration (minutes)"].sum()
spell_tab["count share"] = spell_tab["possession"] / spell_tab["possession"].sum()
print(spell_tab.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 2.6))
for k, (key, name) in enumerate((("count share", "number of possessions"), ("time share", "time on the ball"))):
    left = 0.0
    for team in ("H", "A"):
        v = float(spell_tab.loc[team, key])
        ax.barh(k, 100 * v, left=left, color=COL[team], height=0.55, edgecolor=COL["surface"], linewidth=2,
                label={"H": "Barcelona (H)", "A": "Athletic (A)"}[team] if k == 0 else None)
        ax.text(left + 50 * v, k, "{:.0f}%".format(100 * v), ha="center", va="center", fontsize=9, color="white")
        left += 100 * v
ax.axvline(50, color=COL["text"], linewidth=0.8)
ax.set_yticks([0, 1], ["number of possessions", "time in possession"])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.grid(axis="y", visible=False)
ax.set_xlabel("% (predictions over the whole match)")
ax.set_title("Unbalanced in duration, balanced in count", pad=18)
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.38), ncol=2)
plt.tight_layout()
plt.show()

### 9b.3 Error diagnosis for post-processing

spellF1 punishes **missed** possessions (recall) and **spurious** possessions (precision). To choose the right
post-processing, we look at where the errors happen:

- **Left, public window (diagnosis, not tuning):** every GT possession is drawn by its duration and its best IoU with a
  predicted possession of the same team. Possessions below the 0.3 line never match.
- **Right, whole match:** the context of every predicted possession: framed by dead balls, starting with a switch from
  the opponent, or following a long ball.

**Finding.** Almost every missed GT possession is very short (± 1 s): a brief interception that is immediately won
back. The Viterbi switching cost deliberately holds such possessions back, because in automatic tracking a brief
opponent touch is hard to tell apart from pressing. Lowering the cost does produce short possessions, but more of them
are spurious; the cost curve in 6b shows the balance point. Merging short possessions or a median filter does not
improve spellF1 in the external validation, so neither is used.

In [ ]:
def spells_of(arr):
    c_ = np.flatnonzero(arr[1:] != arr[:-1]) + 1
    a_, b_ = np.r_[0, c_], np.r_[c_, len(arr)]
    return [(int(x0), int(x1), arr[x0]) for x0, x1 in zip(a_, b_) if arr[x0] in ("H", "A")]


gt_sp = spells_of(gt_lab[w0:w1])
pr_sp = spells_of(labels[w0:w1])
gt_iou = []
for x0, x1, t in gt_sp:
    ious = [(min(x1, y1) - max(x0, y0)) / (max(x1, y1) - min(x0, y0)) for y0, y1, u in pr_sp
            if u == t and min(x1, y1) > max(x0, y0)]
    gt_iou.append(((x1 - x0) / FPS, max(ious, default=0.0), t))
gt_iou = pd.DataFrame(gt_iou, columns=["durasi", "iou", "tim"])
print("public GT possessions: {} · matched (IoU ≥ 0.5) {} · missed (IoU < 0.3) {} · median duration of the missed ones {:.1f} s".format(
    len(gt_iou), int((gt_iou.iou >= 0.5).sum()), int((gt_iou.iou < 0.3).sum()),
    gt_iou[gt_iou.iou < 0.3].durasi.median() if (gt_iou.iou < 0.3).any() else float("nan")))

sp_start_g = starts[np.isin(g[starts], ["H", "A"])]
sp_end_g = ends[np.isin(g[starts], ["H", "A"])]
prev_lab = np.where(sp_start_g > 0, g[np.maximum(sp_start_g - 1, 0)], "X")
next_lab = np.where(sp_end_g < len(g), g[np.minimum(sp_end_g, len(g) - 1)], "X")
gidx = np.flatnonzero(GRADED)
after_long = np.array([in_long[max(gidx[s_] - 2 * FPS, 0):gidx[s_]].any() for s_ in sp_start_g])
ctx = pd.Series({"starts after a dead ball (restart)": (prev_lab == "D").mean(),
                 "starts with a switch from the opponent": np.isin(prev_lab, ["H", "A"]).mean(),
                 "ends with a dead ball": (next_lab == "D").mean(),
                 "starts ≤ 2 s after a long ball": after_long.mean(),
                 "durasi < 2 s": (spell_len < 2).mean()})
lb_end_lab = labels[np.minimum(long_balls.end.to_numpy() + FPS, N_FRAMES - 1)]
lb_start_lab = labels[long_balls.start.to_numpy()]
ok_lb = np.isin(lb_start_lab, ["H", "A"]) & np.isin(lb_end_lab, ["H", "A"])
print("long balls followed by a (predicted) possession change: {:.0%} of {}".format(
    (lb_start_lab[ok_lb] != lb_end_lab[ok_lb]).mean(), int(ok_lb.sum())))
print(ctx.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
for team in ("H", "A"):
    sub = gt_iou[gt_iou.tim == team]
    ax.plot(sub.durasi, sub.iou, linestyle="none", marker="o", markersize=8, color=COL[team],
            markeredgecolor=COL["surface"], markeredgewidth=1.5, label={"H": "H possession", "A": "A possession"}[team])
for thr_ in (0.3, 0.7):
    ax.axhline(thr_, color=COL["muted"], linewidth=1, linestyle="--")
ax.set_xscale("log")
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("GT possession duration (s, log scale)")
ax.set_ylabel("best IoU with a prediction")
ax.set_title("Public GT possessions: which ones are missed", pad=18)
note(ax, "dashed lines = IoU threshold range 0.3–0.7 · diagnosis only, not used for tuning")
ax.legend(loc="lower right")
ax = axes[1]
ax.barh(range(len(ctx)), 100 * ctx.to_numpy(), color=COL["violet"], height=0.6)
for k, v in enumerate(ctx.to_numpy()):
    ax.text(100 * v + 1, k, "{:.0f}%".format(100 * v), va="center", fontsize=9, color=COL["text"])
ax.set_yticks(range(len(ctx)), ctx.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of predicted possessions over the whole match")
ax.grid(axis="y", visible=False)
ax.set_title("Context of the predicted possessions", pad=18)
plt.tight_layout()
plt.show()

## 9c. Where the public-GT errors are: ball source, tracking-ball vs YOLO-ball, and video

This section is **diagnosis**, not tuning: no parameter is changed based on the numbers here. The tracking-ball comparison is
recomputed in this notebook exactly as that pipeline ran: the tracking ball and the `D` boundaries before snapping.

1. **Error table:** every run of frames whose label differs from the GT (≥ 0.2 s), with its type, the ball source in
   that run, and whether the tracking-ball pipeline was also wrong there.
2. **Timeline:** GT, tracking-ball prediction, YOLO-ball prediction, and the tracking-ball / YOLO ball source for the public 5 minutes.
3. **Location on the pitch:** the merged ball position while the YOLO-ball prediction is wrong.
4. **Video crops** from the longest error runs (start, middle and end of the run).
5. **Review clip** (`YOLO_CLIP`, tactical video time) with every ball marker and a GT / tracking-ball / YOLO-ball label strip.

In [ ]:
gt_w = gt_lab[w0:w1]
res_trk = pm.evaluate(gt_codes, pm.label_codes(labels_trk))
print("public GT (correctness check): spellF1 tracking-ball {:.3f} → base {:.3f} | predicted possessions tracking-ball {} → base {} | GT {}".format(
    res_trk["spell_f1_mean"], public["spell_f1_mean"], int(res_trk["n_pred_spells"]), public["n_pred_spells"],
    public["n_gt_spells"]))
chg_all = GRADED & (labels != labels_trk)
print("whole match: {} frames ({:.2%} of graded frames) differ tracking-ball → base | {} frames in the public window".format(
    int(chg_all.sum()), chg_all.sum() / N_GRADED, int(chg_all[w0:w1].sum())))


def err_kind(g_, p_):
    if g_ == "D":
        return "D missed (predicted live)"
    if p_ == "D":
        return "D palsu"
    return "wrong team"


def error_runs(pred, src_arr, min_frames=5):
    wrong = pred[w0:w1] != gt_w
    out = []
    for a_, b_ in zip(*runs(wrong)):
        if b_ - a_ < min_frames:
            continue
        seg = slice(w0 + a_, w0 + b_)
        sv = pd.Series(src_arr[seg]).map(SRC_NAMES).value_counts(normalize=True)
        dty = np.hypot(*(ball_raw[seg] - YO_BALL[seg]).T) if YOLO_ON else np.array([np.nan])
        out.append({"start (video)": vtime(w0 + a_), "frame": w0 + a_, "durasi s": round((b_ - a_) / FPS, 2),
                    "GT": gt_w[a_], "prediksi": pred[w0 + a_], "jenis": err_kind(gt_w[a_], pred[w0 + a_]),
                    "ball source": ", ".join("{} {:.0%}".format(k_, v_) for k_, v_ in sv.items()),
                    "|tactical−YOLO| median m": round(float(np.nanmedian(dty)), 1) if np.isfinite(dty).any() else np.nan,
                    "tracking-ball also wrong": bool((labels_trk[seg] != gt_w[a_:b_]).mean() > 0.5)})
    return pd.DataFrame(out)


ERR_FINAL = error_runs(labels, src_poss)
ERR_TRK = error_runs(labels_trk, ball_src)
pd.set_option("display.max_colwidth", 60)
print("\nbase prediction errors in the public window ({} runs, {:.1f} s):".format(len(ERR_FINAL), ERR_FINAL["durasi s"].sum() if len(ERR_FINAL) else 0))
print(ERR_FINAL.drop(columns=["frame"]).to_string(index=False) if len(ERR_FINAL) else "none")
print("\ntracking-ball prediction errors ({} runs, {:.1f} s) by type:".format(len(ERR_TRK), ERR_TRK["durasi s"].sum() if len(ERR_TRK) else 0),
      ERR_TRK.groupby("jenis")["durasi s"].sum().round(1).to_dict() if len(ERR_TRK) else {})

fig, ax = plt.subplots(figsize=(13, 3.3))
rows_tl = [("GT publik", gt_w, None), ("Prediksi tracking-ball", labels_trk[w0:w1], None), ("Prediksi base", labels[w0:w1], None),
           ("Tracking ball", ball_src[w0:w1], "src"), ("YOLO ball", src_poss[w0:w1], "src")]
for row_, (name_, arr_, kind_) in enumerate(rows_tl):
    chg_ = np.flatnonzero(arr_[1:] != arr_[:-1]) + 1
    for a_, b_ in zip(np.r_[0, chg_], np.r_[chg_, len(arr_)]):
        col_ = src_cols[SRC_NAMES[int(arr_[a_])]] if kind_ == "src" else COL.get(arr_[a_], COL["D"])
        ax.broken_barh([(a_ / FPS, (b_ - a_) / FPS)], (row_ - 0.32, 0.64), facecolors=col_, edgecolor="none")
for r_ in ERR_FINAL.itertuples():
    ax.plot([(r_.frame - w0) / FPS] * 2, [1.6, 2.4], color="#d1242f", linewidth=1.2)
ax.set_yticks(range(len(rows_tl)), [r_[0] for r_ in rows_tl])
ax.invert_yaxis()
ax.set_xlim(0, (w1 - w0) / FPS)
ax.set_xlabel("seconds since the start of the public window (video {} – {})".format(vtime(w0), vtime(w1)))
ax.grid(axis="y", visible=False)
ax.set_title("Possession and ball source, public window", pad=18)
note(ax, "red line = start of a base error run · ball-source colours as in figure 2b.6")
ax.legend(handles=[Patch(color=COL["H"], label="H"), Patch(color=COL["A"], label="A"), Patch(color=COL["D"], label="D")]
          + [Patch(color=c_, label=n_) for n_, c_ in src_cols.items()], loc="upper left", bbox_to_anchor=(0, -0.3),
          ncol=9, fontsize=8)
plt.tight_layout()
plt.show()

wrong_final = np.zeros(N_FRAMES, bool)
wrong_final[w0:w1] = labels[w0:w1] != gt_w
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.plot([0, PITCH_L, PITCH_L, 0, 0], [0, 0, PITCH_W, PITCH_W, 0], color=COL["muted"], linewidth=1)
ax.axvline(PITCH_L / 2, color=COL["muted"], linewidth=0.8)
ok_win = np.zeros(N_FRAMES, bool)
ok_win[w0:w1] = True
ax.scatter(*ball_poss[ok_win & ~wrong_final].T, s=2, color=COL["grid"], label="correct")
kinds_ = np.array([err_kind(gt_lab[t_], labels[t_]) if wrong_final[t_] else "" for t_ in range(w0, w1)])
for kind_, col_ in (("wrong team", COL["A"]), ("D missed (predicted live)", COL["violet"]), ("D palsu", COL["yellow"])):
    m_ = np.zeros(N_FRAMES, bool)
    m_[w0:w1] = kinds_ == kind_
    if m_.any():
        ax.scatter(*ball_poss[m_].T, s=6, color=col_, label="{} ({:.1f} s)".format(kind_, m_.sum() / FPS))
ax.set_aspect("equal")
ax.set_xlim(-3, PITCH_L + 3)
ax.set_ylim(PITCH_W + 3, -3)
ax.set_xlabel("x (m)")
ax.set_title("Ball position while the base prediction is wrong, public window", pad=18)
note(ax, "combined YOLO ball · frames without a ball are not drawn")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def label_strip(t_, width, height=96, span_s=8):
    strip = np.full((height, width, 3), 28, np.uint8)
    lo_, hi_ = t_ - span_s * FPS, t_ + span_s * FPS
    x0_, x1_ = 120, width - 12
    bgr = {"H": (214, 120, 42), "A": (52, 104, 235), "D": (169, 178, 180), "X": (60, 60, 60)}
    for r_, (name_, arr_) in enumerate((("GT", gt_lab), ("trk", labels_trk), ("base", labels))):
        y_ = 8 + r_ * 28
        cv2.putText(strip, name_, (12, y_ + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (235, 235, 235), 1, cv2.LINE_AA)
        xs_ = np.linspace(x0_, x1_, hi_ - lo_ + 1).astype(int)
        for k_, f_ in enumerate(range(lo_, hi_)):
            if 0 <= f_ < N_FRAMES:
                cv2.line(strip, (xs_[k_], y_), (xs_[k_], y_ + 22), bgr.get(str(arr_[f_]), (60, 60, 60)), 2)
    xc_ = int(x0_ + (t_ - lo_) / (hi_ - lo_) * (x1_ - x0_))
    cv2.line(strip, (xc_, 2), (xc_, height - 2), (255, 255, 255), 2)
    return strip


def frame_panel(reader_, t_, width=1280):
    img_ = reader_.read(t_)
    if img_ is None:
        return None
    draw_overlay(img_, t_)
    img_ = cv2.resize(img_, (width, int(width * VID_H / VID_W)))
    d_ = np.hypot(*(ball_raw[t_] - YO_BALL[t_])) if YOLO_ON else np.nan
    txt_ = "video {} | frame {} | GT {} | tracking-ball {} | base {} | YOLO ball: {} | tactical-YOLO {}".format(
        vtime(t_), t_, gt_lab[t_], labels_trk[t_], labels[t_], SRC_NAMES[int(src_poss[t_])],
        "{:.1f} m".format(d_) if np.isfinite(d_) else "-")
    cv2.rectangle(img_, (0, 0), (width, 34), (0, 0, 0), -1)
    cv2.putText(img_, txt_, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1, cv2.LINE_AA)
    return np.vstack([img_, label_strip(t_, width)])


def parse_vt(s_):
    m_, sec_ = s_.split(":")
    return int(round((int(m_) * 60 + float(sec_)) * FPS))


if YOLO_ON and len(ERR_FINAL):
    reader = FrameReader(VIDEO_T)
    tiles_ = []
    for _, r_ in ERR_FINAL.sort_values("durasi s", ascending=False).head(4).iterrows():
        n_, f0_ = int(r_["durasi s"] * FPS), int(r_["frame"])
        for t_ in (f0_ + min(5, n_ // 2), f0_ + n_ // 2, f0_ + max(n_ - 6, n_ // 2)):
            img_ = reader.read(t_)
            if img_ is None:
                continue
            draw_overlay(img_, t_, label=False)
            xy_ = ball_poss[t_] if np.isfinite(ball_poss[t_, 0]) else (ball_raw[t_] if finite[t_] else None)
            u_, v_ = m_to_px([t_], xy_[None])[0] if xy_ is not None else (VID_W / 2, VID_H / 2)
            tile_ = crop_zoom(img_, u_, v_, half=110, zoom=2)
            cv2.rectangle(tile_, (0, 0), (440, 40), (0, 0, 0), -1)
            cv2.putText(tile_, "{} GT {} base {} ({})".format(vtime(t_), gt_lab[t_], labels[t_], r_["jenis"]), (6, 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(tile_, "YOLO ball: {}".format(SRC_NAMES[int(src_poss[t_])]), (6, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                        (255, 255, 255), 1, cv2.LINE_AA)
            tiles_.append(tile_)
    out_ = YO_REP / "kesalahan_publik.jpg"
    if mosaic(tiles_, 3, out_):
        show_image(out_, "Longest base error runs in the public window: start · middle · end (2× crops)", width=13)

    if YOLO_SHOW_VIDEO:
        from IPython.display import Video, display
        c0_, c1_ = parse_vt(YOLO_CLIP[0]), parse_vt(YOLO_CLIP[1])
        clip_path = YO_REP / "klip_tinjauan.webm"
        first_ = frame_panel(reader, c0_, width=960)
        writer_ = cv2.VideoWriter(str(clip_path), cv2.VideoWriter_fourcc(*"VP80"), FPS, (first_.shape[1], first_.shape[0]))
        if not writer_.isOpened():
            clip_path = YO_REP / "klip_tinjauan.mp4"
            writer_ = cv2.VideoWriter(str(clip_path), cv2.VideoWriter_fourcc(*"avc1"), FPS, (first_.shape[1], first_.shape[0]))
        writer_.write(first_)
        for t_ in range(c0_ + 1, c1_):
            panel_ = frame_panel(reader, t_, width=960)
            if panel_ is not None:
                writer_.write(panel_)
        writer_.release()
        print("review clip {} – {} ({:.0f} s): {} ({:.1f} MB) (contains match video: keep it local only)".format(
            YOLO_CLIP[0], YOLO_CLIP[1], (c1_ - c0_) / FPS, clip_path.relative_to(ROOT), clip_path.stat().st_size / 1e6))
        display(Video(str(clip_path), embed=True, width=960,
                      mimetype="video/webm" if clip_path.suffix == ".webm" else "video/mp4"))
    reader.close()

### Summary of the checks

Every `check(...)` call in the notebook is collected here. The run stops with an error if any critical check failed;
`WARN` rows are non-critical.

In [ ]:
summary = pd.DataFrame(CHECKS)
print(summary.to_string(index=False))
n_fail = int((summary.status == "FAIL").sum())
assert n_fail == 0, "{} critical checks failed".format(n_fail)
print("\nDONE — {} | sha256 {} | public spellF1 {:.3f} [{:.3f} – {:.3f}]".format(
    path.relative_to(ROOT), DIGEST, public["spell_f1_mean"], public["spell_bootstrap_low"], public["spell_bootstrap_high"]))

## Data scientist's notes

We treat this problem as **time-series segmentation with an object-based metric**. What is scored is the whole
possession, not the frame, so every decision is tested with spellF1, not accuracy.

**What we learned from the data.**

- *A missing ball is not the main problem during live play.* ± 41% of frames without a ball sounds bad, but ± 90% of
  them are stoppages. During live play the ball is visible on ± 92% of frames (± 96% after merging with the broadcast).
  We turned the ball gaps during stoppages into a dead-ball cue.
- *The broadcast is useful once filtered.* A third of the broadcast frames are replays, close-ups or crowd shots. We do
  not use those frames for the ball position (zero weight), but we do use them as a cue that play has stopped. The
  broadcast-to-tactical alignment is stable down to the minute with one offset per half.
- *Physical outliers are rare, structural ones are not.* Track jumps > 20 m/s are only ± 0.02% of steps and balls
  > 60 m/s ± 0.07%; both are already tamed by clipping in the features. The big one is the number of tracks: on ± 13% of
  frames one team has > 11 tracks (ghost / wrong-team tracks), a situation that never occurs in the training data. The
  sensitivity test (6c) shows the model is robust to it, so we do not guess which tracks to drop.
- *Long balls stay tracked.* The difficulty is who wins the ball afterwards, and that is the sequence model's job, not
  preprocessing.
- *The public 5 minutes are too small and too easy.* Dead ball ± 20% (rest of the match ± 49%) and ball available
  ± 90% (rest of the match ± 61%). One possession is worth ± 6 spellF1 points. On the 23 external matches, the score of
  the first 5 minutes is almost uncorrelated with the 90-minute score. We use it as a correctness check, not as a
  selection tool.
- *The imbalance is in duration, not in the number of possessions.* Barcelona keeps the ball longer, but both teams
  have almost the same number of possessions because possessions alternate. The model is H↔A symmetric, so it cannot
  guess based on the team.

**What the model relies on, and where it fails (6c).**

- *Team shape and the player set are the backbone.* Shuffling either group drops spellF1 by ± 0.10–0.13; the local
  ball features by ± 0.05. `n track` carries no information (Δ ≈ 0), but dropping it does not raise the score either.
  Some feature pairs are almost identical (e.g. arrival time vs distance of the nearest player, |r| ± 0.98, section 5).
  Arrival time still adds information because it accounts for the running direction; the network handles the
  redundancy without problems, so we did not prune it.
- *Errors pile up at possession boundaries.* The per-frame error is ± 25% in the first 0.5 s of a possession and drops
  to ± 3% after 10 s. Possessions < 1 s are caught ± 28% of the time, > 10 s ≥ 95%. ± 12% of the true possessions are
  missed completely; their median duration is ± 1.3 s and the ball is visible on only ± 1/3 of their frames. The
  information needed to catch them is simply not in the tracking.
- *Where a possession comes from decides how hard it is.* Possessions after a dead ball are caught ± 89% of the time,
  after a contest ± 70%.
- *The model's confidence is well calibrated* (Brier ± 0.04): the errors are not about the probability scale but
  about the lack of information around a switch.

**How we chose the methods.** Because the GT covers only 5 minutes, we looked for labels elsewhere: 23 openly licensed,
labelled matches, which the competition rules allow. Leave-match-out cross-validation on those matches scores every match
over its full 90 minutes. Every choice of architecture, augmentation, features and decoder cost was made there.
Post-processing was chosen on spellF1: Viterbi with a switching cost (± 0.81) beats a median filter (± 0.78) and
per-frame labels without smoothing (± 0.68). Extra smoothing on top of Viterbi (Gaussian smoothing, confidence clipping,
minimum duration) adds nothing when chosen nested (−0.002), so it is not used. Ideas from the error analysis were tested
with the same protocol and compared per match in pairs: of five new feature groups, only ball motion & arrival time
passed (+0.002…+0.005, consistent over two seeds and three ball conditions). We still report the rejected ones (table
in 6c).

| Component | Method | Evidence |
|---|---|---|
| Ball | tactical + broadcast ball aligned through player positions (replay/close-up at zero weight), bridged at the gap edges + interpolation ≤ 1 s | ball available on ± 59% → ± 63% of graded frames; median distance between the two recordings' balls ± 0.8 m; per-minute offset residual ≈ 0; bridging: 1 s fill error 0.76 → 0.30 m (label-free simulation) |
| Ball from video | YOLO26 trained on automatic labels (trackers agree + COCO YOLO in a 1.5 m window), full-resolution detection, Viterbi tracking, static hot spots filtered; for the possession model only | coverage, error, precision per threshold and the three-source vote on the validation blocks (2b.5); every threshold without GT |
| Anchored YOLO fills | gaps are filled only by YOLO tracks that once agreed with the tracker ball | in gaps with a broadcast ball: anchored tracks agree ± 83%, unanchored ± 2% (2b.5) |
| `D` boundaries | end → restart kick, start → ball crossing a line, ± 3 s window | IDSSE (`BallStatus`): dead F1 0.933 → 0.948, end boundary within ± 0.5 s 44% → 93%, 7/7 matches |
| Data quality | audit of physical & structural outliers, ghost-track sensitivity test | physical outliers ≤ 0.1%; ghost tracks do not change the score (4c, 6c) |
| `D` | collective behaviour + anchors + Viterbi + stoppage-cue requirement; `D` share 0.48 (Altmann et al., 2023) | replay ⇒ dead ball on ± 93% of frames; public GT: live/dead F1 ± 0.97 (for information) |
| H/A | ensemble of 6 sequence models (BiGRU/TCN, player-set encoder) trained on 23 labelled external matches; including ball-motion & arrival-time features | external cross-validation ± 0.81 / 0.84 vs ± 0.61 / 0.63 for the rule-based method; ball-motion features +0.002…+0.005 |
| Decoder | 3-state Viterbi, switching cost 10, 2 s restart prior | cost curve & post-processing comparison on external cross-validation (6b) |

**The team's questions and their answers.**

| Question | Answer | Evidence |
|---|---|---|
| May we train on external data? | Yes. RULES.md lists *"External data"* explicitly; what is forbidden is hand labels and the private GT | section 6 |
| Does the train/evaluation split represent 90+ minutes? | Yes. The split is per match: every test match is scored whole; the first 5 minutes are the part that is *not* representative | 6b, 9b.1 |
| What about Barcelona's dominance (imbalance)? | H↔A symmetric features, possession-weighted loss; the number of H/A possessions is almost balanced | 6b, 9b.2 |
| Does the broadcast contain replays, and what are the angles? | Yes, ± 1/3 of the broadcast frames are replays, close-ups or crowd shots; those frames get zero weight for the ball position | 4b.2 |
| Is the tactical ball small and far away? | It goes missing mainly while the ball is dead; during live play it is visible ± 92% of the time, lowest near the goals and on the far side | 4b.1 |
| Long balls? | Still tracked in the tactical recording (± 98% of frames); ball height is rarely available | 4b.4 |
| Does the CSV contain tactical or broadcast frames? | Tactical, 199,319 rows; public GT = tactical frames 11,185 … 18,684 | section 9 |
| Which metric to focus on? | spellF1; macroF1 barely separates the post-processing variants | 6b |
| Are there outliers in the data? | Physical outliers are very rare (≤ 0.1%) and already clipped in the features; > 11 tracks per team on ± 13% of frames, and the model is robust to it | 4c, 6c |
| Are there uninformative data/features? | `n track` (permutation ≈ 0); a few redundant feature pairs. Dropping them does not raise the score, so they are kept | 5, 6c |
| Where is the model wrong, and what is the fix? | At possession boundaries and in possessions < 2 s, especially while the ball is missing. The fix that worked: ball motion direction + arrival time; six other ideas were rejected with numbers | 6c |
| Can player-position matching pair a broadcast frame with the wrong tactical frame? | Per frame, yes: other frames within ± 1–2 s are often just as similar (especially during dead balls). That is why we search for **one offset per half** (mode over ± 120 samples), not per-frame pairs. No moment > 30 s away passes below 2 m; the distance between the two trackers' balls is smallest exactly at that offset (0.77 m) and rises symmetrically when shifted by ± 1 frame | 2, 4b.3 |
| Does the GT use the tactical or the broadcast timeline? | Tactical: one label per tactical video frame (199,319). The labels come from the organisers' corrected tracking, not from the ATD we received | 9, 6c |
| Does the broadcast match threshold need tuning? | No: thresholds of 0.5–4 m give the same ball agreement. What matters is the transition at the gap edges (bridging the fill edges) | 2 |
| Did we train a model on the video images? | Yes: a YOLO26 ball detector for the tactical camera. Its labels were made by an algorithm (two trackers agree, then a COCO detector points at the ball pixels), not by hand. The possession model is still trained on tracking | 2b |
| When the tactical and broadcast trackers disagree, which one is right? | A third source from the video takes part in the choice: the tactical ball is replaced when YOLO and the broadcast agree against it (vote in 2b.5) | 2b.5, 2b.6 |
| Why does the dead-ball detector not use the YOLO ball? | The `D` detector uses "ball missing" as a stoppage cue; a YOLO ball during stoppages (ball in hands, spare balls) would change that cue without validation. The merged ball is used only to snap the `D` boundaries to the restart kick and the ball going out | 2b, 4 |
| Does the label switch before the ball reaches the other team? | On the external data the model's switches are on time (median difference 0.00 s); snapping switches to "the nearest receiving player" actually lowers spellF1 (−0.002). In the 09:04–09:16 clip the cause is a tactical ball floating 8–18 m away, handled by YOLO (2b). The `SNAP_RECEPTION` option remains available for review | 8, 2b |
| Why is the YOLO ball sometimes 2–3 m off? | Mostly balls in the air: in the image YOLO is right on the ball, but the homography projects it onto the ground | 2b.5, 2b.7 |

**Limitations and next steps.**

- **Very short possessions** (≤ 1 s, e.g. a brief interception) are still often missed, and most of them happen while
  the ball is not visible: no tracking feature can recover them. The ball-motion feature gain is small, consistent with that
  information limit. A trained touch detector and extra training data did not raise the score either.
- **Stoppages for a whistle** (ball not going out) can start `D` 1–3 s late, because the detector waits for the players
  to slow down. Snapping tidied the `D` boundaries to physical events and the whistle rule was added, but stoppages
  without a clear whistle still depend on player behaviour.
- **Two directions remained open after the tracking-based pipeline:**
  1. choosing the right camera when the tactical and broadcast trackers disagree by > 5 m (± 11% of frames with a ball
     in both; in two public-GT errors the broadcast was right), which needs a third source of truth such as a ball
     detected in the video and validated without hand labels. The YOLO ball detector took this direction. Its
     limitations: balls in the air are still projected onto the ground (misplaced), validation covers only frames where
     both trackers see the ball (the easy frames), and the possession model was not retrained on the new
     ball-availability pattern;
  2. control information for very short touches, which positions alone do not contain. The video touch evidence is a
     first step; a more precise "who touched it" signal (e.g. leg pose against the ball) is still open.

## References

1. Altmann, S., Forcher, L., Woll, A., & Härtel, S. (2023). Effective playing time affects physical match performance
   in soccer: An analysis according to playing position. *Biology of Sport*, 40(4), 967–973.
2. Bassek, M., Rein, R., Weber, H., & Memmert, D. (2025). An integrated dataset of spatiotemporal and event data in
   elite soccer. *Scientific Data*, 12, 195.
3. Borghesi, M., Costa, L. D., Morra, L., & Lamberti, F. (2023). Using temporal convolutional networks to estimate
   ball possession in soccer games. *Expert Systems with Applications*, 223, 119780.
4. Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
5. Cioppa, A., Giancola, S., Deliège, A., Kang, L., et al. (2022). SoccerNet-Tracking: Multiple object tracking
   dataset and benchmark in soccer videos. *IEEE/CVF CVPR Workshops*.
6. Deliège, A., Cioppa, A., Giancola, S., et al. (2021). SoccerNet-v2: A dataset and benchmarks for holistic
   understanding of broadcast soccer videos (action spotting, camera calibration, replay grounding). *IEEE/CVF CVPR
   Workshops*.
7. Fernández, J., & Bornn, L. (2018). Wide open spaces: A statistical technique for measuring space creation in
   professional soccer. *MIT Sloan Sports Analytics Conference*.
8. Fisher, A., Rudin, C., & Dominici, F. (2019). All models are wrong, but many are useful: Learning a variable's
   importance by studying an entire class of prediction models simultaneously. *Journal of Machine Learning
   Research*, 20(177), 1–81.
9. Jocher, G., & Qiu, J. (2026). *Ultralytics YOLO26* (software, AGPL-3.0). github.com/ultralytics/ultralytics.
10. Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly
    efficient gradient boosting decision tree. *NeurIPS*.
11. Lea, C., Flynn, M. D., Vidal, R., Reiter, A., & Hager, G. D. (2017). Temporal convolutional networks for action
    segmentation and detection. *IEEE CVPR*, 156–165.
12. Lin, T.-Y., Maire, M., Belongie, S., Hays, J., Perona, P., Ramanan, D., Dollár, P., & Zitnick, C. L. (2014).
    Microsoft COCO: Common objects in context. *ECCV*, 740–755.
13. Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
14. Ötting, M., Langrock, R., & Maruotti, A. (2023). A copula-based multivariate hidden Markov model for modelling
    momentum in football. *Annals of Operations Research* (arXiv:2002.01193).
15. Rabiner, L. R. (1989). A tutorial on hidden Markov models and selected applications in speech recognition.
    *Proceedings of the IEEE*, 77(2), 257–286.
16. Redmon, J., Divvala, S., Girshick, R., & Farhadi, A. (2016). You only look once: Unified, real-time object
    detection. *IEEE CVPR*, 779–788.
17. Smith, L. N., & Topin, N. (2019). Super-convergence: Very fast training of neural networks using large learning
    rates. *Artificial Intelligence and Machine Learning for Multi-Domain Operations Applications* (SPIE).
18. Spearman, W. (2018). Beyond expected goals. *MIT Sloan Sports Analytics Conference*.
19. Vidal-Codina, F., Evans, N., El Fakir, B., & Billingham, J. (2022). Automatic event detection in football using
    tracking data. *Sports Engineering*, 25, 14.
20. Viterbi, A. J. (1967). Error bounds for convolutional codes and an asymptotically optimum decoding algorithm.
    *IEEE Transactions on Information Theory*, 13(2), 260–269.
21. Zaheer, M., Kottur, S., Ravanbakhsh, S., Póczos, B., Salakhutdinov, R., & Smola, A. (2017). Deep sets. *NeurIPS*.

**External data:** SkillCorner Open Data (github.com/SkillCorner/opendata) · Metrica Sports sample data
(github.com/metrica-sports/sample-data) · IDSSE (Bassek et al., 2025; huggingface.co/datasets/pysport/idsse-data,
CC BY 4.0) — used for validation only: the `D` boundaries and the ball-out rule. IDSSE is needed for the ball-out rule
validation cell (section 8d); without it, set `BALLOUT_STAGE = "force"` to apply the already validated variant.
**Competition repository:** More than a Hack 2027 (github.com/metrica-sports/more-than-a-hack-2027).